# Local+global ResNet34: 60 epochs
HPO v3.1-D2 for train and fixed fold-0 validation. 3,524 train / 825 validation, extra 281 training-only, gold excluded.
Fresh ImageNet initialization. Early stopping disabled; BCE scheduler retained. Steven-v4 metrics are audit-only.


In [ ]:
print("STARTUP: writing pinned local+global modules and HPO validation assets", flush=True)
from pathlib import Path
import sys
WORK = Path('/kaggle/working/local_global_resnet34_hpo60')
WORK.mkdir(parents=True, exist_ok=True)
MODULES = {'local_global_model.py': '"""One ResNet pass: native local triplets and study-wide contextual correction."""\nimport math\nimport torch\nfrom local_global_data import MAX_WINDOWS, WindowBatch\n\n\nclass SliceModel(torch.nn.Module):\n    def __init__(self, backbone, pretrained, dropout, targets, top_k_fraction=0.1):\n        super().__init__()\n        import torchvision\n        if backbone not in {"resnet34", "resnet50"}:\n            raise ValueError("Expected ResNet34 or ResNet50")\n        self.backbone = getattr(torchvision.models, backbone)(weights="DEFAULT" if pretrained else None)\n        old = self.backbone.conv1\n        self.backbone.conv1 = torch.nn.Conv2d(6, 64, 7, stride=2, padding=3, bias=False)\n        with torch.no_grad():\n            self.backbone.conv1.weight.zero_()\n            self.backbone.conv1.weight[:, :3].copy_(old.weight)\n        features = self.backbone.fc.in_features\n        self.backbone.fc = torch.nn.Identity()\n        self.head = torch.nn.Sequential(torch.nn.Dropout(dropout), torch.nn.Linear(features, targets))\n        width = 256\n        self.projection = torch.nn.Linear(features, width)\n        self.plane = torch.nn.Embedding(4, width)\n        self.fluid = torch.nn.Embedding(3, width)\n        self.fat = torch.nn.Embedding(3, width)\n        self.series = torch.nn.Embedding(4, width)\n        self.position = torch.nn.Linear(8, width)\n        # Acquisition metadata is a cue, not the dominant initial signal.\n        # Default Embedding initialization has unit variance and would swamp\n        # the projected image features before any useful contextual learning.\n        for embedding in (self.plane, self.fluid, self.fat, self.series):\n            torch.nn.init.normal_(embedding.weight, std=0.02)\n        torch.nn.init.normal_(self.position.weight, std=0.02)\n        torch.nn.init.zeros_(self.position.bias)\n        self.input_norm = torch.nn.LayerNorm(width)\n        self.context = torch.nn.TransformerEncoderLayer(width, 4, dim_feedforward=512,\n                                                       dropout=0.1, batch_first=True, norm_first=True)\n        self.queries = torch.nn.Parameter(torch.randn(targets, width) * 0.02)\n        self.pool = torch.nn.MultiheadAttention(width, 4, dropout=0.1, batch_first=True)\n        self.output_norm = torch.nn.LayerNorm(width)\n        self.correction = torch.nn.Linear(width, 1)\n        # Small, nonzero weights: contextual queries/attention receive gradients\n        # from the first update, without swamping the retained local prediction.\n        torch.nn.init.normal_(self.correction.weight, std=0.005)\n        torch.nn.init.zeros_(self.correction.bias)\n        self.top_k_fraction = float(top_k_fraction)\n        if not 0 < self.top_k_fraction <= 1:\n            raise ValueError("Invalid top-k fraction")\n        self.to(memory_format=torch.channels_last)\n\n    def forward(self, batch, study_of_slice, studies, fixed_windows_per_study=None, fixed_window_valid=None):\n        if not isinstance(batch, WindowBatch) or batch.valid.shape != (studies, MAX_WINDOWS):\n            raise ValueError("Native-triplet WindowBatch required")\n        features = self.backbone(batch.images)\n        logits = self.head(features)\n        # Scatter using CPU-prepared indices: no boolean selection/nonzero CUDA\n        # synchronization per study and no encoder work on padded windows.\n        grouped = logits.new_zeros((studies * MAX_WINDOWS, logits.shape[-1])).index_copy(0, batch.padded_indices, logits)\n        grouped = grouped.view(studies, MAX_WINDOWS, -1).masked_fill(~batch.valid[..., None], -torch.inf)\n        max_k = max(1, math.ceil(MAX_WINDOWS * self.top_k_fraction))\n        values = grouped.topk(max_k, dim=1).values\n        counts = torch.ceil(batch.valid.sum(1) * self.top_k_fraction).long().clamp_min(1)\n        use = torch.arange(max_k, device=logits.device)[None, :, None] < counts[:, None, None]\n        local = values.masked_fill(~use, 0).sum(1) / counts[:, None]\n        meta = batch.metadata\n        discrete = meta[:, :4].long()\n        tokens = (self.projection(features) + self.plane(discrete[:, 0]) + self.fluid(discrete[:, 1])\n                  + self.fat(discrete[:, 2]) + self.series(discrete[:, 3]) + self.position(meta[:, 4:]))\n        tokens = self.input_norm(tokens)\n        grouped_tokens = tokens.new_zeros((studies * MAX_WINDOWS, tokens.shape[-1])).index_copy(0, batch.padded_indices, tokens)\n        grouped_tokens = grouped_tokens.view(studies, MAX_WINDOWS, -1)\n        padding = ~batch.valid\n        context = self.context(grouped_tokens, src_key_padding_mask=padding)\n        query = self.queries[None].expand(studies, -1, -1)\n        pooled, _ = self.pool(query, context, context, key_padding_mask=padding, need_weights=False)\n        delta = 0.1 * self.correction(self.output_norm(pooled)).squeeze(-1)\n        return local + delta\n', 'local_global_data.py': '"""Native-neighbor cache and selected-window batches shared by train and inference."""\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nSCHEMA = "native_triplets_v1"\nMAX_WINDOWS = 48\nMEAN, STD = 0.449, 0.226\n# Runner chooses these only after the same-GPU timing/memory preflight.\nEXECUTION = {"augment_on_gpu": False, "channels_last": False}\n\n\ndef category(value, choices):\n    value = str(value).strip().lower()\n    return choices.index(value) if value in choices else len(choices)\n\n\ndef window_metadata(record):\n    """Coordinates are relative to their own series, never aligned across planes."""\n    n = record["count"]\n    plane = category(record.get("plane"), ["sagittal", "coronal", "axial"])\n    fluid = category(record.get("fluid_sensitive"), ["0", "1"])\n    fat = category(record.get("fat_suppression"), ["0", "1"])\n    geometry = record["order_source"] == "physical"\n    position = np.asarray(record["positions"], dtype=np.float32)\n    gaps = np.asarray(record["neighbor_gaps_mm"], dtype=np.float32)\n    if position.shape != (n,) or gaps.shape != (n, 2):\n        raise ValueError("Metadata and centre counts differ")\n    # Four categorical fields and eight numerical fields. Unknown geometry has\n    # zero gap encodings plus an explicit availability indicator.\n    phases = np.pi * position\n    numeric = np.column_stack((position * 2 - 1, np.sin(phases), np.cos(phases),\n                               np.sin(2 * phases), np.cos(2 * phases),\n                               np.log1p(gaps[:, 0]), np.log1p(gaps[:, 1]),\n                               np.full(n, float(geometry)))).astype(np.float32)\n    discrete = np.tile([plane, fluid, fat, record["slot"]], (n, 1)).astype(np.float32)\n    result = np.column_stack((discrete, numeric))\n    if not np.isfinite(result).all() or (gaps < 0).any():\n        raise ValueError("Invalid position/gap metadata")\n    return result\n\n\nclass NativeStudyCache:\n    """Select centres before materializing triplets; mmap unique slices once."""\n    def __init__(self, manifest):\n        self.manifest = manifest\n        self._parts = {}\n        self.records = {}\n        for sid, rows in manifest.items():\n            rows = sorted(rows, key=lambda r: r["slot"])\n            for r in rows:\n                if r.get("schema") != SCHEMA:\n                    raise ValueError("Native acquired-neighbor cache required; old cache is incompatible")\n                mapping = np.asarray(r["triplet_offsets"], dtype=np.int64)\n                if mapping.shape != (r["count"], 3) or mapping.min() < 0 or mapping.max() >= r["stored_count"]:\n                    raise ValueError("Invalid triplet row mapping")\n            self.records[sid] = (rows, np.cumsum([0] + [r["count"] for r in rows]),\n                                 np.concatenate([window_metadata(r) for r in rows]))\n\n    def part(self, directory, name):\n        key = (directory, name)\n        if key not in self._parts:\n            self._parts[key] = np.load(Path(directory) / name, mmap_mode="r")\n        return self._parts[key]\n\n    def selected(self, sid, keep):\n        rows, boundaries, metadata = self.records[sid]\n        images, supports = [], []\n        for i, r in enumerate(rows):\n            local = keep[(keep >= boundaries[i]) & (keep < boundaries[i + 1])] - boundaries[i]\n            if not len(local):\n                continue\n            offsets = np.asarray(r["triplet_offsets"], dtype=np.int64)[local] + r["row_start"]\n            images.append(self.part(r["directory"], r["part"])[offsets])\n            supports.append(self.part(r["directory"], r["support_part"])[offsets])\n        # Caller supplies sorted indices; row ordering remains slot-major.\n        return np.concatenate(images), np.concatenate(supports), metadata[keep]\n\n\n@dataclass\nclass WindowBatch:\n    images: torch.Tensor\n    metadata: torch.Tensor\n    padded_indices: torch.Tensor\n    valid: torch.Tensor\n\n\ndef augment_images(images, owners, studies, rng, settings):\n    """Baseline random draws; transform intensity and support together.\n\n    No CUDA-to-CPU queries or full-size erasing masks. Mask geometry uses nearest\n    interpolation; intensity uses bilinear. Erasing applies only to intensities.\n    """\n    device = images.device\n    area = rng.uniform(settings["crop_area_min"], 1.0, studies)\n    extent = np.sqrt(area)\n    angles = np.radians(rng.uniform(-settings["rotation_degrees"], settings["rotation_degrees"], studies))\n    margin = (1 - extent) + settings.get("translate_fraction", 0.0)\n    ox = rng.uniform(-1, 1, studies) * margin\n    oy = rng.uniform(-1, 1, studies) * margin\n    theta = np.zeros((studies, 2, 3), dtype=np.float32)\n    theta[:, 0, 0] = theta[:, 1, 1] = np.cos(angles) * extent\n    theta[:, 0, 1], theta[:, 1, 0] = -np.sin(angles) * extent, np.sin(angles) * extent\n    theta[:, 0, 2], theta[:, 1, 2] = ox, oy\n    grid = F.affine_grid(torch.as_tensor(theta, device=device)[owners], images.shape, align_corners=False)\n    intensity = F.grid_sample(images[:, :3], grid, mode="bilinear", align_corners=False)\n    support = F.grid_sample(images[:, 3:], grid, mode="nearest", align_corners=False)\n    gain = torch.as_tensor(rng.uniform(1 - settings["intensity_jitter"], 1 + settings["intensity_jitter"], studies), dtype=torch.float32, device=device)\n    shift = torch.as_tensor(rng.uniform(-settings["brightness_shift"], settings["brightness_shift"], studies), dtype=torch.float32, device=device)\n    intensity.mul_(gain[owners, None, None, None]).add_(shift[owners, None, None, None])\n    erase = rng.random(len(images)) < settings["erase_probability"]\n    side = int(round(images.shape[-1] * settings["erase_fraction"]))\n    for i in np.flatnonzero(erase):\n        top, left = (int(rng.integers(0, images.shape[-1] - side)) for _ in range(2))\n        intensity[i, :, top:top + side, left:left + side] = 0\n    return torch.cat((intensity, support), dim=1)\n\n\ndef prepare_batch(cache, labels, study_ids, slice_cap, rng, device, augmentation=None, sources=None):\n    pixels, metadata, owners, padded, targets, weights = [], [], [], [], [], []\n    valid = np.zeros((len(study_ids), MAX_WINDOWS), dtype=bool)\n    for i, sid in enumerate(study_ids):\n        n = len(cache.records[sid][2])\n        if not 0 < n <= MAX_WINDOWS:\n            raise ValueError(f"Invalid centre count for {sid}: {n}")\n        keep = np.arange(n)\n        if slice_cap is not None and n > slice_cap:\n            keep = (np.sort(rng.choice(n, slice_cap, replace=False)) if rng is not None\n                    else np.linspace(0, n - 1, slice_cap).round().astype(int))\n        images, support, meta = cache.selected(sid, keep)\n        pixels.append(np.concatenate((images, support), axis=1))\n        metadata.append(meta)\n        owners.extend([i] * len(keep))\n        padded.extend(i * MAX_WINDOWS + np.arange(len(keep)))\n        valid[i, :len(keep)] = True\n        targets.append(labels[sid][0]); weights.append(labels[sid][1])\n    # Keep uint8 until transfer: 4x smaller host buffer and PCIe transfer than\n    # moving float32. No extra prefetched batch or permanent device cache.\n    host = torch.from_numpy(np.concatenate(pixels))\n    if device.type == "cuda":\n        host = host.pin_memory()\n    processing_device = device if EXECUTION["augment_on_gpu"] else torch.device("cpu")\n    images = host.to(processing_device, non_blocking=True).float()\n    images[:, :3].div_(255).sub_(MEAN).div_(STD)\n    images[:, 3:].div_(255)\n    owner = torch.tensor(owners, dtype=torch.long, device=processing_device)\n    if rng is not None:\n        images = augment_images(images, owner, len(study_ids), rng, augmentation)\n    images = images.to(device, non_blocking=True)\n    if EXECUTION["channels_last"]:\n        images = images.contiguous(memory_format=torch.channels_last)\n    owner = owner.to(device)\n    batch = WindowBatch(images, torch.as_tensor(np.concatenate(metadata), device=device),\n                        torch.tensor(padded, device=device), torch.as_tensor(valid, device=device))\n    return (batch, owner, torch.as_tensor(np.stack(targets), device=device),\n            torch.as_tensor(np.stack(weights), device=device),\n            torch.as_tensor(np.stack([sources[s] for s in study_ids]), device=device) if sources else None,\n            torch.ones(len(owners), device=device))\n', 'local_global_speed.py': '"""Short paired timing preflight; retain only faster choices within baseline VRAM."""\nimport gc\nimport json\nimport statistics\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom local_global_data import EXECUTION, prepare_batch\n\n\ndef choose_execution(model, cache, labels, train_ids, config, device, output):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    path = output / "execution.json"\n    if device.type != "cuda":\n        EXECUTION.update(augment_on_gpu=False, channels_last=False)\n        model.to(memory_format=torch.contiguous_format)\n        return\n    if path.exists():\n        report = json.loads(path.read_text())\n        if report["gpu"] != torch.cuda.get_device_name(device):\n            raise ValueError("Recovery GPU differs; do not silently change execution settings")\n        EXECUTION.update(report["selected"])\n        model.to(memory_format=torch.channels_last if EXECUTION["channels_last"] else torch.contiguous_format)\n        print("EXECUTION_RECOVERED", json.dumps(report["selected"]), flush=True)\n        return\n    cpu_rng = torch.get_rng_state(); cuda_rng = torch.cuda.get_rng_state_all()\n    initial = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}\n    ids = train_ids[:int(config["batch_studies"])]\n    results = []\n    try:\n        for gpu_aug, channels_last in [(False, False), (False, True), (True, False), (True, True)]:\n            choice = dict(augment_on_gpu=gpu_aug, channels_last=channels_last)\n            EXECUTION.update(choice)\n            model.load_state_dict(initial)\n            model.to(memory_format=torch.channels_last if channels_last else torch.contiguous_format)\n            torch.set_rng_state(cpu_rng); torch.cuda.set_rng_state_all(cuda_rng)\n            rng = np.random.default_rng(config["seed"])\n            optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])\n            scaler = torch.amp.GradScaler("cuda", enabled=config["mixed_precision"])\n            model.train(); durations = []\n            gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()\n            batch = owner = target = weight = pred = cells = loss = None\n            try:\n                for step in range(5):\n                    torch.cuda.synchronize(); start = time.perf_counter()\n                    batch, owner, target, weight, _, _ = prepare_batch(cache, labels, ids, 24, rng, device, config["augmentation"])\n                    optimizer.zero_grad(set_to_none=True)\n                    with torch.autocast("cuda", enabled=config["mixed_precision"]):\n                        pred = model(batch, owner, len(ids))\n                        cells = torch.nn.functional.binary_cross_entropy_with_logits(pred.float(), target, reduction="none")\n                        loss = (cells * weight).sum() / weight.sum()\n                    scaler.scale(loss).backward(); scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip_norm"])\n                    scaler.step(optimizer); scaler.update()\n                    torch.cuda.synchronize(); elapsed = time.perf_counter() - start\n                    if step >= 2: durations.append(elapsed)\n                    del batch, pred, cells, loss\n                result = {**choice, "median_seconds": statistics.median(durations),\n                          "peak_allocated_bytes": torch.cuda.max_memory_allocated(),\n                          "peak_reserved_bytes": torch.cuda.max_memory_reserved(), "ok": True}\n            except torch.cuda.OutOfMemoryError:\n                result = {**choice, "ok": False, "reason": "oom"}\n            finally:\n                # An OOM may leave a partial autograd graph alive. Release it\n                # before the next variant so one failure cannot poison others.\n                batch = owner = target = weight = pred = cells = loss = None\n                optimizer.zero_grad(set_to_none=True); del optimizer, scaler\n                gc.collect(); torch.cuda.empty_cache()\n            results.append(result); print("SPEED_PREFLIGHT", json.dumps(result), flush=True)\n        base = results[0]\n        if not base["ok"]: raise RuntimeError("Batch-8 reference exceeds GPU memory; stop rather than silently change experiment")\n        # Allocated and reserved high-water marks must both stay within the\n        # reference footprint. No larger GPU, batch, prefetched queue or cache.\n        eligible = [r for r in results if r["ok"] and\n                    r["peak_allocated_bytes"] <= base["peak_allocated_bytes"] and\n                    r["peak_reserved_bytes"] <= base["peak_reserved_bytes"]]\n        winner = min(eligible, key=lambda r: r["median_seconds"])\n        selected = {k: winner[k] for k in ("augment_on_gpu", "channels_last")}\n        report = {"gpu": torch.cuda.get_device_name(device), "batch": len(ids), "windows_per_study": 24,\n                  "candidates": results, "selected": selected,\n                  "measured_step_speedup": base["median_seconds"] / winner["median_seconds"],\n                  "note": "Short warm-cache step benchmark; not a claimed epoch speedup."}\n        temporary = path.with_suffix(".json.tmp"); temporary.write_text(json.dumps(report, indent=2) + "\\n"); temporary.replace(path)\n        EXECUTION.update(selected)\n        print("EXECUTION_SELECTED", json.dumps(report), flush=True)\n    finally:\n        model.load_state_dict(initial)\n        model.to(memory_format=torch.channels_last if EXECUTION["channels_last"] else torch.contiguous_format)\n        model.zero_grad(set_to_none=True)\n        torch.set_rng_state(cpu_rng); torch.cuda.set_rng_state_all(cuda_rng)\n        del initial\n        gc.collect(); torch.cuda.empty_cache()\n', 'local_global_runner.py': '"""Pinned baseline loop, native data/model adapters, durable epoch recovery."""\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport sys\nfrom pathlib import Path\n\nfrom efficientnet_v31_runner import prepare_trainer\n\nTRAINER_SHA = "b8f8302319afbcf9970f0cf5b806d8fb78e72ef6e169b276274ddf7cd5e0ba85"\n\n\ndef prepare_local_global(raw, steven_audit=False):\n    if hashlib.sha256(raw.encode()).hexdigest() != TRAINER_SHA:\n        raise ValueError("Baseline trainer differs from the verified reference")\n    text = prepare_trainer(raw)\n    # Replace the memory adapter/backbone class, not the optimization/loss loop.\n    start = text.index("class SliceModel("); end = text.index("\\ndef augment(", start)\n    text = text[:start] + "from local_global_model import SliceModel\\n" + text[end:]\n    start = text.index("def prepare_batch("); end = text.index("\\ndef confident_auc(", start)\n    text = text[:start] + "from local_global_data import NativeStudyCache as StudyCache, prepare_batch\\n" + text[end:]\n    marker = "    # Labels as free parameters"\n    if text.count(marker) != 1: raise ValueError("Unexpected model construction boundary")\n    # Preflight restores model/RNG state and persists its measured execution\n    # choice before any training. Recovery resumes with that same choice.\n    text = text.replace(marker, "    from local_global_speed import choose_execution\\n"\n        "    choose_execution(model, cache, labels, train_ids, config, device, args.output_dir)\\n"\n        "    DURABLE_COMMIT()\\n" + marker)\n    # Inference checkpoints cannot become half-written on interruption.\n    text = text.replace("            torch.save(payload, path)",\n        \'            temporary = path.with_suffix(path.suffix + ".tmp")\\n\'\n        \'            torch.save(payload, temporary)\\n\'\n        \'            temporary.replace(path)\')\n    # The recovery file is authoritative. A first-epoch interruption between\n    # writing history and writing recovery must be replayable, not a refusal.\n    text = text.replace(\'    if history_path.exists() and not (args.output_dir / "recovery.pt").is_file():\\n\'\n                        \'        raise FileExistsError(f"Refusing to overwrite existing run history: {history_path}")\',\n                        \'    if history_path.exists() and not (args.output_dir / "recovery.pt").is_file():\\n\'\n                        \'        print("Replaying uncommitted first epoch; no durable recovery exists.", flush=True)\\n\'\n                        \'        history_path.unlink()\')\n    marker = "            batches += 1"\n    if text.count(marker) != 1: raise ValueError("Unexpected progress boundary")\n    text = text.replace(marker, marker + \'\\n            if batches % 50 == 0 or batches == batch_count:\\n\'\n        \'                print(f"PROGRESS epoch={epoch} batch={batches}/{batch_count}", flush=True)\')\n    text = text.replace(\'"input": "3 adjacent slices + 3 acquired-support masks"\',\n        \'"input": "3 native acquired neighbors + 3 support masks; study-wide contextual residual"\')\n    if steven_audit:\n        # Score the same predictions; this reference never reaches the optimizer,\n        # scheduler, checkpoint selection or early-stopping condition.\n        marker = "        metrics, scores = evaluate()"\n        if text.count(marker) != 1: raise ValueError("Unexpected evaluation boundary")\n        text = text.replace(marker, marker + \'\\n\'\n            \'        audit_targets = np.stack([STEVEN_AUDIT_LABELS[s][0] for s in validation_ids])\\n\'\n            \'        audit_weights = np.stack([STEVEN_AUDIT_LABELS[s][1] for s in validation_ids])\\n\'\n            \'        audit_metrics = confident_auc(audit_targets, audit_weights, scores,\\n\'\n            \'                                      config.get("confident_low", .3), config.get("confident_high", .7))\\n\'\n            \'        audit_p = np.clip(scores, 1e-7, 1 - 1e-7)\\n\'\n            \'        audit_bce = float(((-audit_targets * np.log(audit_p) -\\n\'\n            \'                           (1 - audit_targets) * np.log1p(-audit_p)) * audit_weights).sum() / audit_weights.sum())\\n\'\n            \'        print(f"STEVEN_AUDIT epoch={epoch} AUC={audit_metrics[\\\'macro_auc\\\']:.6f} BCE={audit_bce:.6f} (not used for controls)", flush=True)\')\n        marker = "        history.append(record)"\n        if text.count(marker) != 1: raise ValueError("Unexpected history boundary")\n        text = text.replace(marker,\n            \'        record["steven_audit_macro_auc"] = audit_metrics["macro_auc"]\\n\'\n            \'        record["steven_audit_bce"] = audit_bce\\n\' + marker)\n    ast.parse(text)\n    return text\n\n\ndef run(root=Path("/root/cache"), source=Path("/root/source"), assets=Path("/root/assets"),\n        experiment="local_global_v31_d2_seed2026"):\n    import modal\n    if experiment not in {"local_global_v31_d2_seed2026", "local_global_v31_trainval_d2_seed2026"}:\n        raise ValueError("Unknown local+global experiment")\n    out = root / "experiments" / experiment / "run"; out.mkdir(parents=True, exist_ok=True)\n    audit = json.loads((assets / "asset_hashes.json").read_text())\n    for name, sha in audit.items():\n        if hashlib.sha256((assets / name).read_bytes()).hexdigest() != sha: raise ValueError(f"Asset changed: {name}")\n    config = json.loads((assets / "config.json").read_text())\n    if config.get("learnable_labels") or config.get("learned_weights") or config.get("pairwise_auc_weight", 0):\n        raise ValueError("This architecture experiment uses only unchanged soft BCE")\n    trainer_path = out.parent / "trainer.py"\n    steven_audit = experiment == "local_global_v31_trainval_d2_seed2026"\n    if steven_audit and not (out / "execution.json").exists():\n        # Reuse only execution choices, never baseline weights or optimizer state.\n        (out / "execution.json").write_bytes((assets / "execution.json").read_bytes())\n    trainer_path.write_text(prepare_local_global((source / "scripts/train_cnn_baseline.py").read_text(), steven_audit))\n    sys.path.insert(0, str(source))\n    spec = importlib.util.spec_from_file_location("local_global_trainer", trainer_path)\n    trainer = importlib.util.module_from_spec(spec); sys.modules[spec.name] = trainer; spec.loader.exec_module(trainer)\n    if steven_audit:\n        trainer.STEVEN_AUDIT_LABELS = trainer.read_soft_labels(assets / "steven_audit_labels.json")\n    trainer.DURABLE_COMMIT = modal.Volume.from_name("rsna-cnn-native-triplets-224px-v1").commit\n    cache_dirs = [root / f"shard_{i}/slice_cache" for i in range(4)]\n    for path in cache_dirs:\n        if not json.loads((path / "cache_summary.json").read_text())["complete"]: raise ValueError(f"Incomplete cache: {path}")\n    sys.argv = ["trainer", "--cache", *map(str, cache_dirs), "--soft-labels", str(assets / "labels.json"),\n                "--validation-soft-labels", str(assets / "validation_labels.json"), "--folds", str(assets / "folds.csv"),\n                "--config", str(assets / "config.json"), "--output-dir", str(out), "--validation-fold", "0",\n                "--train-slices-per-study", "24"]\n    trainer.main()\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--experiment", default="local_global_v31_d2_seed2026")\n    run(experiment=parser.parse_args().experiment)\n', 'efficientnet_v31_runner.py': '"""Controlled EfficientNetV2-S runner with epoch-boundary recovery."""\nfrom pathlib import Path\nimport ast\nimport json\nimport csv\nimport torch\n\n\n\nMEMORY_ADAPTER = """\nfrom contextlib import contextmanager, nullcontext\nfrom torch.utils.checkpoint import checkpoint\n\n@contextmanager\ndef isolated_bn_buffers(module):\n    # Recompute against disposable running-stat buffers, then restore original\n    # references. No double update and no in-place mutation of saved buffers.\n    originals = []\n    for child in module.modules():\n        if isinstance(child, torch.nn.modules.batchnorm._BatchNorm):\n            for name in (\'running_mean\', \'running_var\', \'num_batches_tracked\'):\n                value = getattr(child, name)\n                if value is not None:\n                    originals.append((child, name, value))\n                    setattr(child, name, value.clone())\n    try:\n        yield\n    finally:\n        for child, name, value in originals:\n            setattr(child, name, value)\n\nclass MemoryEfficientFeatures(torch.nn.Sequential):\n    def forward(self, x):\n        for block in self:\n            if self.training and torch.is_grad_enabled():\n                x = checkpoint(block, x, use_reentrant=False, preserve_rng_state=True,\n                               context_fn=lambda block=block: (nullcontext(), isolated_bn_buffers(block)))\n            else:\n                x = block(x)\n        return x\n"""\n\ndef prepare_trainer(text: str) -> str:\n    """Patch only the backbone adapter and durable epoch recovery hooks."""\n    start = text.index(\'        old_conv = self.backbone.conv1\')\n    end = text.index(\'        self.head =\', start)\n    text = text[:start] + \'\'\'        self.backbone.features = MemoryEfficientFeatures(*self.backbone.features)\n        old_conv = self.backbone.features[0][0]\n        new_conv = torch.nn.Conv2d(6, old_conv.out_channels, old_conv.kernel_size,\n                                  old_conv.stride, old_conv.padding,\n                                  bias=old_conv.bias is not None)\n        with torch.no_grad():\n            new_conv.weight.zero_()\n            new_conv.weight[:, :3].copy_(old_conv.weight)\n            if old_conv.bias is not None:\n                new_conv.bias.copy_(old_conv.bias)\n        self.backbone.features[0][0] = new_conv\n        features = self.backbone.classifier[1].in_features\n        self.backbone.classifier = torch.nn.Identity()\n\'\'\' + text[end:]\n    text = text.replace(\'    if history_path.exists():\\n        raise FileExistsError\', \'    if history_path.exists() and not (args.output_dir / "recovery.pt").is_file():\\n        raise FileExistsError\')\n    boundary = \'    for epoch in range(int(config["epochs"])):\'\n    assert text.count(boundary) == 1\n    resume = \'\'\'    start_epoch = 0\n    recovery_path = args.output_dir / "recovery.pt"\n    if recovery_path.is_file():\n        recovery = torch.load(recovery_path, map_location="cpu", weights_only=False)\n        if recovery["config"] != config:\n            raise ValueError("Recovery config differs from this experiment")\n        model.load_state_dict(recovery["state_dict"], strict=True)\n        optimizer.load_state_dict(recovery["optimizer"])\n        scheduler.load_state_dict(recovery["scheduler"])\n        scaler.load_state_dict(recovery["scaler"])\n        global_step = recovery["global_step"]\n        history, best = recovery["history"], recovery["best"]\n        best_scores = recovery["best_scores"]\n        stalled, best_loss = recovery["stalled"], recovery["best_loss"]\n        rng.bit_generator.state = recovery["numpy_rng"]\n        torch.set_rng_state(recovery["torch_rng"])\n        torch.cuda.set_rng_state_all(recovery["cuda_rng"])\n        start_epoch = recovery["epoch"] + 1\n        # Roll back a CSV record written before a preemption but not committed.\n        with history_path.open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(history[0]))\n            writer.writeheader()\n            writer.writerows(history)\n        print(f"RESUMED epoch={start_epoch} optimizer_step={global_step} stale={stalled}", flush=True)\n        if (patience and stalled >= patience) or start_epoch >= int(config["epochs"]):\n            print("Recovered a finished run; no additional training.", flush=True)\n            start_epoch = int(config["epochs"])\n            stopped_early = bool(patience and stalled >= patience)\n    for epoch in range(start_epoch, int(config["epochs"])):\n\'\'\'\n    text = text.replace(boundary, resume.rstrip())\n    # Save after checkpoint selection, scheduler stepping, and patience updates.\n    boundary = \'        if patience and stalled >= patience:\'\n    assert text.count(boundary) == 1\n    save = \'\'\'        payload = {\n            "state_dict": model.state_dict(), "config": config, "epoch": epoch,\n            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),\n            "scaler": scaler.state_dict(), "global_step": global_step,\n            "history": history, "best": best, "best_scores": best_scores,\n            "stalled": stalled, "best_loss": best_loss,\n            "numpy_rng": rng.bit_generator.state, "torch_rng": torch.get_rng_state(),\n            "cuda_rng": torch.cuda.get_rng_state_all(),\n        }\n        temporary = args.output_dir / "recovery.tmp"\n        torch.save(payload, temporary)\n        temporary.replace(args.output_dir / "recovery.pt")\n        DURABLE_COMMIT()\n        print(f"CHECKPOINTS_READY epoch={epoch}", flush=True)\n\'\'\'\n    text = text.replace(boundary, save + boundary)\n    memory_adapter = \'\\n\' + MEMORY_ADAPTER + \'\\n\'\n    text = text.replace(\'class SliceModel(torch.nn.Module):\', memory_adapter + \'class SliceModel(torch.nn.Module):\')\n    ast.parse(text)\n    return text\n\n\nTARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\n\ndef prepare_steven_validation(steven_csv: Path, folds_path: Path, gold_split_path: Path,\n                              output_path: Path) -> tuple[int, int]:\n    import numpy as np\n\n    with gold_split_path.open(newline="", encoding="utf-8") as stream:\n        gold_ids = {row["StudyInstanceUID"] for row in csv.DictReader(stream)}\n    with folds_path.open(newline="", encoding="utf-8") as stream:\n        fold_map = {row["study_id"]: int(row["fold"]) for row in csv.DictReader(stream)}\n    validation_ids = {study_id for study_id, fold in fold_map.items() if fold == 0}\n    if len(gold_ids) != 58 or len(validation_ids) != 825 or gold_ids & validation_ids:\n        raise RuntimeError("Fold/gold split dimensions or separation check failed")\n\n    records = []\n    seen: set[str] = set()\n    with steven_csv.open(newline="", encoding="utf-8-sig") as stream:\n        reader = csv.DictReader(stream)\n        if not {"StudyInstanceUID", *TARGETS}.issubset(reader.fieldnames or []):\n            raise ValueError("Steven v4 label file has an unexpected target schema")\n        for row in reader:\n            study_id = row["StudyInstanceUID"]\n            if study_id not in validation_ids:\n                continue\n            if study_id in seen:\n                raise ValueError(f"Duplicate Steven v4 validation row: {study_id}")\n            seen.add(study_id)\n            probabilities, weights = [], []\n            for target in TARGETS:\n                try:\n                    value = float(row[target])\n                except (TypeError, ValueError):\n                    value = float("nan")\n                valid = bool(np.isfinite(value) and 0.0 <= value <= 1.0)\n                probabilities.append(value if valid else None)\n                weights.append(float(valid))\n            records.append({"study_id": study_id, "probabilities": probabilities, "weights": weights})\n    if seen != validation_ids:\n        raise RuntimeError(f"Steven v4 validation coverage mismatch: {len(validation_ids - seen)} missing")\n    output_path.write_text(json.dumps({\n        "version": "steven_v4_fold0_validation_only",\n        "targets_in_order": TARGETS,\n        "labels": records,\n    }, allow_nan=False) + "\\n", encoding="utf-8")\n    return len(gold_ids), len(validation_ids)\n\ndef run() -> None:\n    import importlib.util\n    import sys\n    import csv\n    import hashlib\n    import modal\n    root = Path(\'/root/rsna/persistent_cache/experiments/efficientnet_v31_d2_fold0\')\n    root.mkdir(parents=True, exist_ok=True)\n    source = Path(\'/root/rsna/source\')\n    label_path = root / \'inputs/soft_labels_hpo_v3_1_d2_structured.json\'\n    assert hashlib.sha256(label_path.read_bytes()).hexdigest() == \'38367d62eb9e7c198b69d88e2b9ad419e00fea9d1211b98feae45cd8a386f92d\'\n    labels = json.loads(label_path.read_text())\n    ids = {r[\'study_id\'] for r in labels[\'labels\']}\n    assert len(ids) == 4349 and all(all(w == 1 for w in r[\'weights\']) for r in labels[\'labels\'])\n    with (source / \'configs/report_folds_v5.csv\').open() as f:\n        base = {r[\'study_id\']: int(r[\'fold\']) for r in csv.DictReader(f)}\n    assert len(base) == 4068 and sum(v == 0 for v in base.values()) == 825\n    extra = ids - base.keys()\n    assert len(extra) == 281\n    base.update({s: 5 for s in extra})\n    folds = root / \'folds.csv\'\n    with folds.open(\'w\') as f:\n        w=csv.writer(f); w.writerow([\'study_id\',\'fold\']); w.writerows(sorted(base.items()))\n    with (source / \'configs/gold_split_v1.csv\').open() as f:\n        gold = {r[\'StudyInstanceUID\'] for r in csv.DictReader(f)}\n    assert len(gold) == 58 and not gold & ids\n    validation = root / \'steven_v4_fold0_reference.json\'\n    prepare_steven_validation(root/\'inputs/llm_labels_v4_blend.csv\', folds,\n                              source/\'configs/gold_split_v1.csv\', validation)\n    config = json.loads(Path(\'/root/rsna/reference_config.json\').read_text())\n    config[\'model\'][\'backbone\'] = \'efficientnet_v2_s\'\n    config[\'version\'] = \'hpo_v31_d2_efficientnet_v2_s_fold0\'\n    cp = root / \'config.json\'; cp.write_text(json.dumps(config,indent=2)+\'\\n\')\n    raw=(source/\'scripts/train_cnn_baseline.py\').read_text()\n    trainer_file=root/\'train_efficientnet.py\';trainer_file.write_text(prepare_trainer(raw))\n    spec=importlib.util.spec_from_file_location(\'train_efficientnet\',trainer_file)\n    trainer=importlib.util.module_from_spec(spec);sys.modules[spec.name]=trainer\n    sys.path.insert(0,str(source));spec.loader.exec_module(trainer)\n    volume=modal.Volume.from_name(\'rsna-cnn-topk-support-cache-v1\')\n    trainer.DURABLE_COMMIT = volume.commit\n    run_dir=root/\'run\';run_dir.mkdir(exist_ok=True)\n    cache_dirs=[Path(f\'/root/rsna/persistent_cache/shard_{i}/slice_cache\') for i in range(3)]\n    for d in cache_dirs:\n        for name in [\'manifest.jsonl\',\'cache_summary.json\']+[f\'{kind}_{i:03d}.npy\' for kind in [\'slices\',\'support\'] for i in range(4)]:\n            if not (d/name).is_file(): raise FileNotFoundError(d/name)\n    print(\'EfficientNetV2-S ImageNet; HPO v3.1-D2; batch8; 3524 train / fixed825 validation; T4\',flush=True)\n    sys.argv=[str(trainer_file),\'--cache\',*map(str,cache_dirs),\'--soft-labels\',str(label_path),\n              \'--validation-soft-labels\',str(validation),\'--folds\',str(folds),\'--config\',str(cp),\n              \'--output-dir\',str(run_dir),\'--validation-fold\',\'0\',\'--train-slices-per-study\',\'24\']\n    trainer.main()\n\nif __name__ == \'__main__\':\n    run()\n', 'local_global_resnet34_kaggle.py': '"""Kaggle native local+global ResNet34, HPO trainval, all 60 epochs."""\nimport ast\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport sys\nfrom pathlib import Path\n\nEXPERIMENT = \'local_global_resnet34_hpo60\'\nLABEL_SHA = \'38367d62eb9e7c198b69d88e2b9ad419e00fea9d1211b98feae45cd8a386f92d\'\nTRAINER_SHA = \'b8f8302319afbcf9970f0cf5b806d8fb78e72ef6e169b276274ddf7cd5e0ba85\'\nEXPECTED_STUDIES = [1081,1047,1142,1079]\n\n\ndef exactly_one(paths, description):\n    found = [p for p in paths if p.exists()]\n    if len(found) != 1:\n        raise FileNotFoundError(f\'{description}: expected one fixed input, found {found}; checked {[str(p) for p in paths]}\')\n    return found[0]\n\n\ndef prepare_resnet34_trainer(raw):\n    from local_global_runner import prepare_local_global\n    text = prepare_local_global(raw, steven_audit=True)\n    # Observational checkpoint only: never used to step LR or stop training.\n    marker = \'        persist_epoch(record, epoch)\'\n    assert text.count(marker) == 1\n    text = text.replace(marker, marker + \'\'\'\n        previous_bce = min((r["validation_bce"] for r in history[:-1]), default=float("inf"))\n        if record["validation_bce"] < previous_bce:\n            save_checkpoint({"state_dict": model.state_dict(), "config": config, "epoch": epoch},\n                            args.output_dir / "checkpoint_best_bce.pt")\n\'\'\'.rstrip())\n    marker = \'        history.append(record)\'\n    assert text.count(marker) == 1\n    text = text.replace(marker, \'\'\'        record["bce_generalization_gap"] = record["validation_bce"] - record["train_bce"]\n\'\'\' + marker)\n    # Full epoch-boundary recovery works with patience=0; all loop exits are\n    # external failures or the explicit 60-epoch limit, not metric plateaus.\n    assert \'if (patience and stalled >= patience) or start_epoch >= int(config["epochs"]):\' in text\n    ast.parse(text)\n    return text\n\n\ndef main():\n    print(\'STARTUP: local+global ResNet34/ImageNet; HPO v3.1-D2 train and validation; 60 epochs; early stopping DISABLED.\', flush=True)\n    os.environ[\'CUDA_VISIBLE_DEVICES\'] = \'0\'\n    os.environ[\'OMP_NUM_THREADS\'] = \'4\'\n    os.environ[\'MKL_NUM_THREADS\'] = \'4\'\n    import numpy as np\n    import torch\n    import torchvision\n    from local_global_runner import TRAINER_SHA as pinned_sha\n    assert pinned_sha == TRAINER_SHA\n    inputs = Path(\'/kaggle/input\')\n    work = Path(\'/kaggle/working\') / EXPERIMENT\n    assets = work/\'assets\'; run = work/\'run\'\n    source_file = exactly_one([\n        inputs/\'datasets/tumis001/rsna-cnn-hpo-blend-xla-source-v1/skanee/scripts/train_cnn_baseline.py\',\n        inputs/\'rsna-cnn-hpo-blend-xla-source-v1/skanee/scripts/train_cnn_baseline.py\'], \'baseline source\')\n    source = source_file.parents[1]\n    label_path = exactly_one([\n        inputs/\'datasets/tumis001/rsna-knee-hpo-v31-d2-report-labels/soft_labels_hpo_v3_1_d2_structured.json\',\n        inputs/\'rsna-knee-hpo-v31-d2-report-labels/soft_labels_hpo_v3_1_d2_structured.json\'], \'HPO labels\')\n    if hashlib.sha256(label_path.read_bytes()).hexdigest() != LABEL_SHA:\n        raise ValueError(\'HPO label hash mismatch\')\n    if hashlib.sha256(source_file.read_bytes()).hexdigest() != TRAINER_SHA:\n        raise ValueError(\'Trainer hash mismatch\')\n    locks = json.loads((assets/\'asset_hashes.json\').read_text())\n    for name,digest in locks.items():\n        if hashlib.sha256((assets/name).read_bytes()).hexdigest() != digest: raise ValueError(f\'Asset hash mismatch: {name}\')\n    config = json.loads((assets/\'config.json\').read_text())\n    assert config[\'model\'][\'backbone\'] == \'resnet34\' and config[\'model\'][\'pretrained\']\n    assert config[\'epochs\'] == 60 and config[\'early_stopping_patience\'] == 0 and config[\'batch_studies\'] == 8\n    labels = json.loads(label_path.read_text())\n    label_ids = {r[\'study_id\'] for r in labels[\'labels\']}\n    folds = {r[\'study_id\']:int(r[\'fold\']) for r in csv.DictReader((assets/\'folds.csv\').open())}\n    validation = json.loads((assets/\'validation_labels.json\').read_text())\n    val_ids = {r[\'study_id\'] for r in validation[\'labels\']}\n    assert len(label_ids) == len(labels[\'labels\']) == len(folds) == 4349\n    assert set(folds) == label_ids and val_ids == {s for s in folds if folds[s] == 0} and len(val_ids) == 825\n    assert len(label_ids-val_ids) == 3524\n    by_id = {r[\'study_id\']:r for r in labels[\'labels\']}\n    assert all(r == by_id[r[\'study_id\']] for r in validation[\'labels\'])\n    assert labels[\'targets_in_order\'] == validation[\'targets_in_order\']\n    base_folds = {r[\'study_id\']:int(r[\'fold\']) for r in csv.DictReader((source/\'configs/report_folds_v5.csv\').open())}\n    added = label_ids-set(base_folds)\n    assert len(added) == 281 and all(folds[s] != 0 for s in added)\n    assert all(folds[s] == f for s,f in base_folds.items())\n    gold_ids = {r[\'StudyInstanceUID\'] for r in csv.DictReader((source/\'configs/gold_split_v1.csv\').open())}\n    assert len(gold_ids) == 58 and not gold_ids & label_ids\n    cache_dirs=[];seen=set()\n    for i in range(4):\n        slug=f\'rsna-cnn-native-triplet-cache-shard-{i}\'\n        tail=Path(f\'native_triplets/shard_{i}/slice_cache\')\n        cache = exactly_one([inputs/\'notebooks/tumis001\'/slug/tail,\n                             inputs/\'kernels/tumis001\'/slug/tail,inputs/slug/tail], f\'native shard {i}\')\n        summary = json.loads((cache/\'cache_summary.json\').read_text())\n        assert summary[\'complete\'] and summary[\'schema\'] == \'native_triplets_v1\' and summary[\'decode_errors\'] == 0\n        assert summary[\'shard\'] == [i,4] and summary[\'studies\'] == EXPECTED_STUDIES[i]\n        assert summary[\'cache\'] == config[\'cache\']\n        records = [json.loads(line) for line in (cache/\'manifest.jsonl\').read_text().splitlines() if line.strip()]\n        ids = {r[\'study_id\'] for r in records}\n        assert len(ids) == summary[\'studies\'] and not ids & seen\n        seen.update(ids)\n        for name in {r[k] for r in records for k in [\'part\',\'support_part\']}:\n            array = np.load(cache/name,mmap_mode=\'r\')\n            assert array.dtype == np.uint8 and array.shape[1:] == (224,224)\n            assert all(r[\'row_start\']+r[\'stored_count\'] <= len(array) for r in records if r[\'part\']==name or r[\'support_part\']==name)\n            del array\n        cache_dirs.append(cache)\n        print(f\'CACHE_READY shard={i} studies={len(ids)} path={cache}\',flush=True)\n    assert seen == label_ids\n    if not torch.cuda.is_available(): raise RuntimeError(\'GPU is required for training\')\n    torch.cuda.set_device(0);torch.set_num_threads(4)\n    print(f\'GPU: {torch.cuda.get_device_name(0)}; torch={torch.__version__}; torchvision={torchvision.__version__}; one GPU / global batch 8\',flush=True)\n    print(\'SPLIT: 3524 train including extra281; fixed825 HPO validation;58gold excluded.\',flush=True)\n    print(\'CONTROLS: no early stopping; HPO BCE LR plateau (min_lr=1e-6); HPO AUC best checkpoint. Steven scores are audit-only.\',flush=True)\n    run.mkdir(parents=True,exist_ok=True)\n    # Same execution choices as the Modal arm, no benchmarking or trial updates.\n    execution={\'gpu\':torch.cuda.get_device_name(0),\'batch\':8,\'windows_per_study\':24,\n               \'selected\':{\'augment_on_gpu\':True,\'channels_last\':False},\n               \'note\':\'Execution fixed to current Modal arm; no speed benchmark run on Kaggle.\'}\n    if not (run/\'execution.json\').exists(): (run/\'execution.json\').write_text(json.dumps(execution,indent=2)+\'\\n\')\n    sys.path.insert(0,str(source))\n    trainer_path=work/\'trainer.py\';trainer_path.write_text(prepare_resnet34_trainer(source_file.read_text()))\n    spec=importlib.util.spec_from_file_location(\'resnet34_kaggle_trainer\',trainer_path)\n    trainer=importlib.util.module_from_spec(spec);sys.modules[spec.name]=trainer;spec.loader.exec_module(trainer)\n    trainer.STEVEN_AUDIT_LABELS=trainer.read_soft_labels(assets/\'steven_audit_labels.json\')\n    trainer.DURABLE_COMMIT=lambda: None  # Kaggle local files; no Modal client/server use.\n    sys.argv=[\'trainer\',\'--cache\',*map(str,cache_dirs),\'--soft-labels\',str(label_path),\n              \'--validation-soft-labels\',str(assets/\'validation_labels.json\'),\'--folds\',str(assets/\'folds.csv\'),\n              \'--config\',str(assets/\'config.json\'),\'--output-dir\',str(run),\'--validation-fold\',\'0\',\'--train-slices-per-study\',\'24\']\n    trainer.main()\n    summary=json.loads((run/\'run_summary.json\').read_text())\n    assert summary[\'epochs_run\'] == 60 and not summary[\'stopped_early\']\n    print(\'ALL_60_EPOCHS_COMPLETE\',flush=True)\n\nif __name__==\'__main__\':main()\n'}
ASSETS = {'folds.csv': 'study_id,fold\r\n1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260,3\r\n1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585,5\r\n1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231,4\r\n1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764,1\r\n1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542,3\r\n1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339,4\r\n1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101,3\r\n1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606,3\r\n1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513,0\r\n1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559,3\r\n1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394,4\r\n1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938,2\r\n1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806,2\r\n1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981,5\r\n1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018,0\r\n1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866,4\r\n1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801,3\r\n1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401,1\r\n1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269,3\r\n1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922,0\r\n1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736,1\r\n1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097,4\r\n1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018,3\r\n1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941,3\r\n1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347,0\r\n1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365,2\r\n1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449,3\r\n1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097,1\r\n1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678,0\r\n1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863,0\r\n1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557,2\r\n1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364,3\r\n1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659,3\r\n1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636,1\r\n1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100,4\r\n1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882,4\r\n1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339,1\r\n1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343,5\r\n1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834,4\r\n1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833,4\r\n1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586,3\r\n1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166,0\r\n1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864,1\r\n1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471,3\r\n1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104,2\r\n1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182,3\r\n1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314,2\r\n1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824,3\r\n1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586,0\r\n1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725,2\r\n1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221,4\r\n1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019,4\r\n1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790,2\r\n1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205,5\r\n1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320,4\r\n1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930,3\r\n1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484,1\r\n1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536,0\r\n1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161,4\r\n1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478,1\r\n1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415,1\r\n1.2.826.0.1.3680043.8.498.10165669527316708372190215034240922433,2\r\n1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973,0\r\n1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787,2\r\n1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197,1\r\n1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292,3\r\n1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364,3\r\n1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507,2\r\n1.2.826.0.1.3680043.8.498.10197515493414564227930855210542221039,1\r\n1.2.826.0.1.3680043.8.498.10197565004992937722841731691775578390,3\r\n1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773,1\r\n1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801,3\r\n1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775,2\r\n1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183,2\r\n1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516,0\r\n1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585,2\r\n1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129,2\r\n1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062,4\r\n1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375,1\r\n1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821,0\r\n1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115,4\r\n1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576,4\r\n1.2.826.0.1.3680043.8.498.10240166440044087554182120832488098189,4\r\n1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326,5\r\n1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414,0\r\n1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307,0\r\n1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833,1\r\n1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531,0\r\n1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203,1\r\n1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286,1\r\n1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766,4\r\n1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873,1\r\n1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903,0\r\n1.2.826.0.1.3680043.8.498.10262024906324456863812446480254600405,3\r\n1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839,4\r\n1.2.826.0.1.3680043.8.498.10271567891603078238024189515805908561,1\r\n1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542,3\r\n1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029,0\r\n1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382,2\r\n1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639,0\r\n1.2.826.0.1.3680043.8.498.10284263494252700630857173843779724077,1\r\n1.2.826.0.1.3680043.8.498.10284446538566255572004044331831825072,3\r\n1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669,4\r\n1.2.826.0.1.3680043.8.498.10299396087685174806129850512856451005,5\r\n1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808,2\r\n1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097,5\r\n1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106,2\r\n1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132,1\r\n1.2.826.0.1.3680043.8.498.10310831394568337664174691281092566272,5\r\n1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738,0\r\n1.2.826.0.1.3680043.8.498.10314240078685804110739316196999571466,1\r\n1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846,4\r\n1.2.826.0.1.3680043.8.498.10321461243762453573005633063753741725,2\r\n1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548,3\r\n1.2.826.0.1.3680043.8.498.10321702681363532511953946967334431952,5\r\n1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315,0\r\n1.2.826.0.1.3680043.8.498.10322853084122920408377349136830993432,2\r\n1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341,5\r\n1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332,3\r\n1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687,5\r\n1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726,1\r\n1.2.826.0.1.3680043.8.498.10336300544286006569730381727202519821,4\r\n1.2.826.0.1.3680043.8.498.10339654093023509742663983374586998753,3\r\n1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936,1\r\n1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306,4\r\n1.2.826.0.1.3680043.8.498.10351168527688244230248760520733020605,1\r\n1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385,2\r\n1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179,1\r\n1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197,1\r\n1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355,2\r\n1.2.826.0.1.3680043.8.498.10361030765327974510262894062047216433,3\r\n1.2.826.0.1.3680043.8.498.10370612191212456931506340727148605247,2\r\n1.2.826.0.1.3680043.8.498.10371162371589200772388569623043753617,3\r\n1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218,4\r\n1.2.826.0.1.3680043.8.498.10378535286770237477615181844634903700,5\r\n1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801,0\r\n1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500,0\r\n1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066,1\r\n1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341,3\r\n1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432,1\r\n1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419,2\r\n1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933,1\r\n1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724,0\r\n1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007,2\r\n1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334,3\r\n1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409,1\r\n1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434,0\r\n1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713,5\r\n1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667,0\r\n1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715,4\r\n1.2.826.0.1.3680043.8.498.10419958821528108176389641942168280311,3\r\n1.2.826.0.1.3680043.8.498.10419993051271996183722192755371692029,3\r\n1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863,0\r\n1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689,3\r\n1.2.826.0.1.3680043.8.498.10423586531262405323219596291234698626,3\r\n1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035,3\r\n1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399,0\r\n1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056,3\r\n1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805,5\r\n1.2.826.0.1.3680043.8.498.10434135978927658622039675788413759069,4\r\n1.2.826.0.1.3680043.8.498.10440858363080910322920496514558347561,3\r\n1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738,4\r\n1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409,3\r\n1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509,1\r\n1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397,2\r\n1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667,0\r\n1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674,1\r\n1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818,4\r\n1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884,2\r\n1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695,0\r\n1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091,0\r\n1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546,1\r\n1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326,4\r\n1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625,1\r\n1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584,1\r\n1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590,2\r\n1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716,0\r\n1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117,3\r\n1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078,1\r\n1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454,4\r\n1.2.826.0.1.3680043.8.498.10511712617129216774539441890488742560,4\r\n1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522,2\r\n1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213,3\r\n1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627,3\r\n1.2.826.0.1.3680043.8.498.10521372086358356219840179458708706015,2\r\n1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946,1\r\n1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196,1\r\n1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018,2\r\n1.2.826.0.1.3680043.8.498.10532708800468111850607752497432656694,4\r\n1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739,2\r\n1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364,5\r\n1.2.826.0.1.3680043.8.498.10536557774352716497356905292652597953,1\r\n1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327,2\r\n1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828,0\r\n1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573,4\r\n1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156,1\r\n1.2.826.0.1.3680043.8.498.10541029960718591409318399557079055927,4\r\n1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801,5\r\n1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216,4\r\n1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251,0\r\n1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953,3\r\n1.2.826.0.1.3680043.8.498.10549360192675575195491716398555148512,3\r\n1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736,0\r\n1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260,2\r\n1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109,1\r\n1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084,0\r\n1.2.826.0.1.3680043.8.498.10580260109117947709069163110100632085,4\r\n1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169,1\r\n1.2.826.0.1.3680043.8.498.10587511485561679190576188107774736852,2\r\n1.2.826.0.1.3680043.8.498.10589889055723532336864750992142411145,3\r\n1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816,0\r\n1.2.826.0.1.3680043.8.498.10590169032156570386146081157598789893,1\r\n1.2.826.0.1.3680043.8.498.10591507262178959553431147886647585916,3\r\n1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628,4\r\n1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137,1\r\n1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798,1\r\n1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146,4\r\n1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290,5\r\n1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196,0\r\n1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241,1\r\n1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514,2\r\n1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403,4\r\n1.2.826.0.1.3680043.8.498.10619486897279334206912764495067756449,3\r\n1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430,3\r\n1.2.826.0.1.3680043.8.498.10624417936475721846835761375334601839,5\r\n1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593,3\r\n1.2.826.0.1.3680043.8.498.10626411974489938664325062396760010912,4\r\n1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971,0\r\n1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477,3\r\n1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506,0\r\n1.2.826.0.1.3680043.8.498.10632177122947017620482277062226288392,4\r\n1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066,4\r\n1.2.826.0.1.3680043.8.498.10648346793256473462304051626410900340,4\r\n1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374,2\r\n1.2.826.0.1.3680043.8.498.10652685418462611085839874321267337079,2\r\n1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658,0\r\n1.2.826.0.1.3680043.8.498.10653432159060774337155662807644609967,3\r\n1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996,5\r\n1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831,3\r\n1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356,2\r\n1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082,2\r\n1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040,3\r\n1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731,1\r\n1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087,4\r\n1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331,0\r\n1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589,4\r\n1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071,5\r\n1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592,2\r\n1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286,1\r\n1.2.826.0.1.3680043.8.498.10699348333280906956391277892869999716,1\r\n1.2.826.0.1.3680043.8.498.10702284568670902901846767166679293654,5\r\n1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080,4\r\n1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811,2\r\n1.2.826.0.1.3680043.8.498.10715956446507906169485719013515169244,3\r\n1.2.826.0.1.3680043.8.498.10724604223190030113012801427752979879,3\r\n1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282,1\r\n1.2.826.0.1.3680043.8.498.10728448245470003618625208276233778895,5\r\n1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557,2\r\n1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171,1\r\n1.2.826.0.1.3680043.8.498.10736346409444614168571281812134757732,2\r\n1.2.826.0.1.3680043.8.498.10738640862162795323010965765443420384,2\r\n1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802,0\r\n1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519,2\r\n1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375,1\r\n1.2.826.0.1.3680043.8.498.10750145216793948201921667945805555093,1\r\n1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445,0\r\n1.2.826.0.1.3680043.8.498.10751624265599812767108610284996090628,2\r\n1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327,1\r\n1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054,1\r\n1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826,3\r\n1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610,4\r\n1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762,3\r\n1.2.826.0.1.3680043.8.498.10765617676329232046176300833850901740,5\r\n1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428,0\r\n1.2.826.0.1.3680043.8.498.10769126730281890056368854871900324448,1\r\n1.2.826.0.1.3680043.8.498.10770877369291591553624215762368014357,5\r\n1.2.826.0.1.3680043.8.498.10774147135673365265606645781662088745,3\r\n1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555,4\r\n1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334,0\r\n1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770,4\r\n1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002,0\r\n1.2.826.0.1.3680043.8.498.10802653832050736637820225444196833450,3\r\n1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077,3\r\n1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588,1\r\n1.2.826.0.1.3680043.8.498.10812894672739619566834776054635588406,2\r\n1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528,2\r\n1.2.826.0.1.3680043.8.498.10824420630965162625242888916805538597,3\r\n1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263,0\r\n1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306,1\r\n1.2.826.0.1.3680043.8.498.10835889400414225749339608096119636738,1\r\n1.2.826.0.1.3680043.8.498.10839126901366636822676830560774505865,5\r\n1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263,0\r\n1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839,0\r\n1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972,4\r\n1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650,3\r\n1.2.826.0.1.3680043.8.498.10850742579000782417673172155504794301,4\r\n1.2.826.0.1.3680043.8.498.10851562687641986545442408470139368318,2\r\n1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636,2\r\n1.2.826.0.1.3680043.8.498.10854012268828835845581966634225522931,2\r\n1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766,4\r\n1.2.826.0.1.3680043.8.498.10855290714402117092990651362697310270,4\r\n1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655,0\r\n1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551,4\r\n1.2.826.0.1.3680043.8.498.10862481204237376580557516731952961213,4\r\n1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385,4\r\n1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617,3\r\n1.2.826.0.1.3680043.8.498.10880553082640412137979450100928382021,2\r\n1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748,2\r\n1.2.826.0.1.3680043.8.498.10886583574130697346788286647281863932,2\r\n1.2.826.0.1.3680043.8.498.10890699078723972693302029683905114834,3\r\n1.2.826.0.1.3680043.8.498.10895521387894821238259709507311341758,1\r\n1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673,1\r\n1.2.826.0.1.3680043.8.498.10902166579916526176697193295886992412,4\r\n1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264,0\r\n1.2.826.0.1.3680043.8.498.10917389623891555306584069980815338466,2\r\n1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881,1\r\n1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433,1\r\n1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235,3\r\n1.2.826.0.1.3680043.8.498.10920269679387171659375091645187772303,2\r\n1.2.826.0.1.3680043.8.498.10924685588962763768106617368647678902,3\r\n1.2.826.0.1.3680043.8.498.10924995713871537837233985073507509566,2\r\n1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359,1\r\n1.2.826.0.1.3680043.8.498.10930374192038896613513882597332665403,1\r\n1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398,0\r\n1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737,1\r\n1.2.826.0.1.3680043.8.498.10933003998197427979325590791540457658,1\r\n1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722,0\r\n1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768,0\r\n1.2.826.0.1.3680043.8.498.10937585701850306334458420860276069606,1\r\n1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626,3\r\n1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409,3\r\n1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670,0\r\n1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514,1\r\n1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575,2\r\n1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245,3\r\n1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415,4\r\n1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528,3\r\n1.2.826.0.1.3680043.8.498.10979105661314213697886164132444977106,2\r\n1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424,4\r\n1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962,0\r\n1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699,2\r\n1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100,2\r\n1.2.826.0.1.3680043.8.498.10994848378662155711176898659205016445,3\r\n1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174,2\r\n1.2.826.0.1.3680043.8.498.10994899269366718796438389394954844979,2\r\n1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480,0\r\n1.2.826.0.1.3680043.8.498.10998144118962608870602180672090998054,4\r\n1.2.826.0.1.3680043.8.498.10998405409474759941744305665540376339,3\r\n1.2.826.0.1.3680043.8.498.11006241238894862275805535376070111510,2\r\n1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256,2\r\n1.2.826.0.1.3680043.8.498.11010004366885474301784150582039137111,2\r\n1.2.826.0.1.3680043.8.498.11011061033863592500429481719734134286,4\r\n1.2.826.0.1.3680043.8.498.11014091506864318694851337553197477906,3\r\n1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310,0\r\n1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088,0\r\n1.2.826.0.1.3680043.8.498.11043081829401721095803441219553837608,2\r\n1.2.826.0.1.3680043.8.498.11043543420859531726426521315399258528,2\r\n1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282,4\r\n1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545,1\r\n1.2.826.0.1.3680043.8.498.11049882927267755769995376553330493298,3\r\n1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008,0\r\n1.2.826.0.1.3680043.8.498.11052656375391331522834364601050945860,2\r\n1.2.826.0.1.3680043.8.498.11052720536194493057619720620119095739,1\r\n1.2.826.0.1.3680043.8.498.11057883757434391643878168168845592475,3\r\n1.2.826.0.1.3680043.8.498.11060834081463491262807709963877480026,4\r\n1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547,0\r\n1.2.826.0.1.3680043.8.498.11067200656721609390997819562401839124,4\r\n1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089,1\r\n1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930,0\r\n1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995,4\r\n1.2.826.0.1.3680043.8.498.11080651962904815250736183346281427445,3\r\n1.2.826.0.1.3680043.8.498.11084010833221855396433326769153950681,2\r\n1.2.826.0.1.3680043.8.498.11090884548990390279582832205294943635,4\r\n1.2.826.0.1.3680043.8.498.11091958567356981151034727948765010881,2\r\n1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233,3\r\n1.2.826.0.1.3680043.8.498.11103894265181149878864749992746236768,1\r\n1.2.826.0.1.3680043.8.498.11110575656494405149601490027444484170,4\r\n1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427,1\r\n1.2.826.0.1.3680043.8.498.11113442398569287228475456000446876768,2\r\n1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324,0\r\n1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213,0\r\n1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911,4\r\n1.2.826.0.1.3680043.8.498.11125718814639515292482773418372267821,4\r\n1.2.826.0.1.3680043.8.498.11126383719852491587081347886000714259,3\r\n1.2.826.0.1.3680043.8.498.11128099896284546522122813576483001027,3\r\n1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290,1\r\n1.2.826.0.1.3680043.8.498.11131210140372132238166068091728877219,4\r\n1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419,4\r\n1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286,0\r\n1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425,0\r\n1.2.826.0.1.3680043.8.498.11148939654362681630811787929668902046,3\r\n1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960,4\r\n1.2.826.0.1.3680043.8.498.11150193626240150189232364321568347937,3\r\n1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456,2\r\n1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758,0\r\n1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033,2\r\n1.2.826.0.1.3680043.8.498.11154238555579687338157479416363307605,1\r\n1.2.826.0.1.3680043.8.498.11156884483511690263238401598901614096,3\r\n1.2.826.0.1.3680043.8.498.11159546221032123495135103648570690656,3\r\n1.2.826.0.1.3680043.8.498.11173951743276817877673759975853741079,2\r\n1.2.826.0.1.3680043.8.498.11180457573346204459956051956213528997,4\r\n1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392,3\r\n1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747,0\r\n1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863,4\r\n1.2.826.0.1.3680043.8.498.11191015483427903713800997920940844550,4\r\n1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131,0\r\n1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618,2\r\n1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878,4\r\n1.2.826.0.1.3680043.8.498.11207013225631251718312189933025529824,3\r\n1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258,0\r\n1.2.826.0.1.3680043.8.498.11208663314213048426907645330143931419,1\r\n1.2.826.0.1.3680043.8.498.11210361780779474361817208167009080083,4\r\n1.2.826.0.1.3680043.8.498.11211598263863419448236436640277320409,4\r\n1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731,0\r\n1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892,0\r\n1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627,0\r\n1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678,5\r\n1.2.826.0.1.3680043.8.498.11219813299300915347039206446596901205,4\r\n1.2.826.0.1.3680043.8.498.11221318524515817857172161854667955115,3\r\n1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884,1\r\n1.2.826.0.1.3680043.8.498.11226582925516370364253861721606580867,5\r\n1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754,4\r\n1.2.826.0.1.3680043.8.498.11231822445970527276203191047278744341,2\r\n1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376,3\r\n1.2.826.0.1.3680043.8.498.11244115315611630362929576340631670750,1\r\n1.2.826.0.1.3680043.8.498.11249981834802444119817546155435759007,1\r\n1.2.826.0.1.3680043.8.498.11254890841796835340329521299102281678,1\r\n1.2.826.0.1.3680043.8.498.11257168136762043265165109052826442974,3\r\n1.2.826.0.1.3680043.8.498.11257869433929994221899994675448655047,5\r\n1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683,0\r\n1.2.826.0.1.3680043.8.498.11262944211104069231500934582412003273,2\r\n1.2.826.0.1.3680043.8.498.11264140845869814478794346490273061499,4\r\n1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204,0\r\n1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885,1\r\n1.2.826.0.1.3680043.8.498.11275711698468745807758222802302685022,3\r\n1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551,2\r\n1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291,0\r\n1.2.826.0.1.3680043.8.498.11277078989123345865112116288626381353,0\r\n1.2.826.0.1.3680043.8.498.11277278232377185347750793496260885546,0\r\n1.2.826.0.1.3680043.8.498.11282162920338602001456704018617788412,3\r\n1.2.826.0.1.3680043.8.498.11282706861378379965012380644125662529,3\r\n1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593,0\r\n1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509,2\r\n1.2.826.0.1.3680043.8.498.11294885816514924203337081832122619605,1\r\n1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644,4\r\n1.2.826.0.1.3680043.8.498.11296867854260286881418279496022384461,4\r\n1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393,3\r\n1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632,4\r\n1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411,0\r\n1.2.826.0.1.3680043.8.498.11312798102268182343541500463824266929,1\r\n1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616,2\r\n1.2.826.0.1.3680043.8.498.11315871586936178043350206344448851055,3\r\n1.2.826.0.1.3680043.8.498.11316091466685605758061778045311969672,3\r\n1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712,0\r\n1.2.826.0.1.3680043.8.498.11334486185870324597335039163116965763,3\r\n1.2.826.0.1.3680043.8.498.11334776960319816844935217444817103072,2\r\n1.2.826.0.1.3680043.8.498.11337616353935430160375524217870968575,1\r\n1.2.826.0.1.3680043.8.498.11338962847958062668991874160027887372,4\r\n1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260,3\r\n1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473,0\r\n1.2.826.0.1.3680043.8.498.11347466940974045182979684641300336177,1\r\n1.2.826.0.1.3680043.8.498.11352722814008170661344291337061231555,1\r\n1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722,0\r\n1.2.826.0.1.3680043.8.498.11362547519125634779422991321337283241,2\r\n1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897,4\r\n1.2.826.0.1.3680043.8.498.11371400503360468962117579439852064607,3\r\n1.2.826.0.1.3680043.8.498.11383056981606331444758741778014516431,1\r\n1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994,1\r\n1.2.826.0.1.3680043.8.498.11386074316342852649653530230563922899,4\r\n1.2.826.0.1.3680043.8.498.11391095418955817124325938917797687542,0\r\n1.2.826.0.1.3680043.8.498.11394861774586578112386386342670500749,1\r\n1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830,0\r\n1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849,0\r\n1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602,5\r\n1.2.826.0.1.3680043.8.498.11406304618685790128709325242250648417,1\r\n1.2.826.0.1.3680043.8.498.11406381777983128446583783158882827226,3\r\n1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512,0\r\n1.2.826.0.1.3680043.8.498.11414215447313658600502835138277811260,4\r\n1.2.826.0.1.3680043.8.498.11420670844338686032030860787688858057,3\r\n1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127,1\r\n1.2.826.0.1.3680043.8.498.11425683153901568541658945026861110110,2\r\n1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597,0\r\n1.2.826.0.1.3680043.8.498.11432815311029789072253610823554308183,4\r\n1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794,1\r\n1.2.826.0.1.3680043.8.498.11440217845998767853986099058965644463,1\r\n1.2.826.0.1.3680043.8.498.11441489276425334980205891153179893797,2\r\n1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623,4\r\n1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579,3\r\n1.2.826.0.1.3680043.8.498.11448865770537436004548718998804636234,2\r\n1.2.826.0.1.3680043.8.498.11451448731901996559834481853733404099,4\r\n1.2.826.0.1.3680043.8.498.11451981256762378203455481812714370193,1\r\n1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769,4\r\n1.2.826.0.1.3680043.8.498.11458488293461896930750018852127694875,1\r\n1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197,0\r\n1.2.826.0.1.3680043.8.498.11465586454035959390661499120751232686,5\r\n1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592,2\r\n1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121,0\r\n1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047,0\r\n1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558,0\r\n1.2.826.0.1.3680043.8.498.11477000011453869751231224714283666904,3\r\n1.2.826.0.1.3680043.8.498.11477583622065195478886525904745372063,4\r\n1.2.826.0.1.3680043.8.498.11482325994550095339373157256174349858,4\r\n1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719,0\r\n1.2.826.0.1.3680043.8.498.11489881286208501558797333161632486099,1\r\n1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434,3\r\n1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046,5\r\n1.2.826.0.1.3680043.8.498.11498239309891927151456629915927889165,1\r\n1.2.826.0.1.3680043.8.498.11498505068787084440256890724559761815,1\r\n1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981,4\r\n1.2.826.0.1.3680043.8.498.11501070108039994411346060488666341726,1\r\n1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588,1\r\n1.2.826.0.1.3680043.8.498.11506642732332295874076832453198901115,2\r\n1.2.826.0.1.3680043.8.498.11506856887791286395737465774061395929,1\r\n1.2.826.0.1.3680043.8.498.11508340006652282767645009271305921426,1\r\n1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664,2\r\n1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433,3\r\n1.2.826.0.1.3680043.8.498.11516842786271998149681118182571061813,4\r\n1.2.826.0.1.3680043.8.498.11520508017843738484108845144418061602,4\r\n1.2.826.0.1.3680043.8.498.11521971289581057766555206729274639736,3\r\n1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388,2\r\n1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983,0\r\n1.2.826.0.1.3680043.8.498.11532517984322509215429681648257299355,4\r\n1.2.826.0.1.3680043.8.498.11533971444023236216745801026152089075,2\r\n1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463,2\r\n1.2.826.0.1.3680043.8.498.11543622448213658856759993374276266341,2\r\n1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974,3\r\n1.2.826.0.1.3680043.8.498.11546127923715190914128277599849591183,0\r\n1.2.826.0.1.3680043.8.498.11547192037231064935759879673500408149,3\r\n1.2.826.0.1.3680043.8.498.11548839502674800051973767797346682106,3\r\n1.2.826.0.1.3680043.8.498.11551347885357242022697196552822735795,5\r\n1.2.826.0.1.3680043.8.498.11551770225828320378339882849245348148,0\r\n1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962,2\r\n1.2.826.0.1.3680043.8.498.11569355546217846610131708883985793793,1\r\n1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425,4\r\n1.2.826.0.1.3680043.8.498.11571057273590444914079281238002921868,1\r\n1.2.826.0.1.3680043.8.498.11574480940878399452466699512074769686,5\r\n1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967,3\r\n1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804,3\r\n1.2.826.0.1.3680043.8.498.11579571696748051185342754881791715610,4\r\n1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552,1\r\n1.2.826.0.1.3680043.8.498.11586616391541918134110642641868079265,3\r\n1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193,3\r\n1.2.826.0.1.3680043.8.498.11590096889491409910425456896187293515,1\r\n1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824,1\r\n1.2.826.0.1.3680043.8.498.11592838054859712596515534341207971997,5\r\n1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789,4\r\n1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584,1\r\n1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202,3\r\n1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457,0\r\n1.2.826.0.1.3680043.8.498.11599387556469808488160280438305325860,1\r\n1.2.826.0.1.3680043.8.498.11604138460812393624691789275263090511,3\r\n1.2.826.0.1.3680043.8.498.11604298807403466096392316117803661479,3\r\n1.2.826.0.1.3680043.8.498.11605266731872772432220415835172557961,2\r\n1.2.826.0.1.3680043.8.498.11610176134207131408691324676304640501,3\r\n1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552,5\r\n1.2.826.0.1.3680043.8.498.11613637647831801354398273334552399874,2\r\n1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849,0\r\n1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378,0\r\n1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097,4\r\n1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397,3\r\n1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612,2\r\n1.2.826.0.1.3680043.8.498.11642228346306175407288158335830376197,4\r\n1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014,1\r\n1.2.826.0.1.3680043.8.498.11646465814751330102194657154003685957,4\r\n1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181,0\r\n1.2.826.0.1.3680043.8.498.11651755040149549966484720727140190853,1\r\n1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843,4\r\n1.2.826.0.1.3680043.8.498.11654064547593365398103107850228629066,4\r\n1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623,0\r\n1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093,2\r\n1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493,2\r\n1.2.826.0.1.3680043.8.498.11669120316759278775792190599036288128,3\r\n1.2.826.0.1.3680043.8.498.11674925658184069242322832524274428408,1\r\n1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847,1\r\n1.2.826.0.1.3680043.8.498.11679981444785137570894759919380932611,4\r\n1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171,2\r\n1.2.826.0.1.3680043.8.498.11685904538570360040472064238371379681,4\r\n1.2.826.0.1.3680043.8.498.11689820638581431730501617718741779835,5\r\n1.2.826.0.1.3680043.8.498.11691245281415120685024733512803575288,3\r\n1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860,3\r\n1.2.826.0.1.3680043.8.498.11703195077261884009442719578918319962,2\r\n1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917,2\r\n1.2.826.0.1.3680043.8.498.11709909941833245598246010905322129256,4\r\n1.2.826.0.1.3680043.8.498.11716605112302071535035323666308656381,4\r\n1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429,0\r\n1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637,0\r\n1.2.826.0.1.3680043.8.498.11720535027022778971695130536590302635,4\r\n1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928,0\r\n1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612,0\r\n1.2.826.0.1.3680043.8.498.11723286647024326363349453996818420374,4\r\n1.2.826.0.1.3680043.8.498.11723945299462627737346907048285143063,3\r\n1.2.826.0.1.3680043.8.498.11725843001854130477287484865097535511,2\r\n1.2.826.0.1.3680043.8.498.11726452645519338341002251323463736083,4\r\n1.2.826.0.1.3680043.8.498.11738872955471637985947490962985266753,3\r\n1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607,0\r\n1.2.826.0.1.3680043.8.498.11752091432368750506312921028559788225,1\r\n1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695,0\r\n1.2.826.0.1.3680043.8.498.11762352496600661264881556836248251400,3\r\n1.2.826.0.1.3680043.8.498.11762851510866726051342964961256680025,1\r\n1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950,1\r\n1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015,2\r\n1.2.826.0.1.3680043.8.498.11776647434143676427722616249587614803,4\r\n1.2.826.0.1.3680043.8.498.11779387602988880691008088661075117406,3\r\n1.2.826.0.1.3680043.8.498.11783597160745641720078978406968189488,0\r\n1.2.826.0.1.3680043.8.498.11785138333495400389510795613252286069,1\r\n1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663,2\r\n1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269,0\r\n1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483,0\r\n1.2.826.0.1.3680043.8.498.11791567830371683660578052737702299124,2\r\n1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033,0\r\n1.2.826.0.1.3680043.8.498.11796547950032543541071896156852778697,2\r\n1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762,1\r\n1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855,4\r\n1.2.826.0.1.3680043.8.498.11803762390999881908101130422708589537,4\r\n1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758,0\r\n1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991,4\r\n1.2.826.0.1.3680043.8.498.11807494462283790976935891042637502434,3\r\n1.2.826.0.1.3680043.8.498.11812607848520072826145426247813670653,1\r\n1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969,3\r\n1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108,0\r\n1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200,3\r\n1.2.826.0.1.3680043.8.498.11836929166973549727078374306175992474,4\r\n1.2.826.0.1.3680043.8.498.11848828525319749834876185415845381479,4\r\n1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417,5\r\n1.2.826.0.1.3680043.8.498.11860731961337988611847638299761007498,2\r\n1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862,0\r\n1.2.826.0.1.3680043.8.498.11867198902144602395671904591496233129,4\r\n1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697,2\r\n1.2.826.0.1.3680043.8.498.11868389442143514705330687746274512632,4\r\n1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011,0\r\n1.2.826.0.1.3680043.8.498.11878847808072119549790690649256090471,1\r\n1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762,0\r\n1.2.826.0.1.3680043.8.498.11881735311125899170335963782017197184,2\r\n1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229,0\r\n1.2.826.0.1.3680043.8.498.11892908237449514945544380023820626319,2\r\n1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794,4\r\n1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744,0\r\n1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673,1\r\n1.2.826.0.1.3680043.8.498.11902464730418099436837163594625541861,2\r\n1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369,0\r\n1.2.826.0.1.3680043.8.498.11905397531490569218559534495408776672,0\r\n1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734,3\r\n1.2.826.0.1.3680043.8.498.11909779134152823550896593167929611759,5\r\n1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708,0\r\n1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707,0\r\n1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851,4\r\n1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715,0\r\n1.2.826.0.1.3680043.8.498.11919420571588879483419578078755803351,4\r\n1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066,4\r\n1.2.826.0.1.3680043.8.498.11920531053482540304184657218030242597,4\r\n1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182,1\r\n1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920,2\r\n1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028,2\r\n1.2.826.0.1.3680043.8.498.11927352926669104658144933891780551483,5\r\n1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890,2\r\n1.2.826.0.1.3680043.8.498.11935116791150327546384719051023540185,0\r\n1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770,3\r\n1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140,0\r\n1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014,1\r\n1.2.826.0.1.3680043.8.498.11944762815616453054528091721553484857,4\r\n1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769,0\r\n1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600,0\r\n1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277,4\r\n1.2.826.0.1.3680043.8.498.11958117172819754862618612170999909480,0\r\n1.2.826.0.1.3680043.8.498.11964826300307458382604479841253457304,5\r\n1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317,0\r\n1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451,3\r\n1.2.826.0.1.3680043.8.498.11979034558288454336812613626594510186,3\r\n1.2.826.0.1.3680043.8.498.11979906027101464432999416052202616662,4\r\n1.2.826.0.1.3680043.8.498.11980187124986749407584543247586384443,1\r\n1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970,1\r\n1.2.826.0.1.3680043.8.498.11981170645605720774509861722988500673,0\r\n1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499,2\r\n1.2.826.0.1.3680043.8.498.11991512352474459753052051071717622771,3\r\n1.2.826.0.1.3680043.8.498.11992172807590757302730513420827032389,3\r\n1.2.826.0.1.3680043.8.498.11999008448480965933785988718780864948,2\r\n1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473,5\r\n1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878,0\r\n1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792,0\r\n1.2.826.0.1.3680043.8.498.12009816058271421855131593296751372723,4\r\n1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084,2\r\n1.2.826.0.1.3680043.8.498.12014959202771141042815134335571931814,2\r\n1.2.826.0.1.3680043.8.498.12015406188790105426257083591807220760,3\r\n1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122,2\r\n1.2.826.0.1.3680043.8.498.12031315219947933845048665397149229969,0\r\n1.2.826.0.1.3680043.8.498.12034579945815402722591687609951730205,2\r\n1.2.826.0.1.3680043.8.498.12036404333388388978317335770611404933,2\r\n1.2.826.0.1.3680043.8.498.12041392088066705782960819598258792338,3\r\n1.2.826.0.1.3680043.8.498.12041491173612341571223481616114183729,3\r\n1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857,0\r\n1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179,3\r\n1.2.826.0.1.3680043.8.498.12051169109587120248372991913002670813,3\r\n1.2.826.0.1.3680043.8.498.12054099347723802635102026371218576198,3\r\n1.2.826.0.1.3680043.8.498.12063742398421783626420335095128808816,3\r\n1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953,0\r\n1.2.826.0.1.3680043.8.498.12067625141402105065337444349201280606,1\r\n1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017,3\r\n1.2.826.0.1.3680043.8.498.12071998470084955703027522152040042364,4\r\n1.2.826.0.1.3680043.8.498.12073401429398042797505820525732905336,3\r\n1.2.826.0.1.3680043.8.498.12092744899746753313905432642371563786,2\r\n1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491,0\r\n1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205,0\r\n1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515,1\r\n1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415,0\r\n1.2.826.0.1.3680043.8.498.12106775587428595108771304868101362618,1\r\n1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051,4\r\n1.2.826.0.1.3680043.8.498.12110510829608190094905588610131531103,3\r\n1.2.826.0.1.3680043.8.498.12117377043894182951345966247899775612,5\r\n1.2.826.0.1.3680043.8.498.12130605331118953242364684736005454581,2\r\n1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761,3\r\n1.2.826.0.1.3680043.8.498.12131866860554871145836544571379633595,4\r\n1.2.826.0.1.3680043.8.498.12132814192991105841939278718608095213,2\r\n1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990,1\r\n1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614,0\r\n1.2.826.0.1.3680043.8.498.12139829070290803654867114374799823711,0\r\n1.2.826.0.1.3680043.8.498.12148236440353424874546037143134438088,2\r\n1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267,4\r\n1.2.826.0.1.3680043.8.498.12153755823296803843819361055386265595,5\r\n1.2.826.0.1.3680043.8.498.12154188739734505793699974626991872621,3\r\n1.2.826.0.1.3680043.8.498.12158630368692962312115489365152319457,4\r\n1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297,0\r\n1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169,2\r\n1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045,4\r\n1.2.826.0.1.3680043.8.498.12166235678892893545054546864889743251,5\r\n1.2.826.0.1.3680043.8.498.12168045829469202472124733625621943469,3\r\n1.2.826.0.1.3680043.8.498.12170753305862973517190756418865414984,2\r\n1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851,0\r\n1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217,0\r\n1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581,3\r\n1.2.826.0.1.3680043.8.498.12176469158911896593955662590695392044,3\r\n1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211,0\r\n1.2.826.0.1.3680043.8.498.12181591358572249491885248089245858515,2\r\n1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302,0\r\n1.2.826.0.1.3680043.8.498.12185615973620890770158935333033805789,2\r\n1.2.826.0.1.3680043.8.498.12186494601308291174548814564387764515,4\r\n1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408,0\r\n1.2.826.0.1.3680043.8.498.12192455583120681639582915365754780698,4\r\n1.2.826.0.1.3680043.8.498.12195007606427082866449558290311905939,5\r\n1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360,3\r\n1.2.826.0.1.3680043.8.498.12200672323103829956274910258854933266,1\r\n1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796,0\r\n1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681,0\r\n1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753,4\r\n1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968,0\r\n1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986,3\r\n1.2.826.0.1.3680043.8.498.12219658385745696203053079500246399481,4\r\n1.2.826.0.1.3680043.8.498.12221892384948220776502422531414659289,2\r\n1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735,1\r\n1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465,1\r\n1.2.826.0.1.3680043.8.498.12232127448545053821926845887026738988,3\r\n1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615,4\r\n1.2.826.0.1.3680043.8.498.12238865510201989224648662550299925120,3\r\n1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494,4\r\n1.2.826.0.1.3680043.8.498.12242172267695588655664456078003635837,4\r\n1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136,0\r\n1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817,0\r\n1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014,0\r\n1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392,0\r\n1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327,1\r\n1.2.826.0.1.3680043.8.498.12257119852150088212071954292747252243,3\r\n1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341,3\r\n1.2.826.0.1.3680043.8.498.12259896940707176389739553110492954399,0\r\n1.2.826.0.1.3680043.8.498.12260661730837055872432103268905182876,3\r\n1.2.826.0.1.3680043.8.498.12261175520056712414700866621816882336,4\r\n1.2.826.0.1.3680043.8.498.12264607993101717201407955391437731134,1\r\n1.2.826.0.1.3680043.8.498.12266054872802015665014176014323258615,1\r\n1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065,3\r\n1.2.826.0.1.3680043.8.498.12266644932420958618394322959047940542,4\r\n1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039,2\r\n1.2.826.0.1.3680043.8.498.12271616365050313084031931009756086606,2\r\n1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367,0\r\n1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249,2\r\n1.2.826.0.1.3680043.8.498.12274801503555668613400469130773239683,1\r\n1.2.826.0.1.3680043.8.498.12275265458583331749472987758046974406,4\r\n1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205,1\r\n1.2.826.0.1.3680043.8.498.12283662652762881783845275662580546217,3\r\n1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132,1\r\n1.2.826.0.1.3680043.8.498.12285469859900617016859868410323043734,4\r\n1.2.826.0.1.3680043.8.498.12286894057773964720268768529280193085,3\r\n1.2.826.0.1.3680043.8.498.12294073960417956629256115458135748872,1\r\n1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729,4\r\n1.2.826.0.1.3680043.8.498.12295443776523723913480216844110882937,4\r\n1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387,3\r\n1.2.826.0.1.3680043.8.498.12300749089664733313051543117302244508,1\r\n1.2.826.0.1.3680043.8.498.12301945990161670321760708884248055936,3\r\n1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199,2\r\n1.2.826.0.1.3680043.8.498.12305281328019552390691110900015627668,3\r\n1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573,4\r\n1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632,4\r\n1.2.826.0.1.3680043.8.498.12308357850075929758995438211317707857,3\r\n1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150,3\r\n1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097,0\r\n1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947,4\r\n1.2.826.0.1.3680043.8.498.12328313245396930020974053819810033945,3\r\n1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914,1\r\n1.2.826.0.1.3680043.8.498.12332727488899501354972751168284397151,3\r\n1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081,5\r\n1.2.826.0.1.3680043.8.498.12335573784261147388984372283481919933,5\r\n1.2.826.0.1.3680043.8.498.12345064031074704673835101507426902246,1\r\n1.2.826.0.1.3680043.8.498.12347007036329783363431172840583325937,1\r\n1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578,0\r\n1.2.826.0.1.3680043.8.498.12348875283993064999589090039392140219,2\r\n1.2.826.0.1.3680043.8.498.12348960754791091583201828167607825033,1\r\n1.2.826.0.1.3680043.8.498.12349656752404066719917550972333542508,1\r\n1.2.826.0.1.3680043.8.498.12352132666379687393012255589355771387,1\r\n1.2.826.0.1.3680043.8.498.12354685390005531691919754048141542648,2\r\n1.2.826.0.1.3680043.8.498.12355011301785011370279005103251815187,4\r\n1.2.826.0.1.3680043.8.498.12356611369549236159671896247094661767,4\r\n1.2.826.0.1.3680043.8.498.12361144153852256276011948786791830435,4\r\n1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269,4\r\n1.2.826.0.1.3680043.8.498.12366238437963147962850341556508931973,2\r\n1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380,0\r\n1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790,0\r\n1.2.826.0.1.3680043.8.498.12377852033407536077897684568488365785,2\r\n1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636,4\r\n1.2.826.0.1.3680043.8.498.12381325071746339154385415220972069782,1\r\n1.2.826.0.1.3680043.8.498.12381909868368799280462187344958674397,4\r\n1.2.826.0.1.3680043.8.498.12383716855639375235119728470042467411,3\r\n1.2.826.0.1.3680043.8.498.12386255059403233724886571725482313876,4\r\n1.2.826.0.1.3680043.8.498.12387075993162655781372091860026675548,3\r\n1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910,0\r\n1.2.826.0.1.3680043.8.498.12387667155732116394489547491673280062,4\r\n1.2.826.0.1.3680043.8.498.12390186846317493876612228407734068726,3\r\n1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894,0\r\n1.2.826.0.1.3680043.8.498.12406043058701411628555010402350032739,1\r\n1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266,3\r\n1.2.826.0.1.3680043.8.498.12408314059030096510457440879771212020,4\r\n1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696,5\r\n1.2.826.0.1.3680043.8.498.12408945963789246646229147558387793270,4\r\n1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831,0\r\n1.2.826.0.1.3680043.8.498.12411396124304166984011280826756258628,0\r\n1.2.826.0.1.3680043.8.498.12414869188691502885668778427120766321,4\r\n1.2.826.0.1.3680043.8.498.12426381920136706157912507136730458010,1\r\n1.2.826.0.1.3680043.8.498.12431272351851438889289190308550697909,1\r\n1.2.826.0.1.3680043.8.498.12432974885645948103184031984290883090,4\r\n1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232,0\r\n1.2.826.0.1.3680043.8.498.12437154858167700438432674983971817217,0\r\n1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876,0\r\n1.2.826.0.1.3680043.8.498.12438775479300781065075835270622538230,3\r\n1.2.826.0.1.3680043.8.498.12450858400030959723767712312663176092,1\r\n1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656,2\r\n1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080,0\r\n1.2.826.0.1.3680043.8.498.12471165461877621368840037945454256612,0\r\n1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896,1\r\n1.2.826.0.1.3680043.8.498.12473755232610784318190599222252893949,4\r\n1.2.826.0.1.3680043.8.498.12481913908649439189738319460342358360,2\r\n1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084,0\r\n1.2.826.0.1.3680043.8.498.12484700311993345120015691857089576898,3\r\n1.2.826.0.1.3680043.8.498.12487362549907644519947672428024977201,5\r\n1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699,3\r\n1.2.826.0.1.3680043.8.498.12490034912643658601552092574630438642,1\r\n1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348,3\r\n1.2.826.0.1.3680043.8.498.12491747080684524905643593142141672951,4\r\n1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074,4\r\n1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525,0\r\n1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072,1\r\n1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038,3\r\n1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919,3\r\n1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696,1\r\n1.2.826.0.1.3680043.8.498.12509825237087140646629745447753158673,5\r\n1.2.826.0.1.3680043.8.498.12509944331769453135724043962410129357,5\r\n1.2.826.0.1.3680043.8.498.12510856755988712246013743789183840748,4\r\n1.2.826.0.1.3680043.8.498.12514357232100339485873453417784586592,1\r\n1.2.826.0.1.3680043.8.498.12515322634403509077334846234426636136,1\r\n1.2.826.0.1.3680043.8.498.12518297594498173981937681848554641851,1\r\n1.2.826.0.1.3680043.8.498.12520203592477518476890010185037272575,3\r\n1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342,0\r\n1.2.826.0.1.3680043.8.498.12523676732876902778577970655429060858,3\r\n1.2.826.0.1.3680043.8.498.12525752631743097782762806859112218620,3\r\n1.2.826.0.1.3680043.8.498.12526590714362529606046044104426807158,4\r\n1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219,1\r\n1.2.826.0.1.3680043.8.498.12531442175319757740745664049001806494,1\r\n1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942,0\r\n1.2.826.0.1.3680043.8.498.12534628800787651008876070514080610643,2\r\n1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111,0\r\n1.2.826.0.1.3680043.8.498.12548176380550230498070609559486512004,4\r\n1.2.826.0.1.3680043.8.498.12552701911592685137269544109687472988,2\r\n1.2.826.0.1.3680043.8.498.12553010776519233077752150342664160826,1\r\n1.2.826.0.1.3680043.8.498.12553455350653672209409408595948190982,3\r\n1.2.826.0.1.3680043.8.498.12554824497579383319980187351290254727,1\r\n1.2.826.0.1.3680043.8.498.12559687473583953266143897736224215680,1\r\n1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616,0\r\n1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395,4\r\n1.2.826.0.1.3680043.8.498.12569762036474456642024618586300354737,2\r\n1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454,4\r\n1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785,1\r\n1.2.826.0.1.3680043.8.498.12574270269124650675934073761741661292,2\r\n1.2.826.0.1.3680043.8.498.12576055921678075813718486704978381993,1\r\n1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298,1\r\n1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431,0\r\n1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184,5\r\n1.2.826.0.1.3680043.8.498.12583872770399266463538638801797827468,0\r\n1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918,0\r\n1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691,3\r\n1.2.826.0.1.3680043.8.498.12607203615180268117341271176820357242,2\r\n1.2.826.0.1.3680043.8.498.12607659092357834912029375690348886159,5\r\n1.2.826.0.1.3680043.8.498.12607959449008110729566106776800628990,3\r\n1.2.826.0.1.3680043.8.498.12614097867016983076979065551985251584,3\r\n1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574,5\r\n1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842,4\r\n1.2.826.0.1.3680043.8.498.12620718517255799151283445360332799702,3\r\n1.2.826.0.1.3680043.8.498.12625250919597111075072308824921101918,3\r\n1.2.826.0.1.3680043.8.498.12626383124142146001532197239671686442,4\r\n1.2.826.0.1.3680043.8.498.12629372332421954368597330267888483345,2\r\n1.2.826.0.1.3680043.8.498.12630423349626505319792444053874011718,2\r\n1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172,0\r\n1.2.826.0.1.3680043.8.498.12632129640405661298623288977632682289,5\r\n1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491,4\r\n1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719,4\r\n1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582,1\r\n1.2.826.0.1.3680043.8.498.12642641661988102031826128876502303807,2\r\n1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635,0\r\n1.2.826.0.1.3680043.8.498.12647791319443598318497312837147189300,2\r\n1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265,0\r\n1.2.826.0.1.3680043.8.498.12655324218817524683138594937984186560,1\r\n1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625,2\r\n1.2.826.0.1.3680043.8.498.12660901572462562819575002681831361427,1\r\n1.2.826.0.1.3680043.8.498.12668708646283106269005674386248135220,2\r\n1.2.826.0.1.3680043.8.498.12673561338227595483297605409292531492,1\r\n1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095,1\r\n1.2.826.0.1.3680043.8.498.12674930973831274527829261576087154644,3\r\n1.2.826.0.1.3680043.8.498.12676359576477452816896379524598090410,2\r\n1.2.826.0.1.3680043.8.498.12688954493547892047021455671738310186,4\r\n1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274,1\r\n1.2.826.0.1.3680043.8.498.12690080124553846006638565600585469969,3\r\n1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630,2\r\n1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922,0\r\n1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111,4\r\n1.2.826.0.1.3680043.8.498.12699751030439008569144814819343161608,2\r\n1.2.826.0.1.3680043.8.498.12700591955254168977239658917260628326,4\r\n1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803,0\r\n1.2.826.0.1.3680043.8.498.12701097867143790763644826659974337046,0\r\n1.2.826.0.1.3680043.8.498.12704063145595847725421427571860589471,1\r\n1.2.826.0.1.3680043.8.498.12709854414521475738435319009767427434,2\r\n1.2.826.0.1.3680043.8.498.12715067947657150279232238280712066972,2\r\n1.2.826.0.1.3680043.8.498.12715130507546263968763238702574191950,5\r\n1.2.826.0.1.3680043.8.498.12715402081094788880283826571710131940,1\r\n1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849,0\r\n1.2.826.0.1.3680043.8.498.12718966639501121738042125605376714231,2\r\n1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688,0\r\n1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405,4\r\n1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956,4\r\n1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122,4\r\n1.2.826.0.1.3680043.8.498.12727488524631885827798129418558343386,2\r\n1.2.826.0.1.3680043.8.498.12728397615513347615111391662798012012,4\r\n1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492,0\r\n1.2.826.0.1.3680043.8.498.12748840878706905309050091576449488666,3\r\n1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067,0\r\n1.2.826.0.1.3680043.8.498.12759704450656741777120047828560448622,5\r\n1.2.826.0.1.3680043.8.498.12761906443846433581503942530016459978,2\r\n1.2.826.0.1.3680043.8.498.12762076104929080571464647825161185643,4\r\n1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361,2\r\n1.2.826.0.1.3680043.8.498.12765434812409174436720333864278447770,3\r\n1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908,0\r\n1.2.826.0.1.3680043.8.498.12769887673556938425710414718385118283,1\r\n1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535,4\r\n1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270,0\r\n1.2.826.0.1.3680043.8.498.12775105605170844918620305197829216665,2\r\n1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333,0\r\n1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949,0\r\n1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180,2\r\n1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802,1\r\n1.2.826.0.1.3680043.8.498.12785792890626338008419452097492192088,3\r\n1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532,2\r\n1.2.826.0.1.3680043.8.498.12792431109288033212787919118858331455,5\r\n1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212,5\r\n1.2.826.0.1.3680043.8.498.12799849540231315910448269889175806934,1\r\n1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358,1\r\n1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010,2\r\n1.2.826.0.1.3680043.8.498.12806792356922865929896103306611295151,5\r\n1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043,4\r\n1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652,0\r\n1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996,3\r\n1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035,0\r\n1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046,1\r\n1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174,4\r\n1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569,2\r\n1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751,0\r\n1.2.826.0.1.3680043.8.498.12826828527176716109949704557150950675,0\r\n1.2.826.0.1.3680043.8.498.12827120140904847104302376671076396695,5\r\n1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052,3\r\n1.2.826.0.1.3680043.8.498.12836702174475748025129787471026627810,4\r\n1.2.826.0.1.3680043.8.498.12837030643153472711780894512565629507,3\r\n1.2.826.0.1.3680043.8.498.12839498544183083639548298117995964432,2\r\n1.2.826.0.1.3680043.8.498.12862782592303338109631225504988756108,3\r\n1.2.826.0.1.3680043.8.498.12864336642218457767597053831458716404,4\r\n1.2.826.0.1.3680043.8.498.12866937766198610064474201436305532400,1\r\n1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655,0\r\n1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502,0\r\n1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608,0\r\n1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832,0\r\n1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039,0\r\n1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155,2\r\n1.2.826.0.1.3680043.8.498.12914677763148740041755977075799271057,3\r\n1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664,0\r\n1.2.826.0.1.3680043.8.498.12919303621342250989179219420795277559,1\r\n1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265,3\r\n1.2.826.0.1.3680043.8.498.12936296973456045538701035923586653502,3\r\n1.2.826.0.1.3680043.8.498.12938500386086171684983091332136233036,4\r\n1.2.826.0.1.3680043.8.498.12942677240800343332600884989364651588,2\r\n1.2.826.0.1.3680043.8.498.12946163460972553911280889205685745877,2\r\n1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544,3\r\n1.2.826.0.1.3680043.8.498.12949322114105642048256996295428966924,3\r\n1.2.826.0.1.3680043.8.498.12950639892345812884896649629409199101,3\r\n1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935,0\r\n1.2.826.0.1.3680043.8.498.12957252711278416684461267069498115130,3\r\n1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111,1\r\n1.2.826.0.1.3680043.8.498.12958999546496283565938189274771483491,1\r\n1.2.826.0.1.3680043.8.498.12961116776122838195555132309258963046,3\r\n1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626,2\r\n1.2.826.0.1.3680043.8.498.12968535455086298222684461157825070395,2\r\n1.2.826.0.1.3680043.8.498.12970315889871740202726744767179863179,2\r\n1.2.826.0.1.3680043.8.498.12978172997053500029791786614906570285,5\r\n1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299,1\r\n1.2.826.0.1.3680043.8.498.12981413356524881858859912294078498486,4\r\n1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172,5\r\n1.2.826.0.1.3680043.8.498.12985683732818911488433522743768046254,1\r\n1.2.826.0.1.3680043.8.498.12986028147682493711912242081052787346,1\r\n1.2.826.0.1.3680043.8.498.12986517636774565418070546261705439006,2\r\n1.2.826.0.1.3680043.8.498.12987428318535758055829100395092244687,3\r\n1.2.826.0.1.3680043.8.498.12992013816638129896738209015383985569,2\r\n1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013,0\r\n1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808,2\r\n1.2.826.0.1.3680043.8.498.13009271274636013067662786774946029285,4\r\n1.2.826.0.1.3680043.8.498.13015502057686077045104167339480215528,2\r\n1.2.826.0.1.3680043.8.498.13016684730504329799460698013153308088,1\r\n1.2.826.0.1.3680043.8.498.13017199074506191657941905498813292773,1\r\n1.2.826.0.1.3680043.8.498.13022266764623819477228425331705640702,4\r\n1.2.826.0.1.3680043.8.498.13022585036130454941964190427558363602,1\r\n1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466,1\r\n1.2.826.0.1.3680043.8.498.13024625496981667037628275235696732200,3\r\n1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870,4\r\n1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594,4\r\n1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852,3\r\n1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589,0\r\n1.2.826.0.1.3680043.8.498.13028226019192925363616760332442456953,1\r\n1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363,0\r\n1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229,2\r\n1.2.826.0.1.3680043.8.498.13035632998746261521316395720102873141,5\r\n1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999,1\r\n1.2.826.0.1.3680043.8.498.13054766954092919774634499780241210835,3\r\n1.2.826.0.1.3680043.8.498.13062438493893996542058869514390908492,2\r\n1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508,0\r\n1.2.826.0.1.3680043.8.498.13065120731888368919957856813755701101,0\r\n1.2.826.0.1.3680043.8.498.13068696303639759696458130704554709416,0\r\n1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974,4\r\n1.2.826.0.1.3680043.8.498.13072908567275370866813748886351904593,1\r\n1.2.826.0.1.3680043.8.498.13073387272228394257811861905056479438,2\r\n1.2.826.0.1.3680043.8.498.13077357855769903335728068420884303993,3\r\n1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769,5\r\n1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703,0\r\n1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421,3\r\n1.2.826.0.1.3680043.8.498.13079170182386765602129976624023222742,2\r\n1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749,4\r\n1.2.826.0.1.3680043.8.498.13087770868099843948960419968576882974,4\r\n1.2.826.0.1.3680043.8.498.13088550794087945662322609097092458487,2\r\n1.2.826.0.1.3680043.8.498.13092094949394905924922534008771583745,3\r\n1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456,3\r\n1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894,0\r\n1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987,4\r\n1.2.826.0.1.3680043.8.498.13099702881219862576749493772702888064,0\r\n1.2.826.0.1.3680043.8.498.13102058401501684064017656674827527261,1\r\n1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786,0\r\n1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590,0\r\n1.2.826.0.1.3680043.8.498.13113623966675790946816351311283701141,2\r\n1.2.826.0.1.3680043.8.498.13114228294359876639109410575915394712,1\r\n1.2.826.0.1.3680043.8.498.13114693814781737495494700902699619138,4\r\n1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127,0\r\n1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285,0\r\n1.2.826.0.1.3680043.8.498.13118532243906060645114391749986421707,1\r\n1.2.826.0.1.3680043.8.498.13125910783461518266627225743585018143,4\r\n1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841,1\r\n1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048,4\r\n1.2.826.0.1.3680043.8.498.13128352910661672966143173193241171100,3\r\n1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142,3\r\n1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773,3\r\n1.2.826.0.1.3680043.8.498.13130534736322715018024786213131944607,1\r\n1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258,3\r\n1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660,0\r\n1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032,3\r\n1.2.826.0.1.3680043.8.498.13136367866260644484250529322937554927,1\r\n1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545,5\r\n1.2.826.0.1.3680043.8.498.13137510961466386836387239510287919268,1\r\n1.2.826.0.1.3680043.8.498.13137549927896953291645459480590055925,3\r\n1.2.826.0.1.3680043.8.498.13139684164906828131777150061532685584,3\r\n1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116,2\r\n1.2.826.0.1.3680043.8.498.13146466304138609448483938844487385775,1\r\n1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270,3\r\n1.2.826.0.1.3680043.8.498.13148805278538458401210722493048566510,2\r\n1.2.826.0.1.3680043.8.498.13150780733560495803861509587935093776,0\r\n1.2.826.0.1.3680043.8.498.13150794288945922409613136250303663502,5\r\n1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956,1\r\n1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387,1\r\n1.2.826.0.1.3680043.8.498.13155572156594271049520969168607160312,2\r\n1.2.826.0.1.3680043.8.498.13157776420553094741176082047621957775,1\r\n1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032,4\r\n1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296,0\r\n1.2.826.0.1.3680043.8.498.13162284470388627411596358366248494046,3\r\n1.2.826.0.1.3680043.8.498.13162479706670045075896993021883562680,2\r\n1.2.826.0.1.3680043.8.498.13163727085668231782866823012535268245,5\r\n1.2.826.0.1.3680043.8.498.13165770394160595162984722917452661205,4\r\n1.2.826.0.1.3680043.8.498.13167727300538353052482338786183678684,4\r\n1.2.826.0.1.3680043.8.498.13172295193446466338347273836250944090,3\r\n1.2.826.0.1.3680043.8.498.13173050135651736408720687361834045815,1\r\n1.2.826.0.1.3680043.8.498.13173781757294564816839795307893228067,3\r\n1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710,0\r\n1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937,2\r\n1.2.826.0.1.3680043.8.498.13179169600887013946657470830682778717,1\r\n1.2.826.0.1.3680043.8.498.13179829911816163852815103052496326166,1\r\n1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369,0\r\n1.2.826.0.1.3680043.8.498.13199635020145400856987271814458645896,2\r\n1.2.826.0.1.3680043.8.498.13200225436827306542001129042739578441,5\r\n1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075,1\r\n1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495,1\r\n1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145,0\r\n1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434,2\r\n1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605,0\r\n1.2.826.0.1.3680043.8.498.13214267416673058581197714011913363750,1\r\n1.2.826.0.1.3680043.8.498.13222515831888962408632292783773537312,2\r\n1.2.826.0.1.3680043.8.498.13224933005220329091792052916905776921,1\r\n1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226,2\r\n1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477,4\r\n1.2.826.0.1.3680043.8.498.13232501202175905676335982676386953813,3\r\n1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655,4\r\n1.2.826.0.1.3680043.8.498.13240749152416621711688053465035013133,3\r\n1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308,2\r\n1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169,0\r\n1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501,3\r\n1.2.826.0.1.3680043.8.498.13256173302993031630968402868348974108,3\r\n1.2.826.0.1.3680043.8.498.13261595972961358158990410740461308054,4\r\n1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025,0\r\n1.2.826.0.1.3680043.8.498.13262473718294425031790270527413368245,2\r\n1.2.826.0.1.3680043.8.498.13262583253812134425581922368942846684,1\r\n1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733,1\r\n1.2.826.0.1.3680043.8.498.13267708980958529833316446819156908381,2\r\n1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636,0\r\n1.2.826.0.1.3680043.8.498.13273890409523605456488117581706115075,3\r\n1.2.826.0.1.3680043.8.498.13280017707177438711374726939683941320,3\r\n1.2.826.0.1.3680043.8.498.13283411144589724334502427926341646116,3\r\n1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634,4\r\n1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662,2\r\n1.2.826.0.1.3680043.8.498.13288244567806030859347115222642155186,1\r\n1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982,1\r\n1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156,5\r\n1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314,4\r\n1.2.826.0.1.3680043.8.498.13295633458001436592005068637719195786,4\r\n1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518,4\r\n1.2.826.0.1.3680043.8.498.13309299049346932814529962458932580753,3\r\n1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875,0\r\n1.2.826.0.1.3680043.8.498.13314276151860609710431717811242659651,4\r\n1.2.826.0.1.3680043.8.498.13319886927127591971322856062062654146,4\r\n1.2.826.0.1.3680043.8.498.13321746459276191365724686301267421531,2\r\n1.2.826.0.1.3680043.8.498.13323187196249612936800308722675884644,2\r\n1.2.826.0.1.3680043.8.498.13328029743559327467301971917273397645,2\r\n1.2.826.0.1.3680043.8.498.13330694332190836651711643932953274573,2\r\n1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121,2\r\n1.2.826.0.1.3680043.8.498.13335786652653204791740906118815827207,3\r\n1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030,0\r\n1.2.826.0.1.3680043.8.498.13353249891620291438488727876474802879,1\r\n1.2.826.0.1.3680043.8.498.13353409340862821090624127486679097099,3\r\n1.2.826.0.1.3680043.8.498.13353966209257569249620562863006006441,3\r\n1.2.826.0.1.3680043.8.498.13357923062946813820390731946535198170,2\r\n1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604,0\r\n1.2.826.0.1.3680043.8.498.13361877959630120404875545203408927433,4\r\n1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567,1\r\n1.2.826.0.1.3680043.8.498.13363856251607349805111249441972460575,3\r\n1.2.826.0.1.3680043.8.498.13367691581890020433564928882602648123,3\r\n1.2.826.0.1.3680043.8.498.13367740087756379568826046059252678700,5\r\n1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388,0\r\n1.2.826.0.1.3680043.8.498.13369443704765101449274062863864122737,1\r\n1.2.826.0.1.3680043.8.498.13371001997511542011416510364359588752,2\r\n1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717,4\r\n1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141,0\r\n1.2.826.0.1.3680043.8.498.13383972749798909973472489456126521194,2\r\n1.2.826.0.1.3680043.8.498.13390551515761269433524925969561757962,2\r\n1.2.826.0.1.3680043.8.498.13391735462825209934041441767886593395,3\r\n1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680,3\r\n1.2.826.0.1.3680043.8.498.13399254446600380420635633478666850540,1\r\n1.2.826.0.1.3680043.8.498.13400273795954308944359576841942016981,4\r\n1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385,1\r\n1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965,0\r\n1.2.826.0.1.3680043.8.498.13440909104569690641314836192184434461,5\r\n1.2.826.0.1.3680043.8.498.13455212888114594010716499984611216646,1\r\n1.2.826.0.1.3680043.8.498.13459473184684263663596121399737033933,5\r\n1.2.826.0.1.3680043.8.498.13460078631594032326006554321483799928,3\r\n1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785,1\r\n1.2.826.0.1.3680043.8.498.13461073016327661159616957998589287085,1\r\n1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185,2\r\n1.2.826.0.1.3680043.8.498.13505671694940677161128783863912245059,2\r\n1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226,0\r\n1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636,4\r\n1.2.826.0.1.3680043.8.498.13616982533395291590655841043722813973,1\r\n1.2.826.0.1.3680043.8.498.13660439785395658826946769630921591235,1\r\n1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179,4\r\n1.2.826.0.1.3680043.8.498.13737588520456064856233576393747389054,4\r\n1.2.826.0.1.3680043.8.498.13762716088334302039143651507177887969,1\r\n1.2.826.0.1.3680043.8.498.13829111847660458141056077151781118782,1\r\n1.2.826.0.1.3680043.8.498.13843368750899904654034226115826130084,3\r\n1.2.826.0.1.3680043.8.498.13879163301346984697246311226647565712,1\r\n1.2.826.0.1.3680043.8.498.13915367834258386094001181500560177470,2\r\n1.2.826.0.1.3680043.8.498.13958765761660666865188384031150827763,2\r\n1.2.826.0.1.3680043.8.498.13972325309264527611989585341636140156,1\r\n1.2.826.0.1.3680043.8.498.13975027824128345697756313869165838055,4\r\n1.2.826.0.1.3680043.8.498.14024129019390041106306062001538710978,2\r\n1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791,0\r\n1.2.826.0.1.3680043.8.498.14061227335828384505403558788548348446,0\r\n1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847,2\r\n1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596,3\r\n1.2.826.0.1.3680043.8.498.14137431039800236510749401303688858447,4\r\n1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704,2\r\n1.2.826.0.1.3680043.8.498.14218113768651197803980865495163189825,3\r\n1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878,3\r\n1.2.826.0.1.3680043.8.498.14236519693915284584208989450687591759,2\r\n1.2.826.0.1.3680043.8.498.14300107110014472682373589824643789093,2\r\n1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838,3\r\n1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575,0\r\n1.2.826.0.1.3680043.8.498.14377917653321813299237274871837695586,1\r\n1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532,3\r\n1.2.826.0.1.3680043.8.498.14453847272947043746856632410254555895,1\r\n1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127,3\r\n1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166,0\r\n1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268,5\r\n1.2.826.0.1.3680043.8.498.14490539215064287912254286565666891557,5\r\n1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791,0\r\n1.2.826.0.1.3680043.8.498.14511862461248267537051973802039959211,1\r\n1.2.826.0.1.3680043.8.498.14558486491688263090686005096911018438,1\r\n1.2.826.0.1.3680043.8.498.14679140388328648617386311272361396555,2\r\n1.2.826.0.1.3680043.8.498.14728068688166107022558957899468489735,5\r\n1.2.826.0.1.3680043.8.498.14744714658078107007743063095375646143,5\r\n1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570,0\r\n1.2.826.0.1.3680043.8.498.14776241668061173271058268311868302014,1\r\n1.2.826.0.1.3680043.8.498.14806659119591146961784818305031900863,1\r\n1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770,0\r\n1.2.826.0.1.3680043.8.498.14850534390650184292244380816212794996,1\r\n1.2.826.0.1.3680043.8.498.14958560978176121375035839707004667832,1\r\n1.2.826.0.1.3680043.8.498.14959561028578335448289993045623217861,4\r\n1.2.826.0.1.3680043.8.498.14959732862523218997149041503831001633,4\r\n1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069,3\r\n1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904,0\r\n1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418,4\r\n1.2.826.0.1.3680043.8.498.15065556607080211420503043544482923211,2\r\n1.2.826.0.1.3680043.8.498.15090991392324602559982474708414427621,2\r\n1.2.826.0.1.3680043.8.498.15103255966832759925770334719703425755,2\r\n1.2.826.0.1.3680043.8.498.15133818248365103660456294775831248835,2\r\n1.2.826.0.1.3680043.8.498.15219410096632334470101257478715035071,5\r\n1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247,4\r\n1.2.826.0.1.3680043.8.498.15297444019822621504849851092382560980,3\r\n1.2.826.0.1.3680043.8.498.15326951535563258414956521742264374802,5\r\n1.2.826.0.1.3680043.8.498.15337067581747365729183643142512558411,3\r\n1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738,0\r\n1.2.826.0.1.3680043.8.498.15376535778246978844757121776560020722,4\r\n1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885,0\r\n1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351,2\r\n1.2.826.0.1.3680043.8.498.15444316810805746453392291780593934328,2\r\n1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269,0\r\n1.2.826.0.1.3680043.8.498.15486463042236628073590077209793828847,2\r\n1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397,4\r\n1.2.826.0.1.3680043.8.498.15539098084847502447238233198010208924,1\r\n1.2.826.0.1.3680043.8.498.15595595490910196952568459534537629865,5\r\n1.2.826.0.1.3680043.8.498.15599161312748548709240395406977042093,3\r\n1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742,2\r\n1.2.826.0.1.3680043.8.498.15709232000390534671340322807442769184,1\r\n1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840,4\r\n1.2.826.0.1.3680043.8.498.15851463019383290210760064201595021029,2\r\n1.2.826.0.1.3680043.8.498.15863325224755490561125835019647303556,4\r\n1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764,0\r\n1.2.826.0.1.3680043.8.498.16046164723877343553376490423345098256,1\r\n1.2.826.0.1.3680043.8.498.16051798246004436152597344963060310360,3\r\n1.2.826.0.1.3680043.8.498.16101435445990180482665354366724730305,3\r\n1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094,4\r\n1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081,0\r\n1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832,1\r\n1.2.826.0.1.3680043.8.498.16200518336931321995379999168737787921,3\r\n1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943,0\r\n1.2.826.0.1.3680043.8.498.16258190393569006141285550986533848849,3\r\n1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452,3\r\n1.2.826.0.1.3680043.8.498.16338296537380784459604897543069625390,4\r\n1.2.826.0.1.3680043.8.498.16364873390453802195568603605191835957,1\r\n1.2.826.0.1.3680043.8.498.16370303687278282143772106415664171358,1\r\n1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331,3\r\n1.2.826.0.1.3680043.8.498.16469256144078887607437578587242258846,2\r\n1.2.826.0.1.3680043.8.498.16483470188406187000599644150600279596,3\r\n1.2.826.0.1.3680043.8.498.16527747624688737126118546487228060848,3\r\n1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367,3\r\n1.2.826.0.1.3680043.8.498.16538738179840493095338324009830673843,3\r\n1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835,0\r\n1.2.826.0.1.3680043.8.498.16581426215518472805200227992550139190,3\r\n1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350,0\r\n1.2.826.0.1.3680043.8.498.16657553974040122719239506008948241297,3\r\n1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849,5\r\n1.2.826.0.1.3680043.8.498.16674237773422474998796191258459643061,3\r\n1.2.826.0.1.3680043.8.498.16691383858003633107641297162554868325,0\r\n1.2.826.0.1.3680043.8.498.16709565360569239280032743619008187278,4\r\n1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779,3\r\n1.2.826.0.1.3680043.8.498.16718788321951483456195507236392554704,4\r\n1.2.826.0.1.3680043.8.498.16724013341359693140331158148551346369,2\r\n1.2.826.0.1.3680043.8.498.16774450022563283092653506225603012645,5\r\n1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414,0\r\n1.2.826.0.1.3680043.8.498.16843381581429596234606436682284624187,3\r\n1.2.826.0.1.3680043.8.498.16845451648760595922758166878121325861,1\r\n1.2.826.0.1.3680043.8.498.16862596867845295946732698348903803700,4\r\n1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153,4\r\n1.2.826.0.1.3680043.8.498.16886658450293145802897654768506439494,4\r\n1.2.826.0.1.3680043.8.498.17049315826418217889213226790962592213,2\r\n1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817,3\r\n1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835,5\r\n1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653,0\r\n1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119,0\r\n1.2.826.0.1.3680043.8.498.17093025979426350838724553020541643719,4\r\n1.2.826.0.1.3680043.8.498.17107696939614797679012793547913769707,2\r\n1.2.826.0.1.3680043.8.498.17123244662958741977525806700986256210,2\r\n1.2.826.0.1.3680043.8.498.17146868851921828837871671912815843012,4\r\n1.2.826.0.1.3680043.8.498.17160610480070701553118850947260201608,1\r\n1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165,1\r\n1.2.826.0.1.3680043.8.498.17265093113586420061761042467476063876,2\r\n1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509,4\r\n1.2.826.0.1.3680043.8.498.17336223563862025807066700226971845206,4\r\n1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823,1\r\n1.2.826.0.1.3680043.8.498.17415936504684075438800895036740822145,2\r\n1.2.826.0.1.3680043.8.498.17420673628956367425739018933968443620,3\r\n1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519,4\r\n1.2.826.0.1.3680043.8.498.17517950899674150719043097786728121608,3\r\n1.2.826.0.1.3680043.8.498.17547781127981007962008163248942714330,2\r\n1.2.826.0.1.3680043.8.498.17596657287050550465463272509976106854,4\r\n1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319,3\r\n1.2.826.0.1.3680043.8.498.17662880041629379035049367252086125592,5\r\n1.2.826.0.1.3680043.8.498.17663005209256897892831319144686413853,5\r\n1.2.826.0.1.3680043.8.498.17691941055331862184525411493110144433,3\r\n1.2.826.0.1.3680043.8.498.17741907690913585163443565102293192244,1\r\n1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650,3\r\n1.2.826.0.1.3680043.8.498.17769942698581399165275319502954100325,3\r\n1.2.826.0.1.3680043.8.498.17813878802252660525798053577039470087,3\r\n1.2.826.0.1.3680043.8.498.17836815413824722543229791044312174885,4\r\n1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916,0\r\n1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366,3\r\n1.2.826.0.1.3680043.8.498.17909590711893200514107254988453660857,4\r\n1.2.826.0.1.3680043.8.498.17931021655186901394796168474942485728,1\r\n1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467,0\r\n1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730,1\r\n1.2.826.0.1.3680043.8.498.18080511167840608448833484000937072258,1\r\n1.2.826.0.1.3680043.8.498.18090147030578547920423840630147682826,3\r\n1.2.826.0.1.3680043.8.498.18092766733837308394607532608319746043,1\r\n1.2.826.0.1.3680043.8.498.18113123512696698441392990570166201548,3\r\n1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285,0\r\n1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009,0\r\n1.2.826.0.1.3680043.8.498.18176586296602067352845573876750324761,1\r\n1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584,2\r\n1.2.826.0.1.3680043.8.498.18230847745299902998425838090653825745,1\r\n1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124,0\r\n1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080,0\r\n1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569,0\r\n1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067,3\r\n1.2.826.0.1.3680043.8.498.18389676218363070462526838642303976332,2\r\n1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908,4\r\n1.2.826.0.1.3680043.8.498.18509793799833391995987656714798904411,4\r\n1.2.826.0.1.3680043.8.498.18518393141873346710197645305647124682,1\r\n1.2.826.0.1.3680043.8.498.18529656610752887218162968403101035390,2\r\n1.2.826.0.1.3680043.8.498.18592535981451766048952107991674227570,2\r\n1.2.826.0.1.3680043.8.498.18614704377104425868545800677445972261,3\r\n1.2.826.0.1.3680043.8.498.18625377919309205857120774497713011583,1\r\n1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748,0\r\n1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829,5\r\n1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006,4\r\n1.2.826.0.1.3680043.8.498.18695727379446620258431748063749176639,2\r\n1.2.826.0.1.3680043.8.498.18721466442792788108431096912844719622,1\r\n1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566,1\r\n1.2.826.0.1.3680043.8.498.18792568294433896490648198192877693850,1\r\n1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334,4\r\n1.2.826.0.1.3680043.8.498.18825393854436144614202928260751418496,1\r\n1.2.826.0.1.3680043.8.498.18848481788456572234982139368351538716,4\r\n1.2.826.0.1.3680043.8.498.18879049235810379709186936621085221483,0\r\n1.2.826.0.1.3680043.8.498.18899625751876258520147203904985857313,2\r\n1.2.826.0.1.3680043.8.498.18931107448953327254073392032141677797,0\r\n1.2.826.0.1.3680043.8.498.18968967345606818331567480514437075646,3\r\n1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874,4\r\n1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295,0\r\n1.2.826.0.1.3680043.8.498.19104350033985745058396432133828748480,2\r\n1.2.826.0.1.3680043.8.498.19124879088354547836118593585646705198,2\r\n1.2.826.0.1.3680043.8.498.19178214677492694135853596920291058397,3\r\n1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548,0\r\n1.2.826.0.1.3680043.8.498.19194090255958016414082788444327981898,3\r\n1.2.826.0.1.3680043.8.498.19197037365998103318720742154440921358,1\r\n1.2.826.0.1.3680043.8.498.19198152070740205434848422424805073748,1\r\n1.2.826.0.1.3680043.8.498.19291375342005405073321673638971145777,4\r\n1.2.826.0.1.3680043.8.498.19342628541833173544083802327412975225,4\r\n1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841,1\r\n1.2.826.0.1.3680043.8.498.19419878169415950073207624537163068568,2\r\n1.2.826.0.1.3680043.8.498.19439950168933299231685815631786334293,2\r\n1.2.826.0.1.3680043.8.498.19486520578867272329833751210055639515,1\r\n1.2.826.0.1.3680043.8.498.19489875090732464985703624680756212359,3\r\n1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506,5\r\n1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182,4\r\n1.2.826.0.1.3680043.8.498.19573195389434178432956976770560221462,3\r\n1.2.826.0.1.3680043.8.498.19592989024994864431870593504844309885,3\r\n1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296,0\r\n1.2.826.0.1.3680043.8.498.19624674714110931897302481276137067638,5\r\n1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310,5\r\n1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031,2\r\n1.2.826.0.1.3680043.8.498.19788612491138642031531719942303619648,3\r\n1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955,4\r\n1.2.826.0.1.3680043.8.498.19821498229702528191367700655079805824,5\r\n1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246,1\r\n1.2.826.0.1.3680043.8.498.19832308119617081676615236082965891208,3\r\n1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530,0\r\n1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760,0\r\n1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001,1\r\n1.2.826.0.1.3680043.8.498.19895710326297085463251295629948896948,3\r\n1.2.826.0.1.3680043.8.498.19925337578871911327225897926650188274,3\r\n1.2.826.0.1.3680043.8.498.19965044457637713843792494926652048462,2\r\n1.2.826.0.1.3680043.8.498.19977273190231806393528180968361858122,4\r\n1.2.826.0.1.3680043.8.498.20023559426196341254845485444587607391,1\r\n1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390,2\r\n1.2.826.0.1.3680043.8.498.20123813962665544845986028832159432587,4\r\n1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830,0\r\n1.2.826.0.1.3680043.8.498.20156361590688072955579948030116579099,4\r\n1.2.826.0.1.3680043.8.498.20184541870535223885310764061146422870,1\r\n1.2.826.0.1.3680043.8.498.20192024727432864451309151144591371229,4\r\n1.2.826.0.1.3680043.8.498.20196165276643741937915346646793735100,2\r\n1.2.826.0.1.3680043.8.498.20204907530808218314127264658715054041,4\r\n1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284,4\r\n1.2.826.0.1.3680043.8.498.20266523169651039493799975212435349766,3\r\n1.2.826.0.1.3680043.8.498.20269296702024202621734017056347194499,1\r\n1.2.826.0.1.3680043.8.498.20301796413163444642416918108994411350,1\r\n1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850,0\r\n1.2.826.0.1.3680043.8.498.20411068889491114730824265411248922690,1\r\n1.2.826.0.1.3680043.8.498.20427402498958253193571475926580142499,4\r\n1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372,4\r\n1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007,1\r\n1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886,0\r\n1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465,0\r\n1.2.826.0.1.3680043.8.498.20522330861615570894210641293344468905,3\r\n1.2.826.0.1.3680043.8.498.20587514285427671071934631922521946816,4\r\n1.2.826.0.1.3680043.8.498.20600318722531772866761132262772105640,2\r\n1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785,0\r\n1.2.826.0.1.3680043.8.498.20690227395174139480327772367312879333,2\r\n1.2.826.0.1.3680043.8.498.20733151481921261415295204209910573639,3\r\n1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784,0\r\n1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413,1\r\n1.2.826.0.1.3680043.8.498.20850785960533583051456342090773230767,5\r\n1.2.826.0.1.3680043.8.498.20852351697358589606507109300359572624,1\r\n1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537,0\r\n1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270,0\r\n1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254,0\r\n1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430,3\r\n1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615,4\r\n1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954,0\r\n1.2.826.0.1.3680043.8.498.21024377993947321291280779867493407199,4\r\n1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101,0\r\n1.2.826.0.1.3680043.8.498.21116046804798247223618072736212944615,1\r\n1.2.826.0.1.3680043.8.498.21135201439483157418374033420156525658,3\r\n1.2.826.0.1.3680043.8.498.21142867813782396036287673290033987859,0\r\n1.2.826.0.1.3680043.8.498.21239630702461934662855021164119114203,5\r\n1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849,5\r\n1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454,1\r\n1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596,3\r\n1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283,2\r\n1.2.826.0.1.3680043.8.498.21403015320378798778538823075316902483,2\r\n1.2.826.0.1.3680043.8.498.21413285047211074156709106352750752275,1\r\n1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958,0\r\n1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871,1\r\n1.2.826.0.1.3680043.8.498.21520719119040558609068952147312604007,1\r\n1.2.826.0.1.3680043.8.498.21556073772857867149576001337847113661,3\r\n1.2.826.0.1.3680043.8.498.21558570608416990974711670673431746511,1\r\n1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030,5\r\n1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044,0\r\n1.2.826.0.1.3680043.8.498.21606196343708804451390109412118036330,3\r\n1.2.826.0.1.3680043.8.498.21634792491972658759276025394685307053,2\r\n1.2.826.0.1.3680043.8.498.21636337451527049130190928852193267826,1\r\n1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137,2\r\n1.2.826.0.1.3680043.8.498.21769352980698937182060933429354012104,4\r\n1.2.826.0.1.3680043.8.498.21798342483930597563659425189400765869,4\r\n1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437,0\r\n1.2.826.0.1.3680043.8.498.21868734450232158209414868901539475548,2\r\n1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673,0\r\n1.2.826.0.1.3680043.8.498.21904421326144238681996713485932410596,4\r\n1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910,4\r\n1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762,0\r\n1.2.826.0.1.3680043.8.498.21991252201684723900045301917670383228,4\r\n1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295,0\r\n1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537,4\r\n1.2.826.0.1.3680043.8.498.22027914323057444746912555313058666950,4\r\n1.2.826.0.1.3680043.8.498.22053727963380087706894183799325067544,1\r\n1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805,2\r\n1.2.826.0.1.3680043.8.498.22088734384869383587217556574382319654,1\r\n1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965,0\r\n1.2.826.0.1.3680043.8.498.22126824950877890715014783370071292997,1\r\n1.2.826.0.1.3680043.8.498.22135282054716841452811687331325271982,3\r\n1.2.826.0.1.3680043.8.498.22137222939487357840166329770642627189,3\r\n1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844,2\r\n1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129,0\r\n1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643,1\r\n1.2.826.0.1.3680043.8.498.22338400095583099753286935111647807038,3\r\n1.2.826.0.1.3680043.8.498.22357754074308826479280023393343731000,1\r\n1.2.826.0.1.3680043.8.498.22394208178597637441149549895512290013,0\r\n1.2.826.0.1.3680043.8.498.22398041676780235097751294563297953841,4\r\n1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005,1\r\n1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645,2\r\n1.2.826.0.1.3680043.8.498.22426390571898968732573184721547490379,5\r\n1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060,2\r\n1.2.826.0.1.3680043.8.498.22480218016816209183384996714060953859,2\r\n1.2.826.0.1.3680043.8.498.22508124641661887430614105328316053644,0\r\n1.2.826.0.1.3680043.8.498.22583985355672251619550173090948161714,3\r\n1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128,4\r\n1.2.826.0.1.3680043.8.498.22594742481654038247097269358877852869,3\r\n1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046,3\r\n1.2.826.0.1.3680043.8.498.22617494262793136174349268379191631799,5\r\n1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020,1\r\n1.2.826.0.1.3680043.8.498.22656222254273662937226093008714426490,4\r\n1.2.826.0.1.3680043.8.498.22678957404909201599340396039237525550,2\r\n1.2.826.0.1.3680043.8.498.22684505799921376181913295621462268931,2\r\n1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423,0\r\n1.2.826.0.1.3680043.8.498.22867776500261605752195206933939172136,3\r\n1.2.826.0.1.3680043.8.498.22886251629508830700774506697670139848,1\r\n1.2.826.0.1.3680043.8.498.22938598188283919466040132624303635817,1\r\n1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053,1\r\n1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170,0\r\n1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449,0\r\n1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134,0\r\n1.2.826.0.1.3680043.8.498.23063675880057833741455102729816364091,5\r\n1.2.826.0.1.3680043.8.498.23065868481290440008286663193810913414,2\r\n1.2.826.0.1.3680043.8.498.23073694267101759922738121732233014980,2\r\n1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416,4\r\n1.2.826.0.1.3680043.8.498.23094733099955217022059584418468076048,3\r\n1.2.826.0.1.3680043.8.498.23107903961514228607587963030481481744,4\r\n1.2.826.0.1.3680043.8.498.23166300530486600338569917595744444657,2\r\n1.2.826.0.1.3680043.8.498.23182005205627494484490011351673355568,1\r\n1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738,0\r\n1.2.826.0.1.3680043.8.498.23394601328148358087542766577727876635,3\r\n1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932,3\r\n1.2.826.0.1.3680043.8.498.23422918444346962998731376196027420200,4\r\n1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288,1\r\n1.2.826.0.1.3680043.8.498.23518857399095666263158269685495375084,2\r\n1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254,4\r\n1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663,0\r\n1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899,3\r\n1.2.826.0.1.3680043.8.498.23703817737293327679300330167234268929,3\r\n1.2.826.0.1.3680043.8.498.23713320796530316370039202981138376061,1\r\n1.2.826.0.1.3680043.8.498.23719982492392272262684855675016475265,2\r\n1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088,0\r\n1.2.826.0.1.3680043.8.498.23757007145576852496599816072431075057,2\r\n1.2.826.0.1.3680043.8.498.23822313132245448173452915228147497501,2\r\n1.2.826.0.1.3680043.8.498.23833933822461809237193998300076847210,2\r\n1.2.826.0.1.3680043.8.498.23849672490426589132734390202638283865,2\r\n1.2.826.0.1.3680043.8.498.23911658810234305013984976739924353704,4\r\n1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277,3\r\n1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366,0\r\n1.2.826.0.1.3680043.8.498.23986051274386630509877205174270047762,4\r\n1.2.826.0.1.3680043.8.498.24012076387083114283116476459425015413,1\r\n1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530,3\r\n1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916,0\r\n1.2.826.0.1.3680043.8.498.24126422042626612740219325272793737656,2\r\n1.2.826.0.1.3680043.8.498.24209643225939781032698894992032106781,3\r\n1.2.826.0.1.3680043.8.498.24219103446788192052503003052696523200,0\r\n1.2.826.0.1.3680043.8.498.24220732394554303343522053195582322788,1\r\n1.2.826.0.1.3680043.8.498.24267153760943721146187182369810590631,4\r\n1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405,2\r\n1.2.826.0.1.3680043.8.498.24291465113360632954006568021942978765,1\r\n1.2.826.0.1.3680043.8.498.24302889622350917085868032747474929504,4\r\n1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557,1\r\n1.2.826.0.1.3680043.8.498.24369332907731805266450307536793220148,2\r\n1.2.826.0.1.3680043.8.498.24432243084670627958434695475653885175,3\r\n1.2.826.0.1.3680043.8.498.24478195415117310702720951368429189290,4\r\n1.2.826.0.1.3680043.8.498.24502332819381293694020055836255028614,2\r\n1.2.826.0.1.3680043.8.498.24529300482982370922479679056174585978,1\r\n1.2.826.0.1.3680043.8.498.24555528844026121007453886375651403314,5\r\n1.2.826.0.1.3680043.8.498.24563801799946310644914912532825629118,2\r\n1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503,2\r\n1.2.826.0.1.3680043.8.498.24635589616467409099788838058892886889,2\r\n1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263,5\r\n1.2.826.0.1.3680043.8.498.24679905857546029995247839380830345841,1\r\n1.2.826.0.1.3680043.8.498.24680736086205072406257935876858402593,1\r\n1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910,0\r\n1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038,0\r\n1.2.826.0.1.3680043.8.498.24808246532191535554375304081795021306,1\r\n1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684,0\r\n1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720,5\r\n1.2.826.0.1.3680043.8.498.24872746733270791851241663716306993824,0\r\n1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187,0\r\n1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983,0\r\n1.2.826.0.1.3680043.8.498.25006652485159423287561656594171884310,2\r\n1.2.826.0.1.3680043.8.498.25010661650105533302140829168731585935,2\r\n1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674,0\r\n1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861,2\r\n1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189,0\r\n1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047,1\r\n1.2.826.0.1.3680043.8.498.25156207169523051276979595262558652072,1\r\n1.2.826.0.1.3680043.8.498.25167060436407065195851060078175404789,4\r\n1.2.826.0.1.3680043.8.498.25188196199869378521949969245728443004,5\r\n1.2.826.0.1.3680043.8.498.25196393144875332073285250936506555534,5\r\n1.2.826.0.1.3680043.8.498.25206096958520407302630747466352252249,2\r\n1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724,0\r\n1.2.826.0.1.3680043.8.498.25259588756496584739814764896903786709,4\r\n1.2.826.0.1.3680043.8.498.25266231062774974303723359806535863810,4\r\n1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319,4\r\n1.2.826.0.1.3680043.8.498.25359770839263995354733835365510047109,5\r\n1.2.826.0.1.3680043.8.498.25394815405242666778213609728680620077,4\r\n1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670,0\r\n1.2.826.0.1.3680043.8.498.25440342930186170531037102906687829898,4\r\n1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891,3\r\n1.2.826.0.1.3680043.8.498.25487225187305802150215261512092051199,3\r\n1.2.826.0.1.3680043.8.498.25505153402978560400776672230220616863,1\r\n1.2.826.0.1.3680043.8.498.25512240159357553134500245590935010104,4\r\n1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882,0\r\n1.2.826.0.1.3680043.8.498.25550871519416839807700061317524004550,4\r\n1.2.826.0.1.3680043.8.498.25558526735614916746398919603493693652,4\r\n1.2.826.0.1.3680043.8.498.25604547074218147115077644853162295379,2\r\n1.2.826.0.1.3680043.8.498.25616615589791001678067099907823838797,1\r\n1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505,0\r\n1.2.826.0.1.3680043.8.498.25643897330283020467388814112594686690,4\r\n1.2.826.0.1.3680043.8.498.25650384476012633612528314070894186451,5\r\n1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567,0\r\n1.2.826.0.1.3680043.8.498.25663827732039068221184137540713755716,5\r\n1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487,2\r\n1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629,0\r\n1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103,0\r\n1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096,3\r\n1.2.826.0.1.3680043.8.498.25766387275525303860953321689718246405,1\r\n1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878,3\r\n1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260,1\r\n1.2.826.0.1.3680043.8.498.25815857653380672532983223088817283976,3\r\n1.2.826.0.1.3680043.8.498.25818541516137783435601858274495597736,1\r\n1.2.826.0.1.3680043.8.498.25856396695552621816673203639618612020,2\r\n1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502,0\r\n1.2.826.0.1.3680043.8.498.25904473139226718487560978103077541725,2\r\n1.2.826.0.1.3680043.8.498.25909078145370189195643713208718315003,4\r\n1.2.826.0.1.3680043.8.498.25928779639851562853500186715690156268,4\r\n1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036,0\r\n1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227,0\r\n1.2.826.0.1.3680043.8.498.26002972602557466222471260313257965551,3\r\n1.2.826.0.1.3680043.8.498.26004572902429209855865822977387283547,2\r\n1.2.826.0.1.3680043.8.498.26006372873438224908989402529530833528,2\r\n1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604,3\r\n1.2.826.0.1.3680043.8.498.26066524302907849854339129271283426208,3\r\n1.2.826.0.1.3680043.8.498.26072622476167889001255150030772906081,2\r\n1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568,0\r\n1.2.826.0.1.3680043.8.498.26201626427712146573573952548179938304,2\r\n1.2.826.0.1.3680043.8.498.26257425287121280175418060407044075705,4\r\n1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091,0\r\n1.2.826.0.1.3680043.8.498.26325951519908477683595799515094966069,4\r\n1.2.826.0.1.3680043.8.498.26333849754345698117396119042432557445,3\r\n1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387,5\r\n1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730,5\r\n1.2.826.0.1.3680043.8.498.26408081008482310696738068933718440200,4\r\n1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669,4\r\n1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691,0\r\n1.2.826.0.1.3680043.8.498.26485916396240515006042297316506505324,3\r\n1.2.826.0.1.3680043.8.498.26512333939158074470511885252122191522,1\r\n1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897,3\r\n1.2.826.0.1.3680043.8.498.26542435427465415008860869796504112727,3\r\n1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957,0\r\n1.2.826.0.1.3680043.8.498.26651323353208729423825552073457972796,1\r\n1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112,2\r\n1.2.826.0.1.3680043.8.498.26725172078893286313351418435377708028,1\r\n1.2.826.0.1.3680043.8.498.26792665054823243997838322477587733719,1\r\n1.2.826.0.1.3680043.8.498.26842258746390750314059176647398369242,3\r\n1.2.826.0.1.3680043.8.498.26843088620128166809417361066548447201,5\r\n1.2.826.0.1.3680043.8.498.26848412380762311150802301982233072694,0\r\n1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512,2\r\n1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601,1\r\n1.2.826.0.1.3680043.8.498.26946980494141305258556994153410856428,2\r\n1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004,4\r\n1.2.826.0.1.3680043.8.498.26989584479346775218799121753035575730,4\r\n1.2.826.0.1.3680043.8.498.27020884753639915245324093232218617967,3\r\n1.2.826.0.1.3680043.8.498.27023892731021470476500135946317920771,3\r\n1.2.826.0.1.3680043.8.498.27025989808002468335294585261291982661,4\r\n1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194,0\r\n1.2.826.0.1.3680043.8.498.27059975738703546675398620281849285863,0\r\n1.2.826.0.1.3680043.8.498.27157074709305277871045901905120405881,3\r\n1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246,1\r\n1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858,0\r\n1.2.826.0.1.3680043.8.498.27186001621007168581611489439038285508,1\r\n1.2.826.0.1.3680043.8.498.27191574009501966309382307682020135126,5\r\n1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620,1\r\n1.2.826.0.1.3680043.8.498.27197413293500684424735285039923168926,1\r\n1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285,4\r\n1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545,0\r\n1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018,0\r\n1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355,0\r\n1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211,3\r\n1.2.826.0.1.3680043.8.498.27324284859290040148888098857880927353,2\r\n1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486,3\r\n1.2.826.0.1.3680043.8.498.27398034409980350921804398626171135707,2\r\n1.2.826.0.1.3680043.8.498.27422742748182689233651649793019570511,4\r\n1.2.826.0.1.3680043.8.498.27456622705361831116491505615162380604,1\r\n1.2.826.0.1.3680043.8.498.27460982183399364904497832933739191826,4\r\n1.2.826.0.1.3680043.8.498.27474440682441854303485403899867525864,2\r\n1.2.826.0.1.3680043.8.498.27593710915588175306316595263625481601,1\r\n1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063,0\r\n1.2.826.0.1.3680043.8.498.27644321991318957631114727895883331622,1\r\n1.2.826.0.1.3680043.8.498.27653320597564308144389724321115208375,2\r\n1.2.826.0.1.3680043.8.498.27655410895168998898506695863000928400,5\r\n1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780,4\r\n1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482,3\r\n1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924,2\r\n1.2.826.0.1.3680043.8.498.27710718386318170070605046744941324421,2\r\n1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721,3\r\n1.2.826.0.1.3680043.8.498.27723573213501259053792522608339812759,4\r\n1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351,0\r\n1.2.826.0.1.3680043.8.498.27793455627426963285485957211105588905,3\r\n1.2.826.0.1.3680043.8.498.27796604203156439069742212039368371002,4\r\n1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316,2\r\n1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935,3\r\n1.2.826.0.1.3680043.8.498.27921111050680927418542588425335218896,4\r\n1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449,0\r\n1.2.826.0.1.3680043.8.498.27934397251575987048967084987994202973,1\r\n1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227,1\r\n1.2.826.0.1.3680043.8.498.27990965288663885936880638302158503894,3\r\n1.2.826.0.1.3680043.8.498.28011057282528460306257420499560926354,5\r\n1.2.826.0.1.3680043.8.498.28023795363827851445172311810918021563,4\r\n1.2.826.0.1.3680043.8.498.28028890493668487238845659163252231337,2\r\n1.2.826.0.1.3680043.8.498.28040645186226259590182363324146810031,4\r\n1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325,1\r\n1.2.826.0.1.3680043.8.498.28132506470634542229651932141193521661,3\r\n1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241,4\r\n1.2.826.0.1.3680043.8.498.28153286315438977482271934270979508349,3\r\n1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475,0\r\n1.2.826.0.1.3680043.8.498.28220626968612492259133318770141901567,2\r\n1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795,0\r\n1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882,0\r\n1.2.826.0.1.3680043.8.498.28289838818003822677994912735532050038,2\r\n1.2.826.0.1.3680043.8.498.28295265939946677460838974253299313798,1\r\n1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028,0\r\n1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306,3\r\n1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525,4\r\n1.2.826.0.1.3680043.8.498.28360174615866226538417948042981668998,4\r\n1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098,1\r\n1.2.826.0.1.3680043.8.498.28481442608487320644829189127206806471,5\r\n1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948,2\r\n1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737,3\r\n1.2.826.0.1.3680043.8.498.28539130184243787339297053654373783542,2\r\n1.2.826.0.1.3680043.8.498.28579372595938810820078670002733005121,3\r\n1.2.826.0.1.3680043.8.498.28595729988867300278425411189173775382,2\r\n1.2.826.0.1.3680043.8.498.28626044336302232656987963623687293705,1\r\n1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156,3\r\n1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719,0\r\n1.2.826.0.1.3680043.8.498.28638490997780173004029312240334483157,3\r\n1.2.826.0.1.3680043.8.498.28646304373057947277669139986982208473,4\r\n1.2.826.0.1.3680043.8.498.28655385212214986812679045835025619123,2\r\n1.2.826.0.1.3680043.8.498.28670999234453413675502551916979760344,3\r\n1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687,2\r\n1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456,0\r\n1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607,1\r\n1.2.826.0.1.3680043.8.498.28770594655815108010532953376271504596,4\r\n1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154,1\r\n1.2.826.0.1.3680043.8.498.28848511016500781798837621261660844237,2\r\n1.2.826.0.1.3680043.8.498.28873180831637763124652074775631752918,3\r\n1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256,0\r\n1.2.826.0.1.3680043.8.498.28944156754203202831026290463520445489,4\r\n1.2.826.0.1.3680043.8.498.28946722216615442736212359468410891872,3\r\n1.2.826.0.1.3680043.8.498.28951331578236330522646806432827370713,3\r\n1.2.826.0.1.3680043.8.498.28954361146470748868500417103935640742,4\r\n1.2.826.0.1.3680043.8.498.29039872266866623711060910233238440882,2\r\n1.2.826.0.1.3680043.8.498.29055892482211392922525222851162308510,3\r\n1.2.826.0.1.3680043.8.498.29090158364700006304493717187346344597,1\r\n1.2.826.0.1.3680043.8.498.29227028592609694700827329867343028944,4\r\n1.2.826.0.1.3680043.8.498.29280448517432444306374036429824425668,3\r\n1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181,0\r\n1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586,0\r\n1.2.826.0.1.3680043.8.498.29343735048224321966824641432873034291,0\r\n1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665,0\r\n1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001,3\r\n1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595,5\r\n1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496,1\r\n1.2.826.0.1.3680043.8.498.29384016102624554484300773136383156153,1\r\n1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370,3\r\n1.2.826.0.1.3680043.8.498.29443692810539372078922146461173545451,3\r\n1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878,5\r\n1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464,0\r\n1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856,5\r\n1.2.826.0.1.3680043.8.498.29735852958813434703969490187047665950,3\r\n1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569,0\r\n1.2.826.0.1.3680043.8.498.29741673719687535294395668150749894623,1\r\n1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054,1\r\n1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350,0\r\n1.2.826.0.1.3680043.8.498.29826129803278661744835136993610646069,2\r\n1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049,5\r\n1.2.826.0.1.3680043.8.498.29943089198208468441240036404506927883,1\r\n1.2.826.0.1.3680043.8.498.29971686675947075024168289004002445393,3\r\n1.2.826.0.1.3680043.8.498.29976611818313906970687136815346573471,4\r\n1.2.826.0.1.3680043.8.498.30058585660043363937345050478072453854,4\r\n1.2.826.0.1.3680043.8.498.30066410959464367576375502522994033142,3\r\n1.2.826.0.1.3680043.8.498.30104491808697888642424160917715722238,4\r\n1.2.826.0.1.3680043.8.498.30131339052241668380936148362472245259,4\r\n1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922,4\r\n1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073,4\r\n1.2.826.0.1.3680043.8.498.30174064884443577177838026250637902388,2\r\n1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934,4\r\n1.2.826.0.1.3680043.8.498.30222189977108980260768952691472020722,3\r\n1.2.826.0.1.3680043.8.498.30233578499383083757206211752001832047,5\r\n1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600,2\r\n1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057,0\r\n1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098,0\r\n1.2.826.0.1.3680043.8.498.30282025087399390929963075109487390046,1\r\n1.2.826.0.1.3680043.8.498.30330001893437233719772129404417384472,3\r\n1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540,4\r\n1.2.826.0.1.3680043.8.498.30420245540489522189154634720778032533,1\r\n1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762,0\r\n1.2.826.0.1.3680043.8.498.30435446905709054197096833427672604260,1\r\n1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989,2\r\n1.2.826.0.1.3680043.8.498.30517381653283520211188050362394799510,2\r\n1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451,4\r\n1.2.826.0.1.3680043.8.498.30537361712821464927469618193893850465,2\r\n1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622,4\r\n1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612,0\r\n1.2.826.0.1.3680043.8.498.30662407196303880392459287553277761223,4\r\n1.2.826.0.1.3680043.8.498.30672390215843894382866701432377109444,1\r\n1.2.826.0.1.3680043.8.498.30673411238679998084995204090753297949,3\r\n1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177,0\r\n1.2.826.0.1.3680043.8.498.30746167840159843733412534591988948270,3\r\n1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520,2\r\n1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579,1\r\n1.2.826.0.1.3680043.8.498.30800677497901063887487934675082066316,1\r\n1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374,3\r\n1.2.826.0.1.3680043.8.498.30836611607735372299752226690388090391,4\r\n1.2.826.0.1.3680043.8.498.30838586189496669675465778955893367497,2\r\n1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070,0\r\n1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425,0\r\n1.2.826.0.1.3680043.8.498.30942874102560162331831960649006411381,2\r\n1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066,3\r\n1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961,0\r\n1.2.826.0.1.3680043.8.498.30998941785989496591079894962766360976,3\r\n1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961,0\r\n1.2.826.0.1.3680043.8.498.31080452089421812812019413302222437434,1\r\n1.2.826.0.1.3680043.8.498.31082592133742929889611857821843859701,4\r\n1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271,4\r\n1.2.826.0.1.3680043.8.498.31137162827930164367134811210741362416,3\r\n1.2.826.0.1.3680043.8.498.31145453746697192222724674918300432873,0\r\n1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793,0\r\n1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669,0\r\n1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777,3\r\n1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716,0\r\n1.2.826.0.1.3680043.8.498.31218039240665208717816408042644491715,2\r\n1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090,0\r\n1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119,0\r\n1.2.826.0.1.3680043.8.498.31280487369922340170695453174997912050,1\r\n1.2.826.0.1.3680043.8.498.31282366761674650660824706600044115805,5\r\n1.2.826.0.1.3680043.8.498.31284572943302383501947308874558551301,2\r\n1.2.826.0.1.3680043.8.498.31352187408311685267245948214616940728,1\r\n1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486,2\r\n1.2.826.0.1.3680043.8.498.31421989235441217674752676866539112529,3\r\n1.2.826.0.1.3680043.8.498.31461240229123701082444627629841504099,1\r\n1.2.826.0.1.3680043.8.498.31482725509752510642857072190147133461,3\r\n1.2.826.0.1.3680043.8.498.31555333355990829772911515362502000209,5\r\n1.2.826.0.1.3680043.8.498.31565705236407722278921437231164204914,3\r\n1.2.826.0.1.3680043.8.498.31610133883321894510649078615659708431,2\r\n1.2.826.0.1.3680043.8.498.31640159014711750808214354395404624815,4\r\n1.2.826.0.1.3680043.8.498.31745644522310786426860265012373331979,2\r\n1.2.826.0.1.3680043.8.498.31790457319528233269194462675995569260,4\r\n1.2.826.0.1.3680043.8.498.31792234576444017394182974951507838927,1\r\n1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721,3\r\n1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882,3\r\n1.2.826.0.1.3680043.8.498.31842732064173292828619477783097732181,3\r\n1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250,4\r\n1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100,4\r\n1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296,0\r\n1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024,5\r\n1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778,3\r\n1.2.826.0.1.3680043.8.498.32078592787308498852570411454111970699,2\r\n1.2.826.0.1.3680043.8.498.32109377199577933060124357239881959602,1\r\n1.2.826.0.1.3680043.8.498.32138869097959653224534642965199855627,3\r\n1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816,2\r\n1.2.826.0.1.3680043.8.498.32179217289301203414651193280580594771,2\r\n1.2.826.0.1.3680043.8.498.32226253011052584155674401943276961907,2\r\n1.2.826.0.1.3680043.8.498.32241219263926672894433308808653719495,2\r\n1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795,5\r\n1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631,4\r\n1.2.826.0.1.3680043.8.498.32328699307307661181881949763360098439,4\r\n1.2.826.0.1.3680043.8.498.32329374180424300100441619205766536205,2\r\n1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461,0\r\n1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654,0\r\n1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007,2\r\n1.2.826.0.1.3680043.8.498.32488933488399304214586978746578986823,0\r\n1.2.826.0.1.3680043.8.498.32501049887140121289143479773995481982,4\r\n1.2.826.0.1.3680043.8.498.32504785212530110880087745488871181111,4\r\n1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399,1\r\n1.2.826.0.1.3680043.8.498.32521102053110469817073126425955562934,4\r\n1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933,2\r\n1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664,0\r\n1.2.826.0.1.3680043.8.498.32611545463217490575457831874695918229,4\r\n1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461,0\r\n1.2.826.0.1.3680043.8.498.32680606387445562892045224227371304764,2\r\n1.2.826.0.1.3680043.8.498.32704159300269838731722725240928883385,3\r\n1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677,0\r\n1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924,0\r\n1.2.826.0.1.3680043.8.498.32784816158214124982843838883515640569,4\r\n1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358,0\r\n1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753,3\r\n1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133,0\r\n1.2.826.0.1.3680043.8.498.32916544979513354644075485240094539251,2\r\n1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569,0\r\n1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042,0\r\n1.2.826.0.1.3680043.8.498.33085434860684110795966817803985794300,2\r\n1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534,3\r\n1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071,3\r\n1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807,0\r\n1.2.826.0.1.3680043.8.498.33273874534046658504935877940537027702,4\r\n1.2.826.0.1.3680043.8.498.33284228284209148349668722931307588402,3\r\n1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510,2\r\n1.2.826.0.1.3680043.8.498.33324670630108603340869986205298694836,3\r\n1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443,0\r\n1.2.826.0.1.3680043.8.498.33387430823462011959198830568803117633,2\r\n1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086,5\r\n1.2.826.0.1.3680043.8.498.33419086668421177906530808458004386826,1\r\n1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684,5\r\n1.2.826.0.1.3680043.8.498.33432461096351771607862098991677662209,1\r\n1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521,4\r\n1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550,0\r\n1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314,3\r\n1.2.826.0.1.3680043.8.498.33620268340279901383196638228030589893,5\r\n1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941,1\r\n1.2.826.0.1.3680043.8.498.33707148112045883958228656491167647755,2\r\n1.2.826.0.1.3680043.8.498.33709759248055631658322056320849494088,4\r\n1.2.826.0.1.3680043.8.498.33734536545775442043827840341408174347,3\r\n1.2.826.0.1.3680043.8.498.33834227140201599591317512187502720859,3\r\n1.2.826.0.1.3680043.8.498.33841907095190352493353075864604147687,0\r\n1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598,0\r\n1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650,0\r\n1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830,2\r\n1.2.826.0.1.3680043.8.498.33947148016115721563601115463364787679,4\r\n1.2.826.0.1.3680043.8.498.33957165685646407808982253731894148028,5\r\n1.2.826.0.1.3680043.8.498.33990091057243328830745904178686745167,1\r\n1.2.826.0.1.3680043.8.498.33997019840409023169542161093833920630,2\r\n1.2.826.0.1.3680043.8.498.34002185870196144582348922158474418674,1\r\n1.2.826.0.1.3680043.8.498.34040631818813472592501398076075999835,1\r\n1.2.826.0.1.3680043.8.498.34058295828142876023659425586740832405,3\r\n1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999,1\r\n1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514,0\r\n1.2.826.0.1.3680043.8.498.34109091435857467600499949506940613037,3\r\n1.2.826.0.1.3680043.8.498.34126843286927697018624309801437341727,1\r\n1.2.826.0.1.3680043.8.498.34206043272866138622475050809785944941,1\r\n1.2.826.0.1.3680043.8.498.34262242337778924864334788374037992215,1\r\n1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745,0\r\n1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984,1\r\n1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613,0\r\n1.2.826.0.1.3680043.8.498.34404554752241524210538769325381023947,3\r\n1.2.826.0.1.3680043.8.498.34433343440016591348979084714300062954,4\r\n1.2.826.0.1.3680043.8.498.34441662343660776323013762699640240389,4\r\n1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465,2\r\n1.2.826.0.1.3680043.8.498.34501976934829436985900782482912019868,3\r\n1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677,5\r\n1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444,0\r\n1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542,3\r\n1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906,0\r\n1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328,4\r\n1.2.826.0.1.3680043.8.498.34685905030370793639196564723935583035,1\r\n1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455,0\r\n1.2.826.0.1.3680043.8.498.34717974828302316666446325243570202245,3\r\n1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599,2\r\n1.2.826.0.1.3680043.8.498.34753112960188810416088070814158959510,4\r\n1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853,4\r\n1.2.826.0.1.3680043.8.498.34795373994453585401035013232069587413,4\r\n1.2.826.0.1.3680043.8.498.34826100703045042103752460066224521343,5\r\n1.2.826.0.1.3680043.8.498.34858782760614383566891001610985452165,4\r\n1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451,0\r\n1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579,5\r\n1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597,1\r\n1.2.826.0.1.3680043.8.498.35017973257585430544005763350601015208,2\r\n1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521,2\r\n1.2.826.0.1.3680043.8.498.35027750475162395113025853006474122989,1\r\n1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139,0\r\n1.2.826.0.1.3680043.8.498.35090276406828326770336478753776261776,2\r\n1.2.826.0.1.3680043.8.498.35121262802959257590413582180924083425,4\r\n1.2.826.0.1.3680043.8.498.35127678462764233938878060108411012978,3\r\n1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761,3\r\n1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841,0\r\n1.2.826.0.1.3680043.8.498.35257158249961136373696255071228322165,1\r\n1.2.826.0.1.3680043.8.498.35273590880523815297008790288567354883,1\r\n1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909,4\r\n1.2.826.0.1.3680043.8.498.35338198410583984337908932327787063577,4\r\n1.2.826.0.1.3680043.8.498.35526385429946467546636621603749416429,2\r\n1.2.826.0.1.3680043.8.498.35542843700847404584397532402313561503,3\r\n1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990,2\r\n1.2.826.0.1.3680043.8.498.35572306796137927247415883445612561056,1\r\n1.2.826.0.1.3680043.8.498.35633121799529901420381085923067525212,3\r\n1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313,0\r\n1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211,1\r\n1.2.826.0.1.3680043.8.498.35677515667357567818845364400903668190,1\r\n1.2.826.0.1.3680043.8.498.35717294165699019392845830077937768611,4\r\n1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777,0\r\n1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681,2\r\n1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944,1\r\n1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720,0\r\n1.2.826.0.1.3680043.8.498.35826402289277331555736683320175912112,3\r\n1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403,0\r\n1.2.826.0.1.3680043.8.498.35836928928058091851993989259747528067,2\r\n1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201,1\r\n1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278,3\r\n1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314,0\r\n1.2.826.0.1.3680043.8.498.35906584224533966870202042352292878668,1\r\n1.2.826.0.1.3680043.8.498.35923146679164443588381195543533271939,4\r\n1.2.826.0.1.3680043.8.498.35923183633813281697521537744486885300,5\r\n1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734,5\r\n1.2.826.0.1.3680043.8.498.35932368810339127867306228699757809449,2\r\n1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384,2\r\n1.2.826.0.1.3680043.8.498.35961087303991442751747064990497975939,1\r\n1.2.826.0.1.3680043.8.498.35965770865730278787229040370963322662,3\r\n1.2.826.0.1.3680043.8.498.35971292331621735865925455612813197566,2\r\n1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006,1\r\n1.2.826.0.1.3680043.8.498.36013620117872095227314165026090752632,1\r\n1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394,0\r\n1.2.826.0.1.3680043.8.498.36083794505435625841225237415909011547,2\r\n1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565,5\r\n1.2.826.0.1.3680043.8.498.36131369236532190338276701919104598475,4\r\n1.2.826.0.1.3680043.8.498.36158956431673597472934120620368656246,4\r\n1.2.826.0.1.3680043.8.498.36180248571044206939894546783755557323,1\r\n1.2.826.0.1.3680043.8.498.36185754164795053628650116141620545139,3\r\n1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976,1\r\n1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278,3\r\n1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198,0\r\n1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202,2\r\n1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167,0\r\n1.2.826.0.1.3680043.8.498.36315811661139684771871696847516662339,2\r\n1.2.826.0.1.3680043.8.498.36431753977367847587395456605556703040,1\r\n1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300,4\r\n1.2.826.0.1.3680043.8.498.36488342368050295711098478501722217415,5\r\n1.2.826.0.1.3680043.8.498.36521162836904095186654423648490092035,5\r\n1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185,0\r\n1.2.826.0.1.3680043.8.498.36599566036331923007586730435849419272,2\r\n1.2.826.0.1.3680043.8.498.36645945703275956149107297016813148264,3\r\n1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533,1\r\n1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374,0\r\n1.2.826.0.1.3680043.8.498.36688959138536319644194458211515765203,2\r\n1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985,0\r\n1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185,4\r\n1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100,0\r\n1.2.826.0.1.3680043.8.498.36825274340853796392762195174336452919,1\r\n1.2.826.0.1.3680043.8.498.36846391753332367835804699576399553980,1\r\n1.2.826.0.1.3680043.8.498.36891467514776207809384804066253854193,4\r\n1.2.826.0.1.3680043.8.498.36914870739904705183135763739850515617,3\r\n1.2.826.0.1.3680043.8.498.36924091633099385100942228071545017565,1\r\n1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047,0\r\n1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991,2\r\n1.2.826.0.1.3680043.8.498.37005775889857426034942587344620888398,2\r\n1.2.826.0.1.3680043.8.498.37007302166102452703965686598524467188,5\r\n1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876,5\r\n1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868,1\r\n1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727,0\r\n1.2.826.0.1.3680043.8.498.37110727485959679337431199930441321268,4\r\n1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535,3\r\n1.2.826.0.1.3680043.8.498.37210322466522537523056852160944888029,2\r\n1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895,0\r\n1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316,0\r\n1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315,3\r\n1.2.826.0.1.3680043.8.498.37347132459817985668038603215490355489,4\r\n1.2.826.0.1.3680043.8.498.37371365833775442014007234936409739360,3\r\n1.2.826.0.1.3680043.8.498.37390494423735156928337187351900345080,4\r\n1.2.826.0.1.3680043.8.498.37413349933694832641902637847629969234,3\r\n1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338,0\r\n1.2.826.0.1.3680043.8.498.37432328401993499374675847888297416449,2\r\n1.2.826.0.1.3680043.8.498.37438631765770807892914198396852544668,1\r\n1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849,3\r\n1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908,0\r\n1.2.826.0.1.3680043.8.498.37485075271886447709158414670394650122,1\r\n1.2.826.0.1.3680043.8.498.37508036912473928281276902883575142611,5\r\n1.2.826.0.1.3680043.8.498.37510478969810784660124747242051166554,4\r\n1.2.826.0.1.3680043.8.498.37519350121906786941281770770693979271,4\r\n1.2.826.0.1.3680043.8.498.37558426555271635608942179071280432530,1\r\n1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023,5\r\n1.2.826.0.1.3680043.8.498.37580209294522316445963449455348371815,4\r\n1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903,1\r\n1.2.826.0.1.3680043.8.498.37591590536870291237852764995372956522,4\r\n1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597,3\r\n1.2.826.0.1.3680043.8.498.37644754953609976838298110418046213654,1\r\n1.2.826.0.1.3680043.8.498.37695832484726177391252504639155717084,4\r\n1.2.826.0.1.3680043.8.498.37713008080610827760634302982073393454,3\r\n1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255,4\r\n1.2.826.0.1.3680043.8.498.37782918081276484321055690005029111967,1\r\n1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680,1\r\n1.2.826.0.1.3680043.8.498.37838990693631869129590283596686992710,4\r\n1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606,2\r\n1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122,0\r\n1.2.826.0.1.3680043.8.498.37879265071349098151803248077034280617,3\r\n1.2.826.0.1.3680043.8.498.37885316830093262977685802809269466252,1\r\n1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335,2\r\n1.2.826.0.1.3680043.8.498.37936061471366103885185528435327650981,5\r\n1.2.826.0.1.3680043.8.498.37969790195791584440906553207150666923,4\r\n1.2.826.0.1.3680043.8.498.38005916106577944091708997377832932938,1\r\n1.2.826.0.1.3680043.8.498.38010459551726691601986492657296948013,4\r\n1.2.826.0.1.3680043.8.498.38013372055010421273425628837282557767,2\r\n1.2.826.0.1.3680043.8.498.38039631093067555454700266397073080430,2\r\n1.2.826.0.1.3680043.8.498.38056024157303576597111203925091929489,4\r\n1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221,3\r\n1.2.826.0.1.3680043.8.498.38085359363401170553191582038767900323,2\r\n1.2.826.0.1.3680043.8.498.38095098835657518803781210111463995296,2\r\n1.2.826.0.1.3680043.8.498.38104228771638294693272534500734878990,4\r\n1.2.826.0.1.3680043.8.498.38173249623713271734596884672770674290,2\r\n1.2.826.0.1.3680043.8.498.38224750236138877853888930317337541394,4\r\n1.2.826.0.1.3680043.8.498.38233845434092134418342290639986261345,3\r\n1.2.826.0.1.3680043.8.498.38238963443939714344730201261512911003,4\r\n1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169,5\r\n1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389,4\r\n1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262,1\r\n1.2.826.0.1.3680043.8.498.38306262614552497897681886368029860358,2\r\n1.2.826.0.1.3680043.8.498.38325494489236992123219618004392575979,1\r\n1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018,1\r\n1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552,0\r\n1.2.826.0.1.3680043.8.498.38365187857616428023213060377488046375,1\r\n1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003,1\r\n1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059,3\r\n1.2.826.0.1.3680043.8.498.38427680930662301114332563903957471851,5\r\n1.2.826.0.1.3680043.8.498.38457315909459574570206119213798163082,3\r\n1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359,5\r\n1.2.826.0.1.3680043.8.498.38518069950656326194016175319535174299,4\r\n1.2.826.0.1.3680043.8.498.38528511947818361078957998546327084024,3\r\n1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267,0\r\n1.2.826.0.1.3680043.8.498.38567342184301320850258918666303107813,1\r\n1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200,0\r\n1.2.826.0.1.3680043.8.498.38591915802992557518846862828618375715,2\r\n1.2.826.0.1.3680043.8.498.38593351637971248239815933638950449587,4\r\n1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451,3\r\n1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926,0\r\n1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736,0\r\n1.2.826.0.1.3680043.8.498.38701927697650476092773702704871872598,4\r\n1.2.826.0.1.3680043.8.498.38702677049080452580748506476598606517,2\r\n1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219,0\r\n1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604,2\r\n1.2.826.0.1.3680043.8.498.38774313460931748278644354956408578813,3\r\n1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893,2\r\n1.2.826.0.1.3680043.8.498.38796519786809225694393617946244012065,1\r\n1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820,0\r\n1.2.826.0.1.3680043.8.498.38862475907405766248096419059100131863,1\r\n1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060,4\r\n1.2.826.0.1.3680043.8.498.38893522232848525819919092148478879455,0\r\n1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405,4\r\n1.2.826.0.1.3680043.8.498.38900640030483599614088224967855884221,2\r\n1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373,0\r\n1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144,0\r\n1.2.826.0.1.3680043.8.498.38928323954764175259962816410233336718,1\r\n1.2.826.0.1.3680043.8.498.38935694477580522005505074220300062204,1\r\n1.2.826.0.1.3680043.8.498.38940868544480800333574786964232944756,1\r\n1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948,0\r\n1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799,3\r\n1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354,0\r\n1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161,0\r\n1.2.826.0.1.3680043.8.498.39025433784711757736168236829157583043,4\r\n1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427,0\r\n1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752,0\r\n1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053,4\r\n1.2.826.0.1.3680043.8.498.39135770176150551756142329948474103964,3\r\n1.2.826.0.1.3680043.8.498.39190090804401525573012570218955054016,2\r\n1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584,2\r\n1.2.826.0.1.3680043.8.498.39250844287482885227700969308333354206,5\r\n1.2.826.0.1.3680043.8.498.39264696726258404531295961801831089064,1\r\n1.2.826.0.1.3680043.8.498.39275838957651450909114440721012584183,2\r\n1.2.826.0.1.3680043.8.498.39324076659429494696721784912308233388,3\r\n1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709,2\r\n1.2.826.0.1.3680043.8.498.39394842976048577715164357752399606924,1\r\n1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764,0\r\n1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402,0\r\n1.2.826.0.1.3680043.8.498.39507184510584030713933330702725991027,2\r\n1.2.826.0.1.3680043.8.498.39547752108509751520306190293259313911,4\r\n1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912,0\r\n1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611,2\r\n1.2.826.0.1.3680043.8.498.39636073039676814228254715705296535819,3\r\n1.2.826.0.1.3680043.8.498.39642276506855996181597769215091644995,4\r\n1.2.826.0.1.3680043.8.498.39787654910948635229369478006027304578,1\r\n1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917,0\r\n1.2.826.0.1.3680043.8.498.39845300426582562781355154625767485381,1\r\n1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592,0\r\n1.2.826.0.1.3680043.8.498.39895705984068714346274689335576595443,5\r\n1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759,2\r\n1.2.826.0.1.3680043.8.498.39954567501513957418422043089587980092,4\r\n1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473,1\r\n1.2.826.0.1.3680043.8.498.39969071491383654382076256213867635377,4\r\n1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112,1\r\n1.2.826.0.1.3680043.8.498.40041164441539690764994039690598305675,1\r\n1.2.826.0.1.3680043.8.498.40086005762552976451418579375198597540,4\r\n1.2.826.0.1.3680043.8.498.40131899651518057045444739705301716085,1\r\n1.2.826.0.1.3680043.8.498.40154708689850542672289658249059660788,3\r\n1.2.826.0.1.3680043.8.498.40196271329411009311273016603705560443,3\r\n1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754,2\r\n1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693,0\r\n1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446,3\r\n1.2.826.0.1.3680043.8.498.40257126480600927758308319831518876543,5\r\n1.2.826.0.1.3680043.8.498.40269807401373034066229810558300266830,1\r\n1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230,1\r\n1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837,0\r\n1.2.826.0.1.3680043.8.498.40372643537313871452054321931253684619,4\r\n1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818,2\r\n1.2.826.0.1.3680043.8.498.40419759559020255481034237210540871312,1\r\n1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679,2\r\n1.2.826.0.1.3680043.8.498.40437305053062892226978469238653825745,4\r\n1.2.826.0.1.3680043.8.498.40441879276927383122480339472406105572,4\r\n1.2.826.0.1.3680043.8.498.40487147942341753806704765355921322161,3\r\n1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201,5\r\n1.2.826.0.1.3680043.8.498.40547250250566982489293057895661861722,4\r\n1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315,0\r\n1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037,0\r\n1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655,0\r\n1.2.826.0.1.3680043.8.498.40618283771317160262473004328692613267,1\r\n1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938,3\r\n1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872,0\r\n1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394,2\r\n1.2.826.0.1.3680043.8.498.40718200801895968234499223301900799376,2\r\n1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476,3\r\n1.2.826.0.1.3680043.8.498.40765829900885943055669918091033014654,5\r\n1.2.826.0.1.3680043.8.498.40823842819404259265489417470859248075,1\r\n1.2.826.0.1.3680043.8.498.40828786094337186697222449293183309391,2\r\n1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849,1\r\n1.2.826.0.1.3680043.8.498.40898220266193841898869905714214454387,5\r\n1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754,3\r\n1.2.826.0.1.3680043.8.498.40930045208346676628855922755791787949,5\r\n1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116,3\r\n1.2.826.0.1.3680043.8.498.40987687016832998820664054439678138878,1\r\n1.2.826.0.1.3680043.8.498.40994616246264625946610553907227975893,2\r\n1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994,0\r\n1.2.826.0.1.3680043.8.498.41057428195711190931490429649014776496,2\r\n1.2.826.0.1.3680043.8.498.41108421820714108307078614688022098388,5\r\n1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260,0\r\n1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212,0\r\n1.2.826.0.1.3680043.8.498.41197678541534079475667407055098243437,3\r\n1.2.826.0.1.3680043.8.498.41240503248178304281405515398211762506,2\r\n1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213,0\r\n1.2.826.0.1.3680043.8.498.41315359092354940082611381451934245995,3\r\n1.2.826.0.1.3680043.8.498.41326575785634778198143222323166756210,1\r\n1.2.826.0.1.3680043.8.498.41363979247025384202932112601795525615,2\r\n1.2.826.0.1.3680043.8.498.41366889271483886167284378157366992074,3\r\n1.2.826.0.1.3680043.8.498.41382780862853765983216810855531499640,1\r\n1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358,4\r\n1.2.826.0.1.3680043.8.498.41440206520769979754790425253631809212,4\r\n1.2.826.0.1.3680043.8.498.41454762422897656673710145732036115764,1\r\n1.2.826.0.1.3680043.8.498.41564229176277158115386256329157870722,2\r\n1.2.826.0.1.3680043.8.498.41587640208769641326416798167574152049,4\r\n1.2.826.0.1.3680043.8.498.41605033773891953796694219242501252567,1\r\n1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415,0\r\n1.2.826.0.1.3680043.8.498.41623512571866040054960002151226127775,5\r\n1.2.826.0.1.3680043.8.498.41640481150213029810772014344448781554,4\r\n1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414,0\r\n1.2.826.0.1.3680043.8.498.41662013720606493874198348776219875496,3\r\n1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545,1\r\n1.2.826.0.1.3680043.8.498.41696746845418651508748386392327996143,4\r\n1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778,3\r\n1.2.826.0.1.3680043.8.498.41717437729988689273718049162153247404,3\r\n1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601,4\r\n1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535,3\r\n1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537,0\r\n1.2.826.0.1.3680043.8.498.41810826648175757249770990269365542940,3\r\n1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683,0\r\n1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004,0\r\n1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476,0\r\n1.2.826.0.1.3680043.8.498.41856137734822068230231773222820851271,3\r\n1.2.826.0.1.3680043.8.498.41939824564209194493855981805593285528,1\r\n1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480,0\r\n1.2.826.0.1.3680043.8.498.42018410640565840140895012243840256810,1\r\n1.2.826.0.1.3680043.8.498.42018683052544325209168649309206539721,4\r\n1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808,2\r\n1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782,1\r\n1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725,4\r\n1.2.826.0.1.3680043.8.498.42120645248710860870761681999665860884,3\r\n1.2.826.0.1.3680043.8.498.42130173624033946465704266507948122088,1\r\n1.2.826.0.1.3680043.8.498.42162585695624963017992042571207363427,4\r\n1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406,3\r\n1.2.826.0.1.3680043.8.498.42225165085995728538373532053227662192,1\r\n1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947,4\r\n1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633,4\r\n1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275,1\r\n1.2.826.0.1.3680043.8.498.42414269167296543219685897036829241885,2\r\n1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936,3\r\n1.2.826.0.1.3680043.8.498.42486421970612099441197418307958561877,4\r\n1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876,0\r\n1.2.826.0.1.3680043.8.498.42566296562794128096734153388840524012,2\r\n1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276,2\r\n1.2.826.0.1.3680043.8.498.42587087495729026068405824960397052997,4\r\n1.2.826.0.1.3680043.8.498.42598518026540237824479232797503218019,0\r\n1.2.826.0.1.3680043.8.498.42609322631348207494943622999125822017,3\r\n1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006,1\r\n1.2.826.0.1.3680043.8.498.42719592405241698450757802131459694665,1\r\n1.2.826.0.1.3680043.8.498.42768603719459834345001093623865167866,3\r\n1.2.826.0.1.3680043.8.498.42776652100045349404152232788825670056,4\r\n1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394,1\r\n1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775,4\r\n1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214,0\r\n1.2.826.0.1.3680043.8.498.42847376403722341195083350288786052637,3\r\n1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339,3\r\n1.2.826.0.1.3680043.8.498.42896117106271721157113072562739356381,1\r\n1.2.826.0.1.3680043.8.498.42921593713351589642747633505679170597,1\r\n1.2.826.0.1.3680043.8.498.42942844751304929338846069579289911684,3\r\n1.2.826.0.1.3680043.8.498.42965031665568825650795822006244115609,2\r\n1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744,0\r\n1.2.826.0.1.3680043.8.498.42990542863216044717944418712887800295,5\r\n1.2.826.0.1.3680043.8.498.43014288535685961874615794778401152033,5\r\n1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887,0\r\n1.2.826.0.1.3680043.8.498.43065391907984821806263204343792008938,3\r\n1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741,0\r\n1.2.826.0.1.3680043.8.498.43127139928509033152155587499980767364,2\r\n1.2.826.0.1.3680043.8.498.43144544376687968624598661376472625623,2\r\n1.2.826.0.1.3680043.8.498.43256282639827216310036033378453939750,4\r\n1.2.826.0.1.3680043.8.498.43256905750043156063416845850825787417,2\r\n1.2.826.0.1.3680043.8.498.43300763182450509440662392516941842389,4\r\n1.2.826.0.1.3680043.8.498.43325226037168674077736336919956196684,1\r\n1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557,4\r\n1.2.826.0.1.3680043.8.498.43351215447181735764667371462573550267,2\r\n1.2.826.0.1.3680043.8.498.43355923025859051425546451466843844346,3\r\n1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838,0\r\n1.2.826.0.1.3680043.8.498.43432499267536761263202158662502957684,5\r\n1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343,0\r\n1.2.826.0.1.3680043.8.498.43442933855807685981061557274166869461,1\r\n1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490,3\r\n1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736,0\r\n1.2.826.0.1.3680043.8.498.43587191909319947256183198986671106743,3\r\n1.2.826.0.1.3680043.8.498.43587826365647713044374316837330397375,3\r\n1.2.826.0.1.3680043.8.498.43692339773096477132070133629776394739,4\r\n1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163,0\r\n1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822,0\r\n1.2.826.0.1.3680043.8.498.43797037142729611453626414920554246615,5\r\n1.2.826.0.1.3680043.8.498.43832771919542075229536000564653592465,2\r\n1.2.826.0.1.3680043.8.498.43870319732657730742794291443940225011,2\r\n1.2.826.0.1.3680043.8.498.43967752580489865323542041346927244370,1\r\n1.2.826.0.1.3680043.8.498.44002376543340544410495269060824194475,2\r\n1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052,0\r\n1.2.826.0.1.3680043.8.498.44149403030093858908988179411336839587,0\r\n1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208,0\r\n1.2.826.0.1.3680043.8.498.44259687212868598275713310246316207497,3\r\n1.2.826.0.1.3680043.8.498.44261256084247541720465769177956875383,3\r\n1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889,0\r\n1.2.826.0.1.3680043.8.498.44323131330689365651417881352766725366,4\r\n1.2.826.0.1.3680043.8.498.44333089398623004498721836325723633455,2\r\n1.2.826.0.1.3680043.8.498.44350638600163778456163342454087669278,3\r\n1.2.826.0.1.3680043.8.498.44365814642869065241166313231424697474,4\r\n1.2.826.0.1.3680043.8.498.44405554266141576380633014711246118209,3\r\n1.2.826.0.1.3680043.8.498.44417768181231991989438194369583907493,1\r\n1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727,1\r\n1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713,1\r\n1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569,0\r\n1.2.826.0.1.3680043.8.498.44525218699266472141419416162979441275,5\r\n1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428,2\r\n1.2.826.0.1.3680043.8.498.44654558140536370040701171884970036040,2\r\n1.2.826.0.1.3680043.8.498.44670636848627131733157841382853275817,3\r\n1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530,0\r\n1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852,1\r\n1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803,4\r\n1.2.826.0.1.3680043.8.498.44792591585917525576461101442761037507,4\r\n1.2.826.0.1.3680043.8.498.44863396093454506630473164990623458425,3\r\n1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791,2\r\n1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025,0\r\n1.2.826.0.1.3680043.8.498.44879493336213912007673875382158420811,4\r\n1.2.826.0.1.3680043.8.498.44902036121220687805715010480088379768,2\r\n1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010,0\r\n1.2.826.0.1.3680043.8.498.44996883917122941421301655137158093858,3\r\n1.2.826.0.1.3680043.8.498.45033326048521572717771708818514772861,4\r\n1.2.826.0.1.3680043.8.498.45059917085750212913588683242522799083,2\r\n1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032,2\r\n1.2.826.0.1.3680043.8.498.45082393649917309927267478072569185694,4\r\n1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634,1\r\n1.2.826.0.1.3680043.8.498.45170517567841608067443743961197123093,2\r\n1.2.826.0.1.3680043.8.498.45207812477654339877341538582090218293,5\r\n1.2.826.0.1.3680043.8.498.45251306032215692953561852960585282987,1\r\n1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632,0\r\n1.2.826.0.1.3680043.8.498.45286493874121093392171044034263851961,4\r\n1.2.826.0.1.3680043.8.498.45304570280714567513866386380356988876,1\r\n1.2.826.0.1.3680043.8.498.45316453179223822297459860764883030492,3\r\n1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949,3\r\n1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959,0\r\n1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641,0\r\n1.2.826.0.1.3680043.8.498.45368532788186860828867663606651818835,4\r\n1.2.826.0.1.3680043.8.498.45380312679924593883892686293576260616,2\r\n1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159,4\r\n1.2.826.0.1.3680043.8.498.45406212529450909512649737062964080190,2\r\n1.2.826.0.1.3680043.8.498.45454539422542821586939369440135728144,1\r\n1.2.826.0.1.3680043.8.498.45462389009834095091353507592651342877,2\r\n1.2.826.0.1.3680043.8.498.45463321406810606039579747001339703552,4\r\n1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775,0\r\n1.2.826.0.1.3680043.8.498.45504348071514081636909771148066956410,1\r\n1.2.826.0.1.3680043.8.498.45519502487291848581539950159353124045,3\r\n1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117,2\r\n1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642,4\r\n1.2.826.0.1.3680043.8.498.45644703214235636656820126625301033696,1\r\n1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674,0\r\n1.2.826.0.1.3680043.8.498.45755198489211010290512228906807170657,3\r\n1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121,1\r\n1.2.826.0.1.3680043.8.498.45811964949115542204306831813385709206,2\r\n1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426,1\r\n1.2.826.0.1.3680043.8.498.45865541491386404818767748310365919685,4\r\n1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549,0\r\n1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381,2\r\n1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614,3\r\n1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663,0\r\n1.2.826.0.1.3680043.8.498.46016776602328665980436110567854612595,4\r\n1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397,4\r\n1.2.826.0.1.3680043.8.498.46051119885630602872050329412008545893,4\r\n1.2.826.0.1.3680043.8.498.46052469424267861803032899866428547156,1\r\n1.2.826.0.1.3680043.8.498.46123761840464615814604226041496278995,3\r\n1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662,0\r\n1.2.826.0.1.3680043.8.498.46222691720198967161306520058359099856,4\r\n1.2.826.0.1.3680043.8.498.46228189366201925238527911905072482095,2\r\n1.2.826.0.1.3680043.8.498.46259494895672269475940484605821837768,4\r\n1.2.826.0.1.3680043.8.498.46268820642949526111807948525363811264,4\r\n1.2.826.0.1.3680043.8.498.46282234455466954264170065462242039568,0\r\n1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523,0\r\n1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469,1\r\n1.2.826.0.1.3680043.8.498.46309540159823359540686782651264853894,3\r\n1.2.826.0.1.3680043.8.498.46335849264221717416382765091672040855,3\r\n1.2.826.0.1.3680043.8.498.46407639440476738746130651250276012493,4\r\n1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766,0\r\n1.2.826.0.1.3680043.8.498.46452612704507986055186132736451572677,2\r\n1.2.826.0.1.3680043.8.498.46491051379327985199309365945879841064,2\r\n1.2.826.0.1.3680043.8.498.46506161284114284997220751358509505380,4\r\n1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265,2\r\n1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479,1\r\n1.2.826.0.1.3680043.8.498.46713978512945139851373191422321462143,0\r\n1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552,3\r\n1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241,1\r\n1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541,0\r\n1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639,0\r\n1.2.826.0.1.3680043.8.498.46797419946872257347118338518757027995,2\r\n1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949,0\r\n1.2.826.0.1.3680043.8.498.46842734152337184482152392783260161845,1\r\n1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657,3\r\n1.2.826.0.1.3680043.8.498.46893956584698923808118623450380169212,1\r\n1.2.826.0.1.3680043.8.498.46909729013880708074310736136355969956,4\r\n1.2.826.0.1.3680043.8.498.46925959240037679795735705370016021232,1\r\n1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431,0\r\n1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787,0\r\n1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029,3\r\n1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037,0\r\n1.2.826.0.1.3680043.8.498.47080537724499166054609508776778189902,4\r\n1.2.826.0.1.3680043.8.498.47094981922544354899183959858078311972,1\r\n1.2.826.0.1.3680043.8.498.47130872195336642107576758658144949813,4\r\n1.2.826.0.1.3680043.8.498.47239511009810375185594384849101241004,1\r\n1.2.826.0.1.3680043.8.498.47255564202539432049700324486197024855,1\r\n1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642,0\r\n1.2.826.0.1.3680043.8.498.47268599732083036147289040424392838620,3\r\n1.2.826.0.1.3680043.8.498.47336007784831556446570294227732688335,3\r\n1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955,4\r\n1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953,0\r\n1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684,0\r\n1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825,0\r\n1.2.826.0.1.3680043.8.498.47495385306095610608327984787779011133,1\r\n1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246,2\r\n1.2.826.0.1.3680043.8.498.47651715105501347255071941051553742767,4\r\n1.2.826.0.1.3680043.8.498.47655269404919497034805209368250028981,1\r\n1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400,5\r\n1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166,4\r\n1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710,0\r\n1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959,0\r\n1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174,4\r\n1.2.826.0.1.3680043.8.498.47855330524708374523226349383799712473,1\r\n1.2.826.0.1.3680043.8.498.47864423097056282241181943425451597067,2\r\n1.2.826.0.1.3680043.8.498.47870074144891988520855535250618228390,2\r\n1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552,0\r\n1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835,3\r\n1.2.826.0.1.3680043.8.498.48033467661219494489168041469309669510,4\r\n1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541,2\r\n1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331,0\r\n1.2.826.0.1.3680043.8.498.48160569779413970442472500116176876660,0\r\n1.2.826.0.1.3680043.8.498.48308485987218296338173512060246238691,4\r\n1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239,3\r\n1.2.826.0.1.3680043.8.498.48368929760320103375578806071106317953,3\r\n1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776,3\r\n1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799,0\r\n1.2.826.0.1.3680043.8.498.48486018503198264262324670490116120612,3\r\n1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236,3\r\n1.2.826.0.1.3680043.8.498.48521453765449688946535921546652765399,1\r\n1.2.826.0.1.3680043.8.498.48531746264504523370547117933271161633,3\r\n1.2.826.0.1.3680043.8.498.48564780460694033783288540237012068889,3\r\n1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208,0\r\n1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545,0\r\n1.2.826.0.1.3680043.8.498.48750912584073496502856009873745362236,3\r\n1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244,0\r\n1.2.826.0.1.3680043.8.498.48780598610511011488578963416186905560,5\r\n1.2.826.0.1.3680043.8.498.48784244479029704181755878324450207126,4\r\n1.2.826.0.1.3680043.8.498.48818931767017230678539144135307491171,5\r\n1.2.826.0.1.3680043.8.498.48829151552094424673693975665028690142,1\r\n1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619,3\r\n1.2.826.0.1.3680043.8.498.48872083192774100826667097552356058756,4\r\n1.2.826.0.1.3680043.8.498.48918090379630186340047006216221526186,3\r\n1.2.826.0.1.3680043.8.498.48933020429865163393792899740045803587,3\r\n1.2.826.0.1.3680043.8.498.48977699132767849812826572970218325717,1\r\n1.2.826.0.1.3680043.8.498.49025249926859976858200443939208207124,1\r\n1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547,0\r\n1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262,1\r\n1.2.826.0.1.3680043.8.498.49054454369199540406654153762248931799,2\r\n1.2.826.0.1.3680043.8.498.49094276335794811195117175603556929507,4\r\n1.2.826.0.1.3680043.8.498.49163335688297423808344168197551387860,3\r\n1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727,0\r\n1.2.826.0.1.3680043.8.498.49219586198038249030222134813685536295,4\r\n1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329,0\r\n1.2.826.0.1.3680043.8.498.49347024975371539869871751159907613638,2\r\n1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925,1\r\n1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330,4\r\n1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305,3\r\n1.2.826.0.1.3680043.8.498.49443962795025989994487968185070137217,4\r\n1.2.826.0.1.3680043.8.498.49481811111224238761880576486796603139,1\r\n1.2.826.0.1.3680043.8.498.49492260217998945587168161555623476238,1\r\n1.2.826.0.1.3680043.8.498.49495644703047055149965226028097259207,5\r\n1.2.826.0.1.3680043.8.498.49500752658775616590911569693817273205,3\r\n1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662,3\r\n1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513,0\r\n1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654,4\r\n1.2.826.0.1.3680043.8.498.49660659440268002826070581802625582554,5\r\n1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112,1\r\n1.2.826.0.1.3680043.8.498.49703744074809050221928706774732375079,3\r\n1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921,4\r\n1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668,0\r\n1.2.826.0.1.3680043.8.498.49850672111501136862025523789202033809,1\r\n1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912,4\r\n1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657,5\r\n1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125,3\r\n1.2.826.0.1.3680043.8.498.49884243466758407760126403944842240776,1\r\n1.2.826.0.1.3680043.8.498.49887758195164927372912584780941789021,1\r\n1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913,3\r\n1.2.826.0.1.3680043.8.498.49942556484410764813681696157580687854,2\r\n1.2.826.0.1.3680043.8.498.49984662916977328110175445895515541267,3\r\n1.2.826.0.1.3680043.8.498.50051845967696843074391018845332480653,3\r\n1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116,0\r\n1.2.826.0.1.3680043.8.498.50073674465735509400964384004695041387,5\r\n1.2.826.0.1.3680043.8.498.50089015780087008332657793589955395952,3\r\n1.2.826.0.1.3680043.8.498.50107479256735739327080544431929221135,2\r\n1.2.826.0.1.3680043.8.498.50109255875403812845391217314726500341,1\r\n1.2.826.0.1.3680043.8.498.50144715912379960437506930683683218333,3\r\n1.2.826.0.1.3680043.8.498.50182378358892377789975570268512056553,4\r\n1.2.826.0.1.3680043.8.498.50183004658202912273462671442829139110,4\r\n1.2.826.0.1.3680043.8.498.50215409013618353346619153290049205718,2\r\n1.2.826.0.1.3680043.8.498.50220991646207533313828550949660577504,4\r\n1.2.826.0.1.3680043.8.498.50248717336781045512984550029203786086,4\r\n1.2.826.0.1.3680043.8.498.50259348432783983788186173025555513832,1\r\n1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052,5\r\n1.2.826.0.1.3680043.8.498.50287512028986928372350279452197816661,4\r\n1.2.826.0.1.3680043.8.498.50318760104506317341991314680640521601,3\r\n1.2.826.0.1.3680043.8.498.50342726517205151681842826322659275894,4\r\n1.2.826.0.1.3680043.8.498.50392110746914984873009735588879049321,3\r\n1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521,2\r\n1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264,4\r\n1.2.826.0.1.3680043.8.498.50487158921933419035809338652990508020,3\r\n1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395,2\r\n1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550,0\r\n1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536,0\r\n1.2.826.0.1.3680043.8.498.50648934951234229265309905009982297515,1\r\n1.2.826.0.1.3680043.8.498.50666048236278074641860077012751071758,2\r\n1.2.826.0.1.3680043.8.498.50667444467788034101157821576803778774,3\r\n1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158,1\r\n1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306,4\r\n1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679,5\r\n1.2.826.0.1.3680043.8.498.50694733929204626284366818623993960684,4\r\n1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241,5\r\n1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310,0\r\n1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130,0\r\n1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652,1\r\n1.2.826.0.1.3680043.8.498.50813123035221058777203670043892828582,1\r\n1.2.826.0.1.3680043.8.498.50819987120895374623148160166392914169,4\r\n1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784,0\r\n1.2.826.0.1.3680043.8.498.50844227724977857739867646104945090414,1\r\n1.2.826.0.1.3680043.8.498.50848799900564667487859253004538870622,2\r\n1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845,0\r\n1.2.826.0.1.3680043.8.498.50882628969985977810341532747520504841,5\r\n1.2.826.0.1.3680043.8.498.50935247372419775856440294835587827419,2\r\n1.2.826.0.1.3680043.8.498.50969691234820675847736765735133138324,3\r\n1.2.826.0.1.3680043.8.498.50980137249034338585131194965993785201,4\r\n1.2.826.0.1.3680043.8.498.51006813205605862503064447831381009425,1\r\n1.2.826.0.1.3680043.8.498.51054592064718900209123281525185013961,2\r\n1.2.826.0.1.3680043.8.498.51059575642903646008860869691477305022,4\r\n1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102,0\r\n1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053,0\r\n1.2.826.0.1.3680043.8.498.51179760508587500025973916664101120069,2\r\n1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367,3\r\n1.2.826.0.1.3680043.8.498.51297777726540529860433097009654453742,4\r\n1.2.826.0.1.3680043.8.498.51351386796317851725461803790273870995,3\r\n1.2.826.0.1.3680043.8.498.51379389400506715529323532959275254794,1\r\n1.2.826.0.1.3680043.8.498.51426696939581135481763379246368970360,2\r\n1.2.826.0.1.3680043.8.498.51427159850769972151411336025891779423,5\r\n1.2.826.0.1.3680043.8.498.51429653138043597611341459799295143598,1\r\n1.2.826.0.1.3680043.8.498.51469564445249084310991187819410005207,0\r\n1.2.826.0.1.3680043.8.498.51509214978951980397275258374070843967,3\r\n1.2.826.0.1.3680043.8.498.51536234256866063100891626028999490545,2\r\n1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116,1\r\n1.2.826.0.1.3680043.8.498.51676679394093588751371807453793655464,3\r\n1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257,3\r\n1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131,0\r\n1.2.826.0.1.3680043.8.498.51768733026033198941231586462443503827,3\r\n1.2.826.0.1.3680043.8.498.51770131897828080663537020812721042704,2\r\n1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099,3\r\n1.2.826.0.1.3680043.8.498.51809092313496476873962641559638828608,4\r\n1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643,0\r\n1.2.826.0.1.3680043.8.498.51828180600648126726057100018010765613,1\r\n1.2.826.0.1.3680043.8.498.51831304809871808790657397599860793132,0\r\n1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915,0\r\n1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601,2\r\n1.2.826.0.1.3680043.8.498.51886877282987244004912794726043055730,4\r\n1.2.826.0.1.3680043.8.498.51910852739185117358682097116614556910,5\r\n1.2.826.0.1.3680043.8.498.51917043703891091519740084517167861965,5\r\n1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215,1\r\n1.2.826.0.1.3680043.8.498.51981019698056120061499970572334688555,2\r\n1.2.826.0.1.3680043.8.498.51985313438970133264846259111827877590,2\r\n1.2.826.0.1.3680043.8.498.52010165788889734743850946282043771445,2\r\n1.2.826.0.1.3680043.8.498.52010826440192227660457107121818389761,5\r\n1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329,2\r\n1.2.826.0.1.3680043.8.498.52059522418267239652374290810994711032,5\r\n1.2.826.0.1.3680043.8.498.52067134682915759489080529708019359378,3\r\n1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294,1\r\n1.2.826.0.1.3680043.8.498.52110163572817528199125320120779141456,4\r\n1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977,4\r\n1.2.826.0.1.3680043.8.498.52138967568492774495485454357106269286,4\r\n1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207,3\r\n1.2.826.0.1.3680043.8.498.52204390785696944340286127542242341202,2\r\n1.2.826.0.1.3680043.8.498.52330656126848153803758928167944040055,3\r\n1.2.826.0.1.3680043.8.498.52347547859131274836026480407333415929,2\r\n1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065,0\r\n1.2.826.0.1.3680043.8.498.52409113683924904563676007800283978062,1\r\n1.2.826.0.1.3680043.8.498.52435631723314823200374985065756182593,3\r\n1.2.826.0.1.3680043.8.498.52437948508356178175558637950043479411,2\r\n1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152,0\r\n1.2.826.0.1.3680043.8.498.52475562989926669452280424331791927661,2\r\n1.2.826.0.1.3680043.8.498.52499379042723926379822784411787551701,2\r\n1.2.826.0.1.3680043.8.498.52520288400853832269613213204540385687,4\r\n1.2.826.0.1.3680043.8.498.52558822670236188507569589448789513993,2\r\n1.2.826.0.1.3680043.8.498.52599302991259061057950834864441634066,1\r\n1.2.826.0.1.3680043.8.498.52636981391578580436180193211113340410,5\r\n1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650,3\r\n1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535,3\r\n1.2.826.0.1.3680043.8.498.52785983814702054185130109942936184688,1\r\n1.2.826.0.1.3680043.8.498.52844966504845792334740560021996858939,2\r\n1.2.826.0.1.3680043.8.498.52846023545870016653608198538437574931,2\r\n1.2.826.0.1.3680043.8.498.52846236545327527967419125989598675101,5\r\n1.2.826.0.1.3680043.8.498.52849154011665864497632297320961015159,2\r\n1.2.826.0.1.3680043.8.498.52937412433120578046842194545406797236,2\r\n1.2.826.0.1.3680043.8.498.52985748674486376309978868177895755694,4\r\n1.2.826.0.1.3680043.8.498.53025489296610571044246428509154950855,3\r\n1.2.826.0.1.3680043.8.498.53076472372145933727990308381815269368,2\r\n1.2.826.0.1.3680043.8.498.53078461307849490495536549806091931824,2\r\n1.2.826.0.1.3680043.8.498.53080172184945962593713560671746700445,3\r\n1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711,5\r\n1.2.826.0.1.3680043.8.498.53114802640679819441521055029024037716,2\r\n1.2.826.0.1.3680043.8.498.53131430415327479511358970615140982847,4\r\n1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832,0\r\n1.2.826.0.1.3680043.8.498.53156118232564605912649705207438612782,2\r\n1.2.826.0.1.3680043.8.498.53173579822475854424126184867040523127,1\r\n1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146,0\r\n1.2.826.0.1.3680043.8.498.53221818460567216467597044606040911702,2\r\n1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146,3\r\n1.2.826.0.1.3680043.8.498.53271710112621372496324855809391580130,2\r\n1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133,2\r\n1.2.826.0.1.3680043.8.498.53337684515627962545396495541158882262,3\r\n1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501,0\r\n1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263,0\r\n1.2.826.0.1.3680043.8.498.53361558447470435467999344560545063264,1\r\n1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200,4\r\n1.2.826.0.1.3680043.8.498.53393741562201846141435914434030220216,2\r\n1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643,0\r\n1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476,3\r\n1.2.826.0.1.3680043.8.498.53528011450161116437223332448942321596,2\r\n1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454,1\r\n1.2.826.0.1.3680043.8.498.53587920256280027683523152568832253423,4\r\n1.2.826.0.1.3680043.8.498.53628897373416863107623485630264501760,4\r\n1.2.826.0.1.3680043.8.498.53649895712172408476570770924813501118,1\r\n1.2.826.0.1.3680043.8.498.53737349894831365159124471910871800583,3\r\n1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027,0\r\n1.2.826.0.1.3680043.8.498.53792159922324410857755054939229546555,2\r\n1.2.826.0.1.3680043.8.498.53797236075631231042400235117445767744,4\r\n1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883,0\r\n1.2.826.0.1.3680043.8.498.53893586200869613140666806653041177558,5\r\n1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965,3\r\n1.2.826.0.1.3680043.8.498.53963759852017695930808272438606800422,3\r\n1.2.826.0.1.3680043.8.498.53968943793571323467397128102531786394,2\r\n1.2.826.0.1.3680043.8.498.54000782656400326230944526201670909133,1\r\n1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092,0\r\n1.2.826.0.1.3680043.8.498.54049224487950229496345484554581532711,4\r\n1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763,0\r\n1.2.826.0.1.3680043.8.498.54100786607660749952814251186461344633,5\r\n1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424,2\r\n1.2.826.0.1.3680043.8.498.54132814503202891434380002378663763655,3\r\n1.2.826.0.1.3680043.8.498.54138747504377266740838999528964336002,1\r\n1.2.826.0.1.3680043.8.498.54142170846176833568961426171739782214,2\r\n1.2.826.0.1.3680043.8.498.54164004416963296439322252627774426375,2\r\n1.2.826.0.1.3680043.8.498.54202117421897784469771192940597638508,1\r\n1.2.826.0.1.3680043.8.498.54270039578867689669466339980682684126,1\r\n1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043,0\r\n1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383,0\r\n1.2.826.0.1.3680043.8.498.54316764984549863227549326352573342232,2\r\n1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219,0\r\n1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985,0\r\n1.2.826.0.1.3680043.8.498.54394066317295451515247748115888976860,1\r\n1.2.826.0.1.3680043.8.498.54427039222108917725902677899081931510,3\r\n1.2.826.0.1.3680043.8.498.54448516923758211534546401490623824012,1\r\n1.2.826.0.1.3680043.8.498.54451515695681765924346479727204451807,4\r\n1.2.826.0.1.3680043.8.498.54482848766921329667469114743547203929,3\r\n1.2.826.0.1.3680043.8.498.54483132338261866032495309977418409299,1\r\n1.2.826.0.1.3680043.8.498.54539703415639765348239790227512479583,3\r\n1.2.826.0.1.3680043.8.498.54589880111295324708950851057241927191,3\r\n1.2.826.0.1.3680043.8.498.54619696938940618755334224550745735815,4\r\n1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589,4\r\n1.2.826.0.1.3680043.8.498.54662745951809363774369202018332309887,2\r\n1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557,0\r\n1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642,1\r\n1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025,0\r\n1.2.826.0.1.3680043.8.498.54699196872730350880681018674893216245,3\r\n1.2.826.0.1.3680043.8.498.54711709751232783991855540573189433446,2\r\n1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665,0\r\n1.2.826.0.1.3680043.8.498.54761306437713414745748829128854749469,4\r\n1.2.826.0.1.3680043.8.498.54792838438079861454532183998999249213,5\r\n1.2.826.0.1.3680043.8.498.54833384793167149128371137537527194757,1\r\n1.2.826.0.1.3680043.8.498.54866613282412053616997257524944654864,5\r\n1.2.826.0.1.3680043.8.498.55008913229055942530957507434366806883,2\r\n1.2.826.0.1.3680043.8.498.55015061060346597564205577461227720715,2\r\n1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406,3\r\n1.2.826.0.1.3680043.8.498.55061039227197546099231336400032878562,2\r\n1.2.826.0.1.3680043.8.498.55065105486648830377200570585789726800,3\r\n1.2.826.0.1.3680043.8.498.55067956175569778132589128971795053431,1\r\n1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441,2\r\n1.2.826.0.1.3680043.8.498.55183153701332792661581741475876132309,3\r\n1.2.826.0.1.3680043.8.498.55202459816190607876303419067810637245,4\r\n1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593,0\r\n1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763,2\r\n1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886,5\r\n1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587,0\r\n1.2.826.0.1.3680043.8.498.55346033326903331672579723063380636237,2\r\n1.2.826.0.1.3680043.8.498.55397602116058381866334455456729470963,1\r\n1.2.826.0.1.3680043.8.498.55415479446253262335685494509470808516,4\r\n1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246,1\r\n1.2.826.0.1.3680043.8.498.55583505147281684100202399148099429721,4\r\n1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870,0\r\n1.2.826.0.1.3680043.8.498.55653770127824574123236841437998533498,1\r\n1.2.826.0.1.3680043.8.498.55687452977471862155042703116192295932,4\r\n1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518,0\r\n1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792,4\r\n1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642,0\r\n1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856,3\r\n1.2.826.0.1.3680043.8.498.55808772152667135571939417929724526296,3\r\n1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728,2\r\n1.2.826.0.1.3680043.8.498.55856308312040330141050707531982127791,4\r\n1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978,0\r\n1.2.826.0.1.3680043.8.498.55887747667305814945719986974538814207,1\r\n1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285,2\r\n1.2.826.0.1.3680043.8.498.55930031395651221279170435293900769500,1\r\n1.2.826.0.1.3680043.8.498.55971152627433130850649668964887611410,1\r\n1.2.826.0.1.3680043.8.498.55971241672517044174019654998425819738,3\r\n1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659,2\r\n1.2.826.0.1.3680043.8.498.56016606949603829995818303105991224499,2\r\n1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805,4\r\n1.2.826.0.1.3680043.8.498.56054334296747411394367010227992359191,1\r\n1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391,2\r\n1.2.826.0.1.3680043.8.498.56091473614612257999053529249190958828,4\r\n1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628,4\r\n1.2.826.0.1.3680043.8.498.56156923369480999484275191793049573490,3\r\n1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535,0\r\n1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849,2\r\n1.2.826.0.1.3680043.8.498.56216873861595017403772627709751840588,1\r\n1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616,0\r\n1.2.826.0.1.3680043.8.498.56269695574661037525483516531194514456,1\r\n1.2.826.0.1.3680043.8.498.56305445410093481072159375249067050124,2\r\n1.2.826.0.1.3680043.8.498.56385473079196655120993854770623075553,4\r\n1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770,0\r\n1.2.826.0.1.3680043.8.498.56474364950531483257947155855279443851,4\r\n1.2.826.0.1.3680043.8.498.56497268548664917337130798701015887461,4\r\n1.2.826.0.1.3680043.8.498.56506415277488978301157490514745400410,1\r\n1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265,0\r\n1.2.826.0.1.3680043.8.498.56564801707939049324472369774251260607,1\r\n1.2.826.0.1.3680043.8.498.56575942633137496743464648679775042979,3\r\n1.2.826.0.1.3680043.8.498.56628956454784173189908113425348977923,1\r\n1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006,1\r\n1.2.826.0.1.3680043.8.498.56715311294415711253166312546942356882,2\r\n1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879,0\r\n1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860,0\r\n1.2.826.0.1.3680043.8.498.56786556618312090924315296993756843070,2\r\n1.2.826.0.1.3680043.8.498.56796622077427019119540733986070534126,4\r\n1.2.826.0.1.3680043.8.498.56798070549054124854685511360777559033,4\r\n1.2.826.0.1.3680043.8.498.56814987912934301686833132930648230999,4\r\n1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445,1\r\n1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883,1\r\n1.2.826.0.1.3680043.8.498.56882015310689536014061315110969323429,1\r\n1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068,0\r\n1.2.826.0.1.3680043.8.498.56894990686683651282564584582335346350,1\r\n1.2.826.0.1.3680043.8.498.56915079210789596803972894081756325026,3\r\n1.2.826.0.1.3680043.8.498.56939860329243258146102948466697018543,1\r\n1.2.826.0.1.3680043.8.498.56945039212126810522718038626526579591,1\r\n1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754,0\r\n1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680,1\r\n1.2.826.0.1.3680043.8.498.56993108275828399240508695615212151102,4\r\n1.2.826.0.1.3680043.8.498.56995522547527264862064392266001620445,2\r\n1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300,1\r\n1.2.826.0.1.3680043.8.498.57074072196714325300896734167161517369,4\r\n1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499,3\r\n1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856,0\r\n1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852,5\r\n1.2.826.0.1.3680043.8.498.57239450717434070615919278253974598533,3\r\n1.2.826.0.1.3680043.8.498.57316522861709598765962486949846744486,1\r\n1.2.826.0.1.3680043.8.498.57342523828166309133982222435655662667,2\r\n1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475,0\r\n1.2.826.0.1.3680043.8.498.57406640473235989459885313797729459197,2\r\n1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597,3\r\n1.2.826.0.1.3680043.8.498.57537915810250993237120953960404180449,3\r\n1.2.826.0.1.3680043.8.498.57615268093937994500995086275007694564,3\r\n1.2.826.0.1.3680043.8.498.57703608122587911055931213603182679454,2\r\n1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044,5\r\n1.2.826.0.1.3680043.8.498.57775389384191131322519457853517177210,2\r\n1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864,1\r\n1.2.826.0.1.3680043.8.498.57802887045514681373294932749517914959,3\r\n1.2.826.0.1.3680043.8.498.57811199787891897795884073594875380873,3\r\n1.2.826.0.1.3680043.8.498.57829173797358025677240411871600928097,4\r\n1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699,4\r\n1.2.826.0.1.3680043.8.498.57845032486422442404222046144765204905,3\r\n1.2.826.0.1.3680043.8.498.57859543314667538612269642333387175763,5\r\n1.2.826.0.1.3680043.8.498.57882532838957577543516026386452055977,2\r\n1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826,4\r\n1.2.826.0.1.3680043.8.498.57951268100539350409054562078476694936,4\r\n1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348,0\r\n1.2.826.0.1.3680043.8.498.58051085126748057986491277367254533832,4\r\n1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887,5\r\n1.2.826.0.1.3680043.8.498.58167256197067081436345136244032069163,4\r\n1.2.826.0.1.3680043.8.498.58250779958866595505585700517927225230,5\r\n1.2.826.0.1.3680043.8.498.58261214953729037661916565473101636690,5\r\n1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548,1\r\n1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195,0\r\n1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600,0\r\n1.2.826.0.1.3680043.8.498.58346459085553680879151588730895136332,4\r\n1.2.826.0.1.3680043.8.498.58366682083795592678613986311570432419,3\r\n1.2.826.0.1.3680043.8.498.58381033890285289577066078050126251787,2\r\n1.2.826.0.1.3680043.8.498.58404766256608483217059706069436050741,4\r\n1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292,4\r\n1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687,2\r\n1.2.826.0.1.3680043.8.498.58482152742263188503728990365891896621,5\r\n1.2.826.0.1.3680043.8.498.58482779112389764165599701941310919660,4\r\n1.2.826.0.1.3680043.8.498.58522891982092784622515590012748600644,2\r\n1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917,2\r\n1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167,0\r\n1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434,5\r\n1.2.826.0.1.3680043.8.498.58626635384305289939700801774393536415,4\r\n1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101,1\r\n1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058,0\r\n1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656,1\r\n1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485,0\r\n1.2.826.0.1.3680043.8.498.58728013757221328590898166513995249334,2\r\n1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744,3\r\n1.2.826.0.1.3680043.8.498.58749835013162544157291112101941849646,2\r\n1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130,2\r\n1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687,1\r\n1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538,5\r\n1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182,2\r\n1.2.826.0.1.3680043.8.498.58857477075095368361468658035369975572,5\r\n1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456,1\r\n1.2.826.0.1.3680043.8.498.58925843611945635442423587804169242644,1\r\n1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530,3\r\n1.2.826.0.1.3680043.8.498.59021913092219789232890327780064145948,1\r\n1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177,4\r\n1.2.826.0.1.3680043.8.498.59048646289883992375610835845636263714,2\r\n1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305,2\r\n1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769,0\r\n1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768,3\r\n1.2.826.0.1.3680043.8.498.59140342594072117638979402278277248612,4\r\n1.2.826.0.1.3680043.8.498.59254379036462832374262221063106167294,3\r\n1.2.826.0.1.3680043.8.498.59262240088132401543519466539885561578,2\r\n1.2.826.0.1.3680043.8.498.59309239681970150083311665704264518844,3\r\n1.2.826.0.1.3680043.8.498.59394856717216352192037635065925607933,3\r\n1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823,4\r\n1.2.826.0.1.3680043.8.498.59426134288436219215525445839849588348,2\r\n1.2.826.0.1.3680043.8.498.59449702487184611096113620550399616423,3\r\n1.2.826.0.1.3680043.8.498.59475691595504590121311599301170361869,3\r\n1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126,4\r\n1.2.826.0.1.3680043.8.498.59511171909586924989766685687032462369,1\r\n1.2.826.0.1.3680043.8.498.59527896591755229337596505685198957637,2\r\n1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273,3\r\n1.2.826.0.1.3680043.8.498.59646951619367295963072805933618947027,4\r\n1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651,5\r\n1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235,0\r\n1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228,3\r\n1.2.826.0.1.3680043.8.498.59826277112503440743678968423276737254,3\r\n1.2.826.0.1.3680043.8.498.59828592894392286103154555572376381359,2\r\n1.2.826.0.1.3680043.8.498.59835677513721120155185666132462195647,2\r\n1.2.826.0.1.3680043.8.498.59864697179957009967756900038846073439,3\r\n1.2.826.0.1.3680043.8.498.59923507410103964137819195926587517759,2\r\n1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504,4\r\n1.2.826.0.1.3680043.8.498.59938475193950583662125713924156951751,2\r\n1.2.826.0.1.3680043.8.498.59939977812549570931951749553467050226,2\r\n1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378,4\r\n1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197,0\r\n1.2.826.0.1.3680043.8.498.59973351395253648445741322652773810641,4\r\n1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101,4\r\n1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357,0\r\n1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484,2\r\n1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162,0\r\n1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746,4\r\n1.2.826.0.1.3680043.8.498.60227430483444032531479844365716485643,1\r\n1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443,3\r\n1.2.826.0.1.3680043.8.498.60259828181878278590614805828898992241,1\r\n1.2.826.0.1.3680043.8.498.60339775486026531260673620094004244044,5\r\n1.2.826.0.1.3680043.8.498.60347768295306794807599672836772410567,1\r\n1.2.826.0.1.3680043.8.498.60394858470917657607846893765818799720,2\r\n1.2.826.0.1.3680043.8.498.60430475172382878092821699001847797153,1\r\n1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420,4\r\n1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796,0\r\n1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184,4\r\n1.2.826.0.1.3680043.8.498.60465293103574559546982440125572490190,4\r\n1.2.826.0.1.3680043.8.498.60473167624534682552272469123806503959,1\r\n1.2.826.0.1.3680043.8.498.60477027832057450004004942199870813886,1\r\n1.2.826.0.1.3680043.8.498.60492772672040498730937577391401159109,2\r\n1.2.826.0.1.3680043.8.498.60548477793215320533051194155539307724,4\r\n1.2.826.0.1.3680043.8.498.60582762894748063612925320003677861384,2\r\n1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474,2\r\n1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331,0\r\n1.2.826.0.1.3680043.8.498.60633008040453481523409852849000823589,4\r\n1.2.826.0.1.3680043.8.498.60661676782543347731000874708337132276,2\r\n1.2.826.0.1.3680043.8.498.60723378470101669902158229125336455520,1\r\n1.2.826.0.1.3680043.8.498.60736996513214411991331917490659995815,4\r\n1.2.826.0.1.3680043.8.498.60840220723824894619895264673401129352,3\r\n1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592,5\r\n1.2.826.0.1.3680043.8.498.60931716414402855657852057585917605786,2\r\n1.2.826.0.1.3680043.8.498.60965286164497194740320124904525623873,3\r\n1.2.826.0.1.3680043.8.498.61024356179284390979230045743542661223,4\r\n1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558,2\r\n1.2.826.0.1.3680043.8.498.61072566792285434404885416160728089641,3\r\n1.2.826.0.1.3680043.8.498.61140619849833544411463750382539885141,5\r\n1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106,5\r\n1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169,1\r\n1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841,0\r\n1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160,3\r\n1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675,0\r\n1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770,0\r\n1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563,2\r\n1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679,0\r\n1.2.826.0.1.3680043.8.498.61435096069921474761812731150911059491,2\r\n1.2.826.0.1.3680043.8.498.61456481310419978302697858056095086811,3\r\n1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628,4\r\n1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470,3\r\n1.2.826.0.1.3680043.8.498.61529451559250327125552112075119161933,5\r\n1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386,2\r\n1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417,2\r\n1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015,2\r\n1.2.826.0.1.3680043.8.498.61630894340791204767579628554826500984,4\r\n1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799,0\r\n1.2.826.0.1.3680043.8.498.61797111379749329977228618245440305623,4\r\n1.2.826.0.1.3680043.8.498.61812343111344551704536863711676179812,4\r\n1.2.826.0.1.3680043.8.498.61864939963243675140060416958055914981,1\r\n1.2.826.0.1.3680043.8.498.61904856234837992279531381374350177464,1\r\n1.2.826.0.1.3680043.8.498.61934170443339188744573234190617483017,3\r\n1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569,2\r\n1.2.826.0.1.3680043.8.498.61981891095235024266625598999854298547,1\r\n1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893,0\r\n1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008,0\r\n1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950,4\r\n1.2.826.0.1.3680043.8.498.62154969360158054310282857093665911261,2\r\n1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086,1\r\n1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775,4\r\n1.2.826.0.1.3680043.8.498.62217447760619326492634156203608889396,3\r\n1.2.826.0.1.3680043.8.498.62218684089117864632544493263539598403,1\r\n1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534,3\r\n1.2.826.0.1.3680043.8.498.62252176347004905787746821113523889523,3\r\n1.2.826.0.1.3680043.8.498.62329283530383340787096585492217632179,2\r\n1.2.826.0.1.3680043.8.498.62383135836496954297324866452887364097,1\r\n1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752,4\r\n1.2.826.0.1.3680043.8.498.62400625562782184774973349983498962743,4\r\n1.2.826.0.1.3680043.8.498.62406588098851994968957074186291720431,0\r\n1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423,2\r\n1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272,0\r\n1.2.826.0.1.3680043.8.498.62662430565901539554882496013173225758,2\r\n1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983,0\r\n1.2.826.0.1.3680043.8.498.62752755711205966800746994120425231855,4\r\n1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875,4\r\n1.2.826.0.1.3680043.8.498.62776347582771469289738234607517546504,1\r\n1.2.826.0.1.3680043.8.498.62782163204193388751869649423542931657,3\r\n1.2.826.0.1.3680043.8.498.62830031603985608774418534497520663269,2\r\n1.2.826.0.1.3680043.8.498.62918046703989259691005447883746340916,0\r\n1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103,4\r\n1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414,0\r\n1.2.826.0.1.3680043.8.498.62948920639798990757395955917890550512,0\r\n1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807,4\r\n1.2.826.0.1.3680043.8.498.63002559681883387414876870926128159682,3\r\n1.2.826.0.1.3680043.8.498.63005973438879239920248535292375481294,2\r\n1.2.826.0.1.3680043.8.498.63068687840516335034631353955761036465,1\r\n1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067,1\r\n1.2.826.0.1.3680043.8.498.63096952391833769395204944538276970114,4\r\n1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410,5\r\n1.2.826.0.1.3680043.8.498.63126604639589769235241216875934773992,2\r\n1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316,5\r\n1.2.826.0.1.3680043.8.498.63195088465290711580668024587233364997,2\r\n1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360,0\r\n1.2.826.0.1.3680043.8.498.63204735160524958513470115761010950181,5\r\n1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952,2\r\n1.2.826.0.1.3680043.8.498.63223220827342079068491460847713457238,4\r\n1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113,0\r\n1.2.826.0.1.3680043.8.498.63242029756172180508167933519465902827,1\r\n1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540,2\r\n1.2.826.0.1.3680043.8.498.63280826430574305683941548157934410641,0\r\n1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895,5\r\n1.2.826.0.1.3680043.8.498.63360204391136938955220146755331421179,1\r\n1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519,3\r\n1.2.826.0.1.3680043.8.498.63424633611847022112809173449690688015,3\r\n1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166,2\r\n1.2.826.0.1.3680043.8.498.63439188800533762543950349022148547855,2\r\n1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386,1\r\n1.2.826.0.1.3680043.8.498.63519003265636561965405570006747767668,1\r\n1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343,1\r\n1.2.826.0.1.3680043.8.498.63521726454879203860288713455843702564,3\r\n1.2.826.0.1.3680043.8.498.63539809431911516912857504094611479480,3\r\n1.2.826.0.1.3680043.8.498.63550258873504696667112631869753870779,2\r\n1.2.826.0.1.3680043.8.498.63556690547679257097888724096967682731,5\r\n1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941,0\r\n1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258,1\r\n1.2.826.0.1.3680043.8.498.63584266247144369196126771996126173497,0\r\n1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473,3\r\n1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809,3\r\n1.2.826.0.1.3680043.8.498.63726512375511246975989322174096818969,2\r\n1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334,2\r\n1.2.826.0.1.3680043.8.498.63849051301265505051181245915656194652,2\r\n1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906,0\r\n1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566,1\r\n1.2.826.0.1.3680043.8.498.63986652686300452077410299429386420400,4\r\n1.2.826.0.1.3680043.8.498.64022112049842890696609934970222024490,4\r\n1.2.826.0.1.3680043.8.498.64022949342538714157641824612944253302,1\r\n1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316,0\r\n1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567,1\r\n1.2.826.0.1.3680043.8.498.64066465191710322298540005681534170036,2\r\n1.2.826.0.1.3680043.8.498.64109231850522212435708176045583779359,3\r\n1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453,2\r\n1.2.826.0.1.3680043.8.498.64172939203926034064618447795131041647,4\r\n1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373,0\r\n1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734,4\r\n1.2.826.0.1.3680043.8.498.64283618749539713756220539395944025878,3\r\n1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715,0\r\n1.2.826.0.1.3680043.8.498.64325734401816269429960464734504322970,4\r\n1.2.826.0.1.3680043.8.498.64381606512330581256470279413868645789,2\r\n1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477,0\r\n1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676,1\r\n1.2.826.0.1.3680043.8.498.64421299492271548595446260681698412384,1\r\n1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105,1\r\n1.2.826.0.1.3680043.8.498.64487230511522940587342058274736571996,3\r\n1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010,0\r\n1.2.826.0.1.3680043.8.498.64527059976573114931020581328924993069,1\r\n1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724,0\r\n1.2.826.0.1.3680043.8.498.64606794953061143951972249525094490665,4\r\n1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223,0\r\n1.2.826.0.1.3680043.8.498.64652085172505930317660036583152917472,4\r\n1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188,5\r\n1.2.826.0.1.3680043.8.498.64673069370410118387363952155758553567,1\r\n1.2.826.0.1.3680043.8.498.64752580249608101443758975861213065048,1\r\n1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005,0\r\n1.2.826.0.1.3680043.8.498.64897285598346101578442697515294908341,1\r\n1.2.826.0.1.3680043.8.498.64917073507426524387269500871970032783,2\r\n1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525,1\r\n1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183,3\r\n1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461,1\r\n1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643,0\r\n1.2.826.0.1.3680043.8.498.65085954283273471512936799274064949557,2\r\n1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252,1\r\n1.2.826.0.1.3680043.8.498.65124952546314712352837396207287384405,3\r\n1.2.826.0.1.3680043.8.498.65160241567146620406193593170843451582,2\r\n1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455,0\r\n1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325,1\r\n1.2.826.0.1.3680043.8.498.65224808465738083245406235000047993647,2\r\n1.2.826.0.1.3680043.8.498.65244223225926362862970496735749098549,4\r\n1.2.826.0.1.3680043.8.498.65312506382127486486519184894274326380,1\r\n1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816,1\r\n1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460,0\r\n1.2.826.0.1.3680043.8.498.65403984120528711726317048984986599537,2\r\n1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554,0\r\n1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178,4\r\n1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719,0\r\n1.2.826.0.1.3680043.8.498.65530757406333318449497070048719890442,3\r\n1.2.826.0.1.3680043.8.498.65543472488214519069910341591668399186,4\r\n1.2.826.0.1.3680043.8.498.65547889329285432771702699287910809921,3\r\n1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856,1\r\n1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847,3\r\n1.2.826.0.1.3680043.8.498.65667678378249093118816281743956646856,1\r\n1.2.826.0.1.3680043.8.498.65703265163052901613211590083675186626,2\r\n1.2.826.0.1.3680043.8.498.65715827585629489888518528736848969678,2\r\n1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765,0\r\n1.2.826.0.1.3680043.8.498.65763261177572072944828171262268039852,4\r\n1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666,0\r\n1.2.826.0.1.3680043.8.498.65788206282522750507559682036750692505,2\r\n1.2.826.0.1.3680043.8.498.65792625478083832901309715174678358900,2\r\n1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674,0\r\n1.2.826.0.1.3680043.8.498.65834774574423727070722226151408235786,1\r\n1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057,4\r\n1.2.826.0.1.3680043.8.498.65896962360688978581417748865609785541,1\r\n1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345,5\r\n1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980,0\r\n1.2.826.0.1.3680043.8.498.65959182350599581687336889990076028593,3\r\n1.2.826.0.1.3680043.8.498.65977711270942843864266358352913175672,1\r\n1.2.826.0.1.3680043.8.498.66020213859389864645118554257523478921,1\r\n1.2.826.0.1.3680043.8.498.66022464665791424345622894674375661089,1\r\n1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398,4\r\n1.2.826.0.1.3680043.8.498.66065279867975557954375204776779208827,4\r\n1.2.826.0.1.3680043.8.498.66077420344931334696112616504179985591,0\r\n1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887,0\r\n1.2.826.0.1.3680043.8.498.66156015559325316834458423557757995415,4\r\n1.2.826.0.1.3680043.8.498.66199248385068685539872840826351152652,1\r\n1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769,3\r\n1.2.826.0.1.3680043.8.498.66264392187277583747691643146920431469,1\r\n1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084,0\r\n1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755,0\r\n1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177,3\r\n1.2.826.0.1.3680043.8.498.66308195780676793917989368423009849637,1\r\n1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952,4\r\n1.2.826.0.1.3680043.8.498.66327751482076404294387513492434577338,1\r\n1.2.826.0.1.3680043.8.498.66334686028272068369345540785320291695,5\r\n1.2.826.0.1.3680043.8.498.66364462115470823900176770413527217503,1\r\n1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421,3\r\n1.2.826.0.1.3680043.8.498.66404725797392899950170138361835649917,5\r\n1.2.826.0.1.3680043.8.498.66489634638847834550694493585147280754,2\r\n1.2.826.0.1.3680043.8.498.66510565421813721258189101326477318907,2\r\n1.2.826.0.1.3680043.8.498.66596090141798159058634289358454183351,3\r\n1.2.826.0.1.3680043.8.498.66640124635222298792902508374800337726,3\r\n1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709,2\r\n1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853,4\r\n1.2.826.0.1.3680043.8.498.66737859593831436259089736126768464771,4\r\n1.2.826.0.1.3680043.8.498.66805139646641294853509958688742554525,3\r\n1.2.826.0.1.3680043.8.498.66841103065389483583108279900721522084,2\r\n1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524,0\r\n1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127,3\r\n1.2.826.0.1.3680043.8.498.66973089869591928349547584554030589647,1\r\n1.2.826.0.1.3680043.8.498.66993076561387452114335652021593805727,3\r\n1.2.826.0.1.3680043.8.498.67001329390153622359718581388215312710,3\r\n1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269,3\r\n1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924,4\r\n1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713,4\r\n1.2.826.0.1.3680043.8.498.67189969532330535369638450306601659967,5\r\n1.2.826.0.1.3680043.8.498.67208608901061914528631080268819069203,4\r\n1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258,0\r\n1.2.826.0.1.3680043.8.498.67281816624401881515444587844116136597,2\r\n1.2.826.0.1.3680043.8.498.67284681623851829916403311739987186159,2\r\n1.2.826.0.1.3680043.8.498.67374476726034608591738910363595695123,3\r\n1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678,4\r\n1.2.826.0.1.3680043.8.498.67397756161011684282778294948252031539,1\r\n1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784,2\r\n1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595,2\r\n1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142,2\r\n1.2.826.0.1.3680043.8.498.67490059263503953166467514121525025332,2\r\n1.2.826.0.1.3680043.8.498.67498362978754615554767658151787188674,4\r\n1.2.826.0.1.3680043.8.498.67527486841399075490744039058207332725,5\r\n1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234,2\r\n1.2.826.0.1.3680043.8.498.67601368724828563564325678719126190797,2\r\n1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330,1\r\n1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451,2\r\n1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361,1\r\n1.2.826.0.1.3680043.8.498.67690090498828405347380696553164435363,3\r\n1.2.826.0.1.3680043.8.498.67705068860320494485379949506807098172,1\r\n1.2.826.0.1.3680043.8.498.67716063997837501476742746033172868748,4\r\n1.2.826.0.1.3680043.8.498.67739091465243228820766982459725849814,1\r\n1.2.826.0.1.3680043.8.498.67740000270419228186274186410889719154,1\r\n1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848,0\r\n1.2.826.0.1.3680043.8.498.67775301856843451215029610725114809378,3\r\n1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306,0\r\n1.2.826.0.1.3680043.8.498.67817378785531423675449308236680249899,3\r\n1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228,4\r\n1.2.826.0.1.3680043.8.498.67875980316053973061252844698884229168,1\r\n1.2.826.0.1.3680043.8.498.67901792488494108577057563458320460313,4\r\n1.2.826.0.1.3680043.8.498.67902658411525109462089381503197501728,1\r\n1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686,2\r\n1.2.826.0.1.3680043.8.498.67941678374744711267840655419606994796,4\r\n1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879,1\r\n1.2.826.0.1.3680043.8.498.67981157392064878495682078197161506255,3\r\n1.2.826.0.1.3680043.8.498.67981539513814867568631217960473023750,3\r\n1.2.826.0.1.3680043.8.498.68042422993686716523227090971667037309,4\r\n1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669,3\r\n1.2.826.0.1.3680043.8.498.68098431505945078581285587720599762771,4\r\n1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666,0\r\n1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147,0\r\n1.2.826.0.1.3680043.8.498.68131564130881160284492534203572971623,1\r\n1.2.826.0.1.3680043.8.498.68142054543837964766571710567214307830,0\r\n1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606,4\r\n1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436,0\r\n1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179,3\r\n1.2.826.0.1.3680043.8.498.68176786698913278568384612123261229708,3\r\n1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317,3\r\n1.2.826.0.1.3680043.8.498.68236154531625015457518732674625182872,1\r\n1.2.826.0.1.3680043.8.498.68256141451764547710152696919981893952,3\r\n1.2.826.0.1.3680043.8.498.68263925314265871578642679583159577645,2\r\n1.2.826.0.1.3680043.8.498.68264559730756309944466784364591509088,3\r\n1.2.826.0.1.3680043.8.498.68304853223764870957854101604223809218,4\r\n1.2.826.0.1.3680043.8.498.68328371080087298795966933114044350359,3\r\n1.2.826.0.1.3680043.8.498.68328948893840135511720724977892905019,3\r\n1.2.826.0.1.3680043.8.498.68434116759606759586218589635360502038,4\r\n1.2.826.0.1.3680043.8.498.68434323590918408241182252996136001891,2\r\n1.2.826.0.1.3680043.8.498.68435927323644457286635872824563629385,3\r\n1.2.826.0.1.3680043.8.498.68442883836038453562548037358230921748,2\r\n1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150,0\r\n1.2.826.0.1.3680043.8.498.68504220056564878621959394439671931747,2\r\n1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756,2\r\n1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462,0\r\n1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028,4\r\n1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600,0\r\n1.2.826.0.1.3680043.8.498.68593524008570334427931338225473487643,3\r\n1.2.826.0.1.3680043.8.498.68637419175382604890136839259344633105,3\r\n1.2.826.0.1.3680043.8.498.68668620295929607459355154952035421306,2\r\n1.2.826.0.1.3680043.8.498.68707023438918237611039069622720041636,0\r\n1.2.826.0.1.3680043.8.498.68740466085414970518655922465908223436,4\r\n1.2.826.0.1.3680043.8.498.68744926735490285461495285547312308522,5\r\n1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835,0\r\n1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335,3\r\n1.2.826.0.1.3680043.8.498.68799355386870083063665227055340582612,3\r\n1.2.826.0.1.3680043.8.498.68806105227826942108149090508198910591,4\r\n1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157,0\r\n1.2.826.0.1.3680043.8.498.68919687595587004665658610968739852899,3\r\n1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960,0\r\n1.2.826.0.1.3680043.8.498.68969263295435881669720950631709959471,2\r\n1.2.826.0.1.3680043.8.498.68971603226593154050514800137550171537,5\r\n1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840,3\r\n1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812,3\r\n1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819,0\r\n1.2.826.0.1.3680043.8.498.69131433601281694165384475765166382765,3\r\n1.2.826.0.1.3680043.8.498.69141410772327587821159027820329185006,1\r\n1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703,5\r\n1.2.826.0.1.3680043.8.498.69212713525853754921729012449897924087,4\r\n1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455,0\r\n1.2.826.0.1.3680043.8.498.69260280362382086729793018979234418276,2\r\n1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198,1\r\n1.2.826.0.1.3680043.8.498.69311065409015951283858403030798356475,1\r\n1.2.826.0.1.3680043.8.498.69397705155385306818511499752200941968,2\r\n1.2.826.0.1.3680043.8.498.69411539430142524612470544041263824359,3\r\n1.2.826.0.1.3680043.8.498.69417319234704516520173921900954337376,3\r\n1.2.826.0.1.3680043.8.498.69429384137362531712760113705595820981,2\r\n1.2.826.0.1.3680043.8.498.69470130842235635628917024269527211321,1\r\n1.2.826.0.1.3680043.8.498.69495014653333523580255444915082056825,3\r\n1.2.826.0.1.3680043.8.498.69514709065182531331987458626496800737,2\r\n1.2.826.0.1.3680043.8.498.69544085715595825083308172629360899339,2\r\n1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384,0\r\n1.2.826.0.1.3680043.8.498.69632034454628807663252084651278084839,1\r\n1.2.826.0.1.3680043.8.498.69635429172929809539574727887787303282,2\r\n1.2.826.0.1.3680043.8.498.69654898267139720912940475772303404772,5\r\n1.2.826.0.1.3680043.8.498.69665129458102359604411189244405850661,2\r\n1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579,0\r\n1.2.826.0.1.3680043.8.498.69717507929047897681405659943474469018,1\r\n1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813,5\r\n1.2.826.0.1.3680043.8.498.69730556680449158538173708009262014812,5\r\n1.2.826.0.1.3680043.8.498.69755849784902271124335765927125722483,4\r\n1.2.826.0.1.3680043.8.498.69828573582708185393942443598783108800,3\r\n1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494,1\r\n1.2.826.0.1.3680043.8.498.69917156795941398965793962392683899162,2\r\n1.2.826.0.1.3680043.8.498.69930569170269472474738549886709596023,4\r\n1.2.826.0.1.3680043.8.498.69954533310505596128507234525019777805,4\r\n1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063,3\r\n1.2.826.0.1.3680043.8.498.70125113032990356507120856030706104508,5\r\n1.2.826.0.1.3680043.8.498.70129413051088864999976563404818211137,2\r\n1.2.826.0.1.3680043.8.498.70204818275951821519024007466988533363,4\r\n1.2.826.0.1.3680043.8.498.70212141059019236877414201247791694101,1\r\n1.2.826.0.1.3680043.8.498.70271042033022764422915852922035168712,4\r\n1.2.826.0.1.3680043.8.498.70291739279544255146458190540233023337,2\r\n1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063,0\r\n1.2.826.0.1.3680043.8.498.70310491987092072540988486213408328865,1\r\n1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260,0\r\n1.2.826.0.1.3680043.8.498.70375834273924102402706635540898552043,1\r\n1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808,0\r\n1.2.826.0.1.3680043.8.498.70427947434642957729711329062828233047,3\r\n1.2.826.0.1.3680043.8.498.70435722790459313233345196619727661231,3\r\n1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056,2\r\n1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757,0\r\n1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894,1\r\n1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314,0\r\n1.2.826.0.1.3680043.8.498.70531311408047627903498280968772932092,2\r\n1.2.826.0.1.3680043.8.498.70532653867267627708152426358320452518,2\r\n1.2.826.0.1.3680043.8.498.70609649448269244608860767754947143395,4\r\n1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827,1\r\n1.2.826.0.1.3680043.8.498.70691305452244378255646875206864340458,3\r\n1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717,3\r\n1.2.826.0.1.3680043.8.498.70796360440320543674384511859375295769,4\r\n1.2.826.0.1.3680043.8.498.70829434408808897601630953382239533220,5\r\n1.2.826.0.1.3680043.8.498.70831735578941318362881379802693293952,0\r\n1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526,0\r\n1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541,5\r\n1.2.826.0.1.3680043.8.498.70877157427595288430686609728186100169,3\r\n1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522,0\r\n1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266,4\r\n1.2.826.0.1.3680043.8.498.70913971933753517251340687745718346828,3\r\n1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068,0\r\n1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104,0\r\n1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315,0\r\n1.2.826.0.1.3680043.8.498.70948162965271404678418299669321564765,1\r\n1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261,0\r\n1.2.826.0.1.3680043.8.498.70966951020892708303866968206211604316,1\r\n1.2.826.0.1.3680043.8.498.70984418257970716669463829979819981861,2\r\n1.2.826.0.1.3680043.8.498.71001339006150287453095055802545712130,4\r\n1.2.826.0.1.3680043.8.498.71047916312801688293181464016099792199,2\r\n1.2.826.0.1.3680043.8.498.71063550263564056105844586167361750567,3\r\n1.2.826.0.1.3680043.8.498.71084400653875583977681616665093993652,3\r\n1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565,2\r\n1.2.826.0.1.3680043.8.498.71180975285730441100082825359032677931,1\r\n1.2.826.0.1.3680043.8.498.71262022207903748073532915013888861999,2\r\n1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846,0\r\n1.2.826.0.1.3680043.8.498.71355457409240695394093233905149541215,1\r\n1.2.826.0.1.3680043.8.498.71388532549899713881913113571627425347,2\r\n1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184,3\r\n1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237,1\r\n1.2.826.0.1.3680043.8.498.71622107620007706778976946784787365505,1\r\n1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629,3\r\n1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585,1\r\n1.2.826.0.1.3680043.8.498.71671013221669628598634284971059228267,1\r\n1.2.826.0.1.3680043.8.498.71686717249431292334159336816742076478,1\r\n1.2.826.0.1.3680043.8.498.71688651674041466985305836912343513997,4\r\n1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610,2\r\n1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504,3\r\n1.2.826.0.1.3680043.8.498.71709477515434076105531625950744135656,4\r\n1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437,0\r\n1.2.826.0.1.3680043.8.498.71744523263023933066263506201344989825,2\r\n1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366,0\r\n1.2.826.0.1.3680043.8.498.71753603955749189961638324806586359888,2\r\n1.2.826.0.1.3680043.8.498.71766161038805688022754893134126528749,3\r\n1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190,0\r\n1.2.826.0.1.3680043.8.498.71814919885777053270239861047114794237,3\r\n1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914,5\r\n1.2.826.0.1.3680043.8.498.71836684300308056297405632410209248276,1\r\n1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525,3\r\n1.2.826.0.1.3680043.8.498.71898908031968158312013185809213669768,3\r\n1.2.826.0.1.3680043.8.498.71899370657875960028369229847153071577,1\r\n1.2.826.0.1.3680043.8.498.71901039762584748047172479049347609017,4\r\n1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627,4\r\n1.2.826.0.1.3680043.8.498.72008869367139958325145888733651022033,4\r\n1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145,2\r\n1.2.826.0.1.3680043.8.498.72018602547183896629903902899903468287,2\r\n1.2.826.0.1.3680043.8.498.72023797276837219152365428999317131110,3\r\n1.2.826.0.1.3680043.8.498.72095214441332094906401331319197122529,2\r\n1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597,0\r\n1.2.826.0.1.3680043.8.498.72172265826902387784016098389863005666,1\r\n1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641,2\r\n1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666,0\r\n1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774,3\r\n1.2.826.0.1.3680043.8.498.72237065473467452226073540784070938342,2\r\n1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637,2\r\n1.2.826.0.1.3680043.8.498.72252601583562621620422058467969183753,1\r\n1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401,4\r\n1.2.826.0.1.3680043.8.498.72270915206075693056344475438055463113,2\r\n1.2.826.0.1.3680043.8.498.72276631472244076773626607623783737687,4\r\n1.2.826.0.1.3680043.8.498.72313325634455670216423864504103231505,1\r\n1.2.826.0.1.3680043.8.498.72342425990274562150756501476989691572,4\r\n1.2.826.0.1.3680043.8.498.72368645606329000923061796699431437924,1\r\n1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834,2\r\n1.2.826.0.1.3680043.8.498.72395496925770440630088720587484704971,4\r\n1.2.826.0.1.3680043.8.498.72453426317225885428514595399496580896,4\r\n1.2.826.0.1.3680043.8.498.72494404099796908482534991590660770670,1\r\n1.2.826.0.1.3680043.8.498.72495590491484814220060275823761767516,2\r\n1.2.826.0.1.3680043.8.498.72495677804641720406765749568114476096,3\r\n1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203,2\r\n1.2.826.0.1.3680043.8.498.72576069358772864331810785663326683534,3\r\n1.2.826.0.1.3680043.8.498.72587440202779699193620339987034971895,2\r\n1.2.826.0.1.3680043.8.498.72653666963736486779382314125782779004,4\r\n1.2.826.0.1.3680043.8.498.72680485820462987585370556994612687540,2\r\n1.2.826.0.1.3680043.8.498.72697172162233919540021967137047646179,5\r\n1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585,4\r\n1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639,0\r\n1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309,0\r\n1.2.826.0.1.3680043.8.498.72786773761211751423503785260617678955,5\r\n1.2.826.0.1.3680043.8.498.72800659596099468282135738543971261904,2\r\n1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230,0\r\n1.2.826.0.1.3680043.8.498.72880034930990967339546757487847803284,3\r\n1.2.826.0.1.3680043.8.498.72896348955280307080916504120667484813,3\r\n1.2.826.0.1.3680043.8.498.72896586188735703676995894830815493004,3\r\n1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492,4\r\n1.2.826.0.1.3680043.8.498.72937134140407212536042212109839895294,1\r\n1.2.826.0.1.3680043.8.498.72943191222961132416678221885831944435,1\r\n1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331,0\r\n1.2.826.0.1.3680043.8.498.73021428171736137311489171988512484948,3\r\n1.2.826.0.1.3680043.8.498.73033521237848260228708170280586043408,4\r\n1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398,3\r\n1.2.826.0.1.3680043.8.498.73070182506490199787361298167985971577,5\r\n1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826,1\r\n1.2.826.0.1.3680043.8.498.73083786204651635096412213233334934941,2\r\n1.2.826.0.1.3680043.8.498.73101997284755549124715401557235981642,2\r\n1.2.826.0.1.3680043.8.498.73170541731840260065335942048010387599,2\r\n1.2.826.0.1.3680043.8.498.73194579460722181989151410198309642266,3\r\n1.2.826.0.1.3680043.8.498.73198335328613992289471885012751338458,3\r\n1.2.826.0.1.3680043.8.498.73205916973275504240156555900460720571,4\r\n1.2.826.0.1.3680043.8.498.73215265387578908341609746559916587894,4\r\n1.2.826.0.1.3680043.8.498.73231153161312542696053060380619432126,2\r\n1.2.826.0.1.3680043.8.498.73241519025825862373621891294281253158,1\r\n1.2.826.0.1.3680043.8.498.73242978867957397254276973409271434573,4\r\n1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751,1\r\n1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930,4\r\n1.2.826.0.1.3680043.8.498.73275019425494043461771574462523171618,3\r\n1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074,4\r\n1.2.826.0.1.3680043.8.498.73325959289998105909842627876607163467,0\r\n1.2.826.0.1.3680043.8.498.73356831368999268488355650751758108912,2\r\n1.2.826.0.1.3680043.8.498.73394992092536626285935384324191560182,1\r\n1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284,2\r\n1.2.826.0.1.3680043.8.498.73477249559479870799427355988138990349,4\r\n1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803,0\r\n1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320,0\r\n1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615,2\r\n1.2.826.0.1.3680043.8.498.73522298807198784022883320957252530465,4\r\n1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951,0\r\n1.2.826.0.1.3680043.8.498.73545582819102316606081606511277579853,1\r\n1.2.826.0.1.3680043.8.498.73567807253694728043552196136704821457,2\r\n1.2.826.0.1.3680043.8.498.73602922823389605007702521335136347605,3\r\n1.2.826.0.1.3680043.8.498.73616237718955147476241731466078978343,5\r\n1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115,3\r\n1.2.826.0.1.3680043.8.498.73716752081143767997507440278231972668,2\r\n1.2.826.0.1.3680043.8.498.73748007182569068263280098473557128226,5\r\n1.2.826.0.1.3680043.8.498.73792361507751648386401969571717557389,2\r\n1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478,2\r\n1.2.826.0.1.3680043.8.498.73869775636181013203432845292462040922,3\r\n1.2.826.0.1.3680043.8.498.73891117356545433634964184054304516063,2\r\n1.2.826.0.1.3680043.8.498.73983081622010008601849687650985260980,3\r\n1.2.826.0.1.3680043.8.498.73988798714023285209290281409329527637,3\r\n1.2.826.0.1.3680043.8.498.74052473719853343027548507306186291134,1\r\n1.2.826.0.1.3680043.8.498.74078340346189210015785395961373387782,2\r\n1.2.826.0.1.3680043.8.498.74095402538454760179339842746572802563,4\r\n1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520,1\r\n1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031,2\r\n1.2.826.0.1.3680043.8.498.74160604730133050784449689952695940558,2\r\n1.2.826.0.1.3680043.8.498.74177736542522869266952610634091628727,5\r\n1.2.826.0.1.3680043.8.498.74206939384612669977629625310277500302,1\r\n1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186,1\r\n1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871,3\r\n1.2.826.0.1.3680043.8.498.74222284584657051737421776919599206132,5\r\n1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604,0\r\n1.2.826.0.1.3680043.8.498.74243110959141459268838217468465880820,2\r\n1.2.826.0.1.3680043.8.498.74243564762473331581882860440560337725,1\r\n1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520,4\r\n1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465,0\r\n1.2.826.0.1.3680043.8.498.74282280145383357786409764912398881272,1\r\n1.2.826.0.1.3680043.8.498.74332475620045145049628676751586693385,4\r\n1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860,1\r\n1.2.826.0.1.3680043.8.498.74388661917632804671669952747214084502,4\r\n1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326,3\r\n1.2.826.0.1.3680043.8.498.74423888576992387013857540016155410962,2\r\n1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661,0\r\n1.2.826.0.1.3680043.8.498.74471396469132211069109289423495141062,1\r\n1.2.826.0.1.3680043.8.498.74471724874857406578316075456414902954,4\r\n1.2.826.0.1.3680043.8.498.74622031396885197580453304563088863451,2\r\n1.2.826.0.1.3680043.8.498.74622912009754642425258484521795653023,1\r\n1.2.826.0.1.3680043.8.498.74688733778107958348746767768064374883,1\r\n1.2.826.0.1.3680043.8.498.74714018385278253151175853998945654373,4\r\n1.2.826.0.1.3680043.8.498.74725806565437665548056410415178280121,4\r\n1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204,0\r\n1.2.826.0.1.3680043.8.498.74855168460956358728708865599401983005,2\r\n1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738,4\r\n1.2.826.0.1.3680043.8.498.74872419726042767543822107291861307866,2\r\n1.2.826.0.1.3680043.8.498.74914646210033032631540802996162814268,1\r\n1.2.826.0.1.3680043.8.498.74939269621045823332837094516056211606,4\r\n1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312,5\r\n1.2.826.0.1.3680043.8.498.74952769952682175467667812958859010329,3\r\n1.2.826.0.1.3680043.8.498.75067227246352035607675978826177487284,3\r\n1.2.826.0.1.3680043.8.498.75126165133796914746114117844697263697,4\r\n1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716,2\r\n1.2.826.0.1.3680043.8.498.75186274888329123179129240384960518861,3\r\n1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309,0\r\n1.2.826.0.1.3680043.8.498.75276563505367629340309025030148269450,2\r\n1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638,1\r\n1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681,5\r\n1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691,2\r\n1.2.826.0.1.3680043.8.498.75418637314688036152637525301534342254,2\r\n1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669,0\r\n1.2.826.0.1.3680043.8.498.75420186662442688955616630735303321846,5\r\n1.2.826.0.1.3680043.8.498.75427432788337288702609576920198810880,1\r\n1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303,3\r\n1.2.826.0.1.3680043.8.498.75522002047190283617659083787085821064,4\r\n1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798,3\r\n1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554,0\r\n1.2.826.0.1.3680043.8.498.75548650589349966221875964542604002971,2\r\n1.2.826.0.1.3680043.8.498.75577846246542841539613557336025271434,3\r\n1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759,1\r\n1.2.826.0.1.3680043.8.498.75627574440860942023321071382766359827,4\r\n1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347,2\r\n1.2.826.0.1.3680043.8.498.75670996863043899245632198549551296292,4\r\n1.2.826.0.1.3680043.8.498.75688484058613633923943882730405145535,5\r\n1.2.826.0.1.3680043.8.498.75739561647115166735277767560308263446,3\r\n1.2.826.0.1.3680043.8.498.75742195884147990442829552992649204078,3\r\n1.2.826.0.1.3680043.8.498.75746520601616488682662195284941583868,2\r\n1.2.826.0.1.3680043.8.498.75751784078102748563815298069799859542,3\r\n1.2.826.0.1.3680043.8.498.75760698509105388883811155432691667447,2\r\n1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864,4\r\n1.2.826.0.1.3680043.8.498.75800418125597325281807386716279941068,4\r\n1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877,3\r\n1.2.826.0.1.3680043.8.498.75924957745868212343758911856809118662,5\r\n1.2.826.0.1.3680043.8.498.75926011938952404257302292495731660728,3\r\n1.2.826.0.1.3680043.8.498.75976388060969166980873144500539735332,2\r\n1.2.826.0.1.3680043.8.498.75980374267002793040832157122311396174,4\r\n1.2.826.0.1.3680043.8.498.76012896432318288623173593280895772876,3\r\n1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386,5\r\n1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088,0\r\n1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523,3\r\n1.2.826.0.1.3680043.8.498.76107330932071966465869847071480517541,2\r\n1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837,3\r\n1.2.826.0.1.3680043.8.498.76125027090596913549224250979076533606,4\r\n1.2.826.0.1.3680043.8.498.76167153566894354739332364376483514559,4\r\n1.2.826.0.1.3680043.8.498.76190704217737835083438638928129775625,0\r\n1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712,4\r\n1.2.826.0.1.3680043.8.498.76242245340157474968574926079920787199,4\r\n1.2.826.0.1.3680043.8.498.76281930374588438789344170829280438697,3\r\n1.2.826.0.1.3680043.8.498.76344826994380951498651448092980303697,0\r\n1.2.826.0.1.3680043.8.498.76350662767360412185091180512006419379,5\r\n1.2.826.0.1.3680043.8.498.76514087677277144682713362609812138150,3\r\n1.2.826.0.1.3680043.8.498.76514504104121634569792846980451896030,4\r\n1.2.826.0.1.3680043.8.498.76525107909962775502207829870873494024,0\r\n1.2.826.0.1.3680043.8.498.76538737802277241976971950690663079376,0\r\n1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120,0\r\n1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992,0\r\n1.2.826.0.1.3680043.8.498.76620177268652814425535081607587091903,4\r\n1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636,1\r\n1.2.826.0.1.3680043.8.498.76662146598924492573212477881527658730,3\r\n1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323,0\r\n1.2.826.0.1.3680043.8.498.76742750470853901575934097885382242952,2\r\n1.2.826.0.1.3680043.8.498.76757251008849492985513280752011229119,1\r\n1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789,0\r\n1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909,0\r\n1.2.826.0.1.3680043.8.498.76862911430418639277808760660745763387,2\r\n1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737,2\r\n1.2.826.0.1.3680043.8.498.76909970028125318280995002129970165723,3\r\n1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939,1\r\n1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254,2\r\n1.2.826.0.1.3680043.8.498.76993922254007307143291254174266755701,5\r\n1.2.826.0.1.3680043.8.498.77018355356797290743788712046815540223,2\r\n1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509,4\r\n1.2.826.0.1.3680043.8.498.77058587552329832558354544874120592826,1\r\n1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412,4\r\n1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536,0\r\n1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695,0\r\n1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855,0\r\n1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733,3\r\n1.2.826.0.1.3680043.8.498.77183368292104509335580564463404842046,1\r\n1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827,1\r\n1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219,1\r\n1.2.826.0.1.3680043.8.498.77261014467268552758826551926128217025,4\r\n1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044,0\r\n1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917,4\r\n1.2.826.0.1.3680043.8.498.77283634197289382453511269950488893300,1\r\n1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136,0\r\n1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485,1\r\n1.2.826.0.1.3680043.8.498.77306755058630855443644908572157134062,2\r\n1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839,0\r\n1.2.826.0.1.3680043.8.498.77370119801633353780677378448761665685,3\r\n1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012,2\r\n1.2.826.0.1.3680043.8.498.77391801789875882109036004511558148658,1\r\n1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512,1\r\n1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809,0\r\n1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402,4\r\n1.2.826.0.1.3680043.8.498.77442647074650043391818472068835112806,2\r\n1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971,4\r\n1.2.826.0.1.3680043.8.498.77508762938574470575737016745357126587,1\r\n1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320,1\r\n1.2.826.0.1.3680043.8.498.77524507552350448792970399203131998592,5\r\n1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471,0\r\n1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713,3\r\n1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727,1\r\n1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146,0\r\n1.2.826.0.1.3680043.8.498.77593859518024366170735307975173961777,4\r\n1.2.826.0.1.3680043.8.498.77611073647079414772859496272387571059,4\r\n1.2.826.0.1.3680043.8.498.77626578713099735143745396709359607445,4\r\n1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176,0\r\n1.2.826.0.1.3680043.8.498.77716260980321491689913730053309892120,4\r\n1.2.826.0.1.3680043.8.498.77735690782343511872544703464260892488,2\r\n1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581,4\r\n1.2.826.0.1.3680043.8.498.77753753872601674668756777373761660873,4\r\n1.2.826.0.1.3680043.8.498.77760453966349959552328150395676117417,4\r\n1.2.826.0.1.3680043.8.498.77800740811302686332282410929816056283,1\r\n1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387,3\r\n1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398,3\r\n1.2.826.0.1.3680043.8.498.77843765997593346714211135369376676497,2\r\n1.2.826.0.1.3680043.8.498.77889800516853576877948593628420228178,0\r\n1.2.826.0.1.3680043.8.498.77893704208186903913532950495036889026,2\r\n1.2.826.0.1.3680043.8.498.77911973519830915742655727368286737079,5\r\n1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742,2\r\n1.2.826.0.1.3680043.8.498.78003548255041391467292087330671945247,3\r\n1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653,3\r\n1.2.826.0.1.3680043.8.498.78072765790191594685273447833272737429,0\r\n1.2.826.0.1.3680043.8.498.78079900750417018394321006978343945239,2\r\n1.2.826.0.1.3680043.8.498.78083057157844664959843364584162587026,2\r\n1.2.826.0.1.3680043.8.498.78084594560404633421210837175859618437,1\r\n1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456,1\r\n1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411,5\r\n1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952,0\r\n1.2.826.0.1.3680043.8.498.78236036694398253902871510456985685293,1\r\n1.2.826.0.1.3680043.8.498.78238394512811614973689011297363399053,4\r\n1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586,2\r\n1.2.826.0.1.3680043.8.498.78286395605671977872377352305890466464,1\r\n1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268,0\r\n1.2.826.0.1.3680043.8.498.78420763928408680447797586163977368190,2\r\n1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960,3\r\n1.2.826.0.1.3680043.8.498.78475145549489837635803968565380247517,1\r\n1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568,2\r\n1.2.826.0.1.3680043.8.498.78521396404736679127326025240279747196,4\r\n1.2.826.0.1.3680043.8.498.78574437345603052437892513165832297960,2\r\n1.2.826.0.1.3680043.8.498.78646473961447523033955716877524794707,2\r\n1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756,0\r\n1.2.826.0.1.3680043.8.498.78667999747216717251638754803697959701,5\r\n1.2.826.0.1.3680043.8.498.78674598286586525734293563394409166190,1\r\n1.2.826.0.1.3680043.8.498.78685661415919658264222803849222055515,2\r\n1.2.826.0.1.3680043.8.498.78757011872917563830995996191850541453,4\r\n1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052,0\r\n1.2.826.0.1.3680043.8.498.78800983923143929298359366572020370031,0\r\n1.2.826.0.1.3680043.8.498.78808320605816811006382053060951793555,4\r\n1.2.826.0.1.3680043.8.498.78812661021494224637586904573732040856,1\r\n1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231,3\r\n1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949,2\r\n1.2.826.0.1.3680043.8.498.78972123731732056524948076388825106061,4\r\n1.2.826.0.1.3680043.8.498.79021499290691125513421477204330532755,3\r\n1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887,1\r\n1.2.826.0.1.3680043.8.498.79077219037094025366585138112899760966,4\r\n1.2.826.0.1.3680043.8.498.79080521163097116147379189510104170611,2\r\n1.2.826.0.1.3680043.8.498.79087227780630716132147615846840128474,3\r\n1.2.826.0.1.3680043.8.498.79117047831806895544174734229992359986,3\r\n1.2.826.0.1.3680043.8.498.79122189963582241618519333445055548347,4\r\n1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815,1\r\n1.2.826.0.1.3680043.8.498.79224785668205420138951103045759609475,3\r\n1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317,0\r\n1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718,2\r\n1.2.826.0.1.3680043.8.498.79247871688929479411597859576284194165,4\r\n1.2.826.0.1.3680043.8.498.79258497507223824227830886693749460348,1\r\n1.2.826.0.1.3680043.8.498.79267465462008228906382361587050826784,2\r\n1.2.826.0.1.3680043.8.498.79274948549030940175536603063980168192,1\r\n1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942,4\r\n1.2.826.0.1.3680043.8.498.79292862882980756453890778996691993351,2\r\n1.2.826.0.1.3680043.8.498.79326627674334637493320764205397279674,0\r\n1.2.826.0.1.3680043.8.498.79341955397030543255969676347554165142,4\r\n1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079,2\r\n1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347,0\r\n1.2.826.0.1.3680043.8.498.79551456190098389081551755482042135791,1\r\n1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803,4\r\n1.2.826.0.1.3680043.8.498.79565393323323007178523850093527250842,5\r\n1.2.826.0.1.3680043.8.498.79582457162604674356422857305405255763,2\r\n1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807,0\r\n1.2.826.0.1.3680043.8.498.79636443202100988025435536780567446582,2\r\n1.2.826.0.1.3680043.8.498.79648517065511897282938523065537768426,2\r\n1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688,2\r\n1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775,0\r\n1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492,0\r\n1.2.826.0.1.3680043.8.498.79798779812052605703861699094268016306,4\r\n1.2.826.0.1.3680043.8.498.79829033430619597320769579608450393290,4\r\n1.2.826.0.1.3680043.8.498.79977436705480715175480363175316375576,5\r\n1.2.826.0.1.3680043.8.498.79991838737793096019496013540529642900,2\r\n1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668,4\r\n1.2.826.0.1.3680043.8.498.80100785345864355815698147663964005920,4\r\n1.2.826.0.1.3680043.8.498.80146250663015309456767479960653638934,3\r\n1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128,4\r\n1.2.826.0.1.3680043.8.498.80180578697901833764761405542790185180,2\r\n1.2.826.0.1.3680043.8.498.80192355832283305029770183988436644845,4\r\n1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187,0\r\n1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721,1\r\n1.2.826.0.1.3680043.8.498.80275294885165896124489476449616436102,3\r\n1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798,0\r\n1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299,4\r\n1.2.826.0.1.3680043.8.498.80292580287849976694993628109721422478,3\r\n1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111,2\r\n1.2.826.0.1.3680043.8.498.80313580693550912024103221258669390487,1\r\n1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146,1\r\n1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774,1\r\n1.2.826.0.1.3680043.8.498.80391378873630377472192017862302619947,3\r\n1.2.826.0.1.3680043.8.498.80433141990021472112645841723991584182,4\r\n1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268,4\r\n1.2.826.0.1.3680043.8.498.80492324225032367918842505447056622561,3\r\n1.2.826.0.1.3680043.8.498.80514754572353458107286859773300865138,1\r\n1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286,0\r\n1.2.826.0.1.3680043.8.498.80557835452734362083552169469427388003,2\r\n1.2.826.0.1.3680043.8.498.80589796918823112368758999678369292754,4\r\n1.2.826.0.1.3680043.8.498.80612964209397056653334784776523875647,3\r\n1.2.826.0.1.3680043.8.498.80627605463642274852766102874822833365,3\r\n1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307,3\r\n1.2.826.0.1.3680043.8.498.80642348849062865125509773837278900750,1\r\n1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925,1\r\n1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506,0\r\n1.2.826.0.1.3680043.8.498.80661392139668653552110941896475066204,4\r\n1.2.826.0.1.3680043.8.498.80671597537348112977780954910198682272,2\r\n1.2.826.0.1.3680043.8.498.80698510014594772882040509716764819716,3\r\n1.2.826.0.1.3680043.8.498.80726797831040227285328789648680519664,4\r\n1.2.826.0.1.3680043.8.498.80735285867775404153100236162962472952,1\r\n1.2.826.0.1.3680043.8.498.80741358365199721720335324482078866565,1\r\n1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704,3\r\n1.2.826.0.1.3680043.8.498.80851365368114686105558347899990395309,2\r\n1.2.826.0.1.3680043.8.498.80861108320341696163987804408524915999,4\r\n1.2.826.0.1.3680043.8.498.80876235930771882271600826613593202364,5\r\n1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715,0\r\n1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834,1\r\n1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417,0\r\n1.2.826.0.1.3680043.8.498.81012548926053584036386949658594985125,1\r\n1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057,3\r\n1.2.826.0.1.3680043.8.498.81024897463656728358703040759074583549,3\r\n1.2.826.0.1.3680043.8.498.81031057654953109295788159826841315360,5\r\n1.2.826.0.1.3680043.8.498.81096597169632636454986988944050298701,5\r\n1.2.826.0.1.3680043.8.498.81103809943451790668501089584752866199,3\r\n1.2.826.0.1.3680043.8.498.81109405367395078564887780335570099673,0\r\n1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924,3\r\n1.2.826.0.1.3680043.8.498.81248820934296461385324385812581970340,3\r\n1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859,0\r\n1.2.826.0.1.3680043.8.498.81327951282145199298572980264557740321,2\r\n1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359,2\r\n1.2.826.0.1.3680043.8.498.81347841116019279082226143308588516787,2\r\n1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694,2\r\n1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421,0\r\n1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669,0\r\n1.2.826.0.1.3680043.8.498.81464322239381998685473520085495957135,2\r\n1.2.826.0.1.3680043.8.498.81466575994184158172641823163654418018,1\r\n1.2.826.0.1.3680043.8.498.81475058647553697318837239968139077272,5\r\n1.2.826.0.1.3680043.8.498.81495303185504163367736663647489556517,4\r\n1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563,5\r\n1.2.826.0.1.3680043.8.498.81567030584250750745410094870790052181,2\r\n1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166,1\r\n1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233,4\r\n1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086,0\r\n1.2.826.0.1.3680043.8.498.81651132686162360431299931833372141617,1\r\n1.2.826.0.1.3680043.8.498.81651455914141365028015556083080638757,4\r\n1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472,4\r\n1.2.826.0.1.3680043.8.498.81698411330214788362326470979160431674,3\r\n1.2.826.0.1.3680043.8.498.81738571920712657790861120957086010772,3\r\n1.2.826.0.1.3680043.8.498.81740348974208459068937862799489495782,2\r\n1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255,4\r\n1.2.826.0.1.3680043.8.498.81793454171031578492222004629265937656,2\r\n1.2.826.0.1.3680043.8.498.81796385087779065457668769443782458844,2\r\n1.2.826.0.1.3680043.8.498.81797437102217585216278133168092577952,4\r\n1.2.826.0.1.3680043.8.498.81819209302731920404237374320026807282,2\r\n1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661,1\r\n1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236,4\r\n1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167,4\r\n1.2.826.0.1.3680043.8.498.81927409938015158663782061022054347955,3\r\n1.2.826.0.1.3680043.8.498.81957447654112577010261210572986132211,4\r\n1.2.826.0.1.3680043.8.498.81964000393640770822199916465038759069,2\r\n1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277,2\r\n1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172,3\r\n1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894,0\r\n1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678,3\r\n1.2.826.0.1.3680043.8.498.82053853036886620816821481139413594407,4\r\n1.2.826.0.1.3680043.8.498.82074883987134049971242727164506046476,1\r\n1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508,0\r\n1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426,1\r\n1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974,0\r\n1.2.826.0.1.3680043.8.498.82169173362972793731001247413129649263,2\r\n1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181,3\r\n1.2.826.0.1.3680043.8.498.82355456395534835058857499033494900716,4\r\n1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746,0\r\n1.2.826.0.1.3680043.8.498.82440268452598207215458064865523988222,2\r\n1.2.826.0.1.3680043.8.498.82447100001417065190678617013741085481,4\r\n1.2.826.0.1.3680043.8.498.82481056562260046738834822905709207128,3\r\n1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874,3\r\n1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051,5\r\n1.2.826.0.1.3680043.8.498.82548251499640077572026404183402934330,5\r\n1.2.826.0.1.3680043.8.498.82548952153746644951736669119607083230,2\r\n1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162,5\r\n1.2.826.0.1.3680043.8.498.82582558624896662901163827145277490232,2\r\n1.2.826.0.1.3680043.8.498.82587763533601760002619014353812074700,0\r\n1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134,5\r\n1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714,2\r\n1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914,1\r\n1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538,1\r\n1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758,0\r\n1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956,1\r\n1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400,0\r\n1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925,2\r\n1.2.826.0.1.3680043.8.498.82967086493280898295669930903240677729,4\r\n1.2.826.0.1.3680043.8.498.83000639623679673469640458882929412717,1\r\n1.2.826.0.1.3680043.8.498.83028939839767027342482365275441078777,4\r\n1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966,4\r\n1.2.826.0.1.3680043.8.498.83054899303871276184985881638619543362,1\r\n1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783,1\r\n1.2.826.0.1.3680043.8.498.83061811791652434833797599835671351097,4\r\n1.2.826.0.1.3680043.8.498.83108474538388106770920972885900596508,4\r\n1.2.826.0.1.3680043.8.498.83121378128521985713113311344685112825,1\r\n1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239,0\r\n1.2.826.0.1.3680043.8.498.83220896521302064187926748035734200732,3\r\n1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332,3\r\n1.2.826.0.1.3680043.8.498.83237706274929081175032033288272528919,5\r\n1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790,1\r\n1.2.826.0.1.3680043.8.498.83271280040777227673434567660749430744,4\r\n1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638,2\r\n1.2.826.0.1.3680043.8.498.83384018761005700404957421316476952190,1\r\n1.2.826.0.1.3680043.8.498.83456649088980869429062498344457162631,5\r\n1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936,0\r\n1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602,0\r\n1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986,0\r\n1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462,4\r\n1.2.826.0.1.3680043.8.498.83602205818477323486169938182117122488,3\r\n1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220,4\r\n1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057,0\r\n1.2.826.0.1.3680043.8.498.83747418172021065605877274971789603771,1\r\n1.2.826.0.1.3680043.8.498.83847285247209768917850605454140194088,4\r\n1.2.826.0.1.3680043.8.498.83956129905762530953192730040209719910,2\r\n1.2.826.0.1.3680043.8.498.84007436672323587547274248257117331532,2\r\n1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926,4\r\n1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783,0\r\n1.2.826.0.1.3680043.8.498.84062602050757262792437893691591415560,1\r\n1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220,1\r\n1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399,2\r\n1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488,0\r\n1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819,2\r\n1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724,0\r\n1.2.826.0.1.3680043.8.498.84292569246911697141902796190329671261,5\r\n1.2.826.0.1.3680043.8.498.84347051904829814300575579670995018991,1\r\n1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259,2\r\n1.2.826.0.1.3680043.8.498.84477198486144701928775090913243250655,3\r\n1.2.826.0.1.3680043.8.498.84483080389344581106129884869678629612,1\r\n1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475,2\r\n1.2.826.0.1.3680043.8.498.84612593835203397696029057940345763682,4\r\n1.2.826.0.1.3680043.8.498.84630493975910180537831353884518764364,3\r\n1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652,0\r\n1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195,0\r\n1.2.826.0.1.3680043.8.498.84709539128629951937294117586370340042,5\r\n1.2.826.0.1.3680043.8.498.84720476187201404764508451536959273676,4\r\n1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482,0\r\n1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132,5\r\n1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518,0\r\n1.2.826.0.1.3680043.8.498.84857368518703570476541203284655412396,1\r\n1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066,0\r\n1.2.826.0.1.3680043.8.498.84982310626395569925821771375535259394,1\r\n1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547,0\r\n1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548,1\r\n1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798,5\r\n1.2.826.0.1.3680043.8.498.85095389649021071892478435298763928925,3\r\n1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538,4\r\n1.2.826.0.1.3680043.8.498.85137704768728261642637391657270006448,4\r\n1.2.826.0.1.3680043.8.498.85194002080793317087210934629814349622,4\r\n1.2.826.0.1.3680043.8.498.85212353344613461454490252928274988058,4\r\n1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921,0\r\n1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047,4\r\n1.2.826.0.1.3680043.8.498.85333821516157983299252782678788406019,1\r\n1.2.826.0.1.3680043.8.498.85341019006931436802294963713199308935,4\r\n1.2.826.0.1.3680043.8.498.85347263026430684291021533170619978114,1\r\n1.2.826.0.1.3680043.8.498.85391052076760686301971262244561025345,2\r\n1.2.826.0.1.3680043.8.498.85397267100597758920280635204297906286,4\r\n1.2.826.0.1.3680043.8.498.85416107585703628549041626609272707431,2\r\n1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892,0\r\n1.2.826.0.1.3680043.8.498.85454884397223412543050876117416146750,4\r\n1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327,1\r\n1.2.826.0.1.3680043.8.498.85475892602877115812059264719695919613,2\r\n1.2.826.0.1.3680043.8.498.85499036441349866810703945289793057098,3\r\n1.2.826.0.1.3680043.8.498.85545102350084131396266710667808696561,5\r\n1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590,3\r\n1.2.826.0.1.3680043.8.498.85602291032341007550376696695283794665,5\r\n1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762,4\r\n1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533,0\r\n1.2.826.0.1.3680043.8.498.85649172620970194090967653766415467836,4\r\n1.2.826.0.1.3680043.8.498.85663162480758065951465617999516770647,4\r\n1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647,2\r\n1.2.826.0.1.3680043.8.498.85715956158218184426400151797364834203,1\r\n1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221,1\r\n1.2.826.0.1.3680043.8.498.85728170022176096748897349447671687164,3\r\n1.2.826.0.1.3680043.8.498.85735729179216019275685564112535593988,1\r\n1.2.826.0.1.3680043.8.498.85748882309364598407865143890978767674,2\r\n1.2.826.0.1.3680043.8.498.85760360279564672425828346507309104394,3\r\n1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501,0\r\n1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800,1\r\n1.2.826.0.1.3680043.8.498.85879572979834912642178551098379145607,4\r\n1.2.826.0.1.3680043.8.498.85882078780717905717989373451578844863,5\r\n1.2.826.0.1.3680043.8.498.85883562680953255833567595509408888410,3\r\n1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290,5\r\n1.2.826.0.1.3680043.8.498.85932495800565018166769303205378781707,2\r\n1.2.826.0.1.3680043.8.498.85943737587102032370951456635167782343,2\r\n1.2.826.0.1.3680043.8.498.85970222049966384156120176704985355996,4\r\n1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055,1\r\n1.2.826.0.1.3680043.8.498.85992037603186775437749979929724514304,4\r\n1.2.826.0.1.3680043.8.498.85992651772359515416050249463715355943,2\r\n1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091,0\r\n1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827,2\r\n1.2.826.0.1.3680043.8.498.86042942187650835661744645315369706668,4\r\n1.2.826.0.1.3680043.8.498.86067849233330188988240710585112183308,4\r\n1.2.826.0.1.3680043.8.498.86082522186055358984181640226529351801,4\r\n1.2.826.0.1.3680043.8.498.86117639434986594266345773277419881144,2\r\n1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915,1\r\n1.2.826.0.1.3680043.8.498.86216155858324761845740646421500394797,4\r\n1.2.826.0.1.3680043.8.498.86239793008693746155478763153154040128,0\r\n1.2.826.0.1.3680043.8.498.86251314013737658351649446042801810374,3\r\n1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972,3\r\n1.2.826.0.1.3680043.8.498.86273853115722509980053190345047859057,4\r\n1.2.826.0.1.3680043.8.498.86280033511814602594654628677479528508,1\r\n1.2.826.0.1.3680043.8.498.86285715293767810771609523819042157072,3\r\n1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373,2\r\n1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889,0\r\n1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601,0\r\n1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655,4\r\n1.2.826.0.1.3680043.8.498.86405969333062807387894608315897106350,4\r\n1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391,2\r\n1.2.826.0.1.3680043.8.498.86416984885007871128151024515780322767,4\r\n1.2.826.0.1.3680043.8.498.86422343639522250286002195622435564347,3\r\n1.2.826.0.1.3680043.8.498.86435271359723045834079326733478538148,4\r\n1.2.826.0.1.3680043.8.498.86435890312227285216810145658548164773,4\r\n1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543,4\r\n1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133,0\r\n1.2.826.0.1.3680043.8.498.86566584776908143877603164704838586097,3\r\n1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251,0\r\n1.2.826.0.1.3680043.8.498.86625876521226465945114246972542881955,1\r\n1.2.826.0.1.3680043.8.498.86651345701812229172985086531423369421,1\r\n1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623,1\r\n1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291,4\r\n1.2.826.0.1.3680043.8.498.86876540394528304678398111135600894869,2\r\n1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831,0\r\n1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409,3\r\n1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930,0\r\n1.2.826.0.1.3680043.8.498.87018440624359538250658207994111333051,3\r\n1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937,3\r\n1.2.826.0.1.3680043.8.498.87064372531225943854092598173945043861,4\r\n1.2.826.0.1.3680043.8.498.87101276883427329522365304528449003589,3\r\n1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520,0\r\n1.2.826.0.1.3680043.8.498.87185817772180109135714642184874643715,2\r\n1.2.826.0.1.3680043.8.498.87198191797585172912979594638355685739,3\r\n1.2.826.0.1.3680043.8.498.87208714421237491643915928765740747633,4\r\n1.2.826.0.1.3680043.8.498.87214831397290763080243320084625847148,2\r\n1.2.826.0.1.3680043.8.498.87222653734797522858506220026427900177,4\r\n1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105,0\r\n1.2.826.0.1.3680043.8.498.87241650441242408799533998101943024283,5\r\n1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751,2\r\n1.2.826.0.1.3680043.8.498.87262205711949358943515326577886508585,1\r\n1.2.826.0.1.3680043.8.498.87267658933515453963224923798100651549,2\r\n1.2.826.0.1.3680043.8.498.87338030934656808032044071206920470285,2\r\n1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171,0\r\n1.2.826.0.1.3680043.8.498.87413924318793242918916384453755054267,4\r\n1.2.826.0.1.3680043.8.498.87415625824966124550466651648244553892,0\r\n1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155,1\r\n1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927,0\r\n1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757,0\r\n1.2.826.0.1.3680043.8.498.87550037046735743504446944390018580426,1\r\n1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868,3\r\n1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518,4\r\n1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188,4\r\n1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110,4\r\n1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015,5\r\n1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348,0\r\n1.2.826.0.1.3680043.8.498.87742737485142445536023876862256284839,2\r\n1.2.826.0.1.3680043.8.498.87761573479019057997334508266888503487,3\r\n1.2.826.0.1.3680043.8.498.87769640167791104899516835133527936885,1\r\n1.2.826.0.1.3680043.8.498.87797202230328868466634480198668365648,4\r\n1.2.826.0.1.3680043.8.498.87799106106199445436621174506197684035,3\r\n1.2.826.0.1.3680043.8.498.87845297884497964690686366000533570840,3\r\n1.2.826.0.1.3680043.8.498.87876800169766592603600598167358214111,3\r\n1.2.826.0.1.3680043.8.498.87900330305833722536121052961334699743,3\r\n1.2.826.0.1.3680043.8.498.87998832540756872371878613339628615880,2\r\n1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523,4\r\n1.2.826.0.1.3680043.8.498.88064086448060280325724227239585797761,3\r\n1.2.826.0.1.3680043.8.498.88070896604159829163161019117970440189,3\r\n1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905,1\r\n1.2.826.0.1.3680043.8.498.88114918358665488269761388157480523350,5\r\n1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689,0\r\n1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685,0\r\n1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894,4\r\n1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615,0\r\n1.2.826.0.1.3680043.8.498.88439732365857773633852840383740478214,4\r\n1.2.826.0.1.3680043.8.498.88457076333783710706782784856816112502,4\r\n1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466,1\r\n1.2.826.0.1.3680043.8.498.88480827968231439729532202780922145132,2\r\n1.2.826.0.1.3680043.8.498.88505044370271842163477475980178015566,4\r\n1.2.826.0.1.3680043.8.498.88523737187772411911621088333701306485,4\r\n1.2.826.0.1.3680043.8.498.88565389747258713872180859306907392080,4\r\n1.2.826.0.1.3680043.8.498.88570871026022705696415758052703394453,5\r\n1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560,4\r\n1.2.826.0.1.3680043.8.498.88574487918084130981335189023288704909,2\r\n1.2.826.0.1.3680043.8.498.88609906210780568637209291182572648571,4\r\n1.2.826.0.1.3680043.8.498.88620225126852321191690360547272730103,2\r\n1.2.826.0.1.3680043.8.498.88653085234639096911379842849193874077,1\r\n1.2.826.0.1.3680043.8.498.88655485247489827200671636011965281184,2\r\n1.2.826.0.1.3680043.8.498.88679328523895187500714090963655363787,4\r\n1.2.826.0.1.3680043.8.498.88682169866098509464628948503453852928,2\r\n1.2.826.0.1.3680043.8.498.88687994744258163246685527273559741656,3\r\n1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015,0\r\n1.2.826.0.1.3680043.8.498.88726804476694411363564819498296119763,2\r\n1.2.826.0.1.3680043.8.498.88752039058074948027322975455607258941,3\r\n1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997,1\r\n1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212,0\r\n1.2.826.0.1.3680043.8.498.88827276474694165071267273766912871459,4\r\n1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328,2\r\n1.2.826.0.1.3680043.8.498.88872155446935518174304990308380906112,4\r\n1.2.826.0.1.3680043.8.498.88874647535130028956725641820977918615,2\r\n1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177,1\r\n1.2.826.0.1.3680043.8.498.88899519473612920868099376939780201067,2\r\n1.2.826.0.1.3680043.8.498.88962401400598535739494671683916272476,1\r\n1.2.826.0.1.3680043.8.498.88964371740619947193957393682095448256,1\r\n1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800,2\r\n1.2.826.0.1.3680043.8.498.89045629523485194086986785222960864883,5\r\n1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366,3\r\n1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294,0\r\n1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908,4\r\n1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116,0\r\n1.2.826.0.1.3680043.8.498.89190511197564127079697151213577962649,3\r\n1.2.826.0.1.3680043.8.498.89204062904097086723738217311755833877,2\r\n1.2.826.0.1.3680043.8.498.89240184178186399632216878563608101260,3\r\n1.2.826.0.1.3680043.8.498.89247482662611898098489676481780144186,3\r\n1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428,1\r\n1.2.826.0.1.3680043.8.498.89307235192406983202868051697765144920,3\r\n1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530,3\r\n1.2.826.0.1.3680043.8.498.89332108074590559691047812196991368746,2\r\n1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946,1\r\n1.2.826.0.1.3680043.8.498.89350648596674998203116057439419132072,2\r\n1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030,0\r\n1.2.826.0.1.3680043.8.498.89416235692252764852783739140428958560,1\r\n1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497,2\r\n1.2.826.0.1.3680043.8.498.89445342796957655182687412965056270903,2\r\n1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244,0\r\n1.2.826.0.1.3680043.8.498.89569762088162031469299314053759758927,1\r\n1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334,1\r\n1.2.826.0.1.3680043.8.498.89584627944947813918297925813120247226,2\r\n1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646,0\r\n1.2.826.0.1.3680043.8.498.89720428711655200442262503508409163392,0\r\n1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395,0\r\n1.2.826.0.1.3680043.8.498.89746004948304570186940283501211828475,4\r\n1.2.826.0.1.3680043.8.498.89829843399218627892046362324057240535,1\r\n1.2.826.0.1.3680043.8.498.89842390998700046088751402253903762827,3\r\n1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585,4\r\n1.2.826.0.1.3680043.8.498.89935384074170571164295050203650829722,3\r\n1.2.826.0.1.3680043.8.498.89943815722422273305517145510644432987,3\r\n1.2.826.0.1.3680043.8.498.89953731420879076053772747816096629231,5\r\n1.2.826.0.1.3680043.8.498.89967772422936726289461201791871771011,2\r\n1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712,0\r\n1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280,4\r\n1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032,4\r\n1.2.826.0.1.3680043.8.498.90057130466053377217652879986591534745,3\r\n1.2.826.0.1.3680043.8.498.90103210546658057701715546323552886913,1\r\n1.2.826.0.1.3680043.8.498.90159195278632080234412316065469499332,1\r\n1.2.826.0.1.3680043.8.498.90170179734833731758923964833173536172,1\r\n1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947,3\r\n1.2.826.0.1.3680043.8.498.90193654806390401129887447441469319421,1\r\n1.2.826.0.1.3680043.8.498.90206716031776606956011363950625953310,2\r\n1.2.826.0.1.3680043.8.498.90223492515368389331845681759355261030,5\r\n1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704,1\r\n1.2.826.0.1.3680043.8.498.90251533154603802045666102095577144628,1\r\n1.2.826.0.1.3680043.8.498.90264300606047579088362539636050224478,1\r\n1.2.826.0.1.3680043.8.498.90268709354762000249148795667759483876,0\r\n1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634,0\r\n1.2.826.0.1.3680043.8.498.90288598546566291188788375261370919222,2\r\n1.2.826.0.1.3680043.8.498.90299615112155255774026824400461236845,5\r\n1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942,0\r\n1.2.826.0.1.3680043.8.498.90331082101665002495739626395247831768,1\r\n1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164,4\r\n1.2.826.0.1.3680043.8.498.90342120331229511020327052213821820086,2\r\n1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056,2\r\n1.2.826.0.1.3680043.8.498.90446842983860370183814559926724769347,2\r\n1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010,3\r\n1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272,0\r\n1.2.826.0.1.3680043.8.498.90490891782778285565462083536468192410,2\r\n1.2.826.0.1.3680043.8.498.90508461713737190527021531642732695353,1\r\n1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293,0\r\n1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085,1\r\n1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781,3\r\n1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080,0\r\n1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680,2\r\n1.2.826.0.1.3680043.8.498.90679199017361780574384378857945322240,4\r\n1.2.826.0.1.3680043.8.498.90751504670572401790063555695835439317,2\r\n1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839,3\r\n1.2.826.0.1.3680043.8.498.90764713690393618577990540005126771712,2\r\n1.2.826.0.1.3680043.8.498.90771070773751767226565553195436971014,1\r\n1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979,0\r\n1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506,3\r\n1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296,4\r\n1.2.826.0.1.3680043.8.498.90985037792656682958281230441654772668,5\r\n1.2.826.0.1.3680043.8.498.91064647744940506223536204418057554573,1\r\n1.2.826.0.1.3680043.8.498.91106275065657355304239097295699332135,2\r\n1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698,4\r\n1.2.826.0.1.3680043.8.498.91124979914013108948829704350601987380,4\r\n1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089,1\r\n1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454,0\r\n1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645,5\r\n1.2.826.0.1.3680043.8.498.91206973544724011503639050238118945528,1\r\n1.2.826.0.1.3680043.8.498.91220255464004258635083940797356822178,3\r\n1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811,1\r\n1.2.826.0.1.3680043.8.498.91237082716839283641165609666644466929,4\r\n1.2.826.0.1.3680043.8.498.91262503278090288446189575519187653125,3\r\n1.2.826.0.1.3680043.8.498.91323765268136520804399328350809179765,3\r\n1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089,0\r\n1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394,1\r\n1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365,0\r\n1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984,1\r\n1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042,3\r\n1.2.826.0.1.3680043.8.498.91391446167553025113367459399831345295,3\r\n1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785,0\r\n1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637,2\r\n1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999,2\r\n1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579,4\r\n1.2.826.0.1.3680043.8.498.91581717063042044543371480238854954692,1\r\n1.2.826.0.1.3680043.8.498.91606395836371290680039618164703651751,2\r\n1.2.826.0.1.3680043.8.498.91611403539216247154562810076063878924,1\r\n1.2.826.0.1.3680043.8.498.91660278231317256353374962767408532443,2\r\n1.2.826.0.1.3680043.8.498.91686669332744972709173289374472094988,4\r\n1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037,3\r\n1.2.826.0.1.3680043.8.498.91742054227008690570256869200116133922,4\r\n1.2.826.0.1.3680043.8.498.91744384920314309659802891351055961910,2\r\n1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273,0\r\n1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593,1\r\n1.2.826.0.1.3680043.8.498.91883829404782805058767342767479130421,0\r\n1.2.826.0.1.3680043.8.498.91897419008563027309752122760214275182,4\r\n1.2.826.0.1.3680043.8.498.91910585599088493175245713843727208369,4\r\n1.2.826.0.1.3680043.8.498.91913894091933783677568170210064807665,2\r\n1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395,2\r\n1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016,0\r\n1.2.826.0.1.3680043.8.498.91991985280680856830060431668414014465,4\r\n1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069,0\r\n1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745,1\r\n1.2.826.0.1.3680043.8.498.92071199603599654827155753204585082303,4\r\n1.2.826.0.1.3680043.8.498.92126358259118823397932994812520542494,3\r\n1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025,0\r\n1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675,0\r\n1.2.826.0.1.3680043.8.498.92249560662718438332916289759028859881,2\r\n1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671,1\r\n1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164,4\r\n1.2.826.0.1.3680043.8.498.92326467051364762529119362944559591665,4\r\n1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728,4\r\n1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154,3\r\n1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745,1\r\n1.2.826.0.1.3680043.8.498.92445779773398371452908052305534240654,3\r\n1.2.826.0.1.3680043.8.498.92449204325697143336618138322280397765,1\r\n1.2.826.0.1.3680043.8.498.92455257597708674644536154732642099964,3\r\n1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506,2\r\n1.2.826.0.1.3680043.8.498.92597065276819676534178545455668171085,2\r\n1.2.826.0.1.3680043.8.498.92602515666185999130675702469977883616,1\r\n1.2.826.0.1.3680043.8.498.92621720172216795135762712303124932801,3\r\n1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095,0\r\n1.2.826.0.1.3680043.8.498.92650426727723557280117118587545413917,4\r\n1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330,2\r\n1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780,0\r\n1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433,0\r\n1.2.826.0.1.3680043.8.498.92722393862174380736027353110424417588,2\r\n1.2.826.0.1.3680043.8.498.92758635328468215409701857842011454608,4\r\n1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401,3\r\n1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078,0\r\n1.2.826.0.1.3680043.8.498.92902692647544828045768164058889405369,3\r\n1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055,1\r\n1.2.826.0.1.3680043.8.498.93026010065162534828966791702007067882,4\r\n1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305,0\r\n1.2.826.0.1.3680043.8.498.93062749017302143105207135454940053142,5\r\n1.2.826.0.1.3680043.8.498.93096918345463865518663434763610016449,1\r\n1.2.826.0.1.3680043.8.498.93104685852024334015651558514195715956,4\r\n1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014,3\r\n1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945,0\r\n1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677,0\r\n1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053,3\r\n1.2.826.0.1.3680043.8.498.93222021909191248270818335069738080963,1\r\n1.2.826.0.1.3680043.8.498.93230313557251171119475173996942942855,2\r\n1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032,3\r\n1.2.826.0.1.3680043.8.498.93318293884671598055223778443354732793,3\r\n1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706,3\r\n1.2.826.0.1.3680043.8.498.93359387652821116106937882388838959209,1\r\n1.2.826.0.1.3680043.8.498.93385433312281352893457225415812880948,2\r\n1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679,0\r\n1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151,0\r\n1.2.826.0.1.3680043.8.498.93623421864452751381778712192879121271,2\r\n1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233,4\r\n1.2.826.0.1.3680043.8.498.93655619137163635289231840294252859840,2\r\n1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068,0\r\n1.2.826.0.1.3680043.8.498.93719148934772792523193842156217224487,4\r\n1.2.826.0.1.3680043.8.498.93722352425168280299901366261374293722,1\r\n1.2.826.0.1.3680043.8.498.93728117237268000947508816107096388086,1\r\n1.2.826.0.1.3680043.8.498.93747304211180517979657679136059719120,1\r\n1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680,0\r\n1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418,0\r\n1.2.826.0.1.3680043.8.498.93902628119199416703899320994283313812,0\r\n1.2.826.0.1.3680043.8.498.93921993164883084178408030385007465064,3\r\n1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308,0\r\n1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838,4\r\n1.2.826.0.1.3680043.8.498.94002633501154835018549300667479350401,4\r\n1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303,2\r\n1.2.826.0.1.3680043.8.498.94030816415316308210133175314873817171,3\r\n1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612,0\r\n1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693,0\r\n1.2.826.0.1.3680043.8.498.94244885518758327449236878653096161475,1\r\n1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075,4\r\n1.2.826.0.1.3680043.8.498.94255124739558547668304022649099251744,1\r\n1.2.826.0.1.3680043.8.498.94274898137310893945218731717226819541,5\r\n1.2.826.0.1.3680043.8.498.94296943940827583101208235208463322140,1\r\n1.2.826.0.1.3680043.8.498.94328216642444410827846344286227627425,5\r\n1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083,4\r\n1.2.826.0.1.3680043.8.498.94381282809153270156987123429739805206,4\r\n1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606,3\r\n1.2.826.0.1.3680043.8.498.94447683257072169359896104855962933413,0\r\n1.2.826.0.1.3680043.8.498.94448720962054029575571654250994769402,2\r\n1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684,1\r\n1.2.826.0.1.3680043.8.498.94590838906158914501563687877014778914,2\r\n1.2.826.0.1.3680043.8.498.94594009911808653224418094594012527944,1\r\n1.2.826.0.1.3680043.8.498.94601583997736688762302974573903982304,2\r\n1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480,3\r\n1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516,3\r\n1.2.826.0.1.3680043.8.498.94635110520016467882028406590905441740,4\r\n1.2.826.0.1.3680043.8.498.94658317716039594639343437780115140531,5\r\n1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618,1\r\n1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046,4\r\n1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175,0\r\n1.2.826.0.1.3680043.8.498.94747403767627336827372096100476526821,4\r\n1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449,3\r\n1.2.826.0.1.3680043.8.498.94761755148413531477206243804832613511,2\r\n1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589,0\r\n1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472,0\r\n1.2.826.0.1.3680043.8.498.94822220672465914426995315295921716764,4\r\n1.2.826.0.1.3680043.8.498.94921929374845793599006362190119159537,5\r\n1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442,0\r\n1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325,3\r\n1.2.826.0.1.3680043.8.498.94954489705497211843190734684710878341,4\r\n1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224,0\r\n1.2.826.0.1.3680043.8.498.95026524886584356226576936309513335214,0\r\n1.2.826.0.1.3680043.8.498.95030266788619257057351879256576338162,2\r\n1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745,1\r\n1.2.826.0.1.3680043.8.498.95080449640993916851075185393444497308,4\r\n1.2.826.0.1.3680043.8.498.95091624999382167943525022414631622532,4\r\n1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982,0\r\n1.2.826.0.1.3680043.8.498.95142886737239438773421088967802136318,1\r\n1.2.826.0.1.3680043.8.498.95193282490279394602564266574551123732,4\r\n1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380,0\r\n1.2.826.0.1.3680043.8.498.95223549428749744711279985897241419512,1\r\n1.2.826.0.1.3680043.8.498.95256518245659249144569318023469597946,3\r\n1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139,0\r\n1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835,2\r\n1.2.826.0.1.3680043.8.498.95324318295506605557933083155792587297,4\r\n1.2.826.0.1.3680043.8.498.95334149193711661424105715330320414755,3\r\n1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963,1\r\n1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109,0\r\n1.2.826.0.1.3680043.8.498.95433650759419050864053270236035947981,2\r\n1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815,3\r\n1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657,0\r\n1.2.826.0.1.3680043.8.498.95497572581157858069692837655072209020,1\r\n1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948,0\r\n1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885,1\r\n1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310,0\r\n1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375,3\r\n1.2.826.0.1.3680043.8.498.95586477246200285088746818821808017494,4\r\n1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863,5\r\n1.2.826.0.1.3680043.8.498.95601371640669355936785637162110963091,1\r\n1.2.826.0.1.3680043.8.498.95621693348567815920675804584086926261,3\r\n1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698,5\r\n1.2.826.0.1.3680043.8.498.95653228095828585867177997067051877351,3\r\n1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180,0\r\n1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522,1\r\n1.2.826.0.1.3680043.8.498.95720534093121754146413060811354489247,2\r\n1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460,3\r\n1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450,4\r\n1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603,0\r\n1.2.826.0.1.3680043.8.498.95796009319232592137564145023096016872,4\r\n1.2.826.0.1.3680043.8.498.95850844945129153291759207212815927078,3\r\n1.2.826.0.1.3680043.8.498.95864678405973312042815189771902289608,1\r\n1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951,3\r\n1.2.826.0.1.3680043.8.498.95912798959818676102590029534562878721,3\r\n1.2.826.0.1.3680043.8.498.95918254432056542701730614049432316728,2\r\n1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466,0\r\n1.2.826.0.1.3680043.8.498.96051765028927322365509303222455708654,4\r\n1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601,4\r\n1.2.826.0.1.3680043.8.498.96094001324999448553107805455529187871,1\r\n1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716,0\r\n1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916,4\r\n1.2.826.0.1.3680043.8.498.96152894373838410779390571000523933753,1\r\n1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199,4\r\n1.2.826.0.1.3680043.8.498.96174494699876332875347380499122120925,3\r\n1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428,0\r\n1.2.826.0.1.3680043.8.498.96316229168827165128604998083731481211,3\r\n1.2.826.0.1.3680043.8.498.96323485839061204012967074568679195142,4\r\n1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674,3\r\n1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420,3\r\n1.2.826.0.1.3680043.8.498.96483513352751590579274577125356217169,4\r\n1.2.826.0.1.3680043.8.498.96510621665552630060821422211995212854,2\r\n1.2.826.0.1.3680043.8.498.96559819562317669526803257835398470484,3\r\n1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946,1\r\n1.2.826.0.1.3680043.8.498.96562865861483086790615358719391771217,2\r\n1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801,1\r\n1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083,0\r\n1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570,0\r\n1.2.826.0.1.3680043.8.498.96621557839848412555023112236627663339,4\r\n1.2.826.0.1.3680043.8.498.96657465815875587552665111166265086308,2\r\n1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186,0\r\n1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926,0\r\n1.2.826.0.1.3680043.8.498.96846796615932368166905347054245013501,4\r\n1.2.826.0.1.3680043.8.498.96859317329616917022799512740280754327,4\r\n1.2.826.0.1.3680043.8.498.96943946564115188893954628924857155670,3\r\n1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742,4\r\n1.2.826.0.1.3680043.8.498.96947237995559790704304892745226362883,4\r\n1.2.826.0.1.3680043.8.498.96950011369238410949579095568317214261,2\r\n1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623,4\r\n1.2.826.0.1.3680043.8.498.97004483591331502283638578516018997228,4\r\n1.2.826.0.1.3680043.8.498.97020520267554584684160936968569132254,3\r\n1.2.826.0.1.3680043.8.498.97042446513400958172996637321308032556,1\r\n1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644,3\r\n1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523,4\r\n1.2.826.0.1.3680043.8.498.97107472223824971578324157203273471706,3\r\n1.2.826.0.1.3680043.8.498.97135552871905937467775630206570358085,4\r\n1.2.826.0.1.3680043.8.498.97156518925820521351080132062400795929,2\r\n1.2.826.0.1.3680043.8.498.97220604868276521004168045790587997687,5\r\n1.2.826.0.1.3680043.8.498.97234348688223610934999923021953923903,2\r\n1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902,2\r\n1.2.826.0.1.3680043.8.498.97259766233250760343109933426776941754,3\r\n1.2.826.0.1.3680043.8.498.97259977321685658337087695233264517545,4\r\n1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442,0\r\n1.2.826.0.1.3680043.8.498.97274847570547462056218828853253311223,1\r\n1.2.826.0.1.3680043.8.498.97296323335289879687414244774227160450,1\r\n1.2.826.0.1.3680043.8.498.97303893645051596044169857585815133097,5\r\n1.2.826.0.1.3680043.8.498.97348112008590316355392924312704081684,5\r\n1.2.826.0.1.3680043.8.498.97353980946516912216109392482235435606,3\r\n1.2.826.0.1.3680043.8.498.97400307568415006244714024769069514497,4\r\n1.2.826.0.1.3680043.8.498.97464332941382646555656038793589898504,5\r\n1.2.826.0.1.3680043.8.498.97485476951995325021464683747847126869,1\r\n1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373,0\r\n1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369,4\r\n1.2.826.0.1.3680043.8.498.97500632594529766269369977508352975364,5\r\n1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391,1\r\n1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480,0\r\n1.2.826.0.1.3680043.8.498.97564564380731258764291146761742841527,1\r\n1.2.826.0.1.3680043.8.498.97579738268290461886850261405719501384,2\r\n1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028,0\r\n1.2.826.0.1.3680043.8.498.97658595748192892503053963872409769829,3\r\n1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091,0\r\n1.2.826.0.1.3680043.8.498.97694479664965569386338738579573728752,3\r\n1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509,3\r\n1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614,2\r\n1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056,3\r\n1.2.826.0.1.3680043.8.498.97759980486481979519068526243106370997,4\r\n1.2.826.0.1.3680043.8.498.97797693721516872008302328638374252668,1\r\n1.2.826.0.1.3680043.8.498.97830923192341890652952842494018952920,3\r\n1.2.826.0.1.3680043.8.498.97849030682292314880463675622456900933,2\r\n1.2.826.0.1.3680043.8.498.97909881548577311929907613359449243075,1\r\n1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499,2\r\n1.2.826.0.1.3680043.8.498.97964867333695657200856428936254426086,1\r\n1.2.826.0.1.3680043.8.498.98022738778461179495257509215244053198,3\r\n1.2.826.0.1.3680043.8.498.98027050915885651900992366068887550187,3\r\n1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642,1\r\n1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381,2\r\n1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265,3\r\n1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354,2\r\n1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974,1\r\n1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589,0\r\n1.2.826.0.1.3680043.8.498.98112373403331748422888362310572528002,3\r\n1.2.826.0.1.3680043.8.498.98143909986671737130865120569673002676,3\r\n1.2.826.0.1.3680043.8.498.98146155044178593810385943365143374814,1\r\n1.2.826.0.1.3680043.8.498.98151095279541278201351784048265686573,2\r\n1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074,0\r\n1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938,0\r\n1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720,1\r\n1.2.826.0.1.3680043.8.498.98315778542448798493600485116505234132,3\r\n1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526,4\r\n1.2.826.0.1.3680043.8.498.98343185597651455245996667389303915779,1\r\n1.2.826.0.1.3680043.8.498.98351891935507349137982199658103205287,5\r\n1.2.826.0.1.3680043.8.498.98400072082366198211931728015692551635,2\r\n1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225,1\r\n1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104,1\r\n1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764,1\r\n1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070,0\r\n1.2.826.0.1.3680043.8.498.98676079331571304021387833410440136393,1\r\n1.2.826.0.1.3680043.8.498.98752901295449528283322564256444025625,2\r\n1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261,4\r\n1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186,0\r\n1.2.826.0.1.3680043.8.498.98824066826946884510314135448574304233,2\r\n1.2.826.0.1.3680043.8.498.98828753472895685940998888332947119084,4\r\n1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068,0\r\n1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803,0\r\n1.2.826.0.1.3680043.8.498.98890851347757949328159691828768030366,4\r\n1.2.826.0.1.3680043.8.498.98914528123216365374152816424181510781,2\r\n1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934,0\r\n1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874,1\r\n1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659,1\r\n1.2.826.0.1.3680043.8.498.99088247184917460283293617180083591028,3\r\n1.2.826.0.1.3680043.8.498.99108147913292332418681050058312852494,4\r\n1.2.826.0.1.3680043.8.498.99155175553749543768821558247962844052,4\r\n1.2.826.0.1.3680043.8.498.99178010550062268668346041362629896201,4\r\n1.2.826.0.1.3680043.8.498.99237666579371266866432031468578336182,1\r\n1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908,1\r\n1.2.826.0.1.3680043.8.498.99257151613846084492342083136797222759,3\r\n1.2.826.0.1.3680043.8.498.99303079292320085374389086609761974820,2\r\n1.2.826.0.1.3680043.8.498.99309981642816644502709148844743995902,4\r\n1.2.826.0.1.3680043.8.498.99413398303724030328059886749330413303,3\r\n1.2.826.0.1.3680043.8.498.99414163787650487720542840540379270468,1\r\n1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513,3\r\n1.2.826.0.1.3680043.8.498.99441773159476751698123795913150660766,4\r\n1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280,1\r\n1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536,0\r\n1.2.826.0.1.3680043.8.498.99454657594216441037514296544803013407,3\r\n1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928,0\r\n1.2.826.0.1.3680043.8.498.99493257395588095270208592176887619190,2\r\n1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961,0\r\n1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206,0\r\n1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095,1\r\n1.2.826.0.1.3680043.8.498.99635784824711754116112241806442235135,2\r\n1.2.826.0.1.3680043.8.498.99651725851820083142440965238449655973,4\r\n1.2.826.0.1.3680043.8.498.99708702196778120583644711299950439036,4\r\n1.2.826.0.1.3680043.8.498.99710592666731203419714499168506345263,3\r\n1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711,2\r\n1.2.826.0.1.3680043.8.498.99726731706411379395165830480890405115,3\r\n1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506,0\r\n1.2.826.0.1.3680043.8.498.99813269006669714758887909212386431934,5\r\n1.2.826.0.1.3680043.8.498.99826037216341310731701527407057049820,3\r\n1.2.826.0.1.3680043.8.498.99827460918477409069820779642161448917,4\r\n1.2.826.0.1.3680043.8.498.99864345386635409953374818045984558970,4\r\n1.2.826.0.1.3680043.8.498.99864558747514747686151806909794288262,4\r\n1.2.826.0.1.3680043.8.498.99880021280666757817587834481314366555,3\r\n1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290,0\r\n1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209,1\r\n1.2.826.0.1.3680043.8.498.99939100657291954890286900729983287973,0\r\n1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966,0\r\n', 'validation_labels.json': '{"version":"hpo_v31_d2_fixed_fold0_validation","targets_in_order":["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker\'s","Contusion","Fracture"],"labels":[{"study_id":"1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513","probabilities":[0.03071,0.03071,0.03071,0.03071,0.0375,0.0375,0.0375,0.37167,0.25833,0.05,0.02375,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018","probabilities":[0.795,0.0375,0.03786,0.03786,0.1,0.1,0.1,0.83429,0.47125,0.08,0.93857,0.94214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922","probabilities":[0.02917,0.03071,0.66571,0.03071,0.47429,0.05583,0.81143,0.59071,0.27917,0.10095,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347","probabilities":[0.02833,0.02833,0.03,0.03,0.026,0.026,0.026,0.03,0.09,0.05,0.01667,0.01667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678","probabilities":[0.02833,0.02833,0.03,0.03,0.03,0.03,0.03,0.03,0.07333,0.02302,0.02375,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166","probabilities":[0.02917,0.03071,0.94071,0.03071,0.88929,0.951,0.82786,0.49714,0.18833,0.53714,0.48333,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586","probabilities":[0.56167,0.49714,0.80571,0.50286,0.78429,0.589,0.589,0.68,0.92143,0.13853,0.73,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536","probabilities":[0.94214,0.03083,0.03214,0.03214,0.09083,0.09083,0.08357,0.93929,0.32083,0.09292,0.93929,0.15833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03083,0.03083,0.03083,0.03214,0.17688,0.03214,0.64286,0.1325],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.029,0.029,0.03214,0.07583,0.03214,0.03083,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821","probabilities":[0.03083,0.82571,0.03214,0.03214,0.482,0.36333,0.11625,0.67857,0.315,0.03214,0.86143,0.0625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307","probabilities":[0.03083,0.03214,0.319,0.02625,0.45714,0.10167,0.09286,0.04143,0.07833,0.04303,0.03214,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531","probabilities":[0.029,0.029,0.03083,0.03083,0.09083,0.09083,0.09083,0.75571,0.93857,0.09505,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.07583,0.03814,0.03214,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029","probabilities":[0.0363,0.04897,0.28444,0.09105,0.60714,0.18421,0.775,0.74143,0.92143,0.803,0.08042,0.01492],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639","probabilities":[0.029,0.029,0.63786,0.94,0.56214,0.04357,0.76286,0.74429,0.81786,0.2,0.0325,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","probabilities":[0.03083,0.03083,0.93929,0.03214,0.05,0.05,0.05,0.03214,0.09375,0.37333,0.02167,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315","probabilities":[0.36714,0.12071,0.94071,0.37,0.86714,0.755,0.86714,0.82429,0.496,0.59143,0.13833,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03083,0.03083,0.79786,0.03214,0.1025,0.03214,0.04357,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500","probabilities":[0.03083,0.03083,0.03083,0.03083,0.03083,0.03083,0.101,0.18786,0.25875,0.44857,0.02625,0.101],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724","probabilities":[0.30571,0.03357,0.0325,0.0325,0.37857,0.28833,0.231,0.05071,0.09375,0.03357,0.047,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434","probabilities":[0.94214,0.03083,0.93857,0.93857,0.03083,0.03083,0.03214,0.33571,0.2125,0.07881,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667","probabilities":[0.88857,0.03083,0.94214,0.03214,0.03917,0.03917,0.03929,0.49857,0.27667,0.05648,0.055,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863","probabilities":[0.03083,0.03083,0.19143,0.04357,0.03214,0.15786,0.62571,0.37429,0.30333,0.03083,0.029,0.11],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.045,0.45,0.45857,0.14,0.03214,0.03214,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667","probabilities":[0.03083,0.03214,0.93429,0.03214,0.64,0.64,0.64,0.33857,0.21,0.08,0.03083,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695","probabilities":[0.03214,0.03214,0.03214,0.03214,0.44571,0.03214,0.03214,0.44857,0.15333,0.51857,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091","probabilities":[0.03083,0.52,0.03214,0.03214,0.02625,0.02625,0.65714,0.65571,0.225,0.03214,0.93714,0.92286],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716","probabilities":[0.03083,0.03083,0.04357,0.04357,0.03929,0.03929,0.19071,0.38429,0.14,0.04727,0.434,0.30667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828","probabilities":[0.16667,0.16667,0.2,0.16667,0.2,0.16667,0.59071,0.72571,0.46625,0.2,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251","probabilities":[0.03083,0.03083,0.03214,0.03214,0.53143,0.75286,0.09083,0.03214,0.0925,0.04667,0.302,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736","probabilities":[0.03214,0.03214,0.93143,0.03214,0.1,0.05,0.1,0.45857,0.14,0.0593,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084","probabilities":[0.23417,0.51333,0.94071,0.04357,0.79857,0.418,0.52143,0.1,0.1,0.485,0.03833,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196","probabilities":[0.03214,0.03214,0.93857,0.03214,0.86167,0.86167,0.924,0.88857,0.39,0.74429,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971","probabilities":[0.03083,0.03083,0.50143,0.03083,0.02625,0.02625,0.029,0.03214,0.07583,0.03083,0.02625,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506","probabilities":[0.59,0.39167,0.94143,0.04357,0.44857,0.0,0.045,0.06571,0.125,0.04357,0.76429,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658","probabilities":[0.03,0.03,0.03,0.03,0.64571,0.03071,0.79929,0.03,0.07333,0.03,0.16714,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331","probabilities":[0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.57857,0.63857,0.16,0.06216,0.93571,0.121],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","probabilities":[0.03214,0.03214,0.93571,0.03214,0.17583,0.101,0.88714,0.66429,0.92143,0.2,0.152,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428","probabilities":[0.88929,0.29,0.93929,0.1,0.58917,0.1,0.70667,0.42757,0.44,0.86214,0.055,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334","probabilities":[0.03083,0.03083,0.03214,0.93929,0.03083,0.03083,0.08,0.33429,0.25667,0.04891,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.48583,0.03214,0.03214,0.03083,0.03083,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263","probabilities":[0.03083,0.40917,0.94,0.03214,0.68357,0.1125,0.59571,0.05071,0.11875,0.88786,0.04667,0.02667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839","probabilities":[0.03083,0.03214,0.03214,0.03214,0.03917,0.54429,0.71,0.32429,0.21,0.05,0.03083,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655","probabilities":[0.03083,0.03083,0.20429,0.11,0.08357,0.82,0.75857,0.45857,0.89429,0.02917,0.029,0.11],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264","probabilities":[0.36857,0.03917,0.93143,0.50143,0.73714,0.10667,0.17859,0.47143,0.34167,0.10376,0.512,0.03143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398","probabilities":[0.027,0.5775,0.85786,0.03071,0.34333,0.45143,0.52143,0.47286,0.4375,0.78643,0.181,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722","probabilities":[0.94143,0.03071,0.1,0.62,0.05,0.1,0.52143,0.424,0.83,0.08312,0.94,0.60429],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768","probabilities":[0.02917,0.02917,0.31714,0.04214,0.03071,0.03071,0.03071,0.03929,0.07583,0.02833,0.02917,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670","probabilities":[0.02917,0.02917,0.49571,0.03071,0.04667,0.04667,0.02917,0.03,0.07333,0.04113,0.04667,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962","probabilities":[0.94143,0.45143,0.03071,0.03071,0.03071,0.03071,0.03071,0.46,0.27833,0.44857,0.94,0.03875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310","probabilities":[0.37786,0.03917,0.93857,0.74429,0.53429,0.73857,0.73071,0.47286,0.89286,0.80143,0.41571,0.135],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088","probabilities":[0.03083,0.03083,0.875,0.03214,0.03083,0.03214,0.47,0.33429,0.15333,0.08091,0.03071,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008","probabilities":[0.03083,0.36571,0.65857,0.03071,0.027,0.027,0.027,0.03071,0.38812,0.04501,0.12625,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547","probabilities":[0.02917,0.65667,0.03786,0.05,0.02917,0.455,0.41333,0.45857,0.90714,0.08,0.50667,0.24],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930","probabilities":[0.03071,0.03071,0.03071,0.03071,0.03071,0.74429,0.69571,0.45143,0.15667,0.03071,0.03071,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324","probabilities":[0.02917,0.02917,0.03071,0.94,0.03071,0.50143,0.02917,0.03071,0.09125,0.04036,0.04589,0.01321],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213","probabilities":[0.03071,0.03071,0.03071,0.03071,0.01833,0.01833,0.01833,0.03071,0.09125,0.03071,0.01,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286","probabilities":[0.615,0.037,0.56,0.88,0.04929,0.0375,0.04583,0.03929,0.09938,0.0375,0.68714,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","probabilities":[0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.49714,0.03071,0.07417,0.45857,0.16786,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758","probabilities":[0.03071,0.03071,0.03071,0.94214,0.0375,0.51571,0.39917,0.45857,0.90429,0.54,0.92,0.16],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747","probabilities":[0.62571,0.05071,0.94,0.61571,0.63333,0.055,0.53,0.45857,0.91857,0.46,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131","probabilities":[0.03083,0.63333,0.94,0.04357,0.05,0.55167,0.05,0.87929,0.47062,0.03214,0.16,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","probabilities":[0.66214,0.03917,0.51143,0.03929,0.03917,0.03917,0.045,0.45857,0.215,0.05316,0.59,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731","probabilities":[0.03083,0.03083,0.93857,0.03214,0.029,0.029,0.03083,0.70857,0.28333,0.21667,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.029,0.32,0.04714,0.1,0.02,0.0,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683","probabilities":[0.03083,0.029,0.94,0.03083,0.02625,0.02625,0.029,0.03214,0.07583,0.342,0.02625,0.24],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204","probabilities":[0.03083,0.03083,0.03214,0.03214,0.49,0.10417,0.94214,0.45857,0.2425,0.45,0.52286,0.096],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291","probabilities":[0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11277078989123345865112116288626381353","probabilities":[0.46357,0.0325,0.03357,0.0325,0.0325,0.0325,0.0325,0.45857,0.19,0.03357,0.025,0.031],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11277278232377185347750793496260885546","probabilities":[0.62583,0.03917,0.94143,0.03929,0.83643,0.615,0.62,0.825,0.93571,0.2,0.56143,0.03357],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593","probabilities":[0.03083,0.03083,0.84143,0.04143,0.03083,0.03083,0.52,0.04286,0.09667,0.12417,0.02625,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411","probabilities":[0.23833,0.07277,0.93214,0.93214,0.42125,0.68429,0.70417,0.72714,0.325,0.17126,0.11641,0.03579],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712","probabilities":[0.79857,0.03917,0.03929,0.03929,0.03917,0.03917,0.03929,0.47286,0.27667,0.49625,0.7875,0.92167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473","probabilities":[0.03083,0.03214,0.94143,0.93071,0.45857,0.04417,0.85214,0.93929,0.35417,0.03214,0.152,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722","probabilities":[0.03083,0.03083,0.93929,0.03214,0.04286,0.04286,0.04286,0.45857,0.27667,0.51143,0.86143,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11391095418955817124325938917797687542","probabilities":[0.03083,0.75214,0.03214,0.03214,0.039,0.039,0.03929,0.45857,0.27667,0.0538,0.92143,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830","probabilities":[0.03214,0.04667,0.03214,0.34786,0.05,0.10667,0.43071,0.03571,0.1,0.04054,0.03214,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849","probabilities":[0.68167,0.03917,0.67929,0.91071,0.49167,0.136,0.03917,0.80917,0.4965,0.16,0.93571,0.705],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03083,0.49,0.49857,0.03214,0.09375,0.51857,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","probabilities":[0.94,0.03214,0.56071,0.91786,0.02625,0.49,0.03083,0.49857,0.215,0.05554,0.2,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","probabilities":[0.03083,0.03214,0.94,0.03214,0.94143,0.45143,0.44714,0.72571,0.92143,0.14006,0.045,0.024],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121","probabilities":[0.94214,0.394,0.26667,0.26667,0.2,0.2,0.2,0.68,0.35,0.68,0.53333,0.2],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047","probabilities":[0.50833,0.0325,0.03357,0.03357,0.03833,0.50667,0.03833,0.10667,0.1,0.03357,0.03833,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558","probabilities":[0.55643,0.445,0.49,0.03917,0.46571,0.05,0.03917,0.10143,0.125,0.73643,0.0375,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.03083,0.03214,0.09375,0.01,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983","probabilities":[0.03083,0.03083,0.03083,0.03214,0.03083,0.03214,0.03083,0.03214,0.19583,0.03214,0.029,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11546127923715190914128277599849591183","probabilities":[0.03083,0.59,0.03214,0.03214,0.03083,0.03083,0.03214,0.33429,0.25667,0.04569,0.03875,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11551770225828320378339882849245348148","probabilities":[0.03214,0.03083,0.89643,0.31286,0.515,0.18,0.18,0.76143,0.93143,0.77429,0.26417,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457","probabilities":[0.02625,0.02625,0.50143,0.05,0.115,0.08167,0.10667,0.52,0.215,0.05,0.78,0.93],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849","probabilities":[0.03071,0.03071,0.49571,0.49571,0.1,0.05,0.57714,0.73929,0.31417,0.26667,0.42667,0.11],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378","probabilities":[0.027,0.027,0.42,0.03214,0.08143,0.08143,0.08143,0.15786,0.07583,0.02833,0.24417,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181","probabilities":[0.18786,0.54833,0.945,0.49143,0.78667,0.51143,0.51833,0.77429,0.35167,0.03,0.16,0.115],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623","probabilities":[0.027,0.027,0.94143,0.03071,0.036,0.026,0.03,0.45857,0.215,0.02833,0.89286,0.91571],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429","probabilities":[0.94214,0.53571,0.93857,0.85,0.66714,0.598,0.51917,0.57333,0.26667,0.03,0.83286,0.402],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637","probabilities":[0.64,0.48714,0.935,0.82143,0.94214,0.70929,0.31286,0.47429,0.31167,0.47571,0.241,0.02833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","probabilities":[0.03071,0.03071,0.565,0.03071,0.0225,0.0225,0.0225,0.03,0.19333,0.03,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612","probabilities":[0.03071,0.03071,0.49571,0.03071,0.05,0.05,0.1,0.27286,0.26,0.05233,0.82,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607","probabilities":[0.2,0.16667,0.93929,0.495,0.43314,0.74429,0.37625,0.75714,0.93857,0.76,0.12427,0.03689],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695","probabilities":[0.02917,0.45143,0.93857,0.02917,0.035,0.035,0.03625,0.515,0.45,0.02917,0.03,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11783597160745641720078978406968189488","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269","probabilities":[0.02917,0.02917,0.31786,0.91571,0.0375,0.0375,0.0375,0.45714,0.1875,0.05515,0.035,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483","probabilities":[0.03071,0.03071,0.93857,0.03071,0.51333,0.51333,0.51333,0.03,0.09,0.03,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033","probabilities":[0.02917,0.47,0.03071,0.02917,0.04214,0.56286,0.45143,0.03929,0.38938,0.18357,0.02375,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758","probabilities":[0.645,0.03786,0.69357,0.03786,0.68429,0.10667,0.42857,0.88,0.38833,0.03786,0.765,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108","probabilities":[0.40357,0.33643,0.92071,0.1,0.88929,0.53429,0.62667,0.03929,0.24583,0.03786,0.90786,0.1],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862","probabilities":[0.94214,0.03,0.94071,0.03,0.52,0.027,0.69714,0.45857,0.215,0.45857,0.02833,0.02833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","probabilities":[0.94214,0.03071,0.93143,0.03071,0.0575,0.0575,0.0975,0.33333,0.10009,0.07677,0.61429,0.08143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762","probabilities":[0.74333,0.58333,0.94333,0.04929,0.25,0.25,0.46833,0.68714,0.5175,0.2,0.12625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229","probabilities":[0.23929,0.0325,0.93143,0.93571,0.74429,0.87583,0.45714,0.53429,0.5675,0.2,0.135,0.115],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744","probabilities":[0.03071,0.03071,0.49429,0.03071,0.16667,0.04667,0.04667,0.03,0.07333,0.04189,0.02375,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369","probabilities":[0.02917,0.02917,0.93143,0.03071,0.46143,0.02917,0.45714,0.8,0.365,0.08,0.54167,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11905397531490569218559534495408776672","probabilities":[0.02917,0.02917,0.03071,0.03071,0.03625,0.03625,0.035,0.03,0.07333,0.05,0.25625,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708","probabilities":[0.60571,0.55583,0.72286,0.72286,0.41833,0.73786,0.47143,0.68714,0.91429,0.1273,0.82357,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707","probabilities":[0.03214,0.03214,0.94143,0.03214,0.52,0.03083,0.03214,0.45857,0.81571,0.47286,0.04357,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715","probabilities":[0.267,0.039,0.03929,0.74571,0.34857,0.76357,0.14125,0.56833,0.77333,0.2925,0.35417,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11935116791150327546384719051023540185","probabilities":[0.03214,0.03083,0.03214,0.03214,0.05,0.05,0.05,0.44857,0.25667,0.0486,0.03214,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140","probabilities":[0.03083,0.03083,0.03214,0.03214,0.095,0.56286,0.57714,0.33429,0.25667,0.16667,0.16667,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769","probabilities":[0.05,0.05,0.1,0.1,0.47875,0.215,0.2,0.1,0.06195,0.0449,0.60786,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600","probabilities":[0.029,0.51714,0.20071,0.03214,0.66214,0.02625,0.76429,0.04286,0.125,0.03917,0.68583,0.398],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11958117172819754862618612170999909480","probabilities":[0.03214,0.03083,0.03214,0.03214,0.1,0.75667,0.74714,0.57643,0.27917,0.09,0.503,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317","probabilities":[0.03214,0.03214,0.94143,0.03214,0.78583,0.1,0.1,0.68571,0.34833,0.12239,0.92857,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.11981170645605720774509861722988500673","probabilities":[0.03083,0.65143,0.03214,0.03214,0.09083,0.09083,0.08357,0.9325,0.45917,0.07799,0.93786,0.26857],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878","probabilities":[0.03083,0.03083,0.03214,0.03214,0.45357,0.095,0.88786,0.33429,0.21,0.07883,0.03833,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792","probabilities":[0.45833,0.03929,0.50143,0.93071,0.1,0.1,0.47286,0.47143,0.275,0.08656,0.135,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12031315219947933845048665397149229969","probabilities":[0.03083,0.03083,0.93929,0.03214,0.88857,0.49857,0.94214,0.45857,0.19,0.39833,0.039,0.024],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857","probabilities":[0.03214,0.03214,0.03214,0.03214,0.02625,0.02625,0.02625,0.95,0.16356,0.03214,0.93,0.94214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953","probabilities":[0.03214,0.03214,0.24286,0.03214,0.46,0.50833,0.46,0.46,0.23875,0.46,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491","probabilities":[0.515,0.66,0.706,0.2125,0.05,0.05,0.05,0.2,0.07273,0.04584,0.03929,0.031],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205","probabilities":[0.03083,0.03083,0.03571,0.03571,0.03214,0.03214,0.32714,0.03214,0.2125,0.45857,0.03083,0.307],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415","probabilities":[0.039,0.039,0.42714,0.03917,0.03833,0.03875,0.45714,0.74833,0.68143,0.87929,0.03875,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614","probabilities":[0.04448,0.04519,0.565,0.05971,0.15207,0.06898,0.58,0.45857,0.5925,0.06531,0.06272,0.01183],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12139829070290803654867114374799823711","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297","probabilities":[0.19917,0.51167,0.94,0.03929,0.46,0.52,0.34143,0.44429,0.15667,0.09067,0.065,0.02875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851","probabilities":[0.03083,0.03083,0.93857,0.03214,0.57571,0.03214,0.66714,0.52,0.27667,0.88857,0.369,0.14167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217","probabilities":[0.03083,0.03214,0.93857,0.03214,0.49857,0.56167,0.47857,0.45714,0.2125,0.09463,0.03833,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211","probabilities":[0.03214,0.31417,0.93857,0.60143,0.79714,0.20667,0.47286,0.03214,0.2125,0.08646,0.41417,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302","probabilities":[0.03083,0.03083,0.03214,0.11,0.41833,0.04417,0.41833,0.45857,0.1475,0.82643,0.02625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408","probabilities":[0.03214,0.03214,0.93857,0.03214,0.03929,0.03929,0.03929,0.52143,0.2175,0.94,0.029,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796","probabilities":[0.04357,0.03083,0.42929,0.29643,0.515,0.434,0.11583,0.45,0.14625,0.03214,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681","probabilities":[0.02625,0.3225,0.91714,0.03214,0.03833,0.03833,0.03833,0.03214,0.31938,0.07456,0.72286,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.0375,0.0375,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03214,0.51286,0.03214,0.03214,0.07583,0.03214,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817","probabilities":[0.94214,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.94143,0.39,0.08687,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014","probabilities":[0.03214,0.03214,0.49857,0.03214,0.05,0.05,0.56429,0.03214,0.2125,0.63714,0.045,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392","probabilities":[0.03214,0.03214,0.03214,0.03214,0.05,0.05,0.80714,0.32143,0.2125,0.04871,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12259896940707176389739553110492954399","probabilities":[0.03214,0.03083,0.03214,0.03214,0.05,0.05,0.1,0.45,0.2125,0.05,0.03214,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367","probabilities":[0.03083,0.03083,0.93429,0.03214,0.51,0.03214,0.03214,0.45857,0.215,0.1725,0.04667,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097","probabilities":[0.03083,0.03083,0.94071,0.03214,0.43714,0.03083,0.03214,0.72429,0.285,0.10515,0.03833,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578","probabilities":[0.02917,0.59,0.03071,0.03071,0.02917,0.02917,0.03071,0.45857,0.26,0.049,0.03625,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380","probabilities":[0.02917,0.02917,0.03071,0.03071,0.027,0.027,0.73286,0.03071,0.11625,0.04069,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790","probabilities":[0.03071,0.49143,0.78429,0.60714,0.2,0.2,0.87429,0.47286,0.3075,0.2,0.64571,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","probabilities":[0.02917,0.02917,0.07571,0.02917,0.22167,0.74571,0.80714,0.47286,0.79857,0.02917,0.099,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894","probabilities":[0.02375,0.02375,0.02375,0.02375,0.25625,0.432,0.02917,0.03929,0.38938,0.06626,0.01833,0.0075],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831","probabilities":[0.02917,0.02917,0.41714,0.475,0.56667,0.64286,0.18,0.03071,0.15563,0.72714,0.03071,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12411396124304166984011280826756258628","probabilities":[0.02917,0.02917,0.03071,0.03071,0.027,0.08917,0.50286,0.03071,0.09125,0.05,0.34,0.0075],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232","probabilities":[0.04167,0.25667,0.93286,0.24,0.45429,0.31429,0.51333,0.92286,0.20375,0.03214,0.125,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12437154858167700438432674983971817217","probabilities":[0.03071,0.02917,0.49143,0.0,0.44,0.05,0.88143,0.67857,0.93,0.1168,0.03071,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876","probabilities":[0.05274,0.04957,0.93857,0.93857,0.775,0.1,0.775,0.74143,0.325,0.75857,0.11414,0.03421],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080","probabilities":[0.68,0.039,0.92429,0.07,0.17333,0.17333,0.24,0.64929,0.46938,0.08,0.07,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12471165461877621368840037945454256612","probabilities":[0.775,0.03083,0.03214,0.03214,0.039,0.039,0.039,0.03214,0.07583,0.05,0.03214,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084","probabilities":[0.25,0.03917,0.88857,0.35571,0.94143,0.06583,0.304,0.74857,0.3875,0.26667,0.505,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525","probabilities":[0.03083,0.03083,0.03214,0.03214,0.5,0.03875,0.039,0.53429,0.2175,0.03214,0.72286,0.15833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342","probabilities":[0.03214,0.03083,0.03571,0.93857,0.2,0.2,0.03214,0.72571,0.31125,0.2,0.86429,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","probabilities":[0.03083,0.49143,0.85857,0.49429,0.71286,0.715,0.80714,0.67857,0.34833,0.79143,0.142,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111","probabilities":[0.87,0.67429,0.57571,0.1,0.05,0.05,0.05,0.68714,0.93,0.10228,0.93929,0.1],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616","probabilities":[0.03083,0.03214,0.94071,0.03214,0.31071,0.03917,0.60286,0.37429,0.21,0.82571,0.03083,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431","probabilities":[0.03083,0.03083,0.94,0.03214,0.03214,0.03214,0.03214,0.45143,0.26,0.05959,0.0649,0.01137],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12583872770399266463538638801797827468","probabilities":[0.03083,0.03083,0.03071,0.03071,0.029,0.029,0.029,0.45286,0.26167,0.02,0.02167,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918","probabilities":[0.03083,0.03083,0.03429,0.03429,0.03214,0.03214,0.03214,0.05429,0.2,0.02917,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","probabilities":[0.94214,0.03071,0.93857,0.93857,0.02917,0.02917,0.03071,0.03,0.09,0.04405,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635","probabilities":[0.03071,0.03071,0.5,0.35857,0.03071,0.03071,0.03071,0.03,0.07333,0.03,0.05,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265","probabilities":[0.02917,0.45,0.93714,0.03071,0.0375,0.0375,0.0375,0.03214,0.0925,0.08,0.03875,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803","probabilities":[0.23929,0.03083,0.93429,0.22857,0.94071,0.56857,0.59,0.93714,0.93143,0.03143,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12701097867143790763644826659974337046","probabilities":[0.03071,0.02917,0.03071,0.45167,0.05,0.05,0.05,0.05,0.05254,0.03944,0.03,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849","probabilities":[0.20167,0.033,0.94143,0.85,0.94214,0.82714,0.86714,0.47286,0.93214,0.13127,0.352,0.51571],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688","probabilities":[0.03214,0.03214,0.86571,0.03214,0.37333,0.1,0.1,0.45857,0.26,0.0643,0.101,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492","probabilities":[0.03,0.03,0.11,0.11,0.03083,0.03083,0.03083,0.03214,0.15687,0.03214,0.03083,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067","probabilities":[0.03071,0.03071,0.03071,0.03071,0.03143,0.03143,0.03143,0.03071,0.07417,0.03071,0.03071,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908","probabilities":[0.03,0.03,0.03143,0.03143,0.1,0.34333,0.208,0.46,0.15833,0.03143,0.94,0.94143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270","probabilities":[0.36857,0.49857,0.50143,0.03286,0.53286,0.48,0.80357,0.47286,0.27667,0.09293,0.355,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333","probabilities":[0.03071,0.03071,0.94071,0.03071,0.03833,0.03,0.03,0.03071,0.09125,0.03071,0.61857,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949","probabilities":[0.51286,0.03167,0.42714,0.42714,0.668,0.58,0.82,0.50701,0.81571,0.10408,0.04357,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652","probabilities":[0.02917,0.02917,0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.07417,0.02917,0.20786,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035","probabilities":[0.02917,0.02917,0.94071,0.03071,0.02917,0.02917,0.03071,0.03071,0.09125,0.53571,0.01833,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751","probabilities":[0.30571,0.03083,0.94071,0.93143,0.66071,0.0375,0.65143,0.46,0.31167,0.94143,0.345,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12826828527176716109949704557150950675","probabilities":[0.02917,0.02917,0.30429,0.03071,0.027,0.0425,0.406,0.45857,0.19,0.05311,0.01833,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655","probabilities":[0.03214,0.03214,0.49286,0.03214,0.05,0.05,0.05,0.03214,0.09375,0.04351,0.68786,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502","probabilities":[0.03083,0.03083,0.92286,0.03214,0.57286,0.03214,0.93357,0.71143,0.28667,0.11919,0.0786,0.01266],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608","probabilities":[0.03214,0.03214,0.03929,0.93857,0.85,0.78286,0.94143,0.93929,0.85786,0.59714,0.111,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832","probabilities":[0.05,0.05,0.25286,0.05,0.1,0.1,0.1,0.1,0.37333,0.05,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039","probabilities":[0.03214,0.66,0.94333,0.03214,0.055,0.03875,0.03929,0.34143,0.27333,0.08628,0.82,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","probabilities":[0.03214,0.03214,0.49286,0.03214,0.775,0.775,0.775,0.03214,0.11875,0.03214,0.04712,0.01407],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935","probabilities":[0.11076,0.07692,0.11824,0.06822,0.2,0.05447,0.2,0.74143,0.92143,0.10668,0.93571,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589","probabilities":[0.04667,0.03917,0.93143,0.1,0.64429,0.33333,0.64429,0.68,0.92429,0.54714,0.17143,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363","probabilities":[0.03214,0.03214,0.94071,0.54214,0.76357,0.69714,0.88929,0.75071,0.35417,0.45857,0.03833,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508","probabilities":[0.38333,0.21583,0.62,0.31143,0.37333,0.67429,0.57857,0.04286,0.39,0.03929,0.334,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13065120731888368919957856813755701101","probabilities":[0.03083,0.03083,0.03214,0.03214,0.55429,0.61167,0.94214,0.45714,0.2125,0.05983,0.05,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13068696303639759696458130704554709416","probabilities":[0.03083,0.51286,0.94071,0.03214,0.5475,0.04417,0.45143,0.33429,0.15333,0.44714,0.029,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703","probabilities":[0.24286,0.03357,0.03357,0.03357,0.03357,0.03357,0.38429,0.93929,0.93857,0.93929,0.48786,0.03357],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894","probabilities":[0.03214,0.03214,0.93857,0.03214,0.44571,0.03083,0.79857,0.45857,0.27667,0.45857,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13099702881219862576749493772702888064","probabilities":[0.03083,0.28667,0.50143,0.38,0.029,0.029,0.85143,0.85714,0.32,0.45857,0.02625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786","probabilities":[0.03083,0.03083,0.93857,0.03214,0.92571,0.11667,0.63,0.05429,0.09667,0.08,0.04357,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.03083,0.03083,0.07583,0.03807,0.029,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127","probabilities":[0.94214,0.25833,0.94143,0.93857,0.88929,0.67429,0.56143,0.825,0.39062,0.65429,0.84857,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285","probabilities":[0.85917,0.88929,0.03929,0.59,0.54143,0.586,0.78571,0.825,0.3875,0.03214,0.77,0.94214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.04357,0.03214,0.15687,0.03083,0.03083,0.11625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13150780733560495803861509587935093776","probabilities":[0.02917,0.02917,0.93857,0.03071,0.69714,0.03071,0.46,0.94,0.355,0.13593,0.04667,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296","probabilities":[0.03071,0.02917,0.03071,0.03071,0.05,0.05,0.05,0.1,0.05254,0.03931,0.03071,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710","probabilities":[0.03071,0.03071,0.495,0.03071,0.05,0.05,0.05,0.03071,0.09125,0.04129,0.02375,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369","probabilities":[0.03071,0.03071,0.92429,0.03071,0.03071,0.03071,0.78571,0.03071,0.1175,0.02375,0.77143,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145","probabilities":[0.02917,0.02917,0.49286,0.02917,0.02375,0.02375,0.8,0.03071,0.11625,0.08917,0.81143,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605","probabilities":[0.02917,0.48714,0.65857,0.03071,0.615,0.04,0.44857,0.49857,0.15667,0.45714,0.92167,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169","probabilities":[0.02917,0.02917,0.03071,0.56286,0.81857,0.42857,0.82714,0.93786,0.78167,0.13951,0.312,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025","probabilities":[0.05,0.05,0.92929,0.1,0.77417,0.1,0.77417,0.74143,0.92143,0.14321,0.08049,0.01829],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636","probabilities":[0.515,0.03917,0.40929,0.58571,0.574,0.67333,0.93429,0.68143,0.93214,0.2,0.34,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","probabilities":[0.03071,0.03071,0.50357,0.03071,0.05,0.05,0.51857,0.49857,0.92143,0.03,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030","probabilities":[0.029,0.029,0.94,0.03214,0.0375,0.0375,0.03833,0.74,0.28167,0.07338,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604","probabilities":[0.03,0.03,0.03,0.03,0.44857,0.03071,0.03071,0.03,0.07333,0.03,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388","probabilities":[0.03,0.03,0.93929,0.03,0.03071,0.03071,0.03071,0.03,0.07333,0.03,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141","probabilities":[0.029,0.41833,0.94,0.03071,0.45857,0.03083,0.03083,0.45714,0.275,0.45,0.02375,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965","probabilities":[0.02917,0.02917,0.03071,0.03071,0.396,0.04667,0.37083,0.33286,0.43375,0.03,0.01833,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226","probabilities":[0.05,0.05,0.93929,0.1,0.1,0.1,0.59286,0.2,0.09891,0.52,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791","probabilities":[0.63714,0.50167,0.61857,0.53286,0.60857,0.33167,0.67214,0.192,0.312,0.36786,0.455,0.115],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.14061227335828384505403558788548348446","probabilities":[0.02833,0.02833,0.94143,0.03,0.73,0.52,0.52,0.93929,0.35417,0.03,0.16,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575","probabilities":[0.02833,0.02833,0.03,0.03,0.0225,0.0225,0.0225,0.03,0.09,0.05,0.01667,0.026],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166","probabilities":[0.03,0.03,0.03143,0.03143,0.03,0.03,0.03143,0.49714,0.34625,0.21667,0.03,0.03143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791","probabilities":[0.02833,0.02833,0.43857,0.03,0.05,0.05,0.6075,0.03,0.09,0.04309,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570","probabilities":[0.02833,0.02833,0.03,0.42714,0.39429,0.42,0.72857,0.04357,0.11875,0.03,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770","probabilities":[0.026,0.026,0.94143,0.73643,0.80643,0.02833,0.80643,0.45857,0.90857,0.12768,0.02375,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904","probabilities":[0.03214,0.03214,0.93714,0.03214,0.93143,0.13583,0.94143,0.93857,0.38,0.82429,0.04357,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","probabilities":[0.20429,0.415,0.94071,0.18,0.88929,0.65429,0.79786,0.88786,0.5375,0.105,0.63,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885","probabilities":[0.03083,0.03083,0.77857,0.55571,0.02625,0.02625,0.029,0.03214,0.0925,0.256,0.93571,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269","probabilities":[0.0325,0.0325,0.42929,0.03917,0.65714,0.43333,0.69167,0.41571,0.12,0.08715,0.047,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764","probabilities":[0.05,0.0325,0.03357,0.03357,0.03875,0.03875,0.03875,0.63429,0.275,0.05435,0.03357,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081","probabilities":[0.94214,0.45857,0.94143,0.04286,0.614,0.34,0.59286,0.34143,0.81571,0.07831,0.93929,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","probabilities":[0.03214,0.03214,0.93857,0.50143,0.77333,0.77333,0.77333,0.03214,0.09375,0.03214,0.04601,0.01285],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","probabilities":[0.24286,0.44714,0.88786,0.36857,0.73071,0.4875,0.65357,0.47286,0.79833,0.03071,0.29,0.34],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350","probabilities":[0.03071,0.03071,0.03071,0.03071,0.02917,0.02917,0.02917,0.27714,0.13667,0.027,0.02917,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.16691383858003633107641297162554868325","probabilities":[0.94214,0.02917,0.54143,0.03071,0.02917,0.02917,0.03071,0.11571,0.03,0.03,0.03,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","probabilities":[0.02917,0.02917,0.74071,0.94,0.03786,0.03786,0.43143,0.47286,0.89643,0.10268,0.027,0.0075],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653","probabilities":[0.03071,0.03071,0.93929,0.03071,0.16667,0.05,0.16667,0.03071,0.37562,0.0488,0.027,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","probabilities":[0.50357,0.64,0.63214,0.16667,0.70857,0.58083,0.86,0.94143,0.45667,0.08143,0.29,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916","probabilities":[0.03071,0.03071,0.94214,0.03071,0.82643,0.03071,0.87214,0.45929,0.2775,0.45929,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467","probabilities":[0.03071,0.03071,0.93857,0.50143,0.56857,0.30667,0.30667,0.68786,0.49312,0.12455,0.58167,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285","probabilities":[0.027,0.86333,0.03786,0.03786,0.03071,0.03071,0.03071,0.05286,0.215,0.45857,0.09917,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009","probabilities":[0.02917,0.35667,0.93857,0.05,0.45714,0.05,0.03071,0.45857,0.2375,0.05,0.02917,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124","probabilities":[0.81429,0.03071,0.93857,0.93857,0.57667,0.05,0.045,0.51167,0.27333,0.09577,0.02,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080","probabilities":[0.58667,0.0375,0.2,0.91714,0.405,0.715,0.40667,0.68,0.93214,0.10989,0.534,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569","probabilities":[0.41857,0.4475,0.94071,0.50143,0.79714,0.05,0.73571,0.68,0.4925,0.03,0.07333,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748","probabilities":[0.02917,0.02917,0.03,0.03,0.03,0.03,0.03,0.03,0.09,0.05,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18879049235810379709186936621085221483","probabilities":[0.02917,0.02917,0.94071,0.03071,0.85929,0.03786,0.45714,0.51286,0.27667,0.10587,0.07107,0.01289],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.18931107448953327254073392032141677797","probabilities":[0.02917,0.02917,0.93929,0.03071,0.45714,0.02917,0.45714,0.45857,0.15667,0.45857,0.04667,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295","probabilities":[0.03071,0.03071,0.03071,0.03071,0.02917,0.02917,0.02917,0.03,0.09,0.03,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548","probabilities":[0.03083,0.03083,0.03083,0.03083,0.03083,0.43143,0.09083,0.62143,0.14,0.05559,0.58571,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296","probabilities":[0.07793,0.05459,0.08876,0.04742,0.07682,0.04516,0.2,0.26681,0.08949,0.04741,0.93571,0.09573],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530","probabilities":[0.03214,0.49,0.93857,0.03214,0.56,0.05,0.81429,0.74,0.49125,0.38,0.28,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760","probabilities":[0.8,0.039,0.039,0.03875,0.1,0.1,0.1,0.80857,0.29375,0.0818,0.64667,0.93143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830","probabilities":[0.03083,0.03083,0.03429,0.03429,0.03214,0.03214,0.31429,0.03,0.175,0.02833,0.045,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850","probabilities":[0.03,0.03,0.03,0.03,0.08143,0.08833,0.88,0.45714,0.43625,0.03,0.0225,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886","probabilities":[0.03071,0.03071,0.93571,0.03071,0.77333,0.1,0.18797,0.03,0.115,0.03,0.70714,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","probabilities":[0.03214,0.03214,0.93929,0.03214,0.86417,0.86417,0.86417,0.03214,0.11875,0.03214,0.04661,0.01332],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785","probabilities":[0.94214,0.03083,0.94,0.03214,0.0375,0.0375,0.03875,0.45857,0.215,0.24667,0.03917,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784","probabilities":[0.94214,0.08357,0.94071,0.93929,0.45357,0.45357,0.03214,0.03214,0.11875,0.07294,0.04667,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537","probabilities":[0.03083,0.03083,0.93714,0.03214,0.03083,0.03083,0.03214,0.78,0.28333,0.07537,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03917,0.03917,0.03917,0.45857,0.4375,0.03214,0.38643,0.79929],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254","probabilities":[0.03083,0.03214,0.03214,0.03214,0.02625,0.02625,0.029,0.45143,0.14,0.05,0.02167,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954","probabilities":[0.03083,0.03083,0.03214,0.93214,0.03917,0.90917,0.80357,0.45857,0.80143,0.03083,0.029,0.11625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101","probabilities":[0.46071,0.03357,0.93714,0.03357,0.53143,0.49143,0.45143,0.33429,0.27333,0.06186,0.0325,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21142867813782396036287673290033987859","probabilities":[0.03083,0.03083,0.03214,0.66286,0.41857,0.46833,0.64,0.32071,0.15333,0.045,0.03083,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958","probabilities":[0.03214,0.385,0.94071,0.03083,0.87929,0.50857,0.65,0.45857,0.81571,0.80071,0.03875,0.23429],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044","probabilities":[0.18,0.03917,0.93,0.03929,0.22357,0.35571,0.66714,0.34143,0.27333,0.45571,0.46714,0.15833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03917,0.03917,0.85143,0.03214,0.09375,0.44857,0.03083,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673","probabilities":[0.029,0.029,0.86286,0.9175,0.78643,0.73357,0.92917,0.72714,0.925,0.45857,0.02625,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762","probabilities":[0.03083,0.03083,0.88429,0.03214,0.79857,0.10929,0.80714,0.47286,0.27667,0.10591,0.38,0.30667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295","probabilities":[0.23333,0.03929,0.94071,0.34286,0.79786,0.8225,0.80643,0.62714,0.93714,0.80071,0.63083,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","probabilities":[0.94214,0.03214,0.93929,0.94071,0.03917,0.224,0.03929,0.94143,0.3925,0.2,0.93929,0.16],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.45143,0.295,0.14667,0.101,0.09083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.22394208178597637441149549895512290013","probabilities":[0.78429,0.37286,0.86571,0.03929,0.65143,0.63429,0.8,0.93929,0.35417,0.09071,0.77429,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.22508124641661887430614105328316053644","probabilities":[0.03083,0.03083,0.93714,0.03214,0.03083,0.03083,0.03214,0.45143,0.26,0.05928,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","probabilities":[0.44,0.03357,0.93143,0.03357,0.1,0.1,0.1,0.46571,0.27667,0.63143,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170","probabilities":[0.16667,0.16667,0.93417,0.38,0.26667,0.73786,0.26667,0.33333,0.2,0.72714,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449","probabilities":[0.03083,0.03083,0.03929,0.03929,0.56286,0.32429,0.56286,0.03929,0.2125,0.47286,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134","probabilities":[0.03214,0.03214,0.94071,0.92857,0.53643,0.03083,0.84357,0.45857,0.215,0.475,0.03083,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738","probabilities":[0.03214,0.03083,0.57714,0.04643,0.54143,0.3625,0.74643,0.68,0.93143,0.47429,0.03833,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663","probabilities":[0.94583,0.03833,0.94071,0.04929,0.65571,0.225,0.16,0.47286,0.3075,0.47714,0.85857,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088","probabilities":[0.02917,0.02917,0.02917,0.02917,0.02917,0.027,0.027,0.03,0.07333,0.03,0.02833,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366","probabilities":[0.08917,0.60833,0.94,0.1675,0.80643,0.37143,0.37143,0.51143,0.83214,0.59429,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916","probabilities":[0.05,0.05,0.195,0.05,0.1,0.1,0.59286,0.64143,0.45667,0.64286,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24219103446788192052503003052696523200","probabilities":[0.025,0.025,0.93929,0.81286,0.64429,0.05,0.05,0.82357,0.32875,0.69643,0.54714,0.758],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910","probabilities":[0.03071,0.02917,0.03,0.03,0.05,0.05,0.05,0.05,0.05,0.05,0.03,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038","probabilities":[0.02917,0.08917,0.93857,0.93571,0.0375,0.0375,0.03786,0.03214,0.19583,0.07404,0.04829,0.01016],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.01806,0.01833,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24872746733270791851241663716306993824","probabilities":[0.02917,0.02917,0.03071,0.03071,0.03071,0.03071,0.03071,0.45,0.29375,0.14667,0.11375,0.099],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187","probabilities":[0.51571,0.44,0.94,0.03071,0.17333,0.10667,0.1,0.68,0.2925,0.09492,0.035,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","probabilities":[0.03214,0.03214,0.03214,0.03214,0.7775,0.7775,0.823,0.03214,0.4125,0.03214,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189","probabilities":[0.38714,0.0325,0.56714,0.03357,0.03875,0.03875,0.03875,0.03357,0.09375,0.039,0.103,0.03357],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724","probabilities":[0.029,0.029,0.32333,0.155,0.02625,0.02167,0.02625,0.48857,0.21,0.51286,0.02625,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670","probabilities":[0.03214,0.03214,0.65143,0.03214,0.03917,0.49857,0.85286,0.93857,0.33,0.45857,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882","probabilities":[0.465,0.16667,0.50143,0.75143,0.442,0.83714,0.465,0.68714,0.93,0.68714,0.68,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505","probabilities":[0.2,0.2,0.26667,0.93,0.26667,0.855,0.33333,0.74143,0.55667,0.82429,0.17596,0.04229],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567","probabilities":[0.66143,0.24583,0.93143,0.93714,0.76,0.54167,0.805,0.59167,0.38167,0.03929,0.815,0.15833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629","probabilities":[0.03083,0.03083,0.03083,0.03083,0.02625,0.02167,0.03083,0.03214,0.07583,0.029,0.93571,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103","probabilities":[0.03214,0.03214,0.93857,0.45071,0.665,0.08,0.61143,0.03214,0.0925,0.08633,0.93571,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502","probabilities":[0.029,0.029,0.65857,0.04357,0.53857,0.59571,0.54143,0.45857,0.43,0.03214,0.25667,0.11625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036","probabilities":[0.535,0.03929,0.50357,0.50357,0.05,0.05,0.05,0.03929,0.2125,0.04502,0.92643,0.1125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","probabilities":[0.03071,0.03071,0.03071,0.03071,0.60571,0.85143,0.56714,0.46,0.14167,0.03,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568","probabilities":[0.07353,0.0451,0.93143,0.25335,0.74429,0.74429,0.77667,0.72714,0.375,0.74286,0.1111,0.03701],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091","probabilities":[0.50333,0.0375,0.72083,0.03786,0.0325,0.0325,0.037,0.03143,0.115,0.05,0.03143,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691","probabilities":[0.03071,0.03071,0.94,0.03071,0.43,0.05,0.16667,0.36,0.27833,0.09303,0.82071,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957","probabilities":[0.027,0.027,0.61143,0.03071,0.01833,0.01833,0.027,0.46,0.2175,0.78571,0.76143,0.265],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.26848412380762311150802301982233072694","probabilities":[0.02917,0.02917,0.93857,0.03071,0.45143,0.02917,0.02917,0.04143,0.095,0.46,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194","probabilities":[0.24929,0.03214,0.94071,0.93929,0.94143,0.56,0.93357,0.93929,0.38125,0.47857,0.03214,0.03143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27059975738703546675398620281849285863","probabilities":[0.02917,0.02917,0.93857,0.03071,0.02917,0.02917,0.03071,0.03429,0.07833,0.05,0.01833,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858","probabilities":[0.03071,0.03071,0.93857,0.03071,0.10667,0.1,0.1,0.67857,0.46625,0.10036,0.03625,0.024],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545","probabilities":[0.03071,0.03071,0.86714,0.10214,0.59571,0.59571,0.59571,0.46,0.92143,0.72714,0.16667,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018","probabilities":[0.6975,0.515,0.85786,0.03929,0.53667,0.32667,0.305,0.68143,0.51875,0.11335,0.865,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355","probabilities":[0.65,0.38429,0.03929,0.03929,0.23,0.18,0.8,0.68857,0.93214,0.47286,0.90143,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","probabilities":[0.16667,0.16667,0.93875,0.2,0.86786,0.39833,0.94214,0.84,0.38833,0.94143,0.41571,0.58357],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","probabilities":[0.03083,0.17214,0.03214,0.03214,0.08357,0.08357,0.58357,0.33429,0.125,0.04759,0.04667,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449","probabilities":[0.03214,0.03214,0.49286,0.93417,0.1,0.1,0.1,0.72571,0.28167,0.07256,0.03875,0.031],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475","probabilities":[0.03214,0.03917,0.03214,0.03214,0.1,0.1,0.26667,0.94143,0.93929,0.2,0.865,0.49857],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","probabilities":[0.03214,0.03214,0.565,0.03214,0.02625,0.02625,0.02625,0.03214,0.19583,0.03214,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882","probabilities":[0.029,0.029,0.94071,0.93929,0.66929,0.66929,0.85833,0.43714,0.78,0.515,0.04667,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.45857,0.03214,0.19583,0.45857,0.02625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456","probabilities":[0.03083,0.03083,0.93857,0.03214,0.03083,0.03083,0.03214,0.03214,0.19583,0.05,0.11625,0.14167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256","probabilities":[0.08444,0.04114,0.12105,0.0814,0.04725,0.04194,0.05946,0.68,0.1,0.05581,0.17225,0.06214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181","probabilities":[0.69333,0.03917,0.03929,0.03929,0.55,0.4575,0.30333,0.80571,0.93286,0.10231,0.03929,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586","probabilities":[0.03214,0.03917,0.94214,0.63786,0.94214,0.61429,0.61429,0.45857,0.81357,0.94,0.93857,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.29343735048224321966824641432873034291","probabilities":[0.56143,0.03929,0.03357,0.03357,0.43143,0.16667,0.63714,0.40857,0.215,0.46143,0.94,0.94],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665","probabilities":[0.05,0.1,0.04667,0.04667,0.025,0.025,0.2,0.88333,0.4975,0.09667,0.94,0.773],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464","probabilities":[0.03214,0.03214,0.49571,0.49571,0.45833,0.366,0.59714,0.03214,0.07583,0.05,0.158,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569","probabilities":[0.029,0.029,0.81714,0.94143,0.18583,0.039,0.82857,0.45857,0.91,0.45857,0.72286,0.91714],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.32571,0.03214,0.07583,0.03214,0.02625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.15687,0.03214,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","probabilities":[0.05071,0.03917,0.49,0.73643,0.64429,0.62,0.88143,0.03929,0.2125,0.45857,0.705,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612","probabilities":[0.03083,0.2075,0.94143,0.03214,0.82714,0.4025,0.87286,0.45857,0.774,0.03214,0.66833,0.80571],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177","probabilities":[0.03083,0.03083,0.50143,0.03214,0.45857,0.03083,0.57714,0.57429,0.3075,0.45714,0.029,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.029,0.03083,0.03214,0.08688,0.05125,0.045,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425","probabilities":[0.23929,0.58333,0.94,0.93,0.76143,0.62,0.43143,0.47143,0.30625,0.66429,0.472,0.145],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961","probabilities":[0.62333,0.57667,0.59,0.03929,0.85,0.38,0.38,0.68714,0.93,0.59429,0.32,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961","probabilities":[0.03083,0.03214,0.03214,0.03214,0.0375,0.30667,0.03083,0.33429,0.185,0.04624,0.41714,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31145453746697192222724674918300432873","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.03071,0.78143,0.285,0.04667,0.03875,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.03071,0.45,0.25833,0.46,0.02167,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669","probabilities":[0.74214,0.71857,0.03857,0.03857,0.03833,0.03833,0.03833,0.93857,0.23333,0.03071,0.82429,0.71571],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","probabilities":[0.02917,0.02917,0.93929,0.03071,0.027,0.027,0.02917,0.45714,0.1875,0.05856,0.02167,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","probabilities":[0.03071,0.03071,0.93857,0.03071,0.57714,0.05,0.25,0.03071,0.11625,0.74143,0.05205,0.01079],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119","probabilities":[0.02917,0.02917,0.81286,0.03071,0.039,0.039,0.88857,0.88571,0.33,0.82429,0.053,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296","probabilities":[0.02917,0.02917,0.79143,0.03071,0.111,0.111,0.474,0.03071,0.38812,0.82571,0.02917,0.2],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461","probabilities":[0.03071,0.03071,0.03071,0.03071,0.02917,0.02917,0.88,0.03071,0.09125,0.03071,0.04214,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32488933488399304214586978746578986823","probabilities":[0.101,0.33167,0.86571,0.38,0.86714,0.28,0.86714,0.03214,0.18938,0.08528,0.09,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664","probabilities":[0.68786,0.03917,0.82714,0.1,0.51,0.866,0.44333,0.83571,0.56333,0.07,0.37041,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","probabilities":[0.03083,0.645,0.03083,0.432,0.03214,0.03083,0.03214,0.33571,0.2125,0.07584,0.93929,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03083,0.03214,0.09375,0.05,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924","probabilities":[0.03083,0.03083,0.29429,0.03214,0.05,0.485,0.73643,0.45857,0.215,0.08073,0.04667,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358","probabilities":[0.03083,0.48143,0.94,0.652,0.49857,0.478,0.61714,0.04143,0.095,0.84857,0.0125,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03083,0.03083,0.03083,0.03214,0.09375,0.03214,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569","probabilities":[0.56583,0.56,0.93214,0.16667,0.03875,0.03833,0.03875,0.45857,0.27667,0.06318,0.85786,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042","probabilities":[0.03083,0.03083,0.88714,0.03214,0.46,0.03214,0.45429,0.27786,0.0925,0.029,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807","probabilities":[0.03214,0.03214,0.03214,0.03214,0.515,0.515,0.515,0.03214,0.09375,0.03214,0.04824,0.01401],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","probabilities":[0.03733,0.04199,0.93214,0.08328,0.27884,0.11365,0.76083,0.26418,0.11507,0.71643,0.05788,0.01082],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550","probabilities":[0.03083,0.2375,0.63571,0.72286,0.94143,0.94143,0.88857,0.81786,0.92571,0.86429,0.76286,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.33841907095190352493353075864604147687","probabilities":[0.7525,0.03917,0.03929,0.03929,0.1,0.1,0.1,0.68857,0.925,0.08665,0.03929,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598","probabilities":[0.05,0.05,0.565,0.05,0.1,0.05,0.1,0.1,0.37333,0.05,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650","probabilities":[0.48,0.03917,0.66167,0.03929,0.039,0.039,0.03917,0.03929,0.2125,0.039,0.91786,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514","probabilities":[0.05,0.05,0.05,0.05,0.05,0.05,0.05,0.30144,0.08866,0.04241,0.4625,0.94],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","probabilities":[0.03,0.03,0.03,0.03,0.0375,0.0375,0.56286,0.03,0.09,0.03,0.93714,0.07167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613","probabilities":[0.02917,0.02917,0.55429,0.03,0.48857,0.476,0.44857,0.04143,0.215,0.03,0.13833,0.13833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444","probabilities":[0.03,0.03,0.15571,0.03,0.48,0.11429,0.04143,0.33429,0.15333,0.03,0.04143,0.04857],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906","probabilities":[0.94214,0.0975,0.08857,0.91857,0.08857,0.08857,0.08857,0.93857,0.81571,0.1018,0.68857,0.93929],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455","probabilities":[0.03,0.03,0.03,0.03,0.05,0.05,0.05,0.03,0.09,0.05,0.098,0.0225],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451","probabilities":[0.03,0.03,0.03,0.03,0.03,0.03,0.03,0.03,0.07333,0.58429,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139","probabilities":[0.08917,0.37429,0.18571,0.03,0.175,0.026,0.73429,0.34143,0.8,0.05871,0.099,0.11375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841","probabilities":[0.94214,0.08917,0.09071,0.09071,0.03,0.10286,0.03,0.38143,0.15333,0.02917,0.68857,0.85143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313","probabilities":[0.28286,0.03286,0.93143,0.334,0.93286,0.42125,0.73,0.51286,0.4375,0.51429,0.08929,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777","probabilities":[0.05,0.05,0.50143,0.50143,0.1,0.1,0.46,0.20554,0.10184,0.72857,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","probabilities":[0.03071,0.03071,0.03071,0.03071,0.155,0.155,0.0375,0.33571,0.1875,0.04573,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403","probabilities":[0.94214,0.08917,0.94071,0.03071,0.26571,0.29,0.10583,0.79167,0.23167,0.09779,0.099,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314","probabilities":[0.03071,0.49,0.94214,0.03071,0.67714,0.02917,0.59714,0.04143,0.1225,0.52143,0.36667,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.0375,0.37429,0.13667,0.02,0.027,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198","probabilities":[0.02917,0.02917,0.88643,0.03071,0.111,0.111,0.09917,0.88429,0.2775,0.07,0.026,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167","probabilities":[0.5775,0.175,0.94,0.37857,0.56857,0.27833,0.44786,0.05286,0.095,0.03143,0.155,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185","probabilities":[0.93286,0.16357,0.08,0.60143,0.03786,0.037,0.03857,0.45571,0.27333,0.10429,0.71583,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985","probabilities":[0.32286,0.49143,0.93214,0.62643,0.8,0.14667,0.8,0.68,0.93214,0.14789,0.36,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100","probabilities":[0.05,0.05,0.1,0.1,0.1,0.1,0.59071,0.65214,0.48583,0.745,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047","probabilities":[0.328,0.05738,0.94143,0.247,0.94214,0.51625,0.75429,0.39573,0.35,0.2,0.37333,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727","probabilities":[0.03083,0.03083,0.82571,0.03214,0.03083,0.03083,0.31429,0.34143,0.27333,0.03083,0.12875,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.029,0.029,0.45857,0.19,0.05,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316","probabilities":[0.03214,0.03214,0.484,0.03214,0.585,0.05,0.1391,0.45143,0.19,0.45143,0.07026,0.01357],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908","probabilities":[0.029,0.029,0.03214,0.94143,0.08357,0.71429,0.03083,0.37429,0.21,0.03083,0.34833,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122","probabilities":[0.94583,0.03917,0.93857,0.05071,0.55286,0.51667,0.56417,0.47286,0.4375,0.352,0.18417,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552","probabilities":[0.02917,0.02917,0.93857,0.03071,0.64286,0.592,0.86,0.88571,0.93071,0.88571,0.465,0.24],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267","probabilities":[0.02917,0.02917,0.03071,0.03071,0.49071,0.49071,0.49071,0.32643,0.13583,0.05037,0.02917,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200","probabilities":[0.51333,0.45,0.80429,0.04143,0.75714,0.29125,0.77429,0.3,0.815,0.52,0.292,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926","probabilities":[0.02375,0.02375,0.08214,0.08214,0.02375,0.02375,0.139,0.03214,0.36917,0.07333,0.23583,0.0075],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736","probabilities":[0.03071,0.03071,0.5,0.03071,0.1,0.1,0.03071,0.67857,0.315,0.02917,0.0075,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219","probabilities":[0.03071,0.03071,0.93071,0.93071,0.51333,0.51333,0.51333,0.03071,0.09125,0.03071,0.04577,0.01171],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","probabilities":[0.03083,0.03083,0.165,0.03929,0.29714,0.29714,0.80571,0.03071,0.29813,0.03071,0.225,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38893522232848525819919092148478879455","probabilities":[0.11625,0.101,0.03214,0.50143,0.135,0.16667,0.12875,0.03214,0.09375,0.03071,0.02625,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373","probabilities":[0.02917,0.18857,0.94071,0.03071,0.81357,0.67667,0.785,0.45571,0.46,0.795,0.19333,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144","probabilities":[0.49,0.51143,0.93143,0.49143,0.1,0.1,0.1,0.03214,0.19417,0.04668,0.11625,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948","probabilities":[0.94214,0.51143,0.03071,0.03071,0.57286,0.405,0.94143,0.45857,0.265,0.15643,0.93714,0.83667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161","probabilities":[0.81643,0.60667,0.705,0.94143,0.4725,0.6,0.599,0.83571,0.538,0.86357,0.93857,0.94214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427","probabilities":[0.02917,0.02917,0.03071,0.03071,0.03071,0.03071,0.03071,0.09071,0.07583,0.02917,0.73571,0.93143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752","probabilities":[0.76357,0.0375,0.85714,0.03786,0.75143,0.33333,0.33333,0.74857,0.93,0.86214,0.08,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764","probabilities":[0.03071,0.03071,0.93857,0.03071,0.16667,0.05,0.05,0.03071,0.07417,0.04553,0.027,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402","probabilities":[0.84917,0.047,0.925,0.03786,0.035,0.035,0.05167,0.51167,0.435,0.438,0.05167,0.035],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.01806,0.01833,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","probabilities":[0.02833,0.03,0.03,0.03,0.0325,0.0325,0.02833,0.33429,0.185,0.04469,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592","probabilities":[0.02917,0.02917,0.93714,0.03071,0.02375,0.02375,0.518,0.03071,0.09125,0.0468,0.02375,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693","probabilities":[0.03071,0.03786,0.87071,0.10214,0.87143,0.66714,0.85071,0.45857,0.92143,0.45857,0.04214,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837","probabilities":[0.02917,0.29929,0.71571,0.03071,0.01833,0.01833,0.027,0.03071,0.29813,0.0443,0.02625,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315","probabilities":[0.03071,0.03071,0.93214,0.03071,0.1,0.1,0.49857,0.73429,0.31667,0.08,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037","probabilities":[0.54429,0.0375,0.29286,0.635,0.56571,0.584,0.47214,0.44429,0.27667,0.03786,0.065,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.02917,0.03214,0.09375,0.05,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872","probabilities":[0.03071,0.03071,0.94214,0.03071,0.82714,0.04214,0.46071,0.94214,0.93929,0.80071,0.374,0.14167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994","probabilities":[0.03071,0.37571,0.94214,0.93857,0.94286,0.49714,0.68786,0.93857,0.93857,0.03071,0.64667,0.88929],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260","probabilities":[0.1,0.1,0.1,0.1,0.1,0.788,0.6,0.74143,0.92143,0.10307,0.43,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212","probabilities":[0.94286,0.45857,0.865,0.94143,0.12625,0.12625,0.0375,0.34143,0.305,0.2,0.03786,0.03786],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","probabilities":[0.44714,0.3425,0.93929,0.03071,0.02917,0.02917,0.02917,0.04214,0.07417,0.04667,0.1475,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415","probabilities":[0.79917,0.03214,0.94,0.93857,0.89833,0.325,0.26667,0.34286,0.43625,0.03083,0.92583,0.03875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414","probabilities":[0.03214,0.03214,0.49429,0.49429,0.1,0.1,0.1,0.74714,0.34833,0.86143,0.11625,0.11625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537","probabilities":[0.03214,0.03214,0.03214,0.03214,0.56,0.02625,0.64429,0.33429,0.13667,0.03214,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683","probabilities":[0.03214,0.03214,0.19286,0.03214,0.79429,0.70643,0.80286,0.65571,0.31667,0.92286,0.35417,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004","probabilities":[0.03083,0.02667,0.94,0.16667,0.35,0.2,0.25,0.88429,0.455,0.76,0.17667,0.168],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476","probabilities":[0.03083,0.135,0.03214,0.03214,0.03083,0.03083,0.41333,0.88083,0.3845,0.2875,0.86,0.935],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480","probabilities":[0.05,0.05,0.32929,0.05,0.1,0.1,0.57857,0.1,0.05,0.05,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876","probabilities":[0.03929,0.03929,0.1,0.92286,0.1,0.1,0.1,0.71214,0.35417,0.09093,0.815,0.94143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.42598518026540237824479232797503218019","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.45286,0.26167,0.04836,0.024,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214","probabilities":[0.18786,0.03417,0.93143,0.03875,0.6,0.03875,0.85286,0.81786,0.3875,0.62,0.565,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744","probabilities":[0.03083,0.56,0.73,0.03214,0.03083,0.03083,0.73071,0.03929,0.29938,0.03917,0.38571,0.024],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887","probabilities":[0.0325,0.03599,0.03857,0.03857,0.0703,0.06915,0.08536,0.10531,0.05371,0.0395,0.035,0.0925],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741","probabilities":[0.43333,0.03357,0.50357,0.50357,0.1,0.1,0.2,0.03357,0.09375,0.85429,0.83286,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838","probabilities":[0.03083,0.03083,0.03571,0.03571,0.03214,0.03214,0.62357,0.03214,0.0925,0.03083,0.1775,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343","probabilities":[0.94214,0.03083,0.94214,0.94,0.03917,0.03917,0.03929,0.93929,0.3875,0.94143,0.0625,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736","probabilities":[0.26143,0.44714,0.93929,0.50357,0.48929,0.1625,0.09,0.47286,0.4625,0.51714,0.065,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163","probabilities":[0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.57643,0.21875,0.05656,0.93571,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","probabilities":[0.03083,0.03083,0.93929,0.59143,0.366,0.2275,0.225,0.73786,0.4835,0.03214,0.0125,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","probabilities":[0.93429,0.03917,0.54357,0.78429,0.45357,0.367,0.05125,0.45571,0.27333,0.08449,0.27214,0.03929],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44149403030093858908988179411336839587","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.03214,0.44857,0.25667,0.02,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208","probabilities":[0.48583,0.0325,0.58786,0.58786,0.04667,0.04667,0.49,0.94214,0.92143,0.835,0.03357,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889","probabilities":[0.2,0.2,0.43143,0.43143,0.8475,0.8475,0.8475,0.53333,0.33333,0.68357,0.93929,0.93417],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569","probabilities":[0.02917,0.0375,0.94143,0.03071,0.81357,0.745,0.94214,0.88929,0.44312,0.152,0.318,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530","probabilities":[0.02917,0.02917,0.03071,0.02917,0.02917,0.02917,0.02375,0.03,0.035,0.03,0.12875,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025","probabilities":[0.88786,0.03071,0.03071,0.55786,0.02375,0.02375,0.02375,0.03,0.115,0.03,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","probabilities":[0.03071,0.49143,0.84857,0.49429,0.68714,0.64833,0.5,0.03,0.21,0.85286,0.04667,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632","probabilities":[0.05,0.05,0.05,0.05,0.05,0.05,0.1,0.05,0.34,0.05,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959","probabilities":[0.45429,0.03786,0.50143,0.50143,0.34,0.05,0.69333,0.03714,0.21,0.07428,0.42,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641","probabilities":[0.08214,0.03071,0.49143,0.53143,0.16667,0.16667,0.16667,0.03,0.07333,0.16667,0.02375,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","probabilities":[0.94214,0.64,0.03071,0.03071,0.03071,0.03071,0.03071,0.93929,0.35417,0.03,0.93929,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674","probabilities":[0.03214,0.03214,0.94,0.03214,0.56,0.03214,0.52429,0.33571,0.155,0.03214,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663","probabilities":[0.101,0.101,0.03214,0.03214,0.03083,0.03083,0.02,0.47286,0.82,0.45857,0.09083,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662","probabilities":[0.9325,0.945,0.63429,0.74286,0.88857,0.81214,0.73,0.51286,0.34333,0.03214,0.93143,0.93429],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46282234455466954264170065462242039568","probabilities":[0.05,0.142,0.51571,0.12804,0.55833,0.33667,0.79714,0.68,0.90857,0.79429,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523","probabilities":[0.03214,0.09786,0.03214,0.31286,0.39,0.09,0.34429,0.34286,0.14625,0.79429,0.67429,0.225],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766","probabilities":[0.03214,0.03214,0.81286,0.03214,0.37333,0.545,0.60833,0.33571,0.275,0.03214,0.03214,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46713978512945139851373191422321462143","probabilities":[0.09083,0.03083,0.37429,0.94143,0.72571,0.57429,0.39571,0.93857,0.93714,0.03214,0.03875,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639","probabilities":[0.03214,0.03929,0.93929,0.03214,0.515,0.05,0.49143,0.45857,0.15667,0.79286,0.48571,0.87143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949","probabilities":[0.27667,0.039,0.45143,0.93143,0.28929,0.75857,0.69714,0.93857,0.38833,0.91286,0.60714,0.625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","probabilities":[0.05,0.05,0.21714,0.21714,0.12875,0.12875,0.03875,0.03357,0.09375,0.03357,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787","probabilities":[0.03083,0.03083,0.5,0.93929,0.48714,0.56167,0.65429,0.03214,0.29938,0.04667,0.478,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037","probabilities":[0.0325,0.0325,0.03357,0.03357,0.05,0.50917,0.05,0.45857,0.14,0.03357,0.67786,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642","probabilities":[0.53357,0.0525,0.94,0.73643,0.3225,0.46714,0.46,0.45714,0.46125,0.03214,0.49333,0.03875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953","probabilities":[0.03083,0.03083,0.35714,0.03214,0.029,0.029,0.029,0.03929,0.07583,0.05,0.029,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684","probabilities":[0.03214,0.03214,0.49286,0.03214,0.58667,0.05,0.82143,0.03214,0.0925,0.44,0.75571,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825","probabilities":[0.03083,0.03083,0.04286,0.04286,0.03083,0.03083,0.45,0.03929,0.19583,0.45714,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710","probabilities":[0.79083,0.03875,0.93929,0.93714,0.83571,0.74143,0.82167,0.82429,0.45333,0.14301,0.54143,0.25875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959","probabilities":[0.03214,0.03083,0.03214,0.93857,0.41875,0.74286,0.39375,0.75429,0.93857,0.75429,0.03214,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552","probabilities":[0.09083,0.82571,0.12857,0.03083,0.08357,0.08357,0.09071,0.03214,0.16938,0.09083,0.09083,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331","probabilities":[0.03083,0.03083,0.04357,0.04357,0.03833,0.52,0.436,0.44429,0.15333,0.05169,0.478,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.48160569779413970442472500116176876660","probabilities":[0.03214,0.03083,0.49286,0.13333,0.05,0.05,0.42857,0.67857,0.92,0.8,0.08357,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799","probabilities":[0.03214,0.03214,0.03214,0.03214,0.63143,0.63143,0.93357,0.45857,0.215,0.45857,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","probabilities":[0.03083,0.35333,0.94,0.04286,0.039,0.039,0.039,0.38143,0.15333,0.03083,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545","probabilities":[0.03214,0.03214,0.94214,0.03214,0.80571,0.56,0.88857,0.93786,0.83571,0.03214,0.03083,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244","probabilities":[0.03083,0.03214,0.03214,0.03214,0.0375,0.0375,0.03083,0.33429,0.25667,0.04534,0.03083,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547","probabilities":[0.03083,0.03083,0.03214,0.03214,0.0375,0.0375,0.58833,0.04143,0.30125,0.03214,0.315,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727","probabilities":[0.03083,0.29333,0.03214,0.03214,0.03875,0.03875,0.111,0.87143,0.404,0.09592,0.8225,0.33071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329","probabilities":[0.26,0.18583,0.03929,0.92143,0.03875,0.25875,0.1,0.88643,0.49562,0.1975,0.76071,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513","probabilities":[0.03214,0.03214,0.75,0.03214,0.81714,0.8,0.71286,0.88857,0.94071,0.15072,0.03833,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668","probabilities":[0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.03929,0.03214,0.07583,0.03849,0.17214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","probabilities":[0.029,0.029,0.37571,0.03214,0.81857,0.03875,0.66857,0.45714,0.43625,0.07,0.029,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550","probabilities":[0.59167,0.03917,0.22,0.43167,0.292,0.45143,0.46071,0.45857,0.3075,0.45857,0.12875,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","probabilities":[0.03083,0.03214,0.03214,0.03214,0.08357,0.08357,0.37571,0.03214,0.09375,0.03961,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310","probabilities":[0.03214,0.03214,0.49429,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.03214,0.03448,0.01065],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784","probabilities":[0.815,0.50167,0.94143,0.25429,0.745,0.03875,0.05125,0.82714,0.27,0.17334,0.92286,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845","probabilities":[0.03214,0.03214,0.93929,0.03214,0.83643,0.5675,0.1,0.7,0.34833,0.11431,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102","probabilities":[0.03214,0.03083,0.03214,0.03214,0.04627,0.04208,0.06555,0.45857,0.27667,0.04913,0.03214,0.01276],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053","probabilities":[0.06256,0.05844,0.93714,0.11482,0.4,0.06599,0.2,0.74714,0.93,0.75929,0.14667,0.1],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.51469564445249084310991187819410005207","probabilities":[0.03083,0.03083,0.135,0.135,0.03214,0.03214,0.03214,0.03214,0.07583,0.03895,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131","probabilities":[0.94583,0.08357,0.94143,0.50357,0.14839,0.09779,0.08718,0.67643,0.8175,0.80071,0.34698,0.03758],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","probabilities":[0.03214,0.50917,0.03214,0.03214,0.03875,0.03875,0.03875,0.34286,0.33125,0.03214,0.188,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.51831304809871808790657397599860793132","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.03214,0.03214,0.16667,0.16667,0.02625,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915","probabilities":[0.03214,0.03214,0.49643,0.03214,0.05,0.05,0.53,0.03214,0.07583,0.74071,0.3925,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065","probabilities":[0.03083,0.03083,0.42571,0.5,0.03917,0.03917,0.75429,0.08,0.08,0.03214,0.03833,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152","probabilities":[0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832","probabilities":[0.03214,0.03083,0.03214,0.03214,0.04333,0.03083,0.05667,0.37429,0.13667,0.105,0.03875,0.55857],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146","probabilities":[0.03214,0.26333,0.94143,0.03214,0.94071,0.03083,0.92429,0.93929,0.33125,0.03214,0.27167,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501","probabilities":[0.81571,0.44714,0.94,0.50143,0.85714,0.14,0.52143,0.31,0.34,0.46071,0.84,0.044],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03083,0.03083,0.03083,0.105,0.37688,0.52,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643","probabilities":[0.029,0.029,0.26333,0.26333,0.24929,0.039,0.746,0.53429,0.81571,0.45857,0.16,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.029,0.029,0.03214,0.09375,0.05,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883","probabilities":[0.03083,0.03083,0.03214,0.03083,0.695,0.88786,0.87857,0.93929,0.23417,0.03214,0.1075,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","probabilities":[0.03083,0.0525,0.88857,0.17214,0.039,0.039,0.03917,0.45571,0.15333,0.03917,0.03214,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763","probabilities":[0.03083,0.03083,0.308,0.32333,0.55583,0.72429,0.83571,0.38143,0.18667,0.03214,0.36667,0.045],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","probabilities":[0.03214,0.805,0.93214,0.03214,0.12875,0.12875,0.27875,0.47286,0.34333,0.72857,0.85571,0.14125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219","probabilities":[0.03083,0.03083,0.05071,0.05071,0.45857,0.029,0.029,0.03214,0.07583,0.03083,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985","probabilities":[0.03083,0.03083,0.93929,0.03214,0.03083,0.03083,0.03214,0.03214,0.09375,0.45857,0.02167,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557","probabilities":[0.03083,0.03083,0.24286,0.03214,0.03214,0.03214,0.88143,0.03214,0.0925,0.03083,0.03083,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025","probabilities":[0.04667,0.03875,0.03929,0.03929,0.039,0.039,0.039,0.45857,0.19,0.04667,0.0375,0.0225],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","probabilities":[0.03083,0.09083,0.93,0.08357,0.02625,0.02167,0.029,0.45857,0.15667,0.04667,0.101,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593","probabilities":[0.55583,0.49143,0.78286,0.82571,0.88143,0.726,0.88143,0.46903,0.44,0.67714,0.2875,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587","probabilities":[0.03083,0.03083,0.03214,0.30571,0.58571,0.03214,0.6,0.33143,0.15333,0.51,0.03083,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870","probabilities":[0.05,0.45167,0.654,0.05,0.1,0.1,0.775,0.2,0.37333,0.05,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518","probabilities":[0.94143,0.59786,0.93929,0.92857,0.65214,0.66917,0.63786,0.34143,0.21,0.07512,0.93929,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642","probabilities":[0.26071,0.03083,0.03214,0.03214,0.585,0.51333,0.75857,0.08,0.08,0.03,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.029,0.101,0.8825,0.38188,0.07367,0.94,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","probabilities":[0.7,0.03917,0.94,0.03786,0.86,0.55857,0.73714,0.82286,0.385,0.03143,0.175,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616","probabilities":[0.02917,0.02917,0.12571,0.03071,0.02833,0.02833,0.03,0.34143,0.25667,0.04643,0.06036,0.01268],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770","probabilities":[0.93286,0.039,0.58429,0.03917,0.03786,0.03786,0.54714,0.49571,0.15333,0.03,0.785,0.91429],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","probabilities":[0.94214,0.03071,0.94143,0.93857,0.2,0.60714,0.2875,0.74857,0.3625,0.03,0.93571,0.08143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879","probabilities":[0.03071,0.56429,0.94214,0.03071,0.51857,0.51857,0.94214,0.93857,0.92857,0.91714,0.262,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860","probabilities":[0.037,0.037,0.62357,0.03786,0.75929,0.2525,0.037,0.45571,0.15333,0.79286,0.87857,0.03875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068","probabilities":[0.20167,0.34,0.64714,0.57714,0.312,0.226,0.73571,0.34571,0.26625,0.85429,0.12083,0.145],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754","probabilities":[0.12875,0.41583,0.72143,0.03929,0.51286,0.1,0.15425,0.93214,0.5475,0.833,0.85286,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856","probabilities":[0.02375,0.02375,0.027,0.027,0.03625,0.03625,0.03625,0.03,0.09,0.04013,0.76714,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475","probabilities":[0.03833,0.03833,0.93214,0.039,0.38333,0.03875,0.0625,0.47286,0.70143,0.09724,0.665,0.854],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","probabilities":[0.525,0.03786,0.93,0.93,0.037,0.037,0.45,0.94,0.93214,0.13793,0.15767,0.03326],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195","probabilities":[0.17643,0.0325,0.11071,0.03786,0.03357,0.03357,0.49857,0.03143,0.07333,0.03167,0.031,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.03071,0.44857,0.15833,0.0225,0.01833,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.03071,0.09125,0.05,0.02167,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058","probabilities":[0.03083,0.03083,0.93857,0.93857,0.52143,0.03214,0.03214,0.03429,0.0975,0.07788,0.04899,0.0108],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485","probabilities":[0.94214,0.03214,0.94143,0.94,0.88786,0.52429,0.57643,0.67857,0.92286,0.14067,0.4175,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235","probabilities":[0.03083,0.03083,0.94,0.93167,0.03929,0.03929,0.45857,0.45857,0.81714,0.16667,0.127,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","probabilities":[0.935,0.03786,0.93786,0.93929,0.525,0.43333,0.10167,0.47286,0.27667,0.03071,0.71643,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357","probabilities":[0.03071,0.03071,0.03071,0.03071,0.03071,0.71857,0.73429,0.46,0.1925,0.44857,0.49357,0.02917],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162","probabilities":[0.03071,0.03071,0.93929,0.10357,0.19333,0.17667,0.51714,0.37167,0.155,0.06102,0.01833,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796","probabilities":[0.02917,0.03071,0.03071,0.03071,0.0325,0.0325,0.0375,0.33571,0.1875,0.04479,0.03071,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331","probabilities":[0.76667,0.0375,0.66071,0.03786,0.50333,0.44857,0.44857,0.07286,0.51938,0.03786,0.1,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841","probabilities":[0.027,0.54643,0.93857,0.03071,0.355,0.50667,0.44857,0.03929,0.2125,0.03071,0.109,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675","probabilities":[0.4125,0.03214,0.03214,0.03214,0.02625,0.02625,0.029,0.51857,0.20375,0.03214,0.029,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","probabilities":[0.03071,0.03071,0.49429,0.94143,0.265,0.04667,0.045,0.27286,0.15667,0.0783,0.02375,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679","probabilities":[0.027,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.38429,0.185,0.32286,0.01833,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799","probabilities":[0.03071,0.03071,0.93857,0.49429,0.1,0.1,0.1,0.33571,0.275,0.03083,0.02375,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893","probabilities":[0.03071,0.03071,0.92429,0.03071,0.45857,0.03071,0.03071,0.45857,0.19,0.03071,0.03071,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008","probabilities":[0.02917,0.03071,0.93929,0.03071,0.49143,0.49143,0.49143,0.34429,0.215,0.08929,0.02375,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62406588098851994968957074186291720431","probabilities":[0.02917,0.02917,0.94143,0.24857,0.57929,0.46167,0.546,0.94143,0.93929,0.835,0.135,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272","probabilities":[0.37571,0.45857,0.77714,0.61571,0.2,0.2,0.2,0.74286,0.49375,0.10761,0.44,0.37333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983","probabilities":[0.02917,0.02917,0.03071,0.03071,0.027,0.027,0.099,0.34143,0.356,0.45857,0.02917,0.11],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62918046703989259691005447883746340916","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.027,0.45857,0.14,0.05,0.93857,0.03625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","probabilities":[0.02375,0.02375,0.94071,0.93857,0.88786,0.08214,0.02917,0.45857,0.27667,0.02917,0.08,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.62948920639798990757395955917890550512","probabilities":[0.03071,0.03786,0.94143,0.03071,0.02917,0.02917,0.027,0.93857,0.32,0.03071,0.16667,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","probabilities":[0.03214,0.03083,0.03214,0.03214,0.09917,0.09917,0.73786,0.03214,0.09375,0.06602,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.63280826430574305683941548157934410641","probabilities":[0.03214,0.03083,0.83429,0.82,0.21,0.73714,0.2,0.75,0.2775,0.10781,0.03214,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941","probabilities":[0.03214,0.15786,0.01333,0.33143,0.03214,0.03083,0.03214,0.44857,0.25667,0.105,0.42167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.63584266247144369196126771996126173497","probabilities":[0.03214,0.03214,0.36714,0.03214,0.51286,0.454,0.35,0.03214,0.09375,0.44857,0.03214,0.15786],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906","probabilities":[0.03083,0.45,0.03214,0.03214,0.03083,0.03083,0.56214,0.45857,0.14,0.03214,0.02625,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316","probabilities":[0.03214,0.03214,0.49429,0.49429,0.05,0.2,0.04667,0.03214,0.09375,0.0426,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373","probabilities":[0.26333,0.039,0.86,0.05071,0.57667,0.36333,0.45,0.04286,0.2625,0.11214,0.15833,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715","probabilities":[0.94214,0.03214,0.94214,0.03214,0.94214,0.85143,0.765,0.72857,0.92143,0.88643,0.225,0.08357],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010","probabilities":[0.03214,0.03214,0.93857,0.03214,0.82714,0.58667,0.87357,0.45857,0.90857,0.03214,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724","probabilities":[0.03083,0.03083,0.03929,0.03929,0.54071,0.24583,0.93429,0.45857,0.91,0.45857,0.04667,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223","probabilities":[0.69714,0.09917,0.03917,0.03917,0.1,0.622,0.64,0.86333,0.28,0.775,0.46143,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005","probabilities":[0.03214,0.03214,0.86786,0.10357,0.03929,0.03214,0.03214,0.03214,0.07583,0.45857,0.04357,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643","probabilities":[0.029,0.024,0.5,0.0,0.55286,0.13333,0.16667,0.31286,0.21,0.342,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455","probabilities":[0.88714,0.03214,0.03214,0.03214,0.10869,0.12228,0.77667,0.03214,0.11875,0.74286,0.93714,0.07928],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554","probabilities":[0.03083,0.03083,0.03214,0.49286,0.03083,0.03083,0.03214,0.03429,0.07833,0.05,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719","probabilities":[0.03214,0.93357,0.03917,0.03917,0.111,0.03875,0.86571,0.68,0.35,0.03214,0.93714,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666","probabilities":[0.03214,0.03214,0.03214,0.93857,0.46071,0.87143,0.87857,0.94143,0.93857,0.13526,0.045,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674","probabilities":[0.03214,0.03083,0.83429,0.03214,0.745,0.1,0.59286,0.68,0.49,0.12259,0.37333,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980","probabilities":[0.03214,0.49143,0.93857,0.03214,0.72143,0.51,0.45857,0.768,0.515,0.1336,0.56929,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.66077420344931334696112616504179985591","probabilities":[0.02917,0.02917,0.92,0.03071,0.58,0.65833,0.76571,0.02917,0.09125,0.02917,0.29,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887","probabilities":[0.59,0.03071,0.03071,0.03071,0.02375,0.02375,0.027,0.03071,0.09125,0.03071,0.11375,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.03083,0.05,0.05,0.05,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755","probabilities":[0.029,0.029,0.93429,0.34833,0.94583,0.3725,0.85214,0.54,0.434,0.59857,0.60143,0.126],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524","probabilities":[0.46071,0.03214,0.93857,0.03214,0.47429,0.53333,0.52143,0.33571,0.1875,0.06051,0.04667,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258","probabilities":[0.03083,0.03083,0.03071,0.18,0.02917,0.02917,0.03071,0.33714,0.19,0.04527,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848","probabilities":[0.03071,0.03071,0.93143,0.03071,0.85071,0.4375,0.43,0.68,0.93143,0.72714,0.27,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306","probabilities":[0.50143,0.037,0.73357,0.39071,0.65333,0.65,0.50417,0.47286,0.15667,0.45857,0.035,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","probabilities":[0.42,0.03071,0.93857,0.03071,0.58,0.58,0.58,0.12841,0.08274,0.03,0.06547,0.01217],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147","probabilities":[0.68143,0.45143,0.02833,0.58071,0.02833,0.39357,0.176,0.34143,0.21,0.04582,0.93714,0.93214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68142054543837964766571710567214307830","probabilities":[0.02917,0.02917,0.05,0.328,0.03071,0.03071,0.03071,0.03,0.09,0.05,0.54167,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436","probabilities":[0.55333,0.03083,0.50143,0.05,0.3125,0.693,0.88143,0.68,0.93143,0.74857,0.27,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150","probabilities":[0.247,0.037,0.93,0.03786,0.191,0.109,0.47333,0.81,0.28,0.25,0.535,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462","probabilities":[0.03071,0.03071,0.03071,0.03071,0.55833,0.55833,0.55833,0.45143,0.19,0.03071,0.01,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600","probabilities":[0.03071,0.03071,0.93786,0.03071,0.05,0.05,0.05,0.03071,0.09125,0.04472,0.01833,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68707023438918237611039069622720041636","probabilities":[0.02917,0.02917,0.03071,0.03071,0.71167,0.0975,0.0375,0.45714,0.155,0.0588,0.81143,0.07],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.58714,0.03071,0.37562,0.03071,0.02917,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","probabilities":[0.88786,0.81214,0.93857,0.03071,0.03625,0.03625,0.03625,0.88857,0.485,0.10398,0.79286,0.94143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960","probabilities":[0.03071,0.03071,0.03071,0.93071,0.05,0.05,0.05,0.03071,0.07417,0.04041,0.027,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819","probabilities":[0.03071,0.03071,0.94071,0.03071,0.94,0.93929,0.93929,0.93857,0.39,0.94071,0.28,0.88571],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455","probabilities":[0.88857,0.02917,0.03071,0.03071,0.45143,0.0,0.56143,0.45714,0.275,0.03071,0.26445,0.03746],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384","probabilities":[0.02917,0.50143,0.57929,0.37714,0.60571,0.1825,0.87357,0.74857,0.92286,0.86143,0.36667,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579","probabilities":[0.03083,0.4525,0.11429,0.03214,0.63571,0.08357,0.34429,0.88643,0.91786,0.11973,0.03083,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063","probabilities":[0.94214,0.029,0.93929,0.03214,0.1,0.1,0.03917,0.93929,0.83643,0.1148,0.03214,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260","probabilities":[0.03083,0.03083,0.03214,0.03214,0.59167,0.59167,0.8225,0.93929,0.3875,0.03214,0.57833,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808","probabilities":[0.03214,0.03214,0.03214,0.03214,0.79571,0.57286,0.85071,0.43571,0.21,0.03214,0.47167,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.09083,0.58571,0.45,0.1875,0.05075,0.0,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314","probabilities":[0.02625,0.02625,0.94143,0.03214,0.88143,0.02625,0.52,0.45857,0.27667,0.03083,0.665,0.81143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70831735578941318362881379802693293952","probabilities":[0.85917,0.1,0.94143,0.38,0.88929,0.37714,0.26917,0.68857,0.385,0.17333,0.56286,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526","probabilities":[0.39583,0.44,0.94143,0.49286,0.2,0.05,0.50071,0.47286,0.2175,0.29,0.132,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.56214,0.03214,0.19583,0.11,0.055,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104","probabilities":[0.03083,0.03083,0.65714,0.65714,0.02625,0.02625,0.02625,0.03214,0.11875,0.77357,0.265,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315","probabilities":[0.03214,0.03214,0.94143,0.03214,0.03917,0.03083,0.50786,0.93929,0.32083,0.03214,0.93929,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261","probabilities":[0.31714,0.04583,0.93214,0.03357,0.44714,0.44714,0.81071,0.51286,0.482,0.03357,0.065,0.045],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846","probabilities":[0.88786,0.685,0.03929,0.03929,0.65071,0.183,0.57143,0.67857,0.34833,0.45714,0.93429,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437","probabilities":[0.03071,0.03071,0.03071,0.03071,0.02917,0.02917,0.37571,0.33286,0.20125,0.33286,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02917,0.02917,0.02917,0.03,0.15833,0.15833,0.11375,0.11375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190","probabilities":[0.21333,0.0375,0.50714,0.0375,0.74143,0.1,0.2,0.78,0.22833,0.74,0.305,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597","probabilities":[0.03071,0.44143,0.03071,0.03071,0.91429,0.03071,0.55857,0.03,0.115,0.50857,0.22857,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666","probabilities":[0.02917,0.02917,0.93714,0.03071,0.02917,0.02917,0.03071,0.03,0.07333,0.05,0.01833,0.0075],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639","probabilities":[0.03083,0.03083,0.93714,0.04286,0.0375,0.0375,0.039,0.45714,0.1875,0.04667,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230","probabilities":[0.03083,0.03083,0.19143,0.04357,0.11625,0.39929,0.101,0.03214,0.19583,0.03083,0.101,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331","probabilities":[0.029,0.029,0.03214,0.03214,0.03083,0.03083,0.111,0.45714,0.1875,0.80143,0.02625,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.73325959289998105909842627876607163467","probabilities":[0.81643,0.03083,0.03214,0.03214,0.029,0.101,0.029,0.8825,0.40688,0.0667,0.93,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803","probabilities":[0.03083,0.03083,0.03214,0.51571,0.03214,0.45857,0.03214,0.04357,0.09375,0.03956,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320","probabilities":[0.03083,0.03083,0.03214,0.87571,0.03917,0.66571,0.78167,0.93929,0.35417,0.10667,0.03875,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951","probabilities":[0.03083,0.03083,0.03083,0.03083,0.03083,0.03083,0.029,0.25143,0.346,0.79357,0.029,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465","probabilities":[0.94214,0.57714,0.93,0.92,0.111,0.111,0.09917,0.34143,0.27333,0.06186,0.93714,0.09071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661","probabilities":[0.37,0.03357,0.84143,0.03917,0.49857,0.045,0.04583,0.45714,0.14625,0.395,0.04667,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","probabilities":[0.03214,0.03214,0.03214,0.03214,0.02625,0.02625,0.09083,0.03214,0.09375,0.03214,0.117,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309","probabilities":[0.029,0.029,0.50143,0.03214,0.4175,0.05,0.56667,0.03214,0.09375,0.04892,0.598,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669","probabilities":[0.09083,0.03083,0.86286,0.02,0.68429,0.36143,0.60714,0.47286,0.15667,0.70714,0.02625,0.029],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554","probabilities":[0.03214,0.0475,0.03214,0.61357,0.03083,0.03083,0.03214,0.86714,0.6275,0.10022,0.93857,0.82714],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088","probabilities":[0.03214,0.03214,0.03214,0.03214,0.85857,0.56,0.85214,0.34571,0.2675,0.039,0.02625,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76190704217737835083438638928129775625","probabilities":[0.03214,0.03214,0.93929,0.37429,0.57857,0.33,0.53857,0.46,0.2175,0.80143,0.08357,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76344826994380951498651448092980303697","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.03083,0.33714,0.26,0.05,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76525107909962775502207829870873494024","probabilities":[0.47929,0.0325,0.93143,0.03357,0.03875,0.03875,0.65714,0.45857,0.215,0.06118,0.03833,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76538737802277241976971950690663079376","probabilities":[0.41,0.45857,0.80714,0.57929,0.65429,0.2,0.2175,0.68,0.92286,0.12882,0.494,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120","probabilities":[0.94214,0.94143,0.03214,0.03214,0.03083,0.03083,0.03214,0.74286,0.92,0.52,0.93714,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992","probabilities":[0.56083,0.03917,0.93,0.04643,0.74429,0.44,0.3775,0.67571,0.93143,0.79857,0.175,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323","probabilities":[0.03214,0.03214,0.86429,0.03083,0.127,0.127,0.436,0.81143,0.28,0.03214,0.05,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789","probabilities":[0.94214,0.45857,0.03214,0.93857,0.04667,0.05,0.03214,0.34143,0.27333,0.0465,0.30625,0.08357],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.63857,0.14333,0.04667,0.03214,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536","probabilities":[0.24286,0.103,0.48571,0.03357,0.59286,0.05,0.2,0.45857,0.27667,0.45857,0.05,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695","probabilities":[0.111,0.194,0.93143,0.26357,0.14667,0.1,0.7075,0.87143,0.23333,0.74143,0.28,0.03875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03083,0.03083,0.45857,0.30143,0.12875,0.03214,0.45857,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044","probabilities":[0.88714,0.03214,0.94143,0.03214,0.845,0.845,0.845,0.47286,0.92143,0.66571,0.15035,0.03537],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136","probabilities":[0.03,0.03071,0.03071,0.03071,0.02917,0.02917,0.645,0.03,0.09,0.03,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839","probabilities":[0.30571,0.03214,0.45071,0.04143,0.80643,0.67143,0.61714,0.53429,0.4375,0.93214,0.328,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809","probabilities":[0.02917,0.037,0.94143,0.04667,0.61786,0.05,0.08,0.33,0.64667,0.09427,0.84429,0.93286],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471","probabilities":[0.02833,0.02833,0.03,0.03,0.02833,0.02833,0.02833,0.33429,0.15833,0.0375,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146","probabilities":[0.40667,0.0375,0.5,0.03786,0.09143,0.09143,0.05286,0.03714,0.09,0.1,0.62786,0.03143],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176","probabilities":[0.027,0.035,0.545,0.045,0.1,0.46,0.46,0.37429,0.345,0.05616,0.045,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.77889800516853576877948593628420228178","probabilities":[0.02833,0.02833,0.1725,0.03,0.026,0.026,0.026,0.64,0.28167,0.86286,0.03,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.78072765790191594685273447833272737429","probabilities":[0.02833,0.02833,0.93857,0.03,0.44571,0.42857,0.42857,0.45857,0.27667,0.06961,0.06312,0.01306],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952","probabilities":[0.02833,0.03,0.03,0.03,0.09286,0.49143,0.87857,0.03,0.09,0.04136,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268","probabilities":[0.33357,0.03917,0.136,0.04286,0.64571,0.044,0.04333,0.47286,0.925,0.66429,0.77857,0.91571],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756","probabilities":[0.03214,0.03214,0.03214,0.03214,0.16667,0.16667,0.16667,0.05,0.05,0.05,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052","probabilities":[0.05,0.05,0.91786,0.1,0.58143,0.05,0.05,0.81,0.22667,0.11291,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.78800983923143929298359366572020370031","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.029,0.03214,0.07583,0.05,0.12875,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317","probabilities":[0.03083,0.03083,0.94143,0.03214,0.59857,0.53571,0.59714,0.49857,0.3475,0.77143,0.055,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.79326627674334637493320764205397279674","probabilities":[0.03083,0.03083,0.93929,0.03214,0.03083,0.03083,0.03214,0.04286,0.2,0.08,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.01813,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807","probabilities":[0.03083,0.03083,0.04286,0.93857,0.03083,0.03083,0.03214,0.04286,0.2625,0.45857,0.029,0.14167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775","probabilities":[0.04982,0.04005,0.62857,0.07939,0.09191,0.04251,0.08324,0.17421,0.08381,0.0464,0.08009,0.01163],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492","probabilities":[0.03071,0.41714,0.93929,0.02625,0.03917,0.039,0.02917,0.45857,0.055,0.80143,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","probabilities":[0.24286,0.03214,0.03083,0.03083,0.31833,0.30143,0.06167,0.05071,0.2125,0.03214,0.1775,0.02875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798","probabilities":[0.02833,0.02833,0.03,0.03,0.02833,0.02833,0.02833,0.33429,0.21875,0.15833,0.0225,0.026],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286","probabilities":[0.25429,0.0325,0.03786,0.93857,0.51643,0.80786,0.0375,0.45857,0.215,0.71143,0.08929,0.14167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506","probabilities":[0.109,0.73286,0.94143,0.92357,0.94214,0.72333,0.5725,0.93857,0.77833,0.22559,0.59786,0.53214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715","probabilities":[0.03071,0.49143,0.03071,0.03071,0.50667,0.50667,0.05,0.105,0.2125,0.45857,0.027,0.02833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417","probabilities":[0.277,0.03929,0.09071,0.93,0.66143,0.6175,0.74714,0.81786,0.39062,0.82714,0.56143,0.75833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.81109405367395078564887780335570099673","probabilities":[0.03071,0.02917,0.93929,0.03071,0.79857,0.03071,0.03071,0.81214,0.4045,0.2,0.76714,0.16667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859","probabilities":[0.02917,0.02917,0.03071,0.03071,0.0375,0.0375,0.45,0.03071,0.09125,0.04667,0.027,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421","probabilities":[0.08627,0.07385,0.57714,0.27667,0.88857,0.08,0.2203,0.82286,0.385,0.17083,0.75167,0.746],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.01807,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086","probabilities":[0.03071,0.03071,0.03071,0.03071,0.2,0.2,0.2,0.03071,0.07417,0.05,0.81143,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894","probabilities":[0.618,0.037,0.03786,0.03786,0.037,0.03625,0.03625,0.45857,0.215,0.04652,0.18417,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508","probabilities":[0.31714,0.03214,0.42714,0.42714,0.215,0.356,0.88214,0.47286,0.27667,0.73,0.42333,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974","probabilities":[0.029,0.63143,0.93143,0.36857,0.21667,0.05,0.52143,0.03214,0.0925,0.07553,0.05125,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746","probabilities":[0.03083,0.03083,0.50286,0.49429,0.45286,0.154,0.45,0.57429,0.15667,0.75214,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.82587763533601760002619014353812074700","probabilities":[0.03214,0.03083,0.03,0.03,0.05,0.05,0.46,0.68,0.31667,0.08489,0.03,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758","probabilities":[0.03214,0.49333,0.91643,0.60857,0.427,0.37125,0.74429,0.74143,0.38333,0.16,0.04667,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400","probabilities":[0.71667,0.57083,0.94143,0.03929,0.88143,0.175,0.88929,0.47286,0.4875,0.1081,0.13417,0.05167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239","probabilities":[0.56143,0.09917,0.43714,0.43714,0.15833,0.15833,0.03833,0.15571,0.19333,0.04667,0.1,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936","probabilities":[0.16667,0.06064,0.19143,0.19143,0.55833,0.83643,0.88417,0.88857,0.911,0.69571,0.50083,0.25875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602","probabilities":[0.03214,0.03214,0.49286,0.03214,0.1,0.1,0.2,0.75,0.35167,0.09556,0.94,0.93857],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986","probabilities":[0.02833,0.02833,0.03,0.03,0.02833,0.02833,0.02833,0.45,0.25833,0.16667,0.0225,0.026],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057","probabilities":[0.02833,0.02833,0.03,0.03,0.02833,0.02833,0.02833,0.03,0.07333,0.04667,0.098,0.1125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783","probabilities":[0.93429,0.85583,0.58786,0.03929,0.03833,0.03833,0.03875,0.19857,0.125,0.53429,0.93929,0.64857],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488","probabilities":[0.03083,0.03083,0.67714,0.03214,0.02625,0.02625,0.47667,0.45714,0.12625,0.04667,0.77429,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724","probabilities":[0.03083,0.03083,0.03214,0.03214,0.51286,0.45143,0.52,0.45857,0.3475,0.03083,0.02625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","probabilities":[0.03214,0.03214,0.49429,0.49429,0.05,0.05,0.05,0.03214,0.09375,0.04265,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03917,0.039,0.94214,0.03214,0.38583,0.44286,0.03214,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","probabilities":[0.03214,0.03214,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.03214,0.1,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518","probabilities":[0.71333,0.03929,0.03929,0.54786,0.453,0.46857,0.42857,0.45571,0.27333,0.03357,0.82786,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","probabilities":[0.03083,0.03083,0.5,0.03083,0.475,0.22857,0.52286,0.45857,0.81786,0.07232,0.04667,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.03214,0.15687,0.03083,0.101,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","probabilities":[0.03083,0.03083,0.03214,0.03214,0.62571,0.45143,0.59429,0.03214,0.09938,0.03083,0.02625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892","probabilities":[0.37214,0.0325,0.36571,0.36571,0.396,0.265,0.54,0.11,0.265,0.03357,0.25875,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533","probabilities":[0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.93929,0.32083,0.03071,0.74786,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501","probabilities":[0.0365,0.03651,0.10112,0.05068,0.07617,0.04583,0.09067,0.12815,0.05816,0.74143,0.05521,0.01205],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091","probabilities":[0.03071,0.03071,0.56333,0.03071,0.02375,0.02375,0.02375,0.03071,0.19417,0.03071,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.86239793008693746155478763153154040128","probabilities":[0.02917,0.02917,0.03214,0.03214,0.05,0.05,0.05,0.2,0.05,0.05,0.03214,0.03083],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889","probabilities":[0.08917,0.0975,0.94071,0.35,0.88929,0.656,0.79,0.93857,0.71,0.09,0.24,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601","probabilities":[0.47833,0.45857,0.87714,0.94143,0.77571,0.248,0.424,0.68,0.92286,0.67714,0.1,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133","probabilities":[0.03083,0.09083,0.03214,0.03214,0.05,0.05,0.54929,0.34714,0.215,0.045,0.027,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","probabilities":[0.03083,0.03083,0.03214,0.03214,0.02625,0.02625,0.02625,0.03214,0.09375,0.05,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","probabilities":[0.03071,0.03071,0.94143,0.61071,0.6175,0.5425,0.41667,0.74857,0.402,0.74143,0.04667,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520","probabilities":[0.613,0.037,0.93214,0.63714,0.41929,0.363,0.33333,0.03929,0.39562,0.40667,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105","probabilities":[0.02917,0.02917,0.03071,0.03071,0.03071,0.32917,0.63286,0.37429,0.21,0.52,0.135,0.13333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","probabilities":[0.03071,0.03071,0.49429,0.03071,0.45143,0.05493,0.10667,0.11212,0.06015,0.03,0.05354,0.01194],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87415625824966124550466651648244553892","probabilities":[0.02917,0.02917,0.37786,0.0,0.027,0.027,0.03071,0.46071,0.19375,0.05153,0.01833,0.0075],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927","probabilities":[0.03071,0.03071,0.08214,0.03071,0.46,0.47714,0.47714,0.03,0.07333,0.03,0.11,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757","probabilities":[0.41857,0.0375,0.03786,0.93214,0.05167,0.61,0.05167,0.35857,0.27667,0.58,0.75857,0.035],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348","probabilities":[0.027,0.81714,0.12714,0.02917,0.84357,0.83083,0.87214,0.45857,0.27667,0.45143,0.18417,0.0075],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.01807,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615","probabilities":[0.08917,0.02917,0.03071,0.03071,0.05,0.05,0.38333,0.44429,0.13375,0.76071,0.04667,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015","probabilities":[0.94214,0.17083,0.94143,0.04214,0.8825,0.1,0.02375,0.75,0.32,0.47429,0.85286,0.61571],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212","probabilities":[0.67667,0.03625,0.24286,0.74071,0.12625,0.12625,0.109,0.88071,0.38375,0.08,0.81571,0.265],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294","probabilities":[0.03,0.43429,0.50143,0.03,0.08857,0.03714,0.03714,0.03,0.245,0.45857,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116","probabilities":[0.02917,0.02917,0.03071,0.03071,0.02375,0.02375,0.02375,0.03071,0.09125,0.05,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030","probabilities":[0.03214,0.03214,0.94143,0.03214,0.88786,0.52,0.88786,0.93929,0.52062,0.475,0.191,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.05429,0.19583,0.03083,0.029,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646","probabilities":[0.03445,0.03821,0.11809,0.07648,0.483,0.483,0.77333,0.22597,0.37333,0.07772,0.0528,0.01397],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89720428711655200442262503508409163392","probabilities":[0.03083,0.03083,0.03214,0.03214,0.34,0.6675,0.0375,0.03929,0.11875,0.06631,0.42929,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","probabilities":[0.031,0.031,0.031,0.031,0.03875,0.03875,0.03875,0.45857,0.14,0.05178,0.925,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712","probabilities":[0.05,0.05,0.93571,0.049,0.45857,0.03833,0.45857,0.88286,0.93571,0.83357,0.83643,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.90268709354762000249148795667759483876","probabilities":[0.03214,0.03083,0.03214,0.55167,0.1,0.1,0.88143,0.264,0.83,0.45571,0.04667,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634","probabilities":[0.03083,0.03083,0.93929,0.03214,0.111,0.03875,0.0375,0.03214,0.11875,0.18583,0.11625,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942","probabilities":[0.57286,0.03917,0.03929,0.03929,0.51857,0.03833,0.51857,0.03929,0.11875,0.03929,0.78143,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272","probabilities":[0.31714,0.03125,0.91571,0.035,0.935,0.63,0.63,0.47286,0.27667,0.10328,0.04667,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","probabilities":[0.37571,0.03929,0.93214,0.30143,0.94071,0.45667,0.61214,0.69,0.90857,0.2,0.03875,0.11875],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080","probabilities":[0.04135,0.05541,0.93929,0.11043,0.81857,0.2605,0.81857,0.75143,0.93143,0.75857,0.08125,0.01391],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979","probabilities":[0.03083,0.03083,0.03214,0.26333,0.03083,0.029,0.03214,0.03214,0.09375,0.03882,0.04917,0.01176],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454","probabilities":[0.94143,0.52,0.65929,0.03214,0.09083,0.08357,0.08357,0.57857,0.2225,0.05991,0.77143,0.14125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089","probabilities":[0.03214,0.03214,0.93857,0.03214,0.87167,0.87167,0.936,0.88857,0.93929,0.74429,0.16667,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365","probabilities":[0.16667,0.2,0.94,0.2,0.3,0.2,0.25,0.79571,0.558,0.2,0.85429,0.135],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785","probabilities":[0.03083,0.03083,0.03929,0.17214,0.76571,0.84857,0.87,0.03214,0.18938,0.03214,0.0525,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273","probabilities":[0.03214,0.03214,0.93214,0.03214,0.76,0.1,0.76,0.03,0.115,0.03,0.05,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.91883829404782805058767342767479130421","probabilities":[0.94214,0.02917,0.94143,0.02917,0.42643,0.42643,0.42643,0.80429,0.33,0.03,0.05071,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","probabilities":[0.03071,0.15643,0.02917,0.93143,0.03786,0.0375,0.03929,0.45571,0.27333,0.10286,0.68643,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069","probabilities":[0.02917,0.49286,0.02917,0.027,0.027,0.027,0.027,0.03,0.07333,0.03,0.027,0.024],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025","probabilities":[0.03,0.03,0.03,0.03,0.05,0.06333,0.05,0.93929,0.93857,0.47429,0.11857,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675","probabilities":[0.02917,0.02917,0.93929,0.03071,0.74429,0.08214,0.08214,0.75571,0.38333,0.88857,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095","probabilities":[0.696,0.75714,0.93857,0.03071,0.05625,0.45,0.035,0.45714,0.275,0.45857,0.80643,0.0325],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780","probabilities":[0.02917,0.02917,0.37571,0.03071,0.78286,0.52286,0.63714,0.1,0.1,0.51714,0.01833,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433","probabilities":[0.11,0.44429,0.92,0.90571,0.41,0.17333,0.434,0.78167,0.38333,0.12295,0.70667,0.939],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078","probabilities":[0.252,0.05124,0.24143,0.92143,0.31917,0.81357,0.88214,0.68,0.35,0.10573,0.51,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305","probabilities":[0.027,0.94143,0.03071,0.02917,0.05,0.07333,0.45857,0.47286,0.4375,0.05896,0.8075,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945","probabilities":[0.03,0.03,0.03,0.03,0.01667,0.01667,0.41167,0.34429,0.43333,0.10286,0.01667,0.02833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677","probabilities":[0.02917,0.02917,0.03071,0.03071,0.52143,0.0325,0.45857,0.39571,0.15333,0.07815,0.045,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679","probabilities":[0.03083,0.03083,0.07857,0.04357,0.03083,0.03083,0.67,0.33571,0.155,0.03083,0.02625,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151","probabilities":[0.03083,0.03083,0.70571,0.15786,0.68214,0.45857,0.60286,0.80857,0.9175,0.14549,0.638,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068","probabilities":[0.03083,0.43286,0.93857,0.37857,0.44714,0.71143,0.82143,0.515,0.4625,0.51333,0.029,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680","probabilities":[0.03083,0.03083,0.17857,0.17857,0.72429,0.039,0.039,0.93929,0.28125,0.10858,0.039,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418","probabilities":[0.03214,0.03214,0.03214,0.03214,0.03083,0.03083,0.72286,0.03214,0.09375,0.03214,0.16929,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93902628119199416703899320994283313812","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.03083,0.45,0.15833,0.02,0.02167,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308","probabilities":[0.03083,0.03083,0.49286,0.03214,0.03833,0.03875,0.05125,0.45714,0.34625,0.45714,0.91571,0.03833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","probabilities":[0.02625,0.02625,0.94143,0.1,0.69429,0.865,0.50917,0.81286,0.81833,0.77286,0.02167,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693","probabilities":[0.41429,0.0325,0.49286,0.16667,0.2,0.2,0.88143,0.68,0.93,0.192,0.152,0.14],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94447683257072169359896104855962933413","probabilities":[0.03083,0.03083,0.03214,0.03214,0.029,0.029,0.029,0.03214,0.09375,0.05,0.02167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175","probabilities":[0.03083,0.03083,0.08214,0.08214,0.03083,0.03083,0.03083,0.03,0.07333,0.02833,0.23583,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589","probabilities":[0.02917,0.02917,0.94,0.50143,0.815,0.57571,0.68583,0.47286,0.92429,0.68,0.47357,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472","probabilities":[0.03071,0.03071,0.93929,0.38143,0.2,0.2,0.2,0.03071,0.09125,0.52143,0.027,0.027],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","probabilities":[0.03071,0.03071,0.93857,0.03071,0.86417,0.86417,0.927,0.88857,0.94,0.08143,0.0798,0.01739],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224","probabilities":[0.07342,0.08422,0.13742,0.07337,0.2,0.05084,0.08579,0.32692,0.09433,0.04624,0.76333,0.92583],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95026524886584356226576936309513335214","probabilities":[0.02917,0.02917,0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.09125,0.05,0.01833,0.02375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982","probabilities":[0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.21214,0.03,0.07333,0.03,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","probabilities":[0.027,0.027,0.24143,0.03071,0.49071,0.41786,0.115,0.03214,0.0925,0.02917,0.027,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139","probabilities":[0.03071,0.03071,0.93857,0.03071,0.03786,0.03786,0.03786,0.03,0.09,0.44857,0.027,0.03071],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109","probabilities":[0.02917,0.037,0.27167,0.27167,0.02375,0.02375,0.024,0.45571,0.13667,0.037,0.72786,0.01833],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657","probabilities":[0.02917,0.02917,0.93857,0.03071,0.027,0.02375,0.51929,0.03,0.09,0.07322,0.32833,0.0],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948","probabilities":[0.02917,0.11917,0.83667,0.88571,0.83571,0.254,0.21,0.45571,0.305,0.0975,0.85071,0.93857],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310","probabilities":[0.03071,0.03071,0.03071,0.93857,0.02375,0.02375,0.02375,0.03,0.09,0.03,0.77143,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180","probabilities":[0.02917,0.02917,0.03071,0.02917,0.02917,0.02917,0.027,0.03071,0.07417,0.03,0.027,0.02917],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603","probabilities":[0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.03071,0.03,0.07333,0.03,0.03,0.03],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466","probabilities":[0.096,0.03083,0.03214,0.03214,0.03214,0.03214,0.03214,0.37571,0.13833,0.51143,0.03214,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716","probabilities":[0.62714,0.03214,0.50286,0.50286,0.59286,0.57143,0.67167,0.35857,0.27667,0.08593,0.17333,0.135],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428","probabilities":[0.73786,0.16571,0.925,0.37714,0.141,0.03833,0.03875,0.81786,0.35417,0.67929,0.925,0.04667],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083","probabilities":[0.79583,0.61857,0.50286,0.785,0.05,0.05,0.16667,0.45714,0.43625,0.0325,0.75857,0.52714],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570","probabilities":[0.09714,0.03357,0.93214,0.49571,0.1,0.1,0.1,0.03357,0.09375,0.04679,0.025,0.0175],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186","probabilities":[0.03214,0.03214,0.49571,0.03214,0.16667,0.05,0.04667,0.03214,0.07583,0.04196,0.02625,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926","probabilities":[0.85583,0.2375,0.945,0.94143,0.70214,0.75083,0.68571,0.88857,0.49,0.03214,0.632,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442","probabilities":[0.03083,0.03083,0.94071,0.03214,0.72429,0.14786,0.81357,0.93929,0.35417,0.14588,0.251,0.02],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373","probabilities":[0.03083,0.03083,0.94143,0.11571,0.88929,0.5825,0.88929,0.45857,0.31,0.34667,0.29167,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480","probabilities":[0.03083,0.03083,0.94333,0.50143,0.85857,0.321,0.20333,0.68,0.19667,0.039,0.55571,0.19917],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028","probabilities":[0.03083,0.03083,0.03214,0.03214,0.03083,0.03083,0.45143,0.29857,0.21,0.05,0.03083,0.01333],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091","probabilities":[0.02625,0.06289,0.50143,0.05,0.74143,0.606,0.73714,0.88083,0.93857,0.07,0.59125,0.02625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589","probabilities":[0.03083,0.17643,0.94417,0.36857,0.88929,0.71143,0.88857,0.94,0.94167,0.03214,0.79071,0.0375],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074","probabilities":[0.29,0.033,0.93857,0.03357,0.03833,0.03833,0.03833,0.035,0.11875,0.414,0.03125,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938","probabilities":[0.67857,0.03214,0.03214,0.03214,0.04417,0.515,0.03214,0.03214,0.09375,0.03663,0.03929,0.03214],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070","probabilities":[0.03083,0.64286,0.03083,0.03083,0.02625,0.02625,0.029,0.03214,0.0925,0.03083,0.68714,0.02167],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186","probabilities":[0.03214,0.03214,0.93929,0.03214,0.7775,0.7775,0.7775,0.03214,0.11875,0.03214,0.04686,0.01292],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068","probabilities":[0.029,0.029,0.94071,0.03214,0.45857,0.44,0.68357,0.33429,0.27333,0.03083,0.277,0.11625],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803","probabilities":[0.18,0.033,0.05,0.05,0.58571,0.03917,0.03929,0.38143,0.15333,0.45857,0.03833,0.025],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934","probabilities":[0.031,0.031,0.031,0.031,0.031,0.031,0.031,0.031,0.031,0.031,0.031,0.031],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536","probabilities":[0.03083,0.03214,0.94214,0.03214,0.85857,0.57286,0.85143,0.47286,0.93214,0.76429,0.03833,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928","probabilities":[0.0735,0.03783,0.09413,0.93214,0.16903,0.7775,0.59571,0.72857,0.2775,0.0959,0.07265,0.01756],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961","probabilities":[0.03083,0.03083,0.92143,0.03214,0.64,0.03214,0.03214,0.70714,0.28167,0.51,0.03833,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","probabilities":[0.88643,0.03214,0.94071,0.03214,0.77667,0.1,0.77667,0.03214,0.11875,0.72857,0.78714,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506","probabilities":[0.029,0.029,0.029,0.029,0.02625,0.02625,0.02625,0.04286,0.0375,0.0375,0.0125,0.0125],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290","probabilities":[0.94583,0.03083,0.03214,0.03214,0.1,0.1,0.07263,0.48359,0.16445,0.07415,0.94071,0.05],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99939100657291954890286900729983287973","probabilities":[0.03357,0.0325,0.05429,0.05429,0.05,0.05,0.46857,0.57643,0.43938,0.05845,0.11571,0.01],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]},{"study_id":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","probabilities":[0.04167,0.0376,0.08926,0.0479,0.07267,0.04539,0.11024,0.16573,0.08258,0.04165,0.08043,0.01776],"weights":[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]}]}\n', 'steven_audit_labels.json': '{"version": "steven_v4_fold0_validation_only", "targets_in_order": ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"], "labels": [{"study_id": "1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.025, 0.025, 0.025, 0.9, 0.325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018", "probabilities": [0.94, 0.015, 0.015, 0.015, 0.25, 0.25, 0.25, 0.98, 0.365, 0.25, 0.97, 0.995], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922", "probabilities": [0.015, 0.015, 0.7, 0.015, 0.9199999999999999, 0.015, 0.98, 0.925, 0.3375, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.13, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.13, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166", "probabilities": [0.015, 0.015, 0.985, 0.015, 0.995, 0.995, 0.995, 0.9, 0.325, 0.9299999999999999, 0.44, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586", "probabilities": [0.9099999999999999, 0.9, 0.46, 0.42, 0.98, 0.925, 0.925, 0.95, 0.95, 0.25, 0.42, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536", "probabilities": [0.995, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.975, 0.3625, 0.25, 0.975, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.79, 0.04], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821", "probabilities": [0.025, 0.97, 0.025, 0.025, 0.64, 0.64, 0.025, 0.94, 0.345, 0.025, 0.95, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307", "probabilities": [0.025, 0.025, 0.135, 0.025, 0.9, 0.04, 0.04, 0.04, 0.145, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.95, 0.97, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.94, 0.25, 0.96, 0.95, 0.95, 0.825, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639", "probabilities": [0.025, 0.025, 0.725, 0.98, 0.925, 0.025, 0.98, 0.97, 0.925, 0.25, 0.05, 0.05], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.25, 0.25, 0.25, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315", "probabilities": [0.41, 0.025, 0.985, 0.42, 0.99, 0.99, 0.99, 0.97, 0.36, 0.94, 0.05, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.975, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.89, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724", "probabilities": [0.41, 0.025, 0.025, 0.025, 0.4, 0.4, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434", "probabilities": [0.995, 0.025, 0.97, 0.97, 0.025, 0.025, 0.025, 0.9, 0.325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667", "probabilities": [0.99, 0.025, 0.995, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863", "probabilities": [0.025, 0.025, 0.05, 0.025, 0.025, 0.025, 0.94, 0.89, 0.32, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9, 0.9099999999999999, 0.32999999999999996, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.94, 0.94, 0.94, 0.64, 0.32, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.025, 0.89, 0.32, 0.9, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091", "probabilities": [0.025, 0.9099999999999999, 0.025, 0.025, 0.025, 0.025, 0.97, 0.95, 0.35, 0.025, 0.96, 0.96], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.675, 0.9099999999999999, 0.32999999999999996, 0.25, 0.35, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.925, 0.94, 0.345, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.98, 0.025, 0.025, 0.1375, 0.25, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.25, 0.25, 0.25, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084", "probabilities": [0.075, 0.9, 0.985, 0.025, 0.98, 0.45, 0.9199999999999999, 0.25, 0.25, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.96, 0.96, 0.96, 0.99, 0.37, 0.97, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506", "probabilities": [0.7, 0.69, 0.99, 0.025, 0.89, 0.25, 0.025, 0.05, 0.15, 0.025, 0.8400000000000001, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.98, 0.015, 0.985, 0.01, 0.13, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.47, 0.9299999999999999, 0.33999999999999997, 0.25, 0.95, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802", "probabilities": [0.025, 0.025, 0.95, 0.025, 0.025, 0.025, 0.98, 0.94, 0.95, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428", "probabilities": [0.995, 0.25, 0.975, 0.25, 0.925, 0.25, 0.96, 0.25, 0.25, 0.975, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334", "probabilities": [0.025, 0.025, 0.025, 0.975, 0.025, 0.025, 0.25, 0.89, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.975, 0.025, 0.025, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263", "probabilities": [0.025, 0.075, 0.98, 0.025, 0.975, 0.025, 0.96, 0.025, 0.1375, 0.985, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.95, 0.96, 0.54, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655", "probabilities": [0.025, 0.025, 0.06, 0.25, 0.025, 0.98, 0.98, 0.9099999999999999, 0.96, 0.015, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264", "probabilities": [0.05, 0.025, 0.97, 0.41, 0.98, 0.25, 0.25, 0.9, 0.325, 0.25, 0.4, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398", "probabilities": [0.015, 0.725, 0.975, 0.015, 0.76, 0.9099999999999999, 0.9199999999999999, 0.9099999999999999, 0.32999999999999996, 0.975, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722", "probabilities": [0.99, 0.015, 0.25, 0.6, 0.25, 0.25, 0.9199999999999999, 0.25, 0.9099999999999999, 0.25, 0.96, 0.5900000000000001], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768", "probabilities": [0.015, 0.015, 0.05, 0.015, 0.015, 0.015, 0.015, 0.025, 0.1375, 0.01, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670", "probabilities": [0.015, 0.015, 0.42, 0.015, 0.25, 0.25, 0.015, 0.01, 0.13, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962", "probabilities": [0.99, 0.9099999999999999, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9199999999999999, 0.33499999999999996, 0.89, 0.98, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310", "probabilities": [0.42500000000000004, 0.025, 0.97, 0.7, 0.9099999999999999, 0.99, 0.985, 0.9099999999999999, 0.95, 0.98, 0.05, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088", "probabilities": [0.025, 0.025, 0.725, 0.025, 0.025, 0.025, 0.76, 0.89, 0.32, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008", "probabilities": [0.025, 0.9099999999999999, 0.6499999999999999, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547", "probabilities": [0.015, 0.95, 0.015, 0.25, 0.015, 0.95, 0.9, 0.9099999999999999, 0.95, 0.25, 0.05, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.97, 0.96, 0.9099999999999999, 0.32999999999999996, 0.015, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324", "probabilities": [0.015, 0.015, 0.015, 0.98, 0.015, 0.9299999999999999, 0.015, 0.015, 0.1325, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286", "probabilities": [0.75, 0.015, 0.44, 0.98, 0.015, 0.015, 0.015, 0.025, 0.1375, 0.015, 0.9, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.015, 0.1325, 0.9099999999999999, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758", "probabilities": [0.015, 0.015, 0.015, 0.995, 0.015, 0.9299999999999999, 0.815, 0.9099999999999999, 0.9299999999999999, 0.95, 0.94, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747", "probabilities": [0.94, 0.025, 0.98, 0.41, 0.9099999999999999, 0.025, 0.9299999999999999, 0.9099999999999999, 0.9299999999999999, 0.9199999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131", "probabilities": [0.025, 0.9099999999999999, 0.98, 0.025, 0.25, 0.75, 0.25, 0.975, 0.3625, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258", "probabilities": [0.7749999999999999, 0.025, 0.34, 0.025, 0.025, 0.025, 0.25, 0.9099999999999999, 0.32999999999999996, 0.25, 0.69, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.025, 0.025, 0.025, 0.95, 0.35, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.75, 0.05, 0.15, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683", "probabilities": [0.025, 0.025, 0.98, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.44000000000000006], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9, 0.025, 0.995, 0.9099999999999999, 0.32999999999999996, 0.9, 0.41, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11277078989123345865112116288626381353", "probabilities": [0.7150000000000001, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11277278232377185347750793496260885546", "probabilities": [0.7749999999999999, 0.025, 0.99, 0.025, 0.995, 0.9299999999999999, 0.96, 0.975, 0.95, 0.25, 0.45, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593", "probabilities": [0.025, 0.025, 0.86, 0.04, 0.025, 0.025, 0.9099999999999999, 0.05, 0.15, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411", "probabilities": [0.1, 0.25, 0.975, 0.975, 0.825, 0.98, 0.9650000000000001, 0.95, 0.35, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712", "probabilities": [0.98, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.825, 0.925, 0.9], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473", "probabilities": [0.025, 0.025, 0.99, 0.9650000000000001, 0.9099999999999999, 0.025, 0.985, 0.975, 0.3625, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.05, 0.05, 0.05, 0.9099999999999999, 0.32999999999999996, 0.9, 0.95, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11391095418955817124325938917797687542", "probabilities": [0.025, 0.975, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.95, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830", "probabilities": [0.025, 0.25, 0.025, 0.375, 0.25, 0.25, 0.7150000000000001, 0.05, 0.15, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849", "probabilities": [0.75, 0.025, 0.725, 0.925, 0.75, 0.25, 0.025, 0.975, 0.3625, 0.25, 0.95, 0.95], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.75, 0.76, 0.025, 0.1375, 0.9, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597", "probabilities": [0.98, 0.025, 0.675, 0.925, 0.025, 0.9, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197", "probabilities": [0.025, 0.025, 0.98, 0.025, 0.99, 0.94, 0.9099999999999999, 0.94, 0.95, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121", "probabilities": [0.995, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.95, 0.35, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047", "probabilities": [0.5900000000000001, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.25, 0.25, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558", "probabilities": [0.735, 0.9099999999999999, 0.41, 0.025, 0.96, 0.25, 0.025, 0.05, 0.15, 0.975, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11546127923715190914128277599849591183", "probabilities": [0.025, 0.97, 0.025, 0.025, 0.025, 0.025, 0.025, 0.89, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11551770225828320378339882849245348148", "probabilities": [0.025, 0.025, 0.7749999999999999, 0.41, 0.925, 0.25, 0.25, 0.95, 0.97, 0.99, 0.425, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457", "probabilities": [0.025, 0.025, 0.41, 0.25, 0.025, 0.025, 0.25, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.99], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849", "probabilities": [0.015, 0.015, 0.42, 0.42, 0.25, 0.25, 0.9299999999999999, 0.935, 0.3425, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378", "probabilities": [0.015, 0.015, 0.41, 0.025, 0.01, 0.01, 0.01, 0.025, 0.1375, 0.01, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181", "probabilities": [0.025, 0.9099999999999999, 0.99, 0.42, 0.98, 0.9, 0.9299999999999999, 0.96, 0.355, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623", "probabilities": [0.015, 0.015, 0.99, 0.015, 0.01, 0.01, 0.01, 0.9099999999999999, 0.32999999999999996, 0.01, 0.75, 0.96], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429", "probabilities": [0.995, 0.97, 0.97, 0.95, 0.96, 0.9299999999999999, 0.935, 0.25, 0.25, 0.01, 0.75, 0.47], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637", "probabilities": [0.94, 0.9099999999999999, 0.995, 0.97, 0.995, 0.9550000000000001, 0.9299999999999999, 0.9199999999999999, 0.33499999999999996, 0.9299999999999999, 0.025, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928", "probabilities": [0.015, 0.015, 0.41, 0.015, 0.01, 0.01, 0.01, 0.01, 0.13, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612", "probabilities": [0.015, 0.015, 0.42, 0.015, 0.25, 0.25, 0.25, 0.41, 0.32999999999999996, 0.25, 0.76, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607", "probabilities": [0.25, 0.25, 0.975, 0.41, 0.25, 0.97, 0.25, 0.96, 0.97, 0.98, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695", "probabilities": [0.015, 0.9099999999999999, 0.97, 0.015, 0.015, 0.015, 0.015, 0.9099999999999999, 0.32999999999999996, 0.015, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11783597160745641720078978406968189488", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269", "probabilities": [0.015, 0.015, 0.325, 0.9099999999999999, 0.015, 0.015, 0.015, 0.9, 0.325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.9, 0.9, 0.9, 0.01, 0.13, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033", "probabilities": [0.015, 0.76, 0.015, 0.015, 0.015, 0.9299999999999999, 0.9099999999999999, 0.025, 0.1375, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758", "probabilities": [0.725, 0.015, 0.725, 0.015, 0.98, 0.25, 0.75, 0.98, 0.365, 0.015, 0.745, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108", "probabilities": [0.665, 0.575, 0.945, 0.25, 0.995, 0.9099999999999999, 0.97, 0.025, 0.1375, 0.015, 0.855, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862", "probabilities": [0.995, 0.01, 0.985, 0.01, 0.9099999999999999, 0.015, 0.97, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011", "probabilities": [0.995, 0.015, 0.97, 0.015, 0.015, 0.015, 0.015, 0.25, 0.25, 0.25, 0.9099999999999999, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762", "probabilities": [0.97, 0.94, 0.98, 0.015, 0.25, 0.25, 0.95, 0.95, 0.35, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229", "probabilities": [0.025, 0.015, 0.97, 0.97, 0.97, 0.995, 0.9, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744", "probabilities": [0.015, 0.015, 0.41, 0.015, 0.25, 0.25, 0.25, 0.01, 0.13, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.9099999999999999, 0.015, 0.9, 0.97, 0.36, 0.25, 0.41, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11905397531490569218559534495408776672", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.25, 0.015, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708", "probabilities": [0.8, 0.7749999999999999, 0.8, 0.8, 0.75, 0.975, 0.8, 0.95, 0.85, 0.25, 0.7849999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.9099999999999999, 0.025, 0.025, 0.9099999999999999, 0.9099999999999999, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715", "probabilities": [0.075, 0.025, 0.025, 0.99, 0.39, 0.995, 0.025, 0.95, 0.6, 0.075, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11935116791150327546384719051023540185", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.89, 0.32, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.9299999999999999, 0.9299999999999999, 0.89, 0.32, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.675, 0.425, 0.25, 0.25, 0.25, 0.25, 0.075, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600", "probabilities": [0.025, 0.94, 0.075, 0.025, 0.975, 0.025, 0.99, 0.05, 0.15, 0.025, 0.425, 0.35], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11958117172819754862618612170999909480", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.75, 0.99, 0.925, 0.3375, 0.25, 0.075, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.925, 0.25, 0.25, 0.94, 0.345, 0.25, 0.95, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.11981170645605720774509861722988500673", "probabilities": [0.025, 0.8, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9650000000000001, 0.35750000000000004, 0.25, 0.9650000000000001, 0.7], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.925, 0.025, 0.985, 0.89, 0.32, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792", "probabilities": [0.65, 0.025, 0.41, 0.9650000000000001, 0.25, 0.25, 0.9099999999999999, 0.9, 0.325, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12031315219947933845048665397149229969", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.99, 0.9099999999999999, 0.995, 0.9099999999999999, 0.32999999999999996, 0.075, 0.075, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.025, 0.9099999999999999, 0.995], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.9199999999999999, 0.9199999999999999, 0.9199999999999999, 0.9199999999999999, 0.33499999999999996, 0.9199999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491", "probabilities": [0.9099999999999999, 0.97, 0.75, 0.325, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205", "probabilities": [0.025, 0.025, 0.05, 0.05, 0.025, 0.025, 0.9199999999999999, 0.025, 0.1375, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.025, 0.025, 0.9, 0.95, 0.9099999999999999, 0.975, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614", "probabilities": [0.25, 0.25, 0.41, 0.25, 0.25, 0.25, 0.95, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12139829070290803654867114374799823711", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297", "probabilities": [0.025, 0.61, 0.98, 0.025, 0.8200000000000001, 0.71, 0.6599999999999999, 0.81, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.9199999999999999, 0.025, 0.99, 0.9099999999999999, 0.32999999999999996, 0.99, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.9099999999999999, 0.9099999999999999, 0.95, 0.9, 0.325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211", "probabilities": [0.025, 0.585, 0.97, 0.41, 0.97, 0.4, 0.9099999999999999, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302", "probabilities": [0.025, 0.025, 0.025, 0.25, 0.9, 0.025, 0.9, 0.9099999999999999, 0.32999999999999996, 0.985, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.025, 0.025, 0.025, 0.9199999999999999, 0.33499999999999996, 0.98, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796", "probabilities": [0.025, 0.025, 0.425, 0.425, 0.9099999999999999, 0.9099999999999999, 0.025, 0.9, 0.325, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681", "probabilities": [0.025, 0.635, 0.9199999999999999, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.7, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817", "probabilities": [0.995, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.99, 0.37, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014", "probabilities": [0.025, 0.025, 0.44, 0.025, 0.25, 0.25, 0.94, 0.025, 0.1375, 0.9199999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.99, 0.75, 0.325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12259896940707176389739553110492954399", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.9, 0.325, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367", "probabilities": [0.025, 0.025, 0.94, 0.025, 0.79, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097", "probabilities": [0.025, 0.025, 0.985, 0.025, 0.76, 0.025, 0.025, 0.96, 0.355, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578", "probabilities": [0.015, 0.97, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.99, 0.015, 0.1325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790", "probabilities": [0.015, 0.9099999999999999, 0.46, 0.4, 0.25, 0.25, 0.99, 0.9099999999999999, 0.32999999999999996, 0.25, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910", "probabilities": [0.015, 0.015, 0.04, 0.015, 0.9099999999999999, 0.98, 0.99, 0.9099999999999999, 0.89, 0.015, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.015, 0.025, 0.1375, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831", "probabilities": [0.015, 0.015, 0.39, 0.465, 0.94, 0.96, 0.25, 0.015, 0.1325, 0.95, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12411396124304166984011280826756258628", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.94, 0.015, 0.1325, 0.25, 0.05, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232", "probabilities": [0.04, 0.04, 0.98, 0.39, 0.9099999999999999, 0.9299999999999999, 0.9, 0.96, 0.355, 0.015, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12437154858167700438432674983971817217", "probabilities": [0.015, 0.015, 0.39, 0.25, 0.25, 0.25, 0.99, 0.94, 0.96, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876", "probabilities": [0.25, 0.25, 0.97, 0.97, 0.96, 0.25, 0.96, 0.95, 0.35, 0.97, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080", "probabilities": [0.98, 0.025, 0.97, 0.25, 0.25, 0.25, 0.25, 0.9650000000000001, 0.35750000000000004, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12471165461877621368840037945454256612", "probabilities": [0.995, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084", "probabilities": [0.05, 0.025, 0.99, 0.41, 0.99, 0.025, 0.25, 0.95, 0.35, 0.25, 0.05, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.025, 0.55, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342", "probabilities": [0.025, 0.025, 0.05, 0.97, 0.25, 0.25, 0.025, 0.94, 0.345, 0.25, 0.97, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942", "probabilities": [0.025, 0.9099999999999999, 0.46, 0.41, 0.95, 0.95, 0.99, 0.94, 0.345, 0.96, 0.05, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111", "probabilities": [0.96, 0.96, 0.5900000000000001, 0.25, 0.25, 0.25, 0.25, 0.95, 0.96, 0.25, 0.975, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616", "probabilities": [0.025, 0.025, 0.985, 0.025, 0.025, 0.025, 0.96, 0.89, 0.32, 0.97, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431", "probabilities": [0.025, 0.025, 0.98, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12583872770399266463538638801797827468", "probabilities": [0.025, 0.025, 0.015, 0.015, 0.025, 0.025, 0.025, 0.9199999999999999, 0.33499999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918", "probabilities": [0.025, 0.025, 0.04, 0.04, 0.025, 0.025, 0.025, 0.05, 0.15, 0.015, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172", "probabilities": [0.995, 0.015, 0.97, 0.97, 0.015, 0.015, 0.015, 0.01, 0.13, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635", "probabilities": [0.015, 0.015, 0.4, 0.39, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.02, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265", "probabilities": [0.015, 0.9, 0.96, 0.015, 0.015, 0.015, 0.015, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803", "probabilities": [0.025, 0.015, 0.99, 0.39, 0.995, 0.97, 0.97, 0.97, 0.97, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12701097867143790763644826659974337046", "probabilities": [0.015, 0.015, 0.015, 0.34, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849", "probabilities": [0.04, 0.025, 0.99, 0.97, 0.995, 0.98, 0.99, 0.9099999999999999, 0.975, 0.25, 0.25, 0.75], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688", "probabilities": [0.025, 0.025, 0.98, 0.025, 0.25, 0.25, 0.25, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492", "probabilities": [0.02, 0.02, 0.25, 0.25, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.02, 0.02, 0.02, 0.015, 0.1325, 0.015, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908", "probabilities": [0.02, 0.02, 0.02, 0.02, 0.02, 0.89, 0.25, 0.9199999999999999, 0.33499999999999996, 0.02, 0.98, 0.99], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270", "probabilities": [0.25, 0.9099999999999999, 0.41, 0.02, 0.95, 0.8400000000000001, 0.975, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333", "probabilities": [0.015, 0.015, 0.985, 0.015, 0.02, 0.02, 0.02, 0.015, 0.1325, 0.015, 0.43, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949", "probabilities": [0.9099999999999999, 0.02, 0.41, 0.41, 0.8, 0.8, 0.99, 0.25, 0.9099999999999999, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035", "probabilities": [0.015, 0.015, 0.985, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.9199999999999999, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751", "probabilities": [0.41, 0.015, 0.985, 0.97, 0.9650000000000001, 0.015, 0.95, 0.9199999999999999, 0.33499999999999996, 0.99, 0.41, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12826828527176716109949704557150950675", "probabilities": [0.015, 0.015, 0.4, 0.015, 0.015, 0.015, 0.9, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655", "probabilities": [0.025, 0.025, 0.4, 0.025, 0.25, 0.25, 0.25, 0.025, 0.1375, 0.25, 0.585, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502", "probabilities": [0.025, 0.025, 0.96, 0.025, 0.9099999999999999, 0.025, 0.985, 0.97, 0.36, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608", "probabilities": [0.025, 0.025, 0.025, 0.97, 0.97, 0.97, 0.99, 0.975, 0.975, 0.9199999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832", "probabilities": [0.25, 0.25, 0.4, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039", "probabilities": [0.025, 0.97, 0.98, 0.025, 0.025, 0.025, 0.025, 0.89, 0.32, 0.25, 0.76, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664", "probabilities": [0.025, 0.025, 0.4, 0.025, 0.96, 0.96, 0.96, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.95, 0.95, 0.25, 0.95, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589", "probabilities": [0.25, 0.025, 0.97, 0.25, 0.95, 0.4, 0.95, 0.95, 0.97, 0.9, 0.25, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363", "probabilities": [0.025, 0.025, 0.985, 0.635, 0.995, 0.97, 0.995, 0.975, 0.3625, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508", "probabilities": [0.44000000000000006, 0.125, 0.6599999999999999, 0.4, 0.25, 0.99, 0.94, 0.05, 0.15, 0.025, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13065120731888368919957856813755701101", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9199999999999999, 0.9299999999999999, 0.995, 0.9, 0.325, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13068696303639759696458130704554709416", "probabilities": [0.025, 0.9099999999999999, 0.985, 0.025, 0.825, 0.025, 0.86, 0.89, 0.32, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703", "probabilities": [0.05, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.975, 0.97, 0.975, 0.525, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.8200000000000001, 0.025, 0.98, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13099702881219862576749493772702888064", "probabilities": [0.025, 0.06, 0.41, 0.41, 0.025, 0.025, 0.98, 0.97, 0.36, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.98, 0.05, 0.9299999999999999, 0.05, 0.15, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127", "probabilities": [0.995, 0.05, 0.99, 0.97, 0.995, 0.96, 0.9199999999999999, 0.975, 0.3625, 0.95, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285", "probabilities": [0.995, 0.995, 0.025, 0.85, 0.96, 0.96, 0.99, 0.975, 0.3625, 0.025, 0.96, 0.995], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13150780733560495803861509587935093776", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.97, 0.015, 0.9199999999999999, 0.98, 0.365, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710", "probabilities": [0.015, 0.015, 0.415, 0.015, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369", "probabilities": [0.015, 0.015, 0.9199999999999999, 0.015, 0.015, 0.015, 0.98, 0.015, 0.1325, 0.015, 0.42, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145", "probabilities": [0.015, 0.015, 0.4, 0.015, 0.015, 0.015, 0.99, 0.015, 0.1325, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605", "probabilities": [0.015, 0.9099999999999999, 0.6499999999999999, 0.015, 0.9450000000000001, 0.25, 0.89, 0.9099999999999999, 0.32999999999999996, 0.9, 0.9, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169", "probabilities": [0.015, 0.015, 0.015, 0.45999999999999996, 0.98, 0.7, 0.99, 0.9650000000000001, 0.81, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025", "probabilities": [0.25, 0.25, 0.9550000000000001, 0.25, 0.9550000000000001, 0.25, 0.9550000000000001, 0.95, 0.95, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636", "probabilities": [0.9099999999999999, 0.025, 0.425, 0.6599999999999999, 0.8500000000000001, 0.95, 0.99, 0.96, 0.975, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875", "probabilities": [0.015, 0.015, 0.425, 0.015, 0.25, 0.25, 0.95, 0.9099999999999999, 0.95, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030", "probabilities": [0.025, 0.025, 0.98, 0.025, 0.025, 0.025, 0.025, 0.94, 0.345, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.89, 0.015, 0.015, 0.01, 0.13, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388", "probabilities": [0.01, 0.01, 0.975, 0.01, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141", "probabilities": [0.025, 0.9, 0.98, 0.015, 0.9099999999999999, 0.025, 0.025, 0.9, 0.325, 0.9, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.25, 0.25, 0.925, 0.88, 0.315, 0.01, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226", "probabilities": [0.25, 0.25, 0.975, 0.25, 0.25, 0.25, 0.94, 0.25, 0.25, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791", "probabilities": [0.97, 0.9099999999999999, 0.42, 0.4, 0.95, 0.9299999999999999, 0.995, 0.25, 0.25, 0.275, 0.075, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.14061227335828384505403558788548348446", "probabilities": [0.01, 0.01, 0.99, 0.01, 0.97, 0.9099999999999999, 0.9099999999999999, 0.975, 0.3625, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.13, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.9, 0.325, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791", "probabilities": [0.01, 0.01, 0.42, 0.01, 0.25, 0.25, 0.9099999999999999, 0.01, 0.13, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570", "probabilities": [0.01, 0.01, 0.01, 0.41, 0.9099999999999999, 0.9099999999999999, 0.96, 0.025, 0.1375, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770", "probabilities": [0.01, 0.01, 0.99, 0.7749999999999999, 0.985, 0.01, 0.985, 0.9099999999999999, 0.96, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904", "probabilities": [0.025, 0.025, 0.96, 0.025, 0.97, 0.525, 0.99, 0.97, 0.36, 0.97, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738", "probabilities": [0.06, 0.175, 0.985, 0.25, 0.995, 0.94, 0.975, 0.985, 0.3675, 0.025, 0.05, 0.05], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885", "probabilities": [0.025, 0.025, 0.65, 0.41, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.95, 0.05], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269", "probabilities": [0.025, 0.025, 0.075, 0.025, 0.96, 0.79, 0.97, 0.69, 0.22, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764", "probabilities": [0.25, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9, 0.325, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081", "probabilities": [0.995, 0.9099999999999999, 0.99, 0.05, 0.9099999999999999, 0.25, 0.94, 0.89, 0.9099999999999999, 0.25, 0.975, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943", "probabilities": [0.025, 0.025, 0.97, 0.41, 0.95, 0.95, 0.95, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835", "probabilities": [0.05, 0.9099999999999999, 0.985, 0.41, 0.975, 0.8500000000000001, 0.9650000000000001, 0.9099999999999999, 0.9099999999999999, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.49, 0.32, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.16691383858003633107641297162554868325", "probabilities": [0.995, 0.015, 0.41, 0.015, 0.015, 0.015, 0.015, 0.04, 0.01, 0.01, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414", "probabilities": [0.015, 0.015, 0.7749999999999999, 0.98, 0.015, 0.015, 0.8, 0.9099999999999999, 0.975, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119", "probabilities": [0.425, 0.94, 0.425, 0.25, 0.95, 0.925, 0.99, 0.99, 0.37, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916", "probabilities": [0.015, 0.015, 0.995, 0.015, 0.975, 0.015, 0.975, 0.915, 0.3325, 0.915, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467", "probabilities": [0.015, 0.015, 0.97, 0.41, 0.9199999999999999, 0.42, 0.42, 0.9550000000000001, 0.35250000000000004, 0.25, 0.41, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285", "probabilities": [0.015, 0.97, 0.015, 0.015, 0.015, 0.015, 0.015, 0.04, 0.145, 0.9099999999999999, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009", "probabilities": [0.015, 0.2, 0.97, 0.25, 0.9, 0.25, 0.015, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124", "probabilities": [0.98, 0.015, 0.97, 0.97, 0.8500000000000001, 0.25, 0.25, 0.89, 0.32, 0.25, 0.02, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080", "probabilities": [0.9099999999999999, 0.015, 0.25, 0.97, 0.52, 0.9650000000000001, 0.52, 0.95, 0.975, 0.25, 0.43, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569", "probabilities": [0.4, 0.075, 0.985, 0.41, 0.97, 0.25, 0.97, 0.95, 0.35, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748", "probabilities": [0.015, 0.015, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.13, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18879049235810379709186936621085221483", "probabilities": [0.015, 0.015, 0.985, 0.015, 0.985, 0.015, 0.9, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.18931107448953327254073392032141677797", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.9, 0.015, 0.9, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.81, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.71, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.95, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530", "probabilities": [0.025, 0.9, 0.97, 0.025, 0.25, 0.25, 0.99, 0.94, 0.345, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760", "probabilities": [0.94, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.96, 0.355, 0.25, 0.25, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830", "probabilities": [0.025, 0.025, 0.04, 0.04, 0.025, 0.025, 0.9099999999999999, 0.01, 0.13, 0.01, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.98, 0.9, 0.325, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886", "probabilities": [0.015, 0.015, 0.95, 0.015, 0.95, 0.25, 0.25, 0.01, 0.13, 0.01, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.975, 0.975, 0.975, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785", "probabilities": [0.995, 0.025, 0.98, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784", "probabilities": [0.995, 0.025, 0.985, 0.975, 0.925, 0.925, 0.025, 0.025, 0.1375, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537", "probabilities": [0.025, 0.025, 0.96, 0.025, 0.025, 0.025, 0.025, 0.95, 0.35, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.025, 0.025, 0.975], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954", "probabilities": [0.025, 0.025, 0.025, 0.975, 0.025, 0.975, 0.975, 0.9099999999999999, 0.9099999999999999, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101", "probabilities": [0.925, 0.025, 0.96, 0.025, 0.94, 0.9099999999999999, 0.9099999999999999, 0.89, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21142867813782396036287673290033987859", "probabilities": [0.025, 0.025, 0.025, 0.75, 0.76, 0.76, 0.94, 0.515, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958", "probabilities": [0.025, 0.55, 0.985, 0.025, 0.975, 0.9099999999999999, 0.95, 0.9099999999999999, 0.9099999999999999, 0.975, 0.025, 0.39999999999999997], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044", "probabilities": [0.05, 0.025, 0.96, 0.025, 0.025, 0.71, 0.98, 0.89, 0.32, 0.89, 0.05, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.98, 0.025, 0.1375, 0.89, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673", "probabilities": [0.025, 0.025, 0.96, 0.875, 0.975, 0.975, 0.995, 0.95, 0.975, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762", "probabilities": [0.025, 0.025, 0.96, 0.025, 0.98, 0.025, 0.99, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295", "probabilities": [0.15000000000000002, 0.025, 0.985, 0.1, 0.975, 0.975, 0.985, 0.95, 0.96, 0.975, 0.425, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965", "probabilities": [0.995, 0.025, 0.975, 0.985, 0.025, 0.44, 0.025, 0.99, 0.37, 0.25, 0.975, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.22394208178597637441149549895512290013", "probabilities": [0.98, 0.9099999999999999, 0.98, 0.025, 0.9299999999999999, 0.9099999999999999, 0.99, 0.975, 0.3625, 0.025, 0.96, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.22508124641661887430614105328316053644", "probabilities": [0.025, 0.025, 0.96, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423", "probabilities": [0.9099999999999999, 0.025, 0.97, 0.025, 0.25, 0.25, 0.25, 0.9099999999999999, 0.32999999999999996, 0.94, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170", "probabilities": [0.25, 0.25, 0.975, 0.25, 0.25, 0.975, 0.25, 0.25, 0.25, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9299999999999999, 0.9, 0.9299999999999999, 0.025, 0.1375, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134", "probabilities": [0.025, 0.025, 0.985, 0.95, 0.925, 0.025, 0.975, 0.9099999999999999, 0.32999999999999996, 0.925, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738", "probabilities": [0.025, 0.025, 0.6, 0.025, 0.45, 0.45, 0.985, 0.95, 0.97, 0.9199999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663", "probabilities": [0.995, 0.02, 0.985, 0.015, 0.95, 0.25, 0.25, 0.9099999999999999, 0.32999999999999996, 0.94, 0.9299999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366", "probabilities": [0.015, 0.9099999999999999, 0.98, 0.41, 0.985, 0.9, 0.9, 0.9, 0.925, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916", "probabilities": [0.25, 0.25, 0.025, 0.25, 0.25, 0.25, 0.94, 0.95, 0.35, 0.96, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24219103446788192052503003052696523200", "probabilities": [0.02, 0.02, 0.975, 0.97, 0.97, 0.25, 0.25, 0.9650000000000001, 0.35750000000000004, 0.9650000000000001, 0.45, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910", "probabilities": [0.015, 0.015, 0.01, 0.01, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038", "probabilities": [0.015, 0.015, 0.97, 0.95, 0.015, 0.015, 0.015, 0.025, 0.1375, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24872746733270791851241663716306993824", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187", "probabilities": [0.75, 0.9099999999999999, 0.98, 0.015, 0.25, 0.25, 0.25, 0.95, 0.35, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.975, 0.975, 0.975, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189", "probabilities": [0.9099999999999999, 0.025, 0.44, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724", "probabilities": [0.025, 0.025, 0.41, 0.41, 0.025, 0.025, 0.025, 0.89, 0.32, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670", "probabilities": [0.025, 0.025, 0.6, 0.025, 0.025, 0.9099999999999999, 0.99, 0.97, 0.36, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882", "probabilities": [0.675, 0.25, 0.41, 0.8, 0.71, 0.98, 0.585, 0.95, 0.96, 0.95, 0.6, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505", "probabilities": [0.25, 0.25, 0.25, 0.96, 0.25, 0.97, 0.25, 0.95, 0.35, 0.96, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567", "probabilities": [0.95, 0.025, 0.97, 0.98, 0.97, 0.97, 0.97, 0.94, 0.345, 0.025, 0.9099999999999999, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.95, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103", "probabilities": [0.025, 0.025, 0.97, 0.425, 0.9, 0.25, 0.95, 0.025, 0.1375, 0.25, 0.95, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502", "probabilities": [0.025, 0.025, 0.6499999999999999, 0.025, 0.94, 0.96, 0.96, 0.9099999999999999, 0.32999999999999996, 0.025, 0.4, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036", "probabilities": [0.75, 0.025, 0.425, 0.425, 0.25, 0.25, 0.25, 0.025, 0.1375, 0.25, 0.935, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.9099999999999999, 0.98, 0.9199999999999999, 0.9199999999999999, 0.33499999999999996, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568", "probabilities": [0.25, 0.25, 0.97, 0.25, 0.97, 0.97, 0.97, 0.95, 0.35, 0.96, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091", "probabilities": [0.41, 0.015, 0.865, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.25, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691", "probabilities": [0.015, 0.015, 0.98, 0.015, 0.25, 0.25, 0.25, 0.9199999999999999, 0.33499999999999996, 0.25, 0.765, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957", "probabilities": [0.015, 0.015, 0.47, 0.015, 0.015, 0.015, 0.015, 0.9199999999999999, 0.33499999999999996, 0.97, 0.45, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.26848412380762311150802301982233072694", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.9099999999999999, 0.015, 0.015, 0.04, 0.145, 0.9199999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194", "probabilities": [0.015, 0.015, 0.985, 0.975, 0.99, 0.9099999999999999, 0.985, 0.975, 0.3625, 0.95, 0.015, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27059975738703546675398620281849285863", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.015, 0.015, 0.015, 0.04, 0.145, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.25, 0.25, 0.25, 0.94, 0.345, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.96, 0.96, 0.96, 0.9199999999999999, 0.95, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018", "probabilities": [0.975, 0.9099999999999999, 0.975, 0.025, 0.6599999999999999, 0.6599999999999999, 0.5, 0.96, 0.355, 0.25, 0.975, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355", "probabilities": [0.9099999999999999, 0.55, 0.025, 0.025, 0.25, 0.25, 0.99, 0.96, 0.975, 0.9099999999999999, 0.9099999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063", "probabilities": [0.25, 0.25, 0.965, 0.25, 0.995, 0.25, 0.995, 0.9299999999999999, 0.365, 0.99, 0.05, 0.725], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.975, 0.89, 0.05, 0.25, 0.05, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449", "probabilities": [0.025, 0.025, 0.4, 0.975, 0.25, 0.25, 0.25, 0.94, 0.345, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.99, 0.975, 0.25, 0.975, 0.55], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882", "probabilities": [0.025, 0.025, 0.985, 0.975, 0.875, 0.875, 0.99, 0.76, 0.76, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.1375, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.35, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181", "probabilities": [0.95, 0.025, 0.025, 0.025, 0.41, 0.41, 0.41, 0.98, 0.98, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586", "probabilities": [0.025, 0.025, 0.995, 0.725, 0.995, 0.9199999999999999, 0.9199999999999999, 0.9099999999999999, 0.815, 0.98, 0.97, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.29343735048224321966824641432873034291", "probabilities": [0.9199999999999999, 0.025, 0.025, 0.025, 0.49, 0.25, 0.83, 0.5599999999999999, 0.32999999999999996, 0.55, 0.98, 0.98], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.025, 0.025, 0.25, 0.99, 0.37, 0.25, 0.98, 0.7749999999999999], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464", "probabilities": [0.025, 0.025, 0.42, 0.42, 0.65, 0.65, 0.97, 0.025, 0.1375, 0.25, 0.41, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569", "probabilities": [0.025, 0.025, 0.94, 0.99, 0.025, 0.025, 0.99, 0.9099999999999999, 0.97, 0.9099999999999999, 0.96, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762", "probabilities": [0.025, 0.025, 0.41, 0.675, 0.97, 0.9, 0.99, 0.025, 0.1375, 0.9099999999999999, 0.845, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.99, 0.97, 0.98, 0.9099999999999999, 0.83, 0.025, 0.47, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.9099999999999999, 0.025, 0.9299999999999999, 0.9099999999999999, 0.32999999999999996, 0.9, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425", "probabilities": [0.025, 0.76, 0.98, 0.96, 0.99, 0.81, 0.8600000000000001, 0.9, 0.325, 0.94, 0.44, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961", "probabilities": [0.85, 0.85, 0.69, 0.025, 0.97, 0.44, 0.44, 0.95, 0.96, 0.85, 0.44, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.42, 0.025, 0.89, 0.32, 0.25, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31145453746697192222724674918300432873", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.96, 0.355, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.325, 0.9199999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669", "probabilities": [0.995, 0.94, 0.02, 0.02, 0.02, 0.02, 0.02, 0.97, 0.36, 0.015, 0.97, 0.75], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.015, 0.015, 0.015, 0.9, 0.325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.9299999999999999, 0.25, 0.25, 0.015, 0.1325, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.025, 0.025, 0.99, 0.97, 0.36, 0.97, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296", "probabilities": [0.015, 0.015, 0.9199999999999999, 0.015, 0.025, 0.025, 0.9099999999999999, 0.015, 0.1325, 0.98, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.98, 0.015, 0.1325, 0.015, 0.015, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32488933488399304214586978746578986823", "probabilities": [0.025, 0.41, 0.98, 0.41, 0.99, 0.94, 0.99, 0.025, 0.1375, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664", "probabilities": [0.985, 0.025, 0.99, 0.25, 0.25, 0.99, 0.9299999999999999, 0.99, 0.37, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461", "probabilities": [0.025, 0.975, 0.025, 0.34, 0.025, 0.025, 0.025, 0.9, 0.325, 0.25, 0.975, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.25, 0.25, 0.975, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358", "probabilities": [0.025, 0.9199999999999999, 0.98, 0.4, 0.9099999999999999, 0.9, 0.94, 0.04, 0.145, 0.98, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569", "probabilities": [0.925, 0.9099999999999999, 0.975, 0.25, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.975, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042", "probabilities": [0.025, 0.025, 0.98, 0.025, 0.9199999999999999, 0.025, 0.9299999999999999, 0.225, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.9099999999999999, 0.9099999999999999, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443", "probabilities": [0.25, 0.25, 0.975, 0.25, 0.25, 0.25, 0.975, 0.25, 0.25, 0.975, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550", "probabilities": [0.025, 0.41, 0.44, 0.45, 0.99, 0.99, 0.99, 0.975, 0.9299999999999999, 0.99, 0.46, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.33841907095190352493353075864604147687", "probabilities": [0.925, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.96, 0.975, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598", "probabilities": [0.25, 0.25, 0.41, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650", "probabilities": [0.86, 0.025, 0.89, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.875, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.025, 0.98], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.015, 0.015, 0.9299999999999999, 0.01, 0.13, 0.01, 0.96, 0.075], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613", "probabilities": [0.015, 0.015, 0.4, 0.01, 0.89, 0.89, 0.89, 0.04, 0.145, 0.01, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.95, 0.01, 0.01, 0.89, 0.32, 0.01, 0.04, 0.04], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906", "probabilities": [0.995, 0.015, 0.01, 0.9299999999999999, 0.01, 0.01, 0.01, 0.97, 0.9099999999999999, 0.25, 0.54, 0.975], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.25, 0.25, 0.25, 0.01, 0.13, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.13, 0.9299999999999999, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139", "probabilities": [0.015, 0.89, 0.4, 0.01, 0.01, 0.01, 0.99, 0.89, 0.9, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841", "probabilities": [0.995, 0.015, 0.025, 0.025, 0.01, 0.01, 0.01, 0.89, 0.32, 0.015, 0.54, 0.98], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313", "probabilities": [0.41, 0.02, 0.97, 0.41, 0.98, 0.865, 0.98, 0.9099999999999999, 0.32999999999999996, 0.9199999999999999, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777", "probabilities": [0.25, 0.25, 0.41, 0.41, 0.25, 0.25, 0.9199999999999999, 0.25, 0.25, 0.96, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.325, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403", "probabilities": [0.995, 0.015, 0.985, 0.015, 0.24, 0.24, 0.015, 0.96, 0.355, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314", "probabilities": [0.015, 0.9, 0.995, 0.015, 0.9299999999999999, 0.015, 0.97, 0.04, 0.145, 0.9199999999999999, 0.25, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.89, 0.32, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.025, 0.025, 0.025, 0.96, 0.355, 0.25, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167", "probabilities": [0.725, 0.015, 0.98, 0.4, 0.87, 0.665, 0.835, 0.04, 0.145, 0.01, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185", "probabilities": [0.98, 0.015, 0.25, 0.35, 0.015, 0.015, 0.02, 0.89, 0.32, 0.01, 0.515, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985", "probabilities": [0.61, 0.71, 0.975, 0.5249999999999999, 0.99, 0.25, 0.99, 0.95, 0.975, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.925, 0.925, 0.3375, 0.975, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047", "probabilities": [0.25, 0.25, 0.99, 0.025, 0.995, 0.825, 0.99, 0.25, 0.4, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727", "probabilities": [0.025, 0.025, 0.85, 0.025, 0.025, 0.025, 0.9099999999999999, 0.89, 0.32, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.9, 0.25, 0.25, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908", "probabilities": [0.025, 0.025, 0.025, 0.99, 0.025, 0.99, 0.025, 0.89, 0.32, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122", "probabilities": [0.995, 0.025, 0.97, 0.025, 0.9099999999999999, 0.9199999999999999, 0.925, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.96, 0.25, 0.99, 0.97, 0.9650000000000001, 0.97, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.905, 0.905, 0.905, 0.885, 0.3175, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200", "probabilities": [0.9, 0.9, 0.7, 0.04, 0.94, 0.525, 0.96, 0.25, 0.905, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.025, 0.1375, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736", "probabilities": [0.015, 0.015, 0.4, 0.015, 0.25, 0.25, 0.015, 0.94, 0.345, 0.015, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219", "probabilities": [0.015, 0.015, 0.9650000000000001, 0.9650000000000001, 0.9, 0.9, 0.9, 0.015, 0.1325, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.6599999999999999, 0.6599999999999999, 0.98, 0.015, 0.1325, 0.015, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38893522232848525819919092148478879455", "probabilities": [0.025, 0.025, 0.025, 0.41, 0.25, 0.25, 0.025, 0.025, 0.1375, 0.015, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373", "probabilities": [0.015, 0.06, 0.985, 0.015, 0.985, 0.9099999999999999, 0.985, 0.89, 0.32, 0.97, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144", "probabilities": [0.9, 0.9, 0.97, 0.39, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948", "probabilities": [0.995, 0.9, 0.015, 0.015, 0.9099999999999999, 0.9, 0.99, 0.9099999999999999, 0.32999999999999996, 0.015, 0.96, 0.96], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161", "probabilities": [0.995, 0.9, 0.675, 0.99, 0.925, 0.94, 0.925, 0.99, 0.37, 0.985, 0.97, 0.995], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.025, 0.1375, 0.015, 0.44, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752", "probabilities": [0.935, 0.015, 0.97, 0.015, 0.97, 0.4, 0.4, 0.95, 0.96, 0.985, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402", "probabilities": [0.985, 0.015, 0.97, 0.015, 0.015, 0.015, 0.015, 0.89, 0.32, 0.9099999999999999, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.015, 0.015, 0.01, 0.89, 0.32, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592", "probabilities": [0.015, 0.015, 0.96, 0.015, 0.015, 0.015, 0.9, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693", "probabilities": [0.015, 0.015, 0.995, 0.015, 1.0, 0.97, 0.975, 0.9099999999999999, 0.95, 0.9099999999999999, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837", "probabilities": [0.015, 0.735, 0.7, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.25, 0.25, 0.9099999999999999, 0.95, 0.35, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037", "probabilities": [0.85, 0.015, 0.4, 0.865, 0.8, 0.8, 0.7250000000000001, 0.81, 0.32999999999999996, 0.015, 0.05, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.025, 0.1375, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872", "probabilities": [0.015, 0.015, 0.995, 0.015, 0.99, 0.015, 0.925, 0.995, 0.975, 0.975, 0.05, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994", "probabilities": [0.015, 0.05, 0.995, 0.97, 1.0, 0.9, 0.985, 0.97, 0.97, 0.015, 0.05, 0.995], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.75, 0.94, 0.95, 0.95, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212", "probabilities": [1.0, 0.9099999999999999, 0.975, 0.99, 0.015, 0.015, 0.015, 0.89, 0.32, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213", "probabilities": [0.99, 0.015, 0.975, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.05, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415", "probabilities": [0.995, 0.025, 0.98, 0.97, 0.875, 0.45, 0.25, 0.9, 0.325, 0.025, 0.875, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414", "probabilities": [0.025, 0.025, 0.41, 0.41, 0.25, 0.25, 0.25, 0.94, 0.345, 0.97, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.97, 0.89, 0.32, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683", "probabilities": [0.025, 0.025, 0.06, 0.025, 0.95, 0.935, 0.96, 0.95, 0.35, 0.96, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004", "probabilities": [0.025, 0.25, 0.98, 0.25, 0.25, 0.25, 0.25, 0.96, 0.355, 0.98, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476", "probabilities": [0.025, 0.25, 0.025, 0.025, 0.025, 0.025, 0.9, 0.975, 0.3625, 0.25, 0.94, 0.9199999999999999], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480", "probabilities": [0.25, 0.25, 0.08499999999999999, 0.25, 0.25, 0.25, 0.94, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876", "probabilities": [0.025, 0.025, 0.25, 0.96, 0.25, 0.25, 0.25, 0.975, 0.3625, 0.25, 0.675, 0.99], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.42598518026540237824479232797503218019", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9199999999999999, 0.33499999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.94, 0.025, 0.99, 0.975, 0.3625, 0.96, 0.41, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744", "probabilities": [0.025, 0.9, 0.85, 0.025, 0.025, 0.025, 0.975, 0.025, 0.1375, 0.025, 0.05, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887", "probabilities": [0.025, 0.25, 0.05, 0.05, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741", "probabilities": [0.55, 0.025, 0.425, 0.425, 0.25, 0.25, 0.25, 0.025, 0.1375, 0.97, 0.85, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838", "probabilities": [0.025, 0.025, 0.05, 0.05, 0.025, 0.025, 0.975, 0.025, 0.1375, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343", "probabilities": [0.995, 0.025, 0.995, 0.98, 0.025, 0.025, 0.025, 0.975, 0.3625, 0.99, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736", "probabilities": [0.05, 0.9099999999999999, 0.975, 0.425, 0.925, 0.39, 0.25, 0.9099999999999999, 0.32999999999999996, 0.94, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.925, 0.3375, 0.25, 0.95, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822", "probabilities": [0.025, 0.025, 0.975, 0.7, 0.65, 0.65, 0.025, 0.925, 0.3375, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052", "probabilities": [0.99, 0.025, 0.425, 0.98, 0.925, 0.425, 0.025, 0.89, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44149403030093858908988179411336839587", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.89, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208", "probabilities": [0.7849999999999999, 0.025, 0.675, 0.675, 0.25, 0.25, 0.85, 0.995, 0.95, 0.985, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889", "probabilities": [0.25, 0.25, 0.05, 0.05, 0.975, 0.975, 0.975, 0.25, 0.25, 0.975, 0.975, 0.975], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569", "probabilities": [0.015, 0.015, 0.99, 0.015, 0.985, 0.975, 0.995, 0.995, 0.3725, 0.25, 0.41, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.015, 0.01, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025", "probabilities": [0.985, 0.015, 0.015, 0.665, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010", "probabilities": [0.015, 0.9099999999999999, 0.44, 0.41, 0.95, 0.95, 0.9199999999999999, 0.01, 0.13, 0.96, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959", "probabilities": [0.6000000000000001, 0.015, 0.41, 0.41, 0.25, 0.25, 0.97, 0.01, 0.13, 0.25, 0.42, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641", "probabilities": [0.015, 0.015, 0.39, 0.39, 0.25, 0.25, 0.25, 0.01, 0.13, 0.25, 0.015, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775", "probabilities": [0.995, 0.94, 0.015, 0.015, 0.015, 0.015, 0.015, 0.975, 0.3625, 0.01, 0.975, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674", "probabilities": [0.025, 0.025, 0.98, 0.025, 0.9099999999999999, 0.025, 0.94, 0.9, 0.325, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.25, 0.9099999999999999, 0.96, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662", "probabilities": [0.89, 0.99, 0.6599999999999999, 0.98, 0.99, 0.975, 0.97, 0.9099999999999999, 0.32999999999999996, 0.025, 0.9199999999999999, 0.99], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46282234455466954264170065462242039568", "probabilities": [0.25, 0.05, 0.41, 0.25, 0.41, 0.41, 0.97, 0.95, 0.9099999999999999, 0.98, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523", "probabilities": [0.025, 0.025, 0.025, 0.41, 0.76, 0.25, 0.81, 0.9, 0.325, 0.98, 0.78, 0.35], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766", "probabilities": [0.025, 0.025, 0.94, 0.025, 0.25, 0.81, 0.94, 0.9, 0.325, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46713978512945139851373191422321462143", "probabilities": [0.025, 0.025, 0.4, 0.99, 0.94, 0.9099999999999999, 0.89, 0.97, 0.96, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.925, 0.25, 0.94, 0.9099999999999999, 0.32999999999999996, 0.97, 0.25, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949", "probabilities": [0.41, 0.025, 0.41, 0.97, 0.025, 0.98, 0.97, 0.98, 0.365, 0.97, 0.44, 0.75], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431", "probabilities": [0.25, 0.25, 0.39, 0.39, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787", "probabilities": [0.025, 0.025, 0.4, 0.975, 0.81, 0.81, 0.99, 0.025, 0.1375, 0.25, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.925, 0.25, 0.9099999999999999, 0.32999999999999996, 0.025, 0.835, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642", "probabilities": [0.675, 0.025, 0.98, 0.675, 0.635, 0.9199999999999999, 0.9199999999999999, 0.9, 0.325, 0.025, 0.4, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953", "probabilities": [0.025, 0.025, 0.05, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684", "probabilities": [0.025, 0.025, 0.4, 0.025, 0.9099999999999999, 0.25, 0.99, 0.025, 0.1375, 0.25, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825", "probabilities": [0.025, 0.025, 0.05, 0.05, 0.025, 0.025, 0.9, 0.025, 0.1375, 0.9, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710", "probabilities": [0.995, 0.025, 0.975, 0.96, 0.99, 0.99, 0.99, 0.97, 0.36, 0.25, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959", "probabilities": [0.025, 0.025, 0.025, 0.97, 0.575, 0.96, 0.575, 0.94, 0.97, 0.94, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552", "probabilities": [0.025, 0.97, 0.05, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.96, 0.9199999999999999, 0.89, 0.32, 0.25, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.48160569779413970442472500116176876660", "probabilities": [0.025, 0.025, 0.4, 0.25, 0.25, 0.25, 0.7, 0.94, 0.94, 0.97, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.94, 0.94, 0.985, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208", "probabilities": [0.025, 0.9, 0.98, 0.05, 0.025, 0.025, 0.025, 0.89, 0.32, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545", "probabilities": [0.025, 0.025, 0.995, 0.025, 0.98, 0.9099999999999999, 0.99, 0.9650000000000001, 0.97, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.89, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.04, 0.145, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727", "probabilities": [0.025, 0.25, 0.025, 0.025, 0.025, 0.025, 0.025, 0.97, 0.36, 0.25, 0.825, 0.975], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329", "probabilities": [0.05, 0.025, 0.025, 0.95, 0.025, 0.025, 0.25, 0.975, 0.3625, 0.25, 0.765, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513", "probabilities": [0.025, 0.025, 0.69, 0.025, 0.97, 0.97, 0.9099999999999999, 0.99, 0.985, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116", "probabilities": [0.025, 0.025, 0.05, 0.025, 0.97, 0.025, 0.97, 0.9, 0.325, 0.1, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550", "probabilities": [0.81, 0.025, 0.41, 0.41, 0.25, 0.9099999999999999, 0.925, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.7, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784", "probabilities": [0.985, 0.9099999999999999, 0.99, 0.41, 0.975, 0.025, 0.025, 0.99, 0.37, 0.25, 0.9099999999999999, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.995, 0.925, 0.25, 0.94, 0.345, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053", "probabilities": [0.25, 0.25, 0.96, 0.25, 0.25, 0.25, 0.25, 0.94, 0.96, 0.975, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.51469564445249084310991187819410005207", "probabilities": [0.025, 0.025, 0.25, 0.25, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131", "probabilities": [0.995, 0.025, 0.99, 0.425, 0.25, 0.25, 0.25, 0.925, 0.925, 0.975, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643", "probabilities": [0.025, 0.925, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9, 0.325, 0.025, 0.25, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.51831304809871808790657397599860793132", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.05, 0.05, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915", "probabilities": [0.025, 0.025, 0.425, 0.025, 0.25, 0.25, 0.45, 0.025, 0.1375, 0.975, 0.7749999999999999, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065", "probabilities": [0.025, 0.025, 0.4, 0.4, 0.025, 0.025, 0.95, 0.25, 0.25, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.05, 0.025, 0.25, 0.89, 0.32, 0.025, 0.025, 0.9099999999999999], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146", "probabilities": [0.025, 0.05, 0.99, 0.025, 0.985, 0.025, 0.97, 0.975, 0.3625, 0.025, 0.05, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501", "probabilities": [0.99, 0.9099999999999999, 0.98, 0.41, 0.97, 0.25, 0.9199999999999999, 0.25, 0.25, 0.925, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643", "probabilities": [0.025, 0.025, 0.41, 0.41, 0.025, 0.025, 0.96, 0.9099999999999999, 0.9099999999999999, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9550000000000001, 0.985, 0.97, 0.975, 0.3625, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.025, 0.025, 0.025, 0.89, 0.32, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763", "probabilities": [0.025, 0.025, 0.41, 0.41, 0.675, 0.98, 0.99, 0.89, 0.32, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383", "probabilities": [0.025, 0.975, 0.975, 0.025, 0.025, 0.025, 0.475, 0.9099999999999999, 0.32999999999999996, 0.96, 0.96, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.025, 0.025, 0.99, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025", "probabilities": [0.25, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665", "probabilities": [0.025, 0.025, 0.96, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593", "probabilities": [0.875, 0.9099999999999999, 0.8200000000000001, 0.8, 0.99, 0.94, 0.99, 0.25, 0.25, 0.96, 0.475, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587", "probabilities": [0.025, 0.025, 0.025, 0.41, 0.94, 0.025, 0.94, 0.5900000000000001, 0.32, 0.89, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870", "probabilities": [0.25, 0.25, 0.41, 0.25, 0.25, 0.25, 0.96, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518", "probabilities": [0.99, 0.925, 0.975, 0.95, 0.925, 0.925, 0.925, 0.89, 0.32, 0.25, 0.975, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642", "probabilities": [0.125, 0.025, 0.025, 0.025, 0.9, 0.9, 0.98, 0.25, 0.25, 0.01, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.985, 0.3675, 0.25, 0.98, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535", "probabilities": [0.96, 0.025, 0.98, 0.015, 0.99, 0.9, 0.97, 0.96, 0.355, 0.01, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616", "probabilities": [0.015, 0.015, 0.06, 0.015, 0.01, 0.01, 0.01, 0.89, 0.32, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770", "probabilities": [0.98, 0.025, 0.7, 0.025, 0.015, 0.015, 0.98, 0.89, 0.32, 0.01, 0.885, 0.95], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265", "probabilities": [0.995, 0.015, 0.99, 0.97, 0.25, 0.94, 0.25, 0.95, 0.35, 0.01, 0.95, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879", "probabilities": [0.015, 0.94, 0.995, 0.015, 0.9, 0.9, 0.995, 0.97, 0.95, 0.99, 0.55, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860", "probabilities": [0.015, 0.015, 0.735, 0.015, 0.985, 0.39, 0.015, 0.89, 0.32, 0.97, 0.75, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068", "probabilities": [0.04, 0.25, 0.6499999999999999, 0.5599999999999999, 0.75, 0.75, 0.96, 0.9, 0.325, 0.97, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754", "probabilities": [0.025, 0.735, 0.94, 0.025, 0.9099999999999999, 0.25, 0.25, 0.975, 0.05, 0.975, 0.94, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.25, 0.9099999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.25, 0.025, 0.025, 0.9099999999999999, 0.9099999999999999, 0.25, 0.475, 0.98], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348", "probabilities": [0.7150000000000001, 0.015, 0.96, 0.96, 0.015, 0.015, 0.85, 0.99, 0.975, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195", "probabilities": [0.025, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9099999999999999, 0.01, 0.13, 0.01, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.89, 0.025, 0.025, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.015, 0.1325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058", "probabilities": [0.025, 0.025, 0.97, 0.97, 0.9199999999999999, 0.025, 0.025, 0.04, 0.145, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485", "probabilities": [0.995, 0.025, 0.99, 0.98, 0.985, 0.94, 0.925, 0.94, 0.96, 0.25, 0.43, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235", "probabilities": [0.025, 0.025, 0.98, 0.96, 0.025, 0.025, 0.9099999999999999, 0.9099999999999999, 0.9199999999999999, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197", "probabilities": [0.98, 0.015, 0.9650000000000001, 0.975, 0.985, 0.94, 0.04, 0.9099999999999999, 0.32999999999999996, 0.015, 0.925, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.97, 0.98, 0.9199999999999999, 0.33499999999999996, 0.89, 0.515, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.44, 0.44, 0.94, 0.9, 0.325, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331", "probabilities": [0.96, 0.015, 0.665, 0.015, 0.89, 0.89, 0.89, 0.05, 0.1375, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841", "probabilities": [0.015, 0.685, 0.97, 0.015, 0.25, 0.9099999999999999, 0.89, 0.025, 0.1375, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675", "probabilities": [0.585, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.325, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770", "probabilities": [0.015, 0.015, 0.41, 0.99, 0.25, 0.25, 0.25, 0.41, 0.32999999999999996, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.89, 0.32, 0.89, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799", "probabilities": [0.015, 0.015, 0.97, 0.41, 0.25, 0.25, 0.25, 0.9, 0.325, 0.025, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893", "probabilities": [0.015, 0.015, 0.9199999999999999, 0.015, 0.9099999999999999, 0.015, 0.015, 0.9099999999999999, 0.32999999999999996, 0.015, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.9099999999999999, 0.9099999999999999, 0.9099999999999999, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62406588098851994968957074186291720431", "probabilities": [0.015, 0.015, 0.99, 0.04, 0.9450000000000001, 0.9099999999999999, 0.9099999999999999, 0.99, 0.975, 0.985, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272", "probabilities": [0.05, 0.9099999999999999, 0.46, 0.41, 0.25, 0.25, 0.25, 0.96, 0.355, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.89, 0.32, 0.9099999999999999, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62918046703989259691005447883746340916", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9099999999999999, 0.32999999999999996, 0.25, 0.97, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414", "probabilities": [0.015, 0.015, 0.985, 0.97, 0.985, 0.015, 0.015, 0.9099999999999999, 0.32999999999999996, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.62948920639798990757395955917890550512", "probabilities": [0.015, 0.015, 0.99, 0.015, 0.015, 0.015, 0.015, 0.97, 0.36, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.975, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.63280826430574305683941548157934410641", "probabilities": [0.025, 0.025, 0.86, 0.86, 0.25, 0.97, 0.25, 0.96, 0.355, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941", "probabilities": [0.025, 0.025, 0.25, 0.34, 0.025, 0.025, 0.025, 0.89, 0.32, 0.025, 0.49, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.63584266247144369196126771996126173497", "probabilities": [0.025, 0.025, 0.4, 0.025, 0.5, 0.5, 0.5, 0.025, 0.1375, 0.89, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906", "probabilities": [0.025, 0.9, 0.025, 0.025, 0.025, 0.025, 0.925, 0.9099999999999999, 0.32999999999999996, 0.025, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316", "probabilities": [0.025, 0.025, 0.41, 0.41, 0.25, 0.3, 0.25, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373", "probabilities": [0.05, 0.025, 0.99, 0.025, 0.8500000000000001, 0.8, 0.8500000000000001, 0.05, 0.15, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715", "probabilities": [0.995, 0.025, 0.995, 0.025, 0.995, 0.98, 0.975, 0.96, 0.95, 0.975, 0.475, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.98, 0.9099999999999999, 0.985, 0.9099999999999999, 0.96, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.925, 0.025, 0.99, 0.9099999999999999, 0.97, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223", "probabilities": [0.97, 0.025, 0.025, 0.025, 0.25, 0.94, 0.94, 0.97, 0.36, 0.96, 0.9099999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643", "probabilities": [0.025, 0.25, 0.4, 0.25, 0.9099999999999999, 0.25, 0.25, 0.39, 0.32, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455", "probabilities": [0.98, 0.025, 0.025, 0.025, 0.25, 0.25, 0.97, 0.025, 0.1375, 0.96, 0.96, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554", "probabilities": [0.025, 0.025, 0.025, 0.4, 0.025, 0.025, 0.025, 0.04, 0.145, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719", "probabilities": [0.025, 0.985, 0.025, 0.025, 0.025, 0.025, 0.98, 0.95, 0.35, 0.025, 0.96, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666", "probabilities": [0.025, 0.025, 0.025, 0.97, 0.925, 0.97, 0.97, 0.99, 0.97, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674", "probabilities": [0.025, 0.025, 0.96, 0.025, 0.975, 0.25, 0.94, 0.95, 0.35, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980", "probabilities": [0.025, 0.9099999999999999, 0.97, 0.025, 0.96, 0.94, 0.9099999999999999, 0.9299999999999999, 0.33999999999999997, 0.25, 0.425, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.66077420344931334696112616504179985591", "probabilities": [0.015, 0.015, 0.94, 0.015, 0.9, 0.96, 0.98, 0.015, 0.1325, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887", "probabilities": [0.7, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755", "probabilities": [0.025, 0.025, 0.99, 0.41, 0.995, 0.44, 0.985, 0.95, 0.35, 0.98, 0.6499999999999999, 0.1], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524", "probabilities": [0.925, 0.015, 0.97, 0.015, 0.9199999999999999, 0.9199999999999999, 0.9199999999999999, 0.9, 0.325, 0.25, 0.25, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258", "probabilities": [0.025, 0.025, 0.015, 0.41, 0.015, 0.015, 0.015, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.975, 0.65, 0.65, 0.95, 0.97, 0.98, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306", "probabilities": [0.9099999999999999, 0.015, 0.725, 0.435, 0.94, 0.94, 0.925, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666", "probabilities": [0.41, 0.015, 0.97, 0.015, 0.95, 0.95, 0.95, 0.25, 0.25, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147", "probabilities": [0.96, 0.9099999999999999, 0.01, 0.725, 0.01, 0.935, 0.25, 0.89, 0.32, 0.25, 0.96, 0.975], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68142054543837964766571710567214307830", "probabilities": [0.015, 0.015, 0.25, 0.25, 0.015, 0.015, 0.015, 0.01, 0.13, 0.25, 0.69, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436", "probabilities": [0.9099999999999999, 0.015, 0.41, 0.25, 0.425, 0.925, 0.99, 0.95, 0.97, 0.95, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150", "probabilities": [0.025, 0.015, 0.96, 0.015, 0.015, 0.015, 0.9, 0.97, 0.36, 0.25, 0.7849999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.79, 0.79, 0.79, 0.9099999999999999, 0.32999999999999996, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600", "probabilities": [0.015, 0.015, 0.9650000000000001, 0.015, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68707023438918237611039069622720041636", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.95, 0.015, 0.015, 0.9, 0.325, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.95, 0.015, 0.1325, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157", "probabilities": [0.985, 0.975, 0.97, 0.015, 0.015, 0.015, 0.015, 0.99, 0.37, 0.25, 0.75, 0.99], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960", "probabilities": [0.015, 0.015, 0.015, 0.9650000000000001, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819", "probabilities": [0.015, 0.015, 0.985, 0.015, 0.98, 0.975, 0.975, 0.97, 0.36, 0.985, 0.25, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455", "probabilities": [0.99, 0.015, 0.015, 0.015, 0.9099999999999999, 0.25, 0.9199999999999999, 0.9, 0.325, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384", "probabilities": [0.015, 0.9299999999999999, 0.425, 0.42, 0.9299999999999999, 0.39, 0.985, 0.95, 0.96, 0.97, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579", "probabilities": [0.025, 0.675, 0.39, 0.025, 0.96, 0.025, 0.9099999999999999, 0.975, 0.925, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063", "probabilities": [0.995, 0.025, 0.975, 0.025, 0.25, 0.25, 0.025, 0.975, 0.825, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.94, 0.94, 0.975, 0.975, 0.3625, 0.025, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.96, 0.9099999999999999, 0.975, 0.89, 0.32, 0.025, 0.45, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.94, 0.9, 0.325, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.99, 0.025, 0.9099999999999999, 0.9099999999999999, 0.32999999999999996, 0.025, 0.475, 0.8], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70831735578941318362881379802693293952", "probabilities": [0.995, 0.25, 0.99, 0.41, 0.995, 0.86, 0.835, 0.96, 0.355, 0.25, 0.46, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526", "probabilities": [0.7150000000000001, 0.9099999999999999, 0.99, 0.4, 0.25, 0.25, 0.925, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.925, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104", "probabilities": [0.025, 0.025, 0.64, 0.64, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.975, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.025, 0.025, 0.975, 0.975, 0.3625, 0.025, 0.975, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261", "probabilities": [0.05, 0.025, 0.975, 0.025, 0.9099999999999999, 0.9099999999999999, 0.975, 0.9099999999999999, 0.32999999999999996, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846", "probabilities": [0.985, 0.95, 0.025, 0.025, 0.975, 0.025, 0.9, 0.94, 0.345, 0.9, 0.94, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.88, 0.315, 0.88, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.025, 0.025, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190", "probabilities": [0.06, 0.015, 0.4, 0.015, 0.95, 0.25, 0.25, 0.94, 0.345, 0.94, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597", "probabilities": [0.015, 0.56, 0.015, 0.015, 0.85, 0.015, 0.7, 0.01, 0.13, 0.88, 0.44, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666", "probabilities": [0.015, 0.015, 0.96, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639", "probabilities": [0.025, 0.025, 0.96, 0.05, 0.025, 0.025, 0.025, 0.9, 0.325, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230", "probabilities": [0.025, 0.025, 0.05, 0.025, 0.025, 0.915, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9, 0.325, 0.98, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.73325959289998105909842627876607163467", "probabilities": [0.995, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.985, 0.3675, 0.25, 0.9099999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803", "probabilities": [0.025, 0.025, 0.025, 0.41, 0.025, 0.9099999999999999, 0.025, 0.025, 0.1375, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320", "probabilities": [0.025, 0.025, 0.025, 0.58, 0.025, 0.76, 0.99, 0.975, 0.3625, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.39, 0.32, 0.975, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465", "probabilities": [0.995, 0.9299999999999999, 0.96, 0.94, 0.025, 0.025, 0.025, 0.89, 0.32, 0.25, 0.96, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661", "probabilities": [0.06, 0.025, 0.96, 0.025, 0.9099999999999999, 0.025, 0.025, 0.9, 0.325, 0.735, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.25, 0.25, 0.96, 0.025, 0.1375, 0.25, 0.65, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669", "probabilities": [0.025, 0.025, 0.96, 0.25, 0.98, 0.9099999999999999, 0.97, 0.9099999999999999, 0.32999999999999996, 0.94, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554", "probabilities": [0.025, 0.025, 0.025, 0.5349999999999999, 0.025, 0.025, 0.025, 0.99, 0.37, 0.25, 0.97, 0.99], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.98, 0.9099999999999999, 0.985, 0.9199999999999999, 0.33499999999999996, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76190704217737835083438638928129775625", "probabilities": [0.025, 0.025, 0.975, 0.45, 0.76, 0.76, 0.76, 0.9199999999999999, 0.33499999999999996, 0.98, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76344826994380951498651448092980303697", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76525107909962775502207829870873494024", "probabilities": [0.7749999999999999, 0.025, 0.97, 0.025, 0.025, 0.025, 0.96, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76538737802277241976971950690663079376", "probabilities": [0.25, 0.9099999999999999, 0.47, 0.425, 0.97, 0.25, 0.25, 0.95, 0.96, 0.25, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120", "probabilities": [0.995, 0.99, 0.025, 0.025, 0.025, 0.025, 0.025, 0.96, 0.94, 0.9099999999999999, 0.96, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992", "probabilities": [0.625, 0.025, 0.96, 0.025, 0.97, 0.6599999999999999, 0.41, 0.95, 0.97, 0.98, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.025, 0.025, 0.9199999999999999, 0.97, 0.36, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789", "probabilities": [0.995, 0.9099999999999999, 0.025, 0.97, 0.25, 0.25, 0.025, 0.89, 0.32, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9299999999999999, 0.33999999999999997, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536", "probabilities": [0.25, 0.025, 0.43, 0.025, 0.94, 0.25, 0.25, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695", "probabilities": [0.025, 0.44, 0.97, 0.425, 0.25, 0.25, 0.975, 0.97, 0.36, 0.95, 0.25, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.74, 0.025, 0.025, 0.35, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044", "probabilities": [0.98, 0.025, 0.99, 0.025, 0.96, 0.96, 0.96, 0.9099999999999999, 0.95, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136", "probabilities": [0.01, 0.015, 0.015, 0.015, 0.015, 0.015, 0.975, 0.01, 0.13, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839", "probabilities": [0.41, 0.015, 0.475, 0.04, 0.985, 0.8400000000000001, 0.8400000000000001, 0.9099999999999999, 0.32999999999999996, 0.975, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809", "probabilities": [0.015, 0.015, 0.99, 0.25, 0.9650000000000001, 0.25, 0.25, 0.89, 0.8200000000000001, 0.25, 0.41, 0.98], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.89, 0.025, 0.025, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146", "probabilities": [0.94, 0.015, 0.4, 0.015, 0.03, 0.03, 0.04, 0.01, 0.13, 0.25, 0.635, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176", "probabilities": [0.015, 0.015, 0.475, 0.25, 0.25, 0.9199999999999999, 0.9199999999999999, 0.74, 0.32, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.77889800516853576877948593628420228178", "probabilities": [0.01, 0.01, 0.175, 0.01, 0.01, 0.01, 0.01, 0.94, 0.345, 0.98, 0.01, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.78072765790191594685273447833272737429", "probabilities": [0.01, 0.01, 0.97, 0.01, 0.8200000000000001, 0.75, 0.75, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.9099999999999999, 0.97, 0.01, 0.13, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268", "probabilities": [0.125, 0.025, 0.25, 0.05, 0.98, 0.05, 0.05, 0.9099999999999999, 0.975, 0.94, 0.94, 0.9099999999999999], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052", "probabilities": [0.25, 0.25, 0.925, 0.25, 0.9099999999999999, 0.25, 0.25, 0.97, 0.36, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.78800983923143929298359366572020370031", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.07500000000000001, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.9299999999999999, 0.9199999999999999, 0.9199999999999999, 0.9099999999999999, 0.32999999999999996, 0.96, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.79326627674334637493320764205397279674", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.025, 0.025, 0.025, 0.05, 0.15, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807", "probabilities": [0.025, 0.025, 0.05, 0.97, 0.025, 0.025, 0.025, 0.05, 0.15, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775", "probabilities": [0.25, 0.25, 0.6, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492", "probabilities": [0.015, 0.9099999999999999, 0.975, 0.025, 0.025, 0.025, 0.015, 0.9099999999999999, 0.025, 0.98, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187", "probabilities": [0.05, 0.015, 0.015, 0.015, 0.6599999999999999, 0.6599999999999999, 0.015, 0.025, 0.1375, 0.015, 0.015, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.89, 0.025, 0.025, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286", "probabilities": [0.05, 0.015, 0.015, 0.98, 0.925, 0.995, 0.015, 0.9099999999999999, 0.32999999999999996, 0.97, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506", "probabilities": [0.015, 0.89, 0.99, 0.915, 0.995, 0.95, 0.95, 0.97, 0.6599999999999999, 0.25, 0.625, 0.5650000000000001], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715", "probabilities": [0.015, 0.9099999999999999, 0.015, 0.015, 0.9099999999999999, 0.9099999999999999, 0.25, 0.025, 0.1375, 0.9099999999999999, 0.015, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417", "probabilities": [0.125, 0.025, 0.025, 0.96, 0.99, 0.885, 0.99, 0.975, 0.3625, 0.99, 0.45, 0.96], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.81109405367395078564887780335570099673", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.98, 0.015, 0.015, 0.975, 0.3625, 0.25, 0.76, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9, 0.015, 0.1325, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421", "probabilities": [0.25, 0.25, 0.41, 0.41, 0.99, 0.25, 0.25, 0.96, 0.355, 0.25, 0.45, 0.96], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.25, 0.9299999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894", "probabilities": [0.9199999999999999, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508", "probabilities": [0.05, 0.015, 0.41, 0.41, 0.5, 0.5, 0.995, 0.9099999999999999, 0.32999999999999996, 0.95, 0.44, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974", "probabilities": [0.025, 0.97, 0.97, 0.41, 0.25, 0.25, 0.9199999999999999, 0.025, 0.1375, 0.25, 0.125, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746", "probabilities": [0.025, 0.025, 0.42, 0.41, 0.94, 0.49, 0.9, 0.9099999999999999, 0.32999999999999996, 0.975, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.82587763533601760002619014353812074700", "probabilities": [0.025, 0.025, 0.01, 0.01, 0.25, 0.25, 0.9199999999999999, 0.95, 0.35, 0.25, 0.01, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758", "probabilities": [0.025, 0.9199999999999999, 0.865, 0.41, 0.765, 0.765, 0.97, 0.95, 0.35, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400", "probabilities": [0.81, 0.865, 0.99, 0.025, 0.99, 0.025, 0.995, 0.9099999999999999, 0.32999999999999996, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239", "probabilities": [0.69, 0.025, 0.04, 0.04, 0.025, 0.025, 0.025, 0.01, 0.13, 0.25, 0.01, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936", "probabilities": [0.25, 0.25, 0.05, 0.05, 0.79, 0.995, 0.995, 0.99, 0.895, 0.96, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602", "probabilities": [0.025, 0.025, 0.4, 0.025, 0.25, 0.25, 0.25, 0.96, 0.355, 0.25, 0.98, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.9, 0.325, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.13, 0.25, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783", "probabilities": [0.99, 0.975, 0.675, 0.025, 0.025, 0.025, 0.025, 0.05, 0.15, 0.96, 0.975, 0.7], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488", "probabilities": [0.025, 0.025, 0.41, 0.025, 0.025, 0.025, 0.95, 0.9, 0.325, 0.25, 0.81, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.9099999999999999, 0.9099999999999999, 0.9099999999999999, 0.32999999999999996, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652", "probabilities": [0.025, 0.025, 0.41, 0.41, 0.25, 0.25, 0.25, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.995, 0.025, 0.1375, 0.9, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518", "probabilities": [0.94, 0.025, 0.025, 0.675, 0.7250000000000001, 0.75, 0.75, 0.89, 0.32, 0.025, 0.815, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066", "probabilities": [0.025, 0.025, 0.4, 0.025, 0.925, 0.39, 0.9099999999999999, 0.9099999999999999, 0.925, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.99, 0.9099999999999999, 0.95, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892", "probabilities": [0.075, 0.025, 0.39, 0.39, 0.25, 0.25, 0.95, 0.25, 0.25, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.975, 0.3625, 0.015, 0.725, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091", "probabilities": [0.015, 0.015, 0.4, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.015, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.86239793008693746155478763153154040128", "probabilities": [0.015, 0.015, 0.025, 0.025, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889", "probabilities": [0.015, 0.015, 0.985, 0.41, 0.995, 0.94, 0.94, 0.97, 0.98, 0.25, 0.43, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601", "probabilities": [0.41, 0.9099999999999999, 0.96, 0.99, 0.97, 0.25, 0.25, 0.95, 0.96, 0.9299999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.25, 0.835, 0.9099999999999999, 0.32999999999999996, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831", "probabilities": [0.015, 0.015, 0.99, 0.425, 0.75, 0.65, 0.65, 0.95, 0.35, 0.95, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520", "probabilities": [0.625, 0.015, 0.975, 0.6, 0.635, 0.635, 0.75, 0.025, 0.1625, 0.26, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.475, 0.89, 0.89, 0.32, 0.9099999999999999, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171", "probabilities": [0.015, 0.015, 0.41, 0.015, 0.9099999999999999, 0.25, 0.25, 0.25, 0.25, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87415625824966124550466651648244553892", "probabilities": [0.015, 0.015, 0.425, 0.25, 0.015, 0.015, 0.015, 0.925, 0.3375, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.69, 0.81, 0.81, 0.01, 0.13, 0.01, 0.4, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757", "probabilities": [0.35, 0.015, 0.015, 0.975, 0.015, 0.96, 0.015, 0.9099999999999999, 0.32999999999999996, 0.95, 0.8, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348", "probabilities": [0.015, 0.96, 0.4, 0.015, 0.975, 0.975, 0.975, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.25, 0.25, 0.9199999999999999, 0.9, 0.325, 0.985, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015", "probabilities": [0.995, 0.015, 0.99, 0.015, 0.985, 0.25, 0.015, 0.97, 0.36, 0.9199999999999999, 0.89, 0.7], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212", "probabilities": [0.97, 0.015, 0.41, 0.7749999999999999, 0.015, 0.015, 0.015, 0.985, 0.3675, 0.25, 0.9099999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294", "probabilities": [0.01, 0.79, 0.41, 0.01, 0.01, 0.01, 0.01, 0.01, 0.13, 0.9099999999999999, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030", "probabilities": [0.025, 0.025, 0.99, 0.025, 0.985, 0.9099999999999999, 0.985, 0.975, 0.3625, 0.925, 0.425, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.05, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.675, 0.675, 0.95, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89720428711655200442262503508409163392", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.25, 0.95, 0.025, 0.025, 0.1375, 0.25, 0.425, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.32999999999999996, 0.25, 0.925, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712", "probabilities": [0.25, 0.25, 0.95, 0.025, 0.81, 0.025, 0.81, 0.95, 0.95, 0.975, 0.875, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.90268709354762000249148795667759483876", "probabilities": [0.025, 0.025, 0.025, 0.05, 0.25, 0.25, 0.99, 0.25, 0.9099999999999999, 0.89, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942", "probabilities": [0.95, 0.025, 0.025, 0.025, 0.9, 0.025, 0.9, 0.025, 0.1375, 0.025, 0.86, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272", "probabilities": [0.05, 0.025, 0.96, 0.025, 0.995, 0.96, 0.96, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293", "probabilities": [0.05, 0.025, 0.975, 0.41, 0.995, 0.96, 0.975, 0.97, 0.86, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080", "probabilities": [0.25, 0.25, 0.975, 0.25, 0.97, 0.25, 0.97, 0.97, 0.97, 0.97, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979", "probabilities": [0.025, 0.025, 0.025, 0.05, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454", "probabilities": [0.99, 0.9099999999999999, 0.725, 0.025, 0.025, 0.025, 0.025, 0.94, 0.345, 0.25, 0.8400000000000001, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089", "probabilities": [0.025, 0.025, 0.97, 0.025, 0.97, 0.97, 0.97, 0.99, 0.975, 0.97, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365", "probabilities": [0.25, 0.25, 0.98, 0.25, 0.25, 0.25, 0.25, 0.95, 0.35, 0.25, 0.95, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.95, 0.96, 0.96, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.97, 0.25, 0.97, 0.01, 0.13, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.91883829404782805058767342767479130421", "probabilities": [0.995, 0.015, 0.99, 0.015, 0.925, 0.925, 0.925, 0.97, 0.36, 0.01, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016", "probabilities": [0.015, 0.015, 0.015, 0.97, 0.015, 0.015, 0.025, 0.89, 0.32, 0.01, 0.5249999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069", "probabilities": [0.015, 0.9199999999999999, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.975, 0.97, 0.9199999999999999, 0.45, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675", "probabilities": [0.015, 0.015, 0.975, 0.015, 0.97, 0.015, 0.015, 0.95, 0.35, 0.99, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095", "probabilities": [0.94, 0.97, 0.97, 0.015, 0.015, 0.9, 0.015, 0.9, 0.325, 0.9099999999999999, 0.935, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780", "probabilities": [0.015, 0.015, 0.05, 0.015, 0.97, 0.9299999999999999, 0.9299999999999999, 0.25, 0.25, 0.89, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433", "probabilities": [0.25, 0.89, 0.94, 0.94, 0.25, 0.25, 0.9099999999999999, 0.95, 0.35, 0.25, 0.46, 0.985], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078", "probabilities": [0.25, 0.25, 0.4, 0.9, 0.515, 0.515, 0.995, 0.95, 0.35, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305", "probabilities": [0.015, 0.99, 0.015, 0.015, 0.25, 0.25, 0.9099999999999999, 0.9099999999999999, 0.32999999999999996, 0.25, 0.7849999999999999, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945", "probabilities": [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.81, 0.9099999999999999, 0.32999999999999996, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.9199999999999999, 0.015, 0.9099999999999999, 0.89, 0.32, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679", "probabilities": [0.025, 0.025, 0.06, 0.025, 0.025, 0.025, 0.96, 0.9, 0.325, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151", "probabilities": [0.025, 0.025, 0.94, 0.025, 0.9650000000000001, 0.9099999999999999, 0.94, 0.95, 0.925, 0.25, 0.75, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068", "probabilities": [0.025, 0.5, 0.97, 0.4, 0.9099999999999999, 0.97, 0.97, 0.9099999999999999, 0.32999999999999996, 0.9, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680", "probabilities": [0.025, 0.025, 0.4, 0.4, 0.98, 0.025, 0.025, 0.975, 0.3625, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.97, 0.025, 0.1375, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93902628119199416703899320994283313812", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9, 0.025, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308", "probabilities": [0.025, 0.025, 0.4, 0.025, 0.025, 0.025, 0.025, 0.9, 0.325, 0.9, 0.9099999999999999, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612", "probabilities": [0.025, 0.025, 0.99, 0.25, 0.95, 0.98, 0.925, 0.98, 0.8500000000000001, 0.98, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693", "probabilities": [0.6499999999999999, 0.025, 0.35, 0.25, 0.25, 0.25, 0.99, 0.95, 0.96, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94447683257072169359896104855962933413", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175", "probabilities": [0.025, 0.025, 0.015, 0.015, 0.025, 0.025, 0.025, 0.01, 0.13, 0.01, 0.015, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589", "probabilities": [0.015, 0.015, 0.98, 0.41, 0.995, 0.97, 0.9550000000000001, 0.9099999999999999, 0.97, 0.95, 0.455, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472", "probabilities": [0.015, 0.015, 0.975, 0.42, 0.25, 0.25, 0.25, 0.015, 0.1325, 0.9199999999999999, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.975, 0.975, 0.975, 0.99, 0.98, 0.01, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.47, 0.975], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95026524886584356226576936309513335214", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.25, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.7150000000000001, 0.01, 0.13, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380", "probabilities": [0.015, 0.015, 0.04, 0.015, 0.755, 0.755, 0.015, 0.025, 0.1375, 0.015, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.89, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109", "probabilities": [0.015, 0.015, 0.05, 0.05, 0.015, 0.015, 0.25, 0.89, 0.32, 0.015, 0.735, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657", "probabilities": [0.015, 0.015, 0.97, 0.015, 0.015, 0.015, 0.9550000000000001, 0.01, 0.13, 0.25, 0.015, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948", "probabilities": [0.015, 0.015, 0.9099999999999999, 0.97, 0.99, 0.61, 0.25, 0.89, 0.32, 0.015, 0.875, 0.97], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310", "probabilities": [0.015, 0.015, 0.015, 0.97, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.94, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.1325, 0.01, 0.015, 0.015], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603", "probabilities": [0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.01, 0.13, 0.01, 0.01, 0.01], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466", "probabilities": [0.25, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9, 0.325, 0.9, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716", "probabilities": [0.99, 0.025, 0.42, 0.42, 0.94, 0.94, 0.94, 0.9099999999999999, 0.32999999999999996, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428", "probabilities": [0.975, 0.42, 0.975, 0.42, 0.025, 0.025, 0.025, 0.975, 0.3625, 0.975, 0.975, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083", "probabilities": [0.975, 0.94, 0.42, 0.85, 0.25, 0.25, 0.25, 0.9, 0.325, 0.025, 0.75, 0.85], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570", "probabilities": [0.35000000000000003, 0.025, 0.975, 0.42, 0.25, 0.25, 0.25, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186", "probabilities": [0.025, 0.025, 0.42, 0.025, 0.25, 0.25, 0.25, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926", "probabilities": [0.975, 0.025, 0.99, 0.99, 0.995, 0.995, 0.99, 0.99, 0.37, 0.025, 0.25, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442", "probabilities": [0.025, 0.025, 0.985, 0.025, 0.98, 0.575, 0.985, 0.975, 0.3625, 0.25, 0.075, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373", "probabilities": [0.025, 0.025, 0.99, 0.4, 0.995, 0.9650000000000001, 0.995, 0.9099999999999999, 0.32999999999999996, 0.9099999999999999, 0.425, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480", "probabilities": [0.025, 0.025, 0.98, 0.41, 0.98, 0.765, 0.8, 0.95, 0.35, 0.025, 0.41, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.64, 0.32, 0.25, 0.025, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091", "probabilities": [0.025, 0.25, 0.41, 0.25, 0.95, 0.81, 0.97, 0.975, 0.97, 0.25, 0.425, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589", "probabilities": [0.025, 0.025, 0.985, 0.41, 0.995, 0.97, 0.99, 0.98, 0.97, 0.025, 0.925, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074", "probabilities": [0.41, 0.025, 0.97, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.89, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938", "probabilities": [0.94, 0.025, 0.025, 0.025, 0.025, 0.9099999999999999, 0.025, 0.025, 0.1375, 0.25, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070", "probabilities": [0.025, 0.96, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.1375, 0.025, 0.81, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186", "probabilities": [0.025, 0.025, 0.975, 0.025, 0.975, 0.975, 0.975, 0.025, 0.1375, 0.025, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068", "probabilities": [0.025, 0.025, 0.985, 0.025, 0.9099999999999999, 0.96, 0.975, 0.89, 0.32, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803", "probabilities": [0.05, 0.025, 0.05, 0.05, 0.97, 0.025, 0.025, 0.89, 0.32, 0.9099999999999999, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536", "probabilities": [0.025, 0.025, 0.995, 0.025, 0.98, 0.9099999999999999, 0.98, 0.9099999999999999, 0.975, 0.97, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928", "probabilities": [0.25, 0.25, 0.25, 0.975, 0.25, 0.975, 0.96, 0.96, 0.355, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961", "probabilities": [0.025, 0.025, 0.95, 0.025, 0.94, 0.025, 0.025, 0.94, 0.345, 0.89, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206", "probabilities": [0.975, 0.025, 0.985, 0.025, 0.97, 0.25, 0.97, 0.025, 0.1375, 0.96, 0.95, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506", "probabilities": [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.05, 0.025, 0.025, 0.025, 0.025], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290", "probabilities": [0.995, 0.025, 0.025, 0.025, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.985, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99939100657291954890286900729983287973", "probabilities": [0.025, 0.025, 0.05, 0.05, 0.25, 0.25, 0.7, 0.925, 0.3375, 0.25, 0.19999999999999998, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, {"study_id": "1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966", "probabilities": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], "weights": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}]}\n', 'config.json': '{\n  "version": "local_global_resnet34_hpo31_trainval_60epochs_seed2026",\n  "cache": {\n    "field_of_view_mm": 140.0,\n    "image_size": 224,\n    "series_slots": 4,\n    "slices_per_slot": 12,\n    "central_band": [\n      0.2,\n      0.8\n    ],\n    "clip_percentiles": [\n      1.0,\n      99.0\n    ],\n    "exclude_order_sources": [\n      "unordered"\n    ]\n  },\n  "model": {\n    "backbone": "resnet34",\n    "pretrained": true,\n    "dropout": 0.4,\n    "pooling": "local_topk_plus_study_context",\n    "top_k_fraction": 0.1,\n    "context_width": 256,\n    "context_heads": 4,\n    "context_layers": 1\n  },\n  "epochs": 60,\n  "batch_studies": 8,\n  "learning_rate": 0.0003,\n  "weight_decay": 0.03,\n  "warmup_fraction": 0.05,\n  "gradient_clip_norm": 1.0,\n  "mixed_precision": true,\n  "validation_fold": 0,\n  "xla_eval_slices": 48,\n  "confident_low": 0.3,\n  "confident_high": 0.7,\n  "seed": 2026,\n  "note": "Current local+global HPO-v3.1-D2 trainval recipe; ResNet34/ImageNet; no early stopping; all60 epochs. Steven audit-only.",\n  "early_stopping_patience": 0,\n  "early_stopping_min_delta": 0.0005,\n  "early_stopping_metric": "auc",\n  "scheduler": {\n    "type": "reduce_on_plateau",\n    "mode": "min",\n    "factor": 0.5,\n    "patience": 2,\n    "threshold": 0.0005,\n    "threshold_mode": "abs",\n    "cooldown": 1,\n    "min_lr": 1e-06\n  },\n  "augmentation": {\n    "crop_area_min": 1.0,\n    "translate_fraction": 0.06,\n    "rotation_degrees": 0.0,\n    "intensity_jitter": 0.15,\n    "brightness_shift": 0.1,\n    "erase_probability": 0.25,\n    "erase_fraction": 0.25\n  }\n}\n', 'asset_hashes.json': '{\n  "folds.csv": "2dd1044adcc54ca4d9bcb9c9fd3b910a6784bd2b99e792bbeaf84a42d417f06a",\n  "validation_labels.json": "7323326a7bfab1c8649561b4a0543852eb0d454fdaf83b4a6e03d055aa619009",\n  "steven_audit_labels.json": "10fdaaab54e67300b91415fb3a5d4f7567d21cceb71ae973df7f3fd7e545a7e5",\n  "config.json": "3503fdc949929c6c276d351609c2e54d46d304f37151829fd5ba76dcffe325b6"\n}\n'}
for name, text in MODULES.items(): (WORK / name).write_text(text)
(WORK / 'assets').mkdir(exist_ok=True)
for name, text in ASSETS.items(): (WORK / 'assets' / name).write_text(text)
sys.path.insert(0, str(WORK))
print("STARTUP: modules/assets ready; only fixed input paths will be checked", flush=True)


In [ ]:
from local_global_resnet34_kaggle import main
main()
